# DIMER Notebook: Image Depth Estimation — Relative Depth and Metric Distance

**Does a convincing depth map tell us how far away things really are?**

**Profile:** `E2E` · **Mode:** `WORKSHOP` · **Notebook specification:** `2.2` · **Status:** Candidate; exact fresh Colab GPU and representative labelled BYOD qualification pending.

This modular, self-paced **Level 2 — Applied tasks** unit follows image segmentation and can be taken independently. Basic Python and Colab familiarity are sufficient; no prior ML or depth-estimation experience is required. Personal completion records are optional notes, **not required submissions**.

**Input → Model → Output:** an RGB photograph → Depth Anything V2 Small or ZoeDepth → one numerical value per pixel. Depth Anything predicts **relative inverse depth** (larger generally means nearer); ZoeDepth predicts **distance in metres**. A colour map displays these numbers but does not establish their accuracy.

You will inspect sensor references, calculate depth errors, distinguish two evaluation protocols, adapt only Depth Anything's neck/head, select with validation scenes, verify a saved adapter in a new process, observe a controlled brightness change, and optionally choose your own brightness factor. ZoeDepth remains frozen throughout.

**Run all:** choose Runtime → Change runtime type → T4 GPU, then Run all in a fresh runtime. No token, upload, manual restart, repository clone, DIMER service or remote inference API is required. This notebook carries its implementation and immutable manifests; public models/data are downloaded anonymously. The host kernel is preserved while an isolated Python 3.12.12 worker executes the models sequentially.

The default sample contains **40 views in six scenes**: 14 train / 6 validation / 20 test, two scenes per role. The data payload is 312,448,846 bytes, plus two model snapshots and the GPU environment. Allow several GiB of storage (an estimate). One measured fresh Colab T4 Run all took 403 s on 2026-09-27, 69 s of it environment setup; your time will vary. The final cell reports this run's time, peak GPU memory and disk use.

**How to use this notebook:** run cells top to bottom (Run all is fine). Cells titled **Infrastructure** may be run without reading them. Optional activities are off by default and are switched on with the form fields in their cells. **Predict** prompts ask you to commit to an answer before running; expandable hints and worked answers follow. New terms are defined under **Key terms** below.

**Roadmap:** inspect data → learn metrics and baselines → freeze/adapt relative depth → verify reload → inspect metric depth → change brightness → conclude → optional own data.

### Key terms

| Term | Meaning in this notebook |
|---|---|
| **Depth / metres** | Distance from the camera to the surface seen at a pixel, in metres. |
| **Inverse depth** | `1 / depth`. Larger values mean nearer surfaces. Depth Anything predicts a *relative* inverse depth: its scale and offset are unknown, so it is not in metres. |
| **Reference depth / sensor mask** | The measured depth from the dataset's scanner, and a true/false map of the pixels where the scanner returned a usable value. |
| **AbsRel** | Mean of `abs(predicted − reference) / reference` over the scored pixels; lower is better. |
| **Delta-1** | Fraction of scored pixels whose prediction is within a factor of 1.25 of the reference; higher is better. |
| **Affine alignment** | Fitting a scale and an offset per image so a relative prediction best matches that image's reference. It uses the answer, so it is an evaluation device, not a calibration. |
| **Backbone** | The large first part of the network that turns pixels into general visual features. It stays frozen (unchanged) here. |
| **Neck and head** | The smaller later layers that combine those features and produce the depth map (the DPT neck and head in Depth Anything). Only these are trained here. |
| **Adapter** | The saved file of trained neck/head weights; it is applied on top of the unchanged base model when reloaded. |
| **AdamW** | The optimisation algorithm that updates the trainable weights from the training loss. |
| **Scale-and-shift-invariant loss** | A training loss that first aligns the prediction's scale and offset to the target, so the model is trained on relative structure rather than metres. |
| **Epoch / selection** | One pass over the training views; selection keeps the epoch (0 = untrained) with the lowest validation AbsRel. |
| **Reload parity** | Loading the saved adapter in a new process and checking that it reproduces the same predictions; it shows the file is complete, not that the model is accurate. |


## 1. Prepare an isolated runtime

**Input:** a fresh Linux x86_64 Colab GPU runtime and carried source. **System:** a verified uv bootstrap and a fully hash-locked Python 3.12.12 environment. **Output:** a separate worker without replacing the notebook kernel's loaded packages.

The three cells below are **Infrastructure**: they are collapsed, and you may run them without studying their implementation. They check the GPU, write and verify the carried source, and build the isolated environment. Expand one (double-click its title) only if you want to read the code. Model weights are checked against per-file size and SHA-256 manifests before loading. The default resources are public; no Hugging Face token is required or exported.

**Notice:** GPU availability and dependency installation. An integrity failure should be investigated, never bypassed. A failed stage stops the workflow; rerunning a prerequisite invalidates downstream completion receipts.

In [ ]:
# @title Infrastructure: check the GPU and create this run's folder
import hashlib
import json
import os
import platform
from pathlib import Path
import subprocess
import time
import urllib.error
import urllib.request
import uuid
import zipfile

SESSION_START = time.perf_counter()
if platform.system() != 'Linux' or platform.machine() != 'x86_64':
    raise RuntimeError('The canonical environment is Linux x86_64 Google Colab with a T4 GPU.')
try:
    gpu = subprocess.run(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], capture_output=True, text=True)
except FileNotFoundError as exc:
    raise RuntimeError('Select Runtime > Change runtime type > T4 GPU before Run all.') from exc
if gpu.returncode:
    raise RuntimeError('Select a fresh T4 GPU runtime; CUDA was not detected.')
print(gpu.stdout.strip())
ROOT = Path.cwd() / 'outputs' / 'depth_estimation' / uuid.uuid4().hex[:12]
ROOT.mkdir(parents=True)

In [ ]:
# @title Infrastructure: write and verify the carried source (about 290 KB; no need to read it)
CARRIED_FILES = {
    'depth_reference/__init__.py': (
        'from .metrics import (\n'
        '    DELTA_THRESHOLD,\n'
        '    EVAL_DEPTH_RANGE_M,\n'
        '    METRIC_DEFINITIONS,\n'
        '    align_inverse_depth,\n'
        '    aligned_abs_rel,\n'
        '    aligned_delta1,\n'
        '    constant_prior,\n'
        '    depth_metrics,\n'
        '    image_metrics,\n'
        '    prior_baselines,\n'
        '    vertical_gradient_prediction,\n'
        ')\n'
        'from .pipeline import (\n'
        '    ARTIFACT_FORMAT,\n'
        '    DEFAULT_TRAINABLE_BLOCKS,\n'
        '    DEFAULT_WEIGHTS_DIR,\n'
        '    DEPTH_KIND,\n'
        '    INPUT_SCHEMA,\n'
        '    MAX_ASPECT_RATIO,\n'
        '    MAX_EVAL_RECORDS,\n'
        '    MAX_IMAGE_SIDE,\n'
        '    MIN_IMAGE_SIDE,\n'
        '    MIN_SCORED_RECORDS,\n'
        '    MODEL_ID,\n'
        '    MODEL_KEY,\n'
        '    MODEL_LICENSE,\n'
        '    MODEL_REVISION,\n'
        '    PARAMETER_COUNT,\n'
        '    POLICY_FROZEN,\n'
        '    POLICY_ZERO_SHOT,\n'
        '    TRANSFORMER_BLOCKS,\n'
        '    DepthAnythingPipeline,\n'
        '    abs_rel,\n'
        '    evaluation_report,\n'
        '    stage_missing_files,\n'
        '    validate_inputs,\n'
        '    verify_snapshot,\n'
        ')\n'
        'from .samples import (\n'
        '    CORPUS_BASE_URL,\n'
        '    CORPUS_BYTES,\n'
        '    CORPUS_COMMIT,\n'
        '    CORPUS_LICENSE,\n'
        '    CORPUS_NAME,\n'
        '    CORPUS_RELEASE,\n'
        '    DOMAINS,\n'
        '    MAX_RECORDS,\n'
        '    MIN_RECORDS,\n'
        '    MIN_VALID_FRACTION,\n'
        '    SAMPLE_RECORDS,\n'
        '    SAMPLE_SPLIT,\n'
        '    build_sample_dataset,\n'
        '    check_split_disjoint,\n'
        '    dataset_digest,\n'
        '    fetch_corpus,\n'
        '    fetch_sample_dataset,\n'
        '    image_digest,\n'
        '    load_byod_dataset,\n'
        '    read_corpus,\n'
        '    scan_summary,\n'
        '    split_dataset,\n'
        '    validate_dataset,\n'
        '    write_dataset_csv,\n'
        ')\n'
        '\n'
        '__all__ = [\n'
        '    "ARTIFACT_FORMAT",\n'
        '    "CORPUS_BASE_URL",\n'
        '    "CORPUS_BYTES",\n'
        '    "CORPUS_COMMIT",\n'
        '    "CORPUS_LICENSE",\n'
        '    "CORPUS_NAME",\n'
        '    "CORPUS_RELEASE",\n'
        '    "DEFAULT_TRAINABLE_BLOCKS",\n'
        '    "DEFAULT_WEIGHTS_DIR",\n'
        '    "DELTA_THRESHOLD",\n'
        '    "EVAL_DEPTH_RANGE_M",\n'
        '    "DEPTH_KIND",\n'
        '    "DOMAINS",\n'
        '    "INPUT_SCHEMA",\n'
        '    "MAX_ASPECT_RATIO",\n'
        '    "MAX_EVAL_RECORDS",\n'
        '    "MAX_IMAGE_SIDE",\n'
        '    "MAX_RECORDS",\n'
        '    "METRIC_DEFINITIONS",\n'
        '    "MIN_IMAGE_SIDE",\n'
        '    "MIN_RECORDS",\n'
        '    "MIN_SCORED_RECORDS",\n'
        '    "MIN_VALID_FRACTION",\n'
        '    "MODEL_ID",\n'
        '    "MODEL_KEY",\n'
        '    "MODEL_LICENSE",\n'
        '    "MODEL_REVISION",\n'
        '    "PARAMETER_COUNT",\n'
        '    "POLICY_FROZEN",\n'
        '    "POLICY_ZERO_SHOT",\n'
        '    "SAMPLE_RECORDS",\n'
        '    "SAMPLE_SPLIT",\n'
        '    "TRANSFORMER_BLOCKS",\n'
        '    "DepthAnythingPipeline",\n'
        '    "abs_rel",\n'
        '    "align_inverse_depth",\n'
        '    "aligned_abs_rel",\n'
        '    "build_sample_dataset",\n'
        '    "check_split_disjoint",\n'
        '    "constant_prior",\n'
        '    "dataset_digest",\n'
        '    "aligned_delta1",\n'
        '    "depth_metrics",\n'
        '    "evaluation_report",\n'
        '    "fetch_corpus",\n'
        '    "fetch_sample_dataset",\n'
        '    "image_digest",\n'
        '    "image_metrics",\n'
        '    "load_byod_dataset",\n'
        '    "prior_baselines",\n'
        '    "read_corpus",\n'
        '    "scan_summary",\n'
        '    "split_dataset",\n'
        '    "stage_missing_files",\n'
        '    "validate_dataset",\n'
        '    "validate_inputs",\n'
        '    "verify_snapshot",\n'
        '    "vertical_gradient_prediction",\n'
        '    "write_dataset_csv",\n'
        ']\n'
    ),
    'depth_reference/metrics.py': (
        '"""Depth metrics for the adaptation contract, implemented here with no external scorer.\n'
        '\n'
        'The model emits **relative inverse depth** (disparity up to an unknown per-image scale and shift), so every\n'
        'prediction is first aligned to the reference by least squares in inverse-depth space over the valid pixels\n'
        '(``1 / depth = s * pred + t``), inverted to metres, and then scored:\n'
        '\n'
        '- **AbsRel** — ``mean(|est - ref| / ref)`` over valid pixels (lower is better; 0 is perfect);\n'
        '- **δ1** — the fraction of valid pixels with ``max(est / ref, ref / est) < 1.25`` (higher is better;\n'
        '  1 is perfect).\n'
        '\n'
        'Only reference pixels inside ``EVAL_DEPTH_RANGE_M`` (0.6..350 m, the DIODE sensor range) are scored, and the\n'
        'aligned inverse depth is floored at ``1 / 350 m`` so a pixel the fit pushes past the far cap counts as 350 m\n'
        'rather than as an unbounded error — the standard cap of relative-depth evaluation. Both metrics are reported\n'
        'per image and as the mean over images (each image weighs the same regardless of its valid pixel count), plus\n'
        'per domain when records carry one. Two trivial priors frame the numbers: the **constant\n'
        "prior** (every pixel at the image's own median reference depth — an oracle constant, the strongest flat\n"
        'guess) and the **vertical-gradient prior** (inverse depth rising linearly from the top row to the bottom row,\n'
        'aligned exactly like a model prediction — "the ground is nearer at the bottom of the frame").\n'
        '"""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'from collections.abc import Mapping, Sequence\n'
        'from typing import Any\n'
        '\n'
        'import numpy as np\n'
        '\n'
        'DELTA_THRESHOLD = 1.25\n'
        'MIN_VALID_PIXELS = 2\n'
        'EVAL_DEPTH_RANGE_M = (0.6, 350.0)  # reference pixels scored; also the far cap of the aligned estimate\n'
        '\n'
        'METRIC_DEFINITIONS = {\n'
        '    "alignment": (\n'
        '        "per image, least-squares scale and shift mapping the prediction onto 1 / reference depth over the "\n'
        '        "valid pixels inside EVAL_DEPTH_RANGE_M; the aligned inverse depth is floored at 1 / the far cap"\n'
        '    ),\n'
        '    "abs_rel": "mean over valid pixels of |aligned estimate - reference| / reference, then mean over images",\n'
        '    "delta1": (\n'
        '        "fraction of valid pixels with max(estimate / reference, reference / estimate) < 1.25, "\n'
        '        "then mean over images"\n'
        '    ),\n'
        '    "per_domain": "the same means restricted to the records of one domain (indoors / outdoor when present)",\n'
        '}\n'
        '\n'
        '\n'
        'def _valid(pred: np.ndarray, depth: np.ndarray, mask: np.ndarray | None) -> np.ndarray:\n'
        '    pred = np.asarray(pred, dtype=np.float64)\n'
        '    depth = np.asarray(depth, dtype=np.float64)\n'
        '    if pred.shape != depth.shape:\n'
        '        raise ValueError(f"shape mismatch: pred {pred.shape} vs depth {depth.shape}")\n'
        '    low, high = EVAL_DEPTH_RANGE_M\n'
        '    valid = np.isfinite(depth) & (depth >= low) & (depth <= high) & np.isfinite(pred)\n'
        '    if mask is not None:\n'
        '        mask = np.asarray(mask, dtype=bool)\n'
        '        if mask.shape != depth.shape:\n'
        '            raise ValueError(f"shape mismatch: mask {mask.shape} vs depth {depth.shape}")\n'
        '        valid &= mask\n'
        '    if int(valid.sum()) < MIN_VALID_PIXELS:\n'
        '        raise ValueError(f"need at least {MIN_VALID_PIXELS} valid reference pixels")\n'
        '    return valid\n'
        '\n'
        '\n'
        'def align_inverse_depth(\n'
        '    pred: np.ndarray, depth: np.ndarray, mask: np.ndarray | None = None\n'
        ') -> tuple[np.ndarray, np.ndarray, dict[str, float]]:\n'
        '    """Return (aligned metric estimate, reference) over valid pixels and the fitted scale / shift."""\n'
        '    valid = _valid(pred, depth, mask)\n'
        '    p = np.asarray(pred, dtype=np.float64)[valid]\n'
        '    ref = np.asarray(depth, dtype=np.float64)[valid]\n'
        '    target = 1.0 / ref\n'
        '    design = np.stack([p, np.ones_like(p)], axis=1)\n'
        '    (scale, shift), *_ = np.linalg.lstsq(design, target, rcond=None)\n'
        '    estimate = 1.0 / np.clip(scale * p + shift, 1.0 / EVAL_DEPTH_RANGE_M[1], None)\n'
        '    return estimate, ref, {"scale": float(scale), "shift": float(shift), "n_valid": int(valid.sum())}\n'
        '\n'
        '\n'
        'def aligned_abs_rel(pred: np.ndarray, depth: np.ndarray, mask: np.ndarray | None = None) -> float:\n'
        '    estimate, ref, _ = align_inverse_depth(pred, depth, mask)\n'
        '    return float(np.mean(np.abs(estimate - ref) / ref))\n'
        '\n'
        '\n'
        'def aligned_delta1(\n'
        '    pred: np.ndarray, depth: np.ndarray, mask: np.ndarray | None = None, *, threshold: float = DELTA_THRESHOLD\n'
        ') -> float:\n'
        '    estimate, ref, _ = align_inverse_depth(pred, depth, mask)\n'
        '    ratio = np.maximum(estimate / ref, ref / estimate)\n'
        '    return float(np.mean(ratio < threshold))\n'
        '\n'
        '\n'
        'def image_metrics(pred: np.ndarray, depth: np.ndarray, mask: np.ndarray | None = None) -> dict[str, Any]:\n'
        '    estimate, ref, fit = align_inverse_depth(pred, depth, mask)\n'
        '    ratio = np.maximum(estimate / ref, ref / estimate)\n'
        '    return {\n'
        '        "abs_rel": float(np.mean(np.abs(estimate - ref) / ref)),\n'
        '        "delta1": float(np.mean(ratio < DELTA_THRESHOLD)),\n'
        '        "n_valid": fit["n_valid"],\n'
        '        "scale": fit["scale"],\n'
        '        "shift": fit["shift"],\n'
        '    }\n'
        '\n'
        '\n'
        'def depth_metrics(preds: Sequence[np.ndarray], records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n'
        '    """Aggregate per-image metrics for aligned predictions over `{id, depth, mask[, domain]}` records."""\n'
        '    if len(preds) != len(records):\n'
        '        raise ValueError(f"{len(preds)} predictions for {len(records)} records")\n'
        '    if not records:\n'
        '        raise ValueError("at least one record is required")\n'
        '    per_image = []\n'
        '    for pred, record in zip(preds, records, strict=True):\n'
        '        row = image_metrics(pred, record["depth"], record.get("mask"))\n'
        '        per_image.append({"id": record["id"], "domain": str(record.get("domain", "unspecified")), **row})\n'
        '    domains = sorted({r["domain"] for r in per_image})\n'
        '    per_domain = {\n'
        '        d: {\n'
        '            "n": sum(1 for r in per_image if r["domain"] == d),\n'
        '            "abs_rel": float(np.mean([r["abs_rel"] for r in per_image if r["domain"] == d])),\n'
        '            "delta1": float(np.mean([r["delta1"] for r in per_image if r["domain"] == d])),\n'
        '        }\n'
        '        for d in domains\n'
        '    }\n'
        '    return {\n'
        '        "n": len(per_image),\n'
        '        "abs_rel": float(np.mean([r["abs_rel"] for r in per_image])),\n'
        '        "delta1": float(np.mean([r["delta1"] for r in per_image])),\n'
        '        "per_domain": per_domain,\n'
        '        "per_image": per_image,\n'
        '        "definitions": dict(METRIC_DEFINITIONS),\n'
        '    }\n'
        '\n'
        '\n'
        'def constant_prior(record: Mapping[str, Any]) -> dict[str, Any]:\n'
        '    """The oracle constant: every pixel at the image\'s own median reference depth (no alignment needed)."""\n'
        '    depth = np.asarray(record["depth"], dtype=np.float64)\n'
        '    valid = _valid(depth, depth, record.get("mask"))\n'
        '    ref = depth[valid]\n'
        '    constant = float(np.median(ref))\n'
        '    ratio = np.maximum(constant / ref, ref / constant)\n'
        '    return {\n'
        '        "abs_rel": float(np.mean(np.abs(constant - ref) / ref)),\n'
        '        "delta1": float(np.mean(ratio < DELTA_THRESHOLD)),\n'
        '        "n_valid": int(valid.sum()),\n'
        '        "constant_m": constant,\n'
        '    }\n'
        '\n'
        '\n'
        'def vertical_gradient_prediction(height: int, width: int) -> np.ndarray:\n'
        '    """Relative inverse depth rising from 0 at the top row to 1 at the bottom row (the ground-plane prior)."""\n'
        '    rows = np.linspace(0.0, 1.0, num=height, dtype=np.float32)\n'
        '    return np.repeat(rows[:, None], width, axis=1)\n'
        '\n'
        '\n'
        'def prior_baselines(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n'
        '    """Both trivial priors scored on the same records as a model, each with the same aggregation."""\n'
        '    constant_rows = []\n'
        '    gradient_preds = []\n'
        '    for record in records:\n'
        '        constant_rows.append(\n'
        '            {"id": record["id"], "domain": str(record.get("domain", "unspecified")), **constant_prior(record)}\n'
        '        )\n'
        '        h, w = np.asarray(record["depth"]).shape\n'
        '        gradient_preds.append(vertical_gradient_prediction(h, w))\n'
        '    gradient = depth_metrics(gradient_preds, records)\n'
        '    return {\n'
        '        "constant_prior": {\n'
        '            "baseline": "every pixel at the image\'s own median reference depth (oracle constant)",\n'
        '            "n": len(constant_rows),\n'
        '            "abs_rel": float(np.mean([r["abs_rel"] for r in constant_rows])),\n'
        '            "delta1": float(np.mean([r["delta1"] for r in constant_rows])),\n'
        '            "per_image": constant_rows,\n'
        '        },\n'
        '        "vertical_gradient_prior": {\n'
        '            "baseline": (\n'
        '                "inverse depth rising linearly from the top row to the bottom row, "\n'
        '                "affine-aligned like a prediction"\n'
        '            ),\n'
        '            **{k: gradient[k] for k in ("n", "abs_rel", "delta1", "per_domain", "per_image")},\n'
        '        },\n'
        '    }\n'
    ),
    'depth_reference/pipeline.py': (
        '"""Monocular relative depth estimation over the pinned Depth Anything V2 Small checkpoint, plus a bounded\n'
        'supervised-adaptation contract.\n'
        '\n'
        'Inference (`predict`) is unchanged: one PIL image in, relative inverse depth at the input resolution out.\n'
        'The adaptation contract (`evaluate`, `adapt`, `save_artifact`, `from_artifact`) scores the model on a\n'
        'validated `{id, image, depth, mask}` dataset by affine-aligned AbsRel and δ1 (`metrics.py`), fine-tunes the\n'
        'DPT neck and head on the frozen backbone (the **frozen policy**) and optionally the last transformer blocks\n'
        'with them (the **unfrozen policy**) under a scale-and-shift-invariant loss, keeps the epoch with the lowest\n'
        'validation AbsRel — where epoch 0 is the untouched checkpoint, the **zero-shot policy**, competing on equal\n'
        'terms — and exports the trained tensors as a safetensors adapter bound to the pinned base weights.\n'
        '"""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import hashlib\n'
        'import json\n'
        'import math\n'
        'import random\n'
        'import time\n'
        'from collections.abc import Callable, Mapping, Sequence\n'
        'from dataclasses import dataclass, field\n'
        'from pathlib import Path\n'
        'from typing import Any\n'
        '\n'
        'import numpy as np\n'
        'from PIL import Image\n'
        '\n'
        'MODEL_ID = "depth-anything/Depth-Anything-V2-Small-hf"\n'
        'MODEL_REVISION = "5426e4f0f36572d16453bbda7a8389317b1bef99"\n'
        'MODEL_LICENSE = "apache-2.0"\n'
        'MODEL_KEY = "depth-anything-v2-small"\n'
        'DEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / "weights" / MODEL_KEY\n'
        'MANIFEST_NAME = "dimer-base-manifest.json"\n'
        '\n'
        '# Input ceilings. The DPT processor rescales the image so that its sides are multiples of 14 close\n'
        "# to 518 px, and the raw prediction is interpolated back to the caller's resolution, so the cost that\n"
        "# grows with the caller's image is the aspect ratio (backbone tokens) and the output interpolation.\n"
        'MAX_IMAGE_SIDE = 4096\n'
        'MIN_IMAGE_SIDE = 14\n'
        'MAX_ASPECT_RATIO = 4.0\n'
        'DEPTH_KIND = "relative"\n'
        'WEIGHTS_FILE = "model.safetensors"\n'
        'WEIGHT_SHA256 = (\n'
        '    "3152477ce0d8d6978d76b995120de97cb5b928701fd0f817769f59e249a16b70"  # manifest digest of WEIGHTS_FILE\n'
        ')\n'
        'PARAMETER_COUNT = 24_785_089  # backbone 22,056,576 + neck 2,700,768 + head 27,745\n'
        'TRANSFORMER_BLOCKS = 12  # DINOv2 ViT-S/14 backbone depth\n'
        'DEFAULT_TRAINABLE_BLOCKS = (\n'
        '    2  # the unfrozen policy trains the last two blocks (3,550,464 parameters) with neck + head\n'
        ')\n'
        'NECK_HEAD_PARAMETERS = 2_728_513  # the DPT neck and head, trained under every adapted policy\n'
        'ADAPTER_PREFIXES = ("neck.", "head.")\n'
        'BLOCK_PREFIX = "backbone.encoder.layer."\n'
        'POLICY_ZERO_SHOT = "zero-shot (no adaptation)"\n'
        'POLICY_FROZEN = "frozen backbone + DPT neck and head"\n'
        'POLICY_UNFROZEN = "unfrozen last {k} blocks + DPT neck and head"\n'
        'MIN_VALID_TARGET_PIXELS = 2  # the affine alignment inside the loss needs at least two valid pixels\n'
        'MAX_EVAL_RECORDS = 2_000\n'
        'MIN_SCORED_RECORDS = 20  # below this a scored dataset is labelled a small sample\n'
        'ARTIFACT_FORMAT = "org.valcorza.depth-anything-v2-small.adapter.v1"\n'
        'ARTIFACT_FORMAT_VERSION = "1.0"\n'
        'ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"\n'
        'ARTIFACT_MANIFEST_NAME = "manifest.json"\n'
        '\n'
        '\n'
        'def _sha256(path: Path) -> str:\n'
        '    digest = hashlib.sha256()\n'
        '    with open(path, "rb") as fh:\n'
        '        for chunk in iter(lambda: fh.read(1 << 20), b""):\n'
        '            digest.update(chunk)\n'
        '    return digest.hexdigest()\n'
        '\n'
        '\n'
        'def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n'
        '    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""\n'
        '    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID:\n'
        '        raise ValueError(f"manifest modelId {manifest.get(\'modelId\')!r} != {MODEL_ID!r}")\n'
        '    if manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(f"manifest revision {manifest.get(\'revision\')!r} != {MODEL_REVISION!r}")\n'
        '    for entry in manifest["files"]:\n'
        '        file_path = root / entry["path"]\n'
        '        if not file_path.is_file():\n'
        '            raise FileNotFoundError(f"snapshot file missing: {file_path}")\n'
        '        size = file_path.stat().st_size\n'
        '        if size != entry["bytes"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: size {size} != manifest {entry[\'bytes\']}")\n'
        '        digest = _sha256(file_path)\n'
        '        if digest != entry["sha256"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: sha256 {digest} != manifest {entry[\'sha256\']}")\n'
        '    return {\n'
        '        "path": str(root),\n'
        '        "model_id": manifest["modelId"],\n'
        '        "revision": manifest["revision"],\n'
        '        "files": len(manifest["files"]),\n'
        '        "total_bytes": manifest.get("totalBytes"),\n'
        '    }\n'
        '\n'
        '\n'
        'def _hub_download(relative_path: str, root: Path) -> None:\n'
        '    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""\n'
        '    from huggingface_hub import hf_hub_download\n'
        '\n'
        '    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n'
        '\n'
        '\n'
        'def stage_missing_files(\n'
        '    path: str | Path | None = None,\n'
        '    *,\n'
        '    allow_download: bool = False,\n'
        '    downloader: Callable[[str, Path], None] | None = None,\n'
        ') -> list[str]:\n'
        '    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but\n'
        '    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""\n'
        '    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(\n'
        '            f"manifest names {manifest.get(\'modelId\')}@{manifest.get(\'revision\')}, "\n'
        '            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"\n'
        '        )\n'
        '    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]\n'
        '    if not missing:\n'
        '        return []\n'
        '    if not allow_download:\n'
        '        raise FileNotFoundError(\n'
        '            f"snapshot at {root} is missing {missing}; "\n'
        '            f"pass allow_download=True to fetch them at {MODEL_REVISION}"\n'
        '        )\n'
        '    fetch = downloader or _hub_download\n'
        '    for relative_path in missing:\n'
        '        fetch(relative_path, root)\n'
        '    return missing\n'
        '\n'
        '\n'
        'def abs_rel(pred: np.ndarray, ref_depth: np.ndarray, *, align: bool = True) -> float:\n'
        '    """Absolute relative error of a relative inverse-depth map against caller-supplied metric depth.\n'
        '\n'
        '    The model emits relative inverse depth (disparity up to an unknown scale and shift), so the\n'
        '    prediction is first aligned to ``1 / ref_depth`` by least squares over valid pixels\n'
        '    (``ref_depth`` inside ``metrics.EVAL_DEPTH_RANGE_M``, 0.6..350 m), inverted to depth with the\n'
        '    aligned inverse depth floored at ``1 / 350 m`` (the far cap of relative-depth evaluation, so a pixel\n'
        '    pushed past it counts as 350 m rather than as an unbounded error), and scored as\n'
        '    ``mean(|est - ref| / ref)``. With ``align=False`` the arrays are compared as given, which is only\n'
        '    meaningful for metric input. `metrics.aligned_abs_rel` is the same computation with a mask argument.\n'
        '    """\n'
        '    from .metrics import EVAL_DEPTH_RANGE_M\n'
        '\n'
        '    pred = np.asarray(pred, dtype=np.float64)\n'
        '    ref_depth = np.asarray(ref_depth, dtype=np.float64)\n'
        '    if pred.shape != ref_depth.shape:\n'
        '        raise ValueError(f"shape mismatch: pred {pred.shape} vs ref {ref_depth.shape}")\n'
        '    low, high = EVAL_DEPTH_RANGE_M\n'
        '    valid = np.isfinite(ref_depth) & (ref_depth >= low) & (ref_depth <= high) & np.isfinite(pred)\n'
        '    if valid.sum() < 2:\n'
        '        raise ValueError(f"need at least 2 valid reference pixels ({low} <= ref_depth <= {high})")\n'
        '    if align:\n'
        '        target = 1.0 / ref_depth[valid]\n'
        '        design = np.stack([pred[valid], np.ones(int(valid.sum()))], axis=1)\n'
        '        (scale, shift), *_ = np.linalg.lstsq(design, target, rcond=None)\n'
        '        estimate = 1.0 / np.clip(scale * pred[valid] + shift, 1.0 / high, None)\n'
        '    else:\n'
        '        estimate = pred[valid]\n'
        '    return float(np.mean(np.abs(estimate - ref_depth[valid]) / ref_depth[valid]))\n'
        '\n'
        '\n'
        'def validate_image(image: Any) -> Image.Image:\n'
        '    """Type- and size-check a caller image and return it as RGB."""\n'
        '    if not isinstance(image, Image.Image):\n'
        '        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")\n'
        '    width, height = image.size\n'
        '    short, long = min(width, height), max(width, height)\n'
        '    if short < MIN_IMAGE_SIDE:\n'
        '        raise ValueError(f"image side {short} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")\n'
        '    if long > MAX_IMAGE_SIDE:\n'
        '        raise ValueError(f"image side {long} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")\n'
        '    if long / short > MAX_ASPECT_RATIO:\n'
        '        raise ValueError(f"aspect ratio {long / short:.2f} > MAX_ASPECT_RATIO {MAX_ASPECT_RATIO}")\n'
        '    return image.convert("RGB")\n'
        '\n'
        '\n'
        'INPUT_SCHEMA: dict[str, Any] = {\n'
        '    "input": "PIL.Image.Image, or a sequence of them for the validation stage; any mode, converted to RGB",\n'
        '    "short_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],\n'
        '    "long_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],\n'
        '    "aspect_ratio": [1.0, MAX_ASPECT_RATIO],\n'
        '    "output": f"{DEPTH_KIND} inverse depth, float32 H x W at the input resolution (larger = nearer)",\n'
        '    "preprocessing": (\n'
        '        "convert to RGB; the DPT processor rescales the sides to multiples of 14 near 518 px and the "\n'
        '        "raw prediction is interpolated back (bicubic) to the input resolution"\n'
        '    ),\n'
        '}\n'
        '\n'
        '\n'
        'def validate_inputs(images: Any, *, names: Sequence[str] | None = None) -> dict[str, Any]:\n'
        '    """Validation stage: return the input manifest (schema, per-input observations, verdict).\n'
        '\n'
        '    Each image is routed through the public ``validate_image`` that ``predict`` itself calls, so a\n'
        '    rejection here raises exactly what ``predict`` would; a caller that wants the finding recorded\n'
        '    catches the exception and stores ``str(exc)`` under ``findings``.\n'
        '    """\n'
        '    batch = [images] if isinstance(images, Image.Image) else images\n'
        '    if not isinstance(batch, Sequence) or isinstance(batch, str | bytes):\n'
        '        raise TypeError("images must be a PIL.Image.Image or a sequence of them")\n'
        '    if len(batch) < 1:\n'
        '        raise ValueError("at least one image is required")\n'
        '    if names is not None and len(names) != len(batch):\n'
        '        raise ValueError("names must have one entry per image")\n'
        '    inputs = []\n'
        '    for index, candidate in enumerate(batch):\n'
        '        rgb = validate_image(candidate)\n'
        '        width, height = rgb.size\n'
        '        long_side, short_side = max(width, height), min(width, height)\n'
        '        inputs.append(\n'
        '            {\n'
        '                "id": names[index] if names else f"image-{index}",\n'
        '                "mode": getattr(candidate, "mode", rgb.mode),\n'
        '                "size": [width, height],\n'
        '                "aspect_ratio": round(long_side / short_side, 3),\n'
        '            }\n'
        '        )\n'
        '    return {\n'
        '        "schema": dict(INPUT_SCHEMA),\n'
        '        "inputs": inputs,\n'
        '        "n_images": len(inputs),\n'
        '        "verdict": "accepted",\n'
        '        "findings": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '\n'
        '\n'
        'def evaluation_report(\n'
        '    result: Mapping[str, Any],\n'
        '    reference_depth: Any | None = None,\n'
        '    *,\n'
        '    sample_kind: str = "synthetic",\n'
        ') -> dict[str, Any]:\n'
        '    """Evaluation stage: a machine-readable report even when nothing is measurable.\n'
        '\n'
        '    With ``reference_depth`` (metric depth in metres, same H x W as the prediction) the report carries\n'
        "    ``abs_rel`` computed by the repository's own helper after least-squares affine alignment in inverse\n"
        '    depth, as sample-sanity evidence. Without it the verdict is ``not-measurable`` and the report says\n'
        '    what ground truth would make the task measurable: relative inverse depth has no intrinsic score.\n'
        '    """\n'
        '    depth = np.asarray(result["depth"])\n'
        '    base = {\n'
        '        "task": "monocular relative depth estimation",\n'
        '        "score_semantics": (\n'
        '            f"{result.get(\'depth_kind\', DEPTH_KIND)} inverse depth with unknown per-image scale and shift: "\n'
        '            "larger is nearer, values are not metres, carry no confidence, and no threshold is shipped"\n'
        '        ),\n'
        '        "sample_kind": sample_kind,\n'
        '        "n_images": 1,\n'
        '        "n_pixels": int(depth.size),\n'
        '        "baselines": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '    if reference_depth is None:\n'
        '        return {\n'
        '            **base,\n'
        '            "metrics": [],\n'
        '            "verdict": "not-measurable",\n'
        '            "reason": "no metric reference depth was supplied for the evaluated image",\n'
        '            "needs": (\n'
        '                "a metric depth map in metres with the same height and width as the image, from a depth "\n'
        '                "sensor, LiDAR, or an RGB-D benchmark, scored with abs_rel(pred, ref_depth, align=True) "\n'
        '                "against a constant-depth or vertical-gradient prior as the trivial baseline"\n'
        '            ),\n'
        '        }\n'
        '    from .metrics import EVAL_DEPTH_RANGE_M\n'
        '\n'
        '    ref = np.asarray(reference_depth, dtype=np.float64)\n'
        '    valid = int((np.isfinite(ref) & (ref >= EVAL_DEPTH_RANGE_M[0]) & (ref <= EVAL_DEPTH_RANGE_M[1])).sum())\n'
        '    return {\n'
        '        **base,\n'
        '        "metrics": [\n'
        '            {\n'
        '                "id": "abs_rel",\n'
        '                "value": abs_rel(depth, ref, align=True),\n'
        '                "align": True,\n'
        '                "n_valid_pixels": valid,\n'
        '                "estimation": (\n'
        '                    "single image, least-squares affine alignment in inverse depth, no dispersion estimate"\n'
        '                ),\n'
        '            }\n'
        '        ],\n'
        '        "verdict": "sample-sanity",\n'
        '        "reason": "one image with caller-supplied metric depth from the tutorial sample; not a benchmark",\n'
        '        "needs": "a held-out set of metric depth maps from the deployment domain for any generalisable claim",\n'
        '    }\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class DepthAnythingPipeline:\n'
        '    """Monocular relative depth estimation over the pinned Depth Anything V2 Small checkpoint."""\n'
        '\n'
        '    _runner: Callable[[Image.Image], np.ndarray]\n'
        '    device: str\n'
        '    source: str = "injected"\n'
        '    _model: Any = field(default=None, repr=False)\n'
        '    _processor: Any = field(default=None, repr=False)\n'
        '    adapter: dict[str, Any] | None = field(default=None, repr=False)\n'
        '\n'
        '    @classmethod\n'
        '    def from_pretrained(\n'
        '        cls,\n'
        '        device: str | None = None,\n'
        '        weights_dir: str | Path | None = None,\n'
        '        allow_download: bool = False,\n'
        '    ) -> DepthAnythingPipeline:\n'
        '        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR\n'
        '        if (root / MANIFEST_NAME).is_file():\n'
        '            stage_missing_files(root, allow_download=allow_download)\n'
        '            verify_snapshot(root)\n'
        '            source, kwargs, origin = str(root), {"local_files_only": True}, "local-snapshot"\n'
        '        elif allow_download:\n'
        '            source, kwargs, origin = MODEL_ID, {}, "hub"\n'
        '        else:\n'
        '            raise FileNotFoundError(\n'
        '                f"no verified snapshot at {root} and allow_download=False; "\n'
        '                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"\n'
        '            )\n'
        '        # Refuse invalid snapshots before importing model libraries.\n'
        '        import torch\n'
        '        from transformers import AutoImageProcessor, AutoModelForDepthEstimation\n'
        '\n'
        '        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n'
        '        processor = AutoImageProcessor.from_pretrained(\n'
        '            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs\n'
        '        )\n'
        '        model = AutoModelForDepthEstimation.from_pretrained(\n'
        '            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs\n'
        '        )\n'
        '        model = model.to(resolved_device).eval()\n'
        '\n'
        '        def runner(image: Image.Image) -> np.ndarray:\n'
        '            inputs = processor(images=image, return_tensors="pt").to(resolved_device)\n'
        '            with torch.inference_mode():\n'
        '                predicted = model(**inputs).predicted_depth\n'
        '            resized = torch.nn.functional.interpolate(\n'
        '                predicted.unsqueeze(1), size=image.size[::-1], mode="bicubic", align_corners=False\n'
        '            )\n'
        '            return resized[0, 0].float().cpu().numpy()\n'
        '\n'
        '        return cls(runner, resolved_device, origin, _model=model, _processor=processor)\n'
        '\n'
        '    def predict(self, image: Image.Image) -> dict[str, Any]:\n'
        '        """Return relative inverse depth as a float32 H x W array at the input resolution."""\n'
        '        rgb = validate_image(image)\n'
        '        depth = np.asarray(self._runner(rgb), dtype=np.float32)\n'
        '        if depth.shape != (rgb.height, rgb.width):\n'
        '            raise RuntimeError(f"backend returned shape {depth.shape}, expected {(rgb.height, rgb.width)}")\n'
        '        return {\n'
        '            "depth": depth,\n'
        '            "depth_kind": DEPTH_KIND,\n'
        '            "depth_min": float(depth.min()),\n'
        '            "depth_max": float(depth.max()),\n'
        '            "height": rgb.height,\n'
        '            "width": rgb.width,\n'
        '            "model_id": MODEL_ID,\n'
        '            "model_revision": MODEL_REVISION,\n'
        '        }\n'
        '\n'
        '    # ---- adaptation ----\n'
        '    def _require_model(self) -> tuple[Any, Any]:\n'
        '        if self._model is None or self._processor is None:\n'
        '            raise ValueError(\n'
        '                "this operation needs a pipeline built with from_pretrained() or from_artifact()"\n'
        '            )\n'
        '        return self._model, self._processor\n'
        '\n'
        '    def _trainable_names(self, trainable_blocks: int) -> list[str]:\n'
        '        """`neck.*` and `head.*` always; plus the last `trainable_blocks` backbone blocks."""\n'
        '        if isinstance(trainable_blocks, bool) or not isinstance(trainable_blocks, int):\n'
        '            raise ValueError(f"trainable_blocks must be an int in 0..{TRANSFORMER_BLOCKS}")\n'
        '        if not 0 <= trainable_blocks <= TRANSFORMER_BLOCKS:\n'
        '            raise ValueError(f"trainable_blocks must be an int in 0..{TRANSFORMER_BLOCKS}")\n'
        '        model, _ = self._require_model()\n'
        '        first = TRANSFORMER_BLOCKS - trainable_blocks\n'
        '        names = []\n'
        '        for name, _p in model.named_parameters():\n'
        '            is_block = name.startswith(BLOCK_PREFIX) and int(name[len(BLOCK_PREFIX) :].split(".")[0]) >= first\n'
        '            if name.startswith(ADAPTER_PREFIXES) or is_block:\n'
        '                names.append(name)\n'
        '        return names\n'
        '\n'
        '    def evaluate(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n'
        '        """Score `predict` on validated `{id, image, depth, mask}` records: affine-aligned AbsRel and δ1."""\n'
        '        from .metrics import depth_metrics\n'
        '        from .samples import validate_dataset\n'
        '\n'
        '        checked = validate_dataset(records, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]\n'
        '        started = time.perf_counter()\n'
        '        preds = [self.predict(r["image"])["depth"] for r in checked]\n'
        '        metrics = depth_metrics(preds, checked)\n'
        '        return {\n'
        '            **metrics,\n'
        '            "policy": self.adapter["policy"] if self.adapter else POLICY_ZERO_SHOT,\n'
        '            "adapted": self.adapter is not None,\n'
        '            "verdict": "measured" if len(checked) >= MIN_SCORED_RECORDS else "measured-small-sample",\n'
        '            "seconds": round(time.perf_counter() - started, 3),\n'
        '            "model_id": MODEL_ID,\n'
        '            "model_revision": MODEL_REVISION,\n'
        '        }\n'
        '\n'
        '    @staticmethod\n'
        '    def _targets(record: Mapping[str, Any], size: tuple[int, int], device: str) -> tuple[Any, Any]:\n'
        '        """Inverse-depth target and validity mask at the model\'s working resolution (nearest resampling)."""\n'
        '        import torch\n'
        '\n'
        '        depth = torch.from_numpy(np.ascontiguousarray(record["depth"], dtype=np.float32))[None, None]\n'
        '        mask = torch.from_numpy(np.ascontiguousarray(record["mask"], dtype=np.float32))[None, None]\n'
        '        depth = torch.nn.functional.interpolate(depth, size=size, mode="nearest")[0, 0]\n'
        '        valid = torch.nn.functional.interpolate(mask, size=size, mode="nearest")[0, 0] > 0.5\n'
        '        valid &= depth > 0\n'
        '        if int(valid.sum()) < MIN_VALID_TARGET_PIXELS:\n'
        '            raise ValueError(\n'
        '                f"record {record.get(\'id\', \'?\')!r} keeps {int(valid.sum())} valid pixels at the working "\n'
        '                f"resolution; at least {MIN_VALID_TARGET_PIXELS} are needed for the affine alignment"\n'
        '            )\n'
        '        return (1.0 / depth.clamp_min(1e-6)).to(device), valid.to(device)\n'
        '\n'
        '    @staticmethod\n'
        '    def _ssi_loss(pred: Any, target: Any, valid: Any) -> Any:\n'
        '        """Scale-and-shift-invariant squared error: the prediction is affinely aligned to the inverse-depth\n'
        '        target over valid pixels in closed form (gradients flow through the fit), residuals normalised by\n'
        '        the target\'s mean magnitude so images at different depths weigh alike."""\n'
        '        p = pred[valid]\n'
        '        t = target[valid]\n'
        '        n = float(p.numel())\n'
        '        sp, st, spp, spt = p.sum(), t.sum(), (p * p).sum(), (p * t).sum()\n'
        '        det = (n * spp - sp * sp).clamp_min(1e-12)\n'
        '        scale = (n * spt - sp * st) / det\n'
        '        shift = (st - scale * sp) / n\n'
        '        residual = (scale * p + shift - t) / (t.abs().mean() + 1e-6)\n'
        '        return (residual * residual).mean()\n'
        '\n'
        '    def adapt(\n'
        '        self,\n'
        '        train: Sequence[Mapping[str, Any]],\n'
        '        val: Sequence[Mapping[str, Any]] | None = None,\n'
        '        *,\n'
        '        trainable_blocks: int = DEFAULT_TRAINABLE_BLOCKS,\n'
        '        head_epochs: int = 2,\n'
        '        epochs: int = 3,\n'
        '        lr: float = 1e-5,\n'
        '        seed: int = 0,\n'
        '        progress: Callable[[dict[str, Any]], None] | None = None,\n'
        '    ) -> dict[str, Any]:\n'
        '        """Bounded supervised adaptation under an explicit policy ladder, selected on validation.\n'
        '\n'
        '        Epoch 0 is the untouched checkpoint (the **zero-shot policy**). Stage A (the **frozen policy**) trains\n'
        '        the DPT neck and head on the frozen backbone for `head_epochs` epochs. Stage B (the **unfrozen\n'
        '        policy**, when `trainable_blocks` > 0 and `epochs` > 0) continues with the last `trainable_blocks`\n'
        '        transformer blocks unfrozen for `epochs` more epochs. Every epoch trains one image at a time with\n'
        '        AdamW (`lr`, weight decay 0.01, gradient clipping 1.0, seeded order, no augmentation) under the\n'
        '        scale-and-shift-invariant loss and is scored on `val` by `evaluate`; the epoch with the **lowest\n'
        '        validation AbsRel** is kept and its tensors restored — so the outcome can be "do not adapt".\n'
        '        Without `val` the final epoch is kept.\n'
        '        """\n'
        '        if isinstance(head_epochs, bool) or not isinstance(head_epochs, int) or not 0 <= head_epochs <= 20:\n'
        '            raise ValueError("head_epochs must be an int in 0..20")\n'
        '        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 0 <= epochs <= 20:\n'
        '            raise ValueError("epochs must be an int in 0..20")\n'
        '        if not isinstance(lr, int | float) or not 0.0 < float(lr) <= 1e-3:\n'
        '            raise ValueError("lr must be in (0, 1e-3]")\n'
        '        if isinstance(trainable_blocks, bool) or not isinstance(trainable_blocks, int):\n'
        '            raise ValueError(f"trainable_blocks must be an int in 0..{TRANSFORMER_BLOCKS}")\n'
        '        if not 0 <= trainable_blocks <= TRANSFORMER_BLOCKS:\n'
        '            raise ValueError(f"trainable_blocks must be an int in 0..{TRANSFORMER_BLOCKS}")\n'
        '        model, processor = self._require_model()\n'
        '        import torch\n'
        '\n'
        '        from .samples import validate_dataset\n'
        '\n'
        '        names_b = self._trainable_names(trainable_blocks)\n'
        '        names_a = [n for n in names_b if n.startswith(ADAPTER_PREFIXES)]\n'
        '        train_records = validate_dataset(train)["records"]\n'
        '        val_records = (\n'
        '            validate_dataset(val, min_records=1, max_records=MAX_EVAL_RECORDS)["records"]\n'
        '            if val is not None\n'
        '            else None\n'
        '        )\n'
        '        params = dict(model.named_parameters())\n'
        '        started = time.perf_counter()\n'
        '\n'
        '        def brief(metrics: Mapping[str, Any] | None) -> dict[str, float] | None:\n'
        '            if metrics is None:\n'
        '                return None\n'
        '            return {\n'
        '                "n": metrics["n"],\n'
        '                "abs_rel": round(metrics["abs_rel"], 6),\n'
        '                "delta1": round(metrics["delta1"], 6),\n'
        '            }\n'
        '\n'
        '        def score() -> dict[str, float] | None:\n'
        '            model.eval()\n'
        '            return brief(self.evaluate(val_records)) if val_records is not None else None\n'
        '\n'
        '        history: list[dict[str, Any]] = [\n'
        '            {"epoch": 0, "stage": POLICY_ZERO_SHOT, "train_loss": None, "val": score()}\n'
        '        ]\n'
        '        best_epoch, best_score = 0, (history[0]["val"]["abs_rel"] if history[0]["val"] else math.inf)\n'
        '        best_state = {n: params[n].detach().clone() for n in names_b}\n'
        '        stages: list[tuple[str, list[str], int]] = [(POLICY_FROZEN, names_a, head_epochs)]\n'
        '        if trainable_blocks > 0 and epochs > 0:\n'
        '            stages.append((POLICY_UNFROZEN.format(k=trainable_blocks), names_b, epochs))\n'
        '        torch.manual_seed(seed)\n'
        '        rng = random.Random(seed)\n'
        '        epoch = 0\n'
        '        initial_state = {n: v.clone() for n, v in best_state.items()}\n'
        '        try:\n'
        '            for stage, names, n_epochs in stages:\n'
        '                if n_epochs == 0:\n'
        '                    continue\n'
        '                for p in model.parameters():\n'
        '                    p.requires_grad_(False)\n'
        '                for n in names:\n'
        '                    params[n].requires_grad_(True)\n'
        '                optimiser = torch.optim.AdamW([params[n] for n in names], lr=float(lr), weight_decay=0.01)\n'
        '                for _ in range(n_epochs):\n'
        '                    epoch += 1\n'
        '                    model.train()\n'
        '                    order = list(range(len(train_records)))\n'
        '                    rng.shuffle(order)\n'
        '                    losses = []\n'
        '                    for index in order:\n'
        '                        record = train_records[index]\n'
        '                        pixel_values = processor(images=record["image"], return_tensors="pt")["pixel_values"]\n'
        '                        pred = model(pixel_values=pixel_values.to(self.device)).predicted_depth[0]\n'
        '                        target, valid = self._targets(record, tuple(pred.shape), self.device)\n'
        '                        loss = self._ssi_loss(pred.float(), target, valid)\n'
        '                        if not torch.isfinite(loss):\n'
        '                            raise RuntimeError(\n'
        '                                f"non-finite loss on record {record[\'id\']!r}; adaptation aborted"\n'
        '                            )\n'
        '                        optimiser.zero_grad(set_to_none=True)\n'
        '                        loss.backward()\n'
        '                        torch.nn.utils.clip_grad_norm_([params[n] for n in names], 1.0)\n'
        '                        optimiser.step()\n'
        '                        losses.append(float(loss.detach()))\n'
        '                    val_metrics = score()\n'
        '                    entry = {\n'
        '                        "epoch": epoch,\n'
        '                        "stage": stage,\n'
        '                        "train_loss": float(np.mean(losses)),\n'
        '                        "val": val_metrics,\n'
        '                    }\n'
        '                    history.append(entry)\n'
        '                    if progress is not None:\n'
        '                        progress(entry)\n'
        '                    current = val_metrics["abs_rel"] if val_metrics else -epoch  # no val: the last epoch wins\n'
        '                    if current < best_score:\n'
        '                        best_epoch, best_score = epoch, current\n'
        '                        best_state = {n: params[n].detach().clone() for n in names_b}\n'
        '        except BaseException:\n'
        '            # Transactional: a failure in training, validation or the progress callback leaves the base\n'
        '            # exactly as it was, frozen, with no adapter attached.\n'
        '            with torch.no_grad():\n'
        '                for n, value in initial_state.items():\n'
        '                    params[n].copy_(value)\n'
        '            for p in model.parameters():\n'
        '                p.requires_grad_(False)\n'
        '            model.eval()\n'
        '            self.adapter = None\n'
        '            raise\n'
        '        with torch.no_grad():\n'
        '            for n, value in best_state.items():\n'
        '                params[n].copy_(value)\n'
        '        for p in model.parameters():\n'
        '            p.requires_grad_(False)\n'
        '        model.eval()\n'
        '        policy = history[best_epoch]["stage"]\n'
        '        self.adapter = {\n'
        '            "policy": policy,\n'
        '            "trainable_blocks": trainable_blocks,\n'
        '            "head_epochs": head_epochs,\n'
        '            "epochs": epochs,\n'
        '            "lr": float(lr),\n'
        '            "seed": seed,\n'
        '            "best_epoch": best_epoch,\n'
        '            "selection": "lowest validation AbsRel (epoch 0 = zero-shot checkpoint)"\n'
        '            if val_records is not None\n'
        '            else "final epoch (no validation split)",\n'
        '            "n_trainable_head": sum(params[n].numel() for n in names_a),\n'
        '            "n_trainable_blocks": sum(\n'
        '                params[n].numel() for n in names_b if not n.startswith(ADAPTER_PREFIXES)\n'
        '            ),\n'
        '            "n_total": sum(p.numel() for p in model.parameters()),\n'
        '            "n_train": len(train_records),\n'
        '            "n_val": len(val_records) if val_records is not None else 0,\n'
        '            "history": history,\n'
        '            "trainable_names": names_b if policy.startswith("unfrozen") else names_a,\n'
        '            "seconds": round(time.perf_counter() - started, 3),\n'
        '        }\n'
        '        return dict(self.adapter)\n'
        '\n'
        '    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:\n'
        '        """Write the DPT neck and head (and any trained block tensors) as safetensors with a manifest naming\n'
        '        the base; the backbone blocks travel only when the unfrozen policy was selected."""\n'
        '        if self.adapter is None:\n'
        '            raise ValueError("nothing to save: call adapt() first")\n'
        '        model, _ = self._require_model()\n'
        '        from safetensors.torch import save_file\n'
        '\n'
        '        out = Path(output_dir)\n'
        '        out.mkdir(parents=True, exist_ok=True)\n'
        '        names = set(self.adapter.get("trainable_names", []))\n'
        '        tensors = {k: v.detach().cpu().contiguous() for k, v in model.state_dict().items() if k in names}\n'
        '        weights_path = out / ARTIFACT_WEIGHTS_NAME\n'
        '        save_file(tensors, str(weights_path), metadata={"format": "pt"})\n'
        '        manifest = {\n'
        '            "format": ARTIFACT_FORMAT,\n'
        '            "format_version": ARTIFACT_FORMAT_VERSION,\n'
        '            "base_model": {\n'
        '                "id": MODEL_ID,\n'
        '                "revision": MODEL_REVISION,\n'
        '                "key": MODEL_KEY,\n'
        '                "weight_file": WEIGHTS_FILE,\n'
        '                "weight_sha256": WEIGHT_SHA256,\n'
        '            },\n'
        '            "adapter": {k: v for k, v in self.adapter.items() if k not in ("history", "trainable_names")},\n'
        '            "no_op": self.adapter["policy"] == POLICY_ZERO_SHOT,\n'
        '            "note": (\n'
        '                "the zero-shot policy was selected: the neck and head tensors below are byte copies of the "\n'
        '                "base and loading this artifact changes nothing"\n'
        '                if self.adapter["policy"] == POLICY_ZERO_SHOT\n'
        '                else "the neck and head tensors, plus the trained backbone blocks under the unfrozen policy"\n'
        '            ),\n'
        '            "history": self.adapter.get("history", []),\n'
        '            "tensors": sorted(tensors),\n'
        '            "files": [\n'
        '                {\n'
        '                    "path": ARTIFACT_WEIGHTS_NAME,\n'
        '                    "bytes": weights_path.stat().st_size,\n'
        '                    "sha256": _sha256(weights_path),\n'
        '                }\n'
        '            ],\n'
        '            "metadata": dict(metadata or {}),\n'
        '        }\n'
        '        (out / ARTIFACT_MANIFEST_NAME).write_text(\n'
        '            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"\n'
        '        )\n'
        '        return out\n'
        '\n'
        '    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> tuple[Path, str, int]:\n'
        '        """Refuse an artifact whose manifest is not exactly the one this pipeline writes: the supported format\n'
        '        and version, the pinned base (id, revision, weight file, digest), exactly one file entry named\n'
        '        `adapter.safetensors` that resolves inside the artifact directory, a canonical policy and an integer\n'
        '        `trainable_blocks` in range. Nothing is deserialised here. The digest check that follows detects\n'
        '        corruption or drift of the weights relative to the adjacent manifest; it is not authenticity against\n'
        '        an actor who can replace both files."""\n'
        '        if manifest.get("format") != ARTIFACT_FORMAT:\n'
        '            raise ValueError(f"artifact format {manifest.get(\'format\')!r} != {ARTIFACT_FORMAT!r}")\n'
        '        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:\n'
        '            raise ValueError(\n'
        '                f"artifact format_version {manifest.get(\'format_version\')!r} is not the supported "\n'
        '                f"{ARTIFACT_FORMAT_VERSION!r}"\n'
        '            )\n'
        '        base = manifest.get("base_model", {})\n'
        '        if (base.get("id"), base.get("revision"), base.get("weight_sha256")) != (\n'
        '            MODEL_ID,\n'
        '            MODEL_REVISION,\n'
        '            WEIGHT_SHA256,\n'
        '        ):\n'
        '            raise ValueError("artifact was adapted from a different base model, revision or weight file")\n'
        '        if base.get("weight_file", WEIGHTS_FILE) != WEIGHTS_FILE:\n'
        '            raise ValueError("artifact was adapted from a different base weight file")\n'
        '        files = manifest.get("files")\n'
        '        if not isinstance(files, list) or len(files) != 1:\n'
        '            raise ValueError("artifact manifest must list exactly one file")\n'
        '        entry = files[0]\n'
        '        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:\n'
        '            raise ValueError(f"artifact manifest must name exactly {ARTIFACT_WEIGHTS_NAME!r}")\n'
        '        weights_path = (root / entry["path"]).resolve()\n'
        '        if weights_path.parent != root.resolve():\n'
        '            raise ValueError("artifact weight path must resolve inside the artifact directory")\n'
        '        adapter = manifest.get("adapter")\n'
        '        if not isinstance(adapter, Mapping):\n'
        '            raise ValueError("artifact manifest has no adapter block")\n'
        '        blocks = adapter.get("trainable_blocks")\n'
        '        if isinstance(blocks, bool) or not isinstance(blocks, int) or not 0 <= blocks <= TRANSFORMER_BLOCKS:\n'
        '            raise ValueError(\n'
        '                f"artifact manifest does not record an integer trainable_blocks in 0..{TRANSFORMER_BLOCKS}"\n'
        '            )\n'
        '        policy = adapter.get("policy")\n'
        '        if policy in (POLICY_ZERO_SHOT, POLICY_FROZEN):\n'
        '            blocks = 0\n'
        '        elif policy != POLICY_UNFROZEN.format(k=blocks) or blocks == 0:\n'
        '            raise ValueError(\n'
        '                f"artifact policy {policy!r} is not a canonical policy for trainable_blocks={blocks}"\n'
        '            )\n'
        '        if not isinstance(manifest.get("tensors"), list):\n'
        '            raise ValueError("artifact manifest must list its tensors")\n'
        '        return weights_path, policy, blocks\n'
        '\n'
        '    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:\n'
        '        """Verify an adapter\'s manifest, digest and exact tensor set **before** deserialising, then overlay\n'
        '        its tensors (a zero-shot artifact must equal the base and changes nothing)."""\n'
        '        root = Path(artifact_dir)\n'
        '        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))\n'
        '        weights_path, policy, blocks = self._check_artifact_manifest(root, manifest)\n'
        '        entry = manifest["files"][0]\n'
        '        if not weights_path.is_file():\n'
        '            raise FileNotFoundError(f"artifact weights missing: {weights_path}")\n'
        '        if _sha256(weights_path) != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: digest or size mismatch; refusing to load")\n'
        '        # The exact tensor set the recorded policy implies: neck and head always, the last `blocks` blocks\n'
        '        # only under the unfrozen policy.\n'
        '        expected = sorted(self._trainable_names(blocks))\n'
        '        if sorted(manifest["tensors"]) != expected:\n'
        '            raise ValueError("artifact tensor list does not match its recorded policy and trainable_blocks")\n'
        '        model, _ = self._require_model()\n'
        '        import torch\n'
        '        from safetensors.torch import load_file\n'
        '\n'
        '        tensors = load_file(str(weights_path))\n'
        '        if sorted(tensors) != expected:\n'
        '            raise ValueError("artifact tensor names differ from its manifest")\n'
        '        state = model.state_dict()\n'
        '        if policy == POLICY_ZERO_SHOT and any(\n'
        '            not torch.equal(value.to(state[key].dtype), state[key].cpu()) for key, value in tensors.items()\n'
        '        ):\n'
        '            raise ValueError("artifact claims the zero-shot policy but its tensors differ from the base")\n'
        '        for key, value in tensors.items():\n'
        '            if key not in state or not key.startswith((*ADAPTER_PREFIXES, BLOCK_PREFIX)):\n'
        '                raise ValueError(\n'
        '                    f"artifact tensor {key} is not an adaptable neck, head or transformer-block tensor"\n'
        '                )\n'
        '            if tuple(value.shape) != tuple(state[key].shape):\n'
        '                raise ValueError(\n'
        '                    f"artifact tensor {key}: shape {tuple(value.shape)} != {tuple(state[key].shape)}"\n'
        '                )\n'
        '        with torch.no_grad():\n'
        '            params = dict(model.named_parameters())\n'
        '            for key, value in tensors.items():\n'
        '                params[key].copy_(value.to(params[key].dtype))\n'
        '        model.eval()\n'
        '        self.adapter = {\n'
        '            **manifest["adapter"],\n'
        '            "trainable_names": sorted(tensors),\n'
        '            "history": manifest.get("history", []),\n'
        '        }\n'
        '        return manifest\n'
        '\n'
        '    @classmethod\n'
        '    def from_artifact(\n'
        '        cls,\n'
        '        artifact_dir: str | Path,\n'
        '        *,\n'
        '        device: str | None = None,\n'
        '        weights_dir: str | Path | None = None,\n'
        '        allow_download: bool = False,\n'
        '    ) -> DepthAnythingPipeline:\n'
        '        pipeline = cls.from_pretrained(device=device, weights_dir=weights_dir, allow_download=allow_download)\n'
        '        pipeline.load_artifact(artifact_dir)\n'
        '        return pipeline\n'
    ),
    'depth_reference/samples.py': (
        '"""Depth-labelled image dataset contract for adapting the relative-depth estimator: the pinned DIODE\n'
        'validation sample, validation, seeded scan-level splitting, BYOD loaders and CSV export.\n'
        '\n'
        'The default dataset is **real** and carries metric ground truth: 40 views from the DIODE validation release\n'
        '(Vasiljevic et al., 2019; CC BY 4.0) — the first two views in file-name order of every one of its 20 scans\n'
        '(10 indoor, 10 outdoor) — chosen a priori on 2026-09-19 and pinned here per file (RGB PNG, depth `.npy` in\n'
        'metres, validity-mask `.npy`) by byte size and SHA-256 as served by the Marigold evaluation mirror of DIODE on\n'
        'the Hugging Face Hub at an immutable commit. Every file is fetched at run time and refused on any byte-size or\n'
        'SHA-256 mismatch; the repository redistributes none of them. Views of one scan share a scene, so the sample is\n'
        'split **by scan**, never by view.\n'
        '\n'
        'A record is ``{id, image, depth, mask}``: a PIL image (or a path to one), a float32 H x W array of metric\n'
        'depth in metres (or a path to a `.npy`) and a boolean H x W validity mask (or a path; missing means\n'
        '``depth > 0``). ``domain`` / ``scene`` / ``scan`` are optional provenance keys; ``scan`` (or ``group``) is the\n'
        'split unit when present.\n'
        '"""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import csv\n'
        'import hashlib\n'
        'import io\n'
        'import json\n'
        'import random\n'
        'import re\n'
        'import urllib.request\n'
        'import zipfile\n'
        'from collections.abc import Mapping, Sequence\n'
        'from pathlib import Path\n'
        'from typing import Any\n'
        '\n'
        'import numpy as np\n'
        'from PIL import Image\n'
        '\n'
        'from .pipeline import MAX_ASPECT_RATIO, MAX_IMAGE_SIDE, MIN_IMAGE_SIDE, MODEL_ID\n'
        '\n'
        'CORPUS_NAME = "DIODE validation views (indoor + outdoor) with metric depth"\n'
        'CORPUS_RELEASE = (\n'
        '    "DIODE validation release as served by the Marigold evaluation mirror "\n'
        '    "(obukhovai/marigold_depth_eval @ 30c5b061, Hugging Face Hub); 40 views pinned 2026-09-19"\n'
        ')\n'
        'CORPUS_COMMIT = "30c5b061d863e383e3ea9fa14555737a199d7ad9"\n'
        'CORPUS_BASE_URL = (\n'
        '    f"https://huggingface.co/datasets/obukhovai/marigold_depth_eval/resolve/{CORPUS_COMMIT}/diode/"\n'
        ')\n'
        'CORPUS_LICENSE = "CC BY 4.0 (DIODE: A Dense Indoor and Outdoor DEpth Dataset, Vasiljevic et al., 2019)"\n'
        'CORPUS_BYTES = 312_448_846\n'
        'DOMAINS = ("indoors", "outdoor")\n'
        'FILE_SUFFIXES = (".png", "_depth.npy", "_depth_mask.npy")\n'
        '\n'
        '# (record id, domain, scene, scan, file stem, then bytes + sha256 of the png, depth .npy and mask .npy)\n'
        'SAMPLE_RECORDS: tuple[tuple[str, str, str, str, str, int, str, int, str, int, str], ...] = (\n'
        '    (\n'
        '        "view-000",\n'
        '        "indoors",\n'
        '        "scene_00019",\n'
        '        "scan_00183",\n'
        '        "00019_00183_indoors_000_010",\n'
        '        1172425,\n'
        '        "71f9713e01a4d9899184b63f293446e62953ee2e526d81e5da8f101f2400ae46",\n'
        '        3145856,\n'
        '        "a2bbbf55765ffec98a75d3a78f9925b28bfe476080d3e8eae86c645db4ac2b15",\n'
        '        3145856,\n'
        '        "e9b5f1cce7c4e8e03a94b006f81ef72d963bfb9b7187f8208f9e92067db823fa",\n'
        '    ),\n'
        '    (\n'
        '        "view-001",\n'
        '        "indoors",\n'
        '        "scene_00019",\n'
        '        "scan_00183",\n'
        '        "00019_00183_indoors_000_040",\n'
        '        1102623,\n'
        '        "ba05f76208099f78dcf483908c7de07ad4d89784ae694aefdbb5a82dc2bfedf1",\n'
        '        3145856,\n'
        '        "c72d44e851b649ad787de4d0a21e5c115660779d4d1f8e054e8af201748c2c39",\n'
        '        3145856,\n'
        '        "304216b2ce8825e529d836e7ebe36dd80090c19b2e620029a158f1dd42a3e42c",\n'
        '    ),\n'
        '    (\n'
        '        "view-002",\n'
        '        "indoors",\n'
        '        "scene_00020",\n'
        '        "scan_00184",\n'
        '        "00020_00184_indoors_050_000",\n'
        '        1142437,\n'
        '        "2641d27768b547e53d4d1dac922b72cb31a0413762bd78b36b898386c150d40e",\n'
        '        3145856,\n'
        '        "d4943f5e28555bfd59e609e7116539f2efd68c8a9378dcf696dca0e078f62004",\n'
        '        3145856,\n'
        '        "789dabf2482fa3e3d4bab67b81d49518e1b618f7ba55b82edae10ac8d18e4855",\n'
        '    ),\n'
        '    (\n'
        '        "view-003",\n'
        '        "indoors",\n'
        '        "scene_00020",\n'
        '        "scan_00184",\n'
        '        "00020_00184_indoors_050_020",\n'
        '        1017065,\n'
        '        "a44432a3ba6bab0f6b4bb4326df5db4347a6eb577f74f83141c59aae6810b6a4",\n'
        '        3145856,\n'
        '        "d5f628673cfcbff12082cfe2dee75c71d8521560486946369dca90b369aa9b5c",\n'
        '        3145856,\n'
        '        "06077d743933e907c17b9e3e2b5146987e87f1b0a2122c784ada4f55a8f8b3f4",\n'
        '    ),\n'
        '    (\n'
        '        "view-004",\n'
        '        "indoors",\n'
        '        "scene_00020",\n'
        '        "scan_00185",\n'
        '        "00020_00185_indoors_000_000",\n'
        '        1208304,\n'
        '        "1bf8317ae6d1ef6c412ff233bda2944568aa7e9169606ed5f694d9c2bc1032f9",\n'
        '        3145856,\n'
        '        "6de0b81413613d2abecea29e327220a6d94d54d5ab1bccca786a9720993a4e30",\n'
        '        3145856,\n'
        '        "afc7dad580ec7d8f41884032f0c97a8bcee31e76d22cc7c2ab988b28daef4dd0",\n'
        '    ),\n'
        '    (\n'
        '        "view-005",\n'
        '        "indoors",\n'
        '        "scene_00020",\n'
        '        "scan_00185",\n'
        '        "00020_00185_indoors_000_020",\n'
        '        1213267,\n'
        '        "a632cec9d93174aaf8adab73a36e9afc12cb08c99c551ce02e97315de2a288d7",\n'
        '        3145856,\n'
        '        "f17ee07bb9b513770b0a98f53c53440957de3b04b162a23991cc92a64cc678ce",\n'
        '        3145856,\n'
        '        "a302df41411a3d947b1a3378f6eb64df763726e84470acd9c47fcaa02f7f4d21",\n'
        '    ),\n'
        '    (\n'
        '        "view-006",\n'
        '        "indoors",\n'
        '        "scene_00020",\n'
        '        "scan_00186",\n'
        '        "00020_00186_indoors_000_000",\n'
        '        1194403,\n'
        '        "2478882bbac47733a075b3871d178e4c27ba432d0f6d2af1341265a8eecf1369",\n'
        '        3145856,\n'
        '        "b4c7533123e6c7204e6b67d4494af0e59bb454b476e552539c2b058509fc2f06",\n'
        '        3145856,\n'
        '        "dbf9d3e124e71fe633411f6e54d58f712095044dd152e3f8d18cbd5c75472c42",\n'
        '    ),\n'
        '    (\n'
        '        "view-007",\n'
        '        "indoors",\n'
        '        "scene_00020",\n'
        '        "scan_00186",\n'
        '        "00020_00186_indoors_020_000",\n'
        '        1171589,\n'
        '        "5a7ffa68320043e62c35319b8b2e971bd29331efef181e92ca04cac17a9f4605",\n'
        '        3145856,\n'
        '        "7b488f809c23c62594406d1e8fafb4857d7da7d6c98571cc0e3a4a10b76bbd71",\n'
        '        3145856,\n'
        '        "403d0dbaa52767792e69a4471c03a6c0dca174dd517dc8c567dd05851c4b3528",\n'
        '    ),\n'
        '    (\n'
        '        "view-008",\n'
        '        "indoors",\n'
        '        "scene_00020",\n'
        '        "scan_00187",\n'
        '        "00020_00187_indoors_000_000",\n'
        '        1258618,\n'
        '        "16bf17dce5d4025536bada42024cd756c7e6f6ee5012a7e7c606039c7e7028db",\n'
        '        3145856,\n'
        '        "77068b8f5e9faa2e6cc89f381720873a36b5631643f3df4cff40d8f59d10ed28",\n'
        '        3145856,\n'
        '        "28d3e14c26011438587539d9e5c4e9fecbd631f02727fd35ac5f533cbabfb8fb",\n'
        '    ),\n'
        '    (\n'
        '        "view-009",\n'
        '        "indoors",\n'
        '        "scene_00020",\n'
        '        "scan_00187",\n'
        '        "00020_00187_indoors_000_020",\n'
        '        1202574,\n'
        '        "e241249dadc41376befa417e33165412a834841334cee9e950729eaecb1243ce",\n'
        '        3145856,\n'
        '        "3ee01eb76928b1cdc49e81a9a7af29797fb4d25c5409754314972da29ed9f9dd",\n'
        '        3145856,\n'
        '        "1ec90f9292cc265e1bd8865757fc09cd119ba4ec9c6cb2863803823b46cc994f",\n'
        '    ),\n'
        '    (\n'
        '        "view-010",\n'
        '        "indoors",\n'
        '        "scene_00021",\n'
        '        "scan_00188",\n'
        '        "00021_00188_indoors_090_000",\n'
        '        981100,\n'
        '        "1040b93aa913ad042231e9b9809293745b02c24c90c0cba706a44c77b7847f56",\n'
        '        3145856,\n'
        '        "b4a9073b489bff9d472576592185733ee7a709a7aa04445f0f43356ee333e248",\n'
        '        3145856,\n'
        '        "940df9212e7cde2ba9225848f57f3aa8d15d6c9dfe2f63bbf3aae00e5c09c282",\n'
        '    ),\n'
        '    (\n'
        '        "view-011",\n'
        '        "indoors",\n'
        '        "scene_00021",\n'
        '        "scan_00188",\n'
        '        "00021_00188_indoors_090_020",\n'
        '        1057542,\n'
        '        "ffe1f5390dca8b043d672a568d28b4f3409276dce13da76f3d18b16e56bb85a9",\n'
        '        3145856,\n'
        '        "d7a5487729ca00dc8ce1f0c08006246143e34b79e8da211b0b4ce81a21db23aa",\n'
        '        3145856,\n'
        '        "2922ae237dc3d890ed80d0cd8d704c61728ced3383cf249c24d2c36246212853",\n'
        '    ),\n'
        '    (\n'
        '        "view-012",\n'
        '        "indoors",\n'
        '        "scene_00021",\n'
        '        "scan_00189",\n'
        '        "00021_00189_indoors_000_000",\n'
        '        1036343,\n'
        '        "fed8f8be2ea611b2d1ce2b75440f029d6b786cef7218bace273e96dc79acf60b",\n'
        '        3145856,\n'
        '        "e71a79b0a7a22fb6a979543809abb2bd7f906487a7b2e9078d4d93cfc562eed7",\n'
        '        3145856,\n'
        '        "4b053fdf65b2123bdcadb43e3629521a8eacce52d7fcd820ccddfb8f3ae39663",\n'
        '    ),\n'
        '    (\n'
        '        "view-013",\n'
        '        "indoors",\n'
        '        "scene_00021",\n'
        '        "scan_00189",\n'
        '        "00021_00189_indoors_000_020",\n'
        '        1075564,\n'
        '        "8dd7d9980c7c79ea74d5438154cbe83fc6e245189b9d36406fc1509d8b7b32e9",\n'
        '        3145856,\n'
        '        "9b5f1be74b6fc3901fffe7dc38be3d3a2662ad92a565e9b1e263d2b5784bbb94",\n'
        '        3145856,\n'
        '        "625221f96afa15cf03e409dcd04910c793f2a187d85b822f834fead3ae601da9",\n'
        '    ),\n'
        '    (\n'
        '        "view-014",\n'
        '        "indoors",\n'
        '        "scene_00021",\n'
        '        "scan_00190",\n'
        '        "00021_00190_indoors_050_000",\n'
        '        1000911,\n'
        '        "4d90cc37cf28095d1aa45668f6f6457eca7f28560a0f62b640b0802a1f5394eb",\n'
        '        3145856,\n'
        '        "c939b634fb09d8a08a111bb6739e4298245dea12171becff202d900b72284fc7",\n'
        '        3145856,\n'
        '        "39e2709c2a01a290103bf55f48d4674ff08e213ac7f42afe7cc0e62cc43bad68",\n'
        '    ),\n'
        '    (\n'
        '        "view-015",\n'
        '        "indoors",\n'
        '        "scene_00021",\n'
        '        "scan_00190",\n'
        '        "00021_00190_indoors_050_030",\n'
        '        996884,\n'
        '        "851a7f44cd3ebfca766a3bd25a818a000277f2752f16c0e1bac0655ba77f6168",\n'
        '        3145856,\n'
        '        "1d11b9096ab6eeca053307d39eb2a7c3db9d1cc529b5c243421c111eca6ea0a1",\n'
        '        6291584,\n'
        '        "8192c4b01dd2c6776cc03f68e9a61f792c9149e27ea7959e362c224193a85349",\n'
        '    ),\n'
        '    (\n'
        '        "view-016",\n'
        '        "indoors",\n'
        '        "scene_00021",\n'
        '        "scan_00191",\n'
        '        "00021_00191_indoors_000_040",\n'
        '        1057445,\n'
        '        "2a7fafdbdfbaf1bbb9363b32b5337a86f391ebdd8cb23a0227d5e6ec3bddc11d",\n'
        '        3145856,\n'
        '        "64609dc44db5769caae0aa5b5a29a92a380ff5aaf9c0173c172a7dda0cd8f42a",\n'
        '        3145856,\n'
        '        "7c2ee20e3128526b51067419562fba26d25ba734d2a92a31467e9b42d190f0a8",\n'
        '    ),\n'
        '    (\n'
        '        "view-017",\n'
        '        "indoors",\n'
        '        "scene_00021",\n'
        '        "scan_00191",\n'
        '        "00021_00191_indoors_010_030",\n'
        '        1053502,\n'
        '        "3a686ca893ba52f2780370737b762080b2cf2ce7201c4311a9f31ed29595f92f",\n'
        '        3145856,\n'
        '        "ece8e9acdeb8601bc111d70b6260d0ea92cea926166dc5b0c1d2bde3f5406b8e",\n'
        '        3145856,\n'
        '        "bdf32493a0ad439062821d9f9f3f3a966454c431d2c4bd4b5d22be89bceb494a",\n'
        '    ),\n'
        '    (\n'
        '        "view-018",\n'
        '        "indoors",\n'
        '        "scene_00021",\n'
        '        "scan_00192",\n'
        '        "00021_00192_indoors_000_000",\n'
        '        1061846,\n'
        '        "28c6d3acbf088975eddc38ab6e5d4162b7f93fba8f5459d0f833c2d30fe89530",\n'
        '        3145856,\n'
        '        "ebbec603a9625ee58cfd861770402c1cd9fafeb5b6126e6a9a7628941bddefce",\n'
        '        3145856,\n'
        '        "0eeeadfb8976b9a77f3ce4351401bec8b1e61ba0a4b2a8397e7328dae738cde5",\n'
        '    ),\n'
        '    (\n'
        '        "view-019",\n'
        '        "indoors",\n'
        '        "scene_00021",\n'
        '        "scan_00192",\n'
        '        "00021_00192_indoors_000_020",\n'
        '        1057090,\n'
        '        "fe69baf50e4801abd3b5f9b2dacd70e33a0ed8c8f4a2d975c980128bf1802e74",\n'
        '        3145856,\n'
        '        "2aba7a7f71b5dbd799a42609aab58ae2b6ad7ca7521dc05db5288b05010b585c",\n'
        '        3145856,\n'
        '        "1263a8da470fcdaad3be5d70034adeac2f2d7b8aa862fa1f962ab908e5a2184f",\n'
        '    ),\n'
        '    (\n'
        '        "view-020",\n'
        '        "outdoor",\n'
        '        "scene_00022",\n'
        '        "scan_00193",\n'
        '        "00022_00193_outdoor_000_000",\n'
        '        1260962,\n'
        '        "f513f607fbdcdbe3545cfc67d90828d81619be157689c1db26826559c1cae0a6",\n'
        '        3145856,\n'
        '        "a145d0f58dfabc71e59bb5831d42b11989d2be3d8bb6ef16ca16449f77907b41",\n'
        '        3145856,\n'
        '        "c06249a9c2d6b204fba67219db98be321f5fa39707b67d3b2e3880fe6a707c55",\n'
        '    ),\n'
        '    (\n'
        '        "view-021",\n'
        '        "outdoor",\n'
        '        "scene_00022",\n'
        '        "scan_00193",\n'
        '        "00022_00193_outdoor_000_020",\n'
        '        1387923,\n'
        '        "2fa8096f650f9bb7327cffa8079d471feaffd11b1a37b3133c456d4bb89356de",\n'
        '        3145856,\n'
        '        "59513f5e83be4ab75dd8a996704fcbb32f1a4b1143c874654d6106781b7386b7",\n'
        '        3145856,\n'
        '        "1ec6d6f79ba218282caf7ae1220b0fa19d06e56e09458b0a85e0825a91545049",\n'
        '    ),\n'
        '    (\n'
        '        "view-022",\n'
        '        "outdoor",\n'
        '        "scene_00022",\n'
        '        "scan_00194",\n'
        '        "00022_00194_outdoor_000_000",\n'
        '        1386881,\n'
        '        "01786324ca6c763a67e32adec512d83d32c295e3243d23fce9fa1b56f8031eb3",\n'
        '        3145856,\n'
        '        "84024306218657d9eea51c2abd54044e6ee6293d0193a7e185ed98b1f302b91e",\n'
        '        6291584,\n'
        '        "7064a6740d486b03a771a14d9801807dfd6dc2e48ac2c0f42417e755a63f067a",\n'
        '    ),\n'
        '    (\n'
        '        "view-023",\n'
        '        "outdoor",\n'
        '        "scene_00022",\n'
        '        "scan_00194",\n'
        '        "00022_00194_outdoor_000_020",\n'
        '        1550903,\n'
        '        "3e759d88c101e158a81eecf16b87c6e96b4ecd3d5d6e912a46af789a1cfd2f1a",\n'
        '        3145856,\n'
        '        "1aa6fd2ef08489b8d0a8547a2fc6093936f524df56967d797363baad7c6f0fff",\n'
        '        3145856,\n'
        '        "933093e8d49a222a2329f89d4adb82a8e4eb9e8a72079d885e8eebf97f31f801",\n'
        '    ),\n'
        '    (\n'
        '        "view-024",\n'
        '        "outdoor",\n'
        '        "scene_00022",\n'
        '        "scan_00195",\n'
        '        "00022_00195_outdoor_000_000",\n'
        '        1354956,\n'
        '        "485ae591b4e9197422d23834e658d3b7c0b2a9a8a1f4d190dc4e8285abcf4b58",\n'
        '        3145856,\n'
        '        "a0eb3b67dbdb6c7a541bc787f8b2c83c927b510477b83ea8eb47e0c0b64520fc",\n'
        '        3145856,\n'
        '        "1425f824f4bd3bfd4bc39f1ef6cff65c24578a79a6a5e56b1813fd68e6405daa",\n'
        '    ),\n'
        '    (\n'
        '        "view-025",\n'
        '        "outdoor",\n'
        '        "scene_00022",\n'
        '        "scan_00195",\n'
        '        "00022_00195_outdoor_000_020",\n'
        '        1312039,\n'
        '        "b505cf1a9e05be9caecc3141b3f6628e2d52aef7bd6768e0495d1fe68f022cc7",\n'
        '        3145856,\n'
        '        "00ca098eb81e758870344c0ea0671c15a75964ebc9afa82e5c842e9edf465b1b",\n'
        '        3145856,\n'
        '        "9c3b3f29d5c27d8ab1a899528fd98d48a608422f5d6878440c56e9452209082d",\n'
        '    ),\n'
        '    (\n'
        '        "view-026",\n'
        '        "outdoor",\n'
        '        "scene_00022",\n'
        '        "scan_00196",\n'
        '        "00022_00196_outdoor_000_010",\n'
        '        1548233,\n'
        '        "aa7cc5973a301d69420c74169d70d850d206adab453a1677c81ad3df8e456c01",\n'
        '        3145856,\n'
        '        "d415a6bb86593c6bb35a6457bbc13cd451206907e6225ca7d37f0171a5a19dc9",\n'
        '        3145856,\n'
        '        "63c640cb2c1d438832cacb039a307141e1a5e426d2f3fd2961e2b4e566215e47",\n'
        '    ),\n'
        '    (\n'
        '        "view-027",\n'
        '        "outdoor",\n'
        '        "scene_00022",\n'
        '        "scan_00196",\n'
        '        "00022_00196_outdoor_000_030",\n'
        '        1581362,\n'
        '        "64683db027716e6d5564c8962e2ef819be8e992a7514c5f0f3f42fa056230504",\n'
        '        3145856,\n'
        '        "bee31f7ca3020b9415e3debff8c219ee3502b104b1844d438a51d84a199979e9",\n'
        '        3145856,\n'
        '        "1aec43f5993a614a4f2b6d7577739615fc7780f13a49fbf899ee24753132e5a0",\n'
        '    ),\n'
        '    (\n'
        '        "view-028",\n'
        '        "outdoor",\n'
        '        "scene_00022",\n'
        '        "scan_00197",\n'
        '        "00022_00197_outdoor_000_000",\n'
        '        1397626,\n'
        '        "7b491f25a7a26e435ef5feb4d90ce5929954a2473e34f76bd70cc2d9a81251c3",\n'
        '        3145856,\n'
        '        "5045c2038970c4c282ffccb141c3afef9027a71523d4796189c59a544abc8847",\n'
        '        3145856,\n'
        '        "a80a8aa9ecadbef495966bc08935e8e02035c382c8ff8d75075646ee14e00191",\n'
        '    ),\n'
        '    (\n'
        '        "view-029",\n'
        '        "outdoor",\n'
        '        "scene_00022",\n'
        '        "scan_00197",\n'
        '        "00022_00197_outdoor_010_010",\n'
        '        1427912,\n'
        '        "ab669c4bf03ca844c125cb3e6d963364f5f14b36d0af9288130d85a174340205",\n'
        '        3145856,\n'
        '        "cd22ceb5a19427d36e1199e167237b0626b468947aa6ec21ef2889b3f687a42e",\n'
        '        3145856,\n'
        '        "69912b6858ed773c453b90993ef0c886b69c3bb3ee43644e54682b1e4f11f4a5",\n'
        '    ),\n'
        '    (\n'
        '        "view-030",\n'
        '        "outdoor",\n'
        '        "scene_00023",\n'
        '        "scan_00198",\n'
        '        "00023_00198_outdoor_000_020",\n'
        '        1196280,\n'
        '        "8e058762aeb6d508cc4a453bb51bb4195596644fdce856199e8ff47289154163",\n'
        '        3145856,\n'
        '        "07f0d1b82b79e37df0afa504b15061a1d801ffb7143da44d98c7e4573c126b30",\n'
        '        3145856,\n'
        '        "fdcca8151892640e098085128f1e34786f79c9ebd8f01c7f14f67aeff5ee529a",\n'
        '    ),\n'
        '    (\n'
        '        "view-031",\n'
        '        "outdoor",\n'
        '        "scene_00023",\n'
        '        "scan_00198",\n'
        '        "00023_00198_outdoor_070_020",\n'
        '        1153475,\n'
        '        "2c6ba4a04b24415855c6d56f9e424e752682c9598ed0a6961a2c095c0fde80e7",\n'
        '        3145856,\n'
        '        "0e1cef73cd8aa9611eee6a36395bda320bdbc5608d9431bcce24e59a1e0908d1",\n'
        '        6291584,\n'
        '        "4571c9f2b44300fbc1a4152b669720d367860f01827cc5512fd939f24965b7a6",\n'
        '    ),\n'
        '    (\n'
        '        "view-032",\n'
        '        "outdoor",\n'
        '        "scene_00023",\n'
        '        "scan_00199",\n'
        '        "00023_00199_outdoor_000_020",\n'
        '        1443825,\n'
        '        "065fdaf432f60cab7d4d85f77f6e8a8da2d8d7f6b618173e61b71c4db79b1b14",\n'
        '        3145856,\n'
        '        "734166fea6643e708cda8075833cffb604a288f4b8790e2c5d758deacb7867d9",\n'
        '        3145856,\n'
        '        "ee02105445049bd21b66b642dae6da7992d410b4a5f7c17397dfef192e12b246",\n'
        '    ),\n'
        '    (\n'
        '        "view-033",\n'
        '        "outdoor",\n'
        '        "scene_00023",\n'
        '        "scan_00199",\n'
        '        "00023_00199_outdoor_010_000",\n'
        '        1401671,\n'
        '        "27ac9a575d259ce2511a02b9c5265e39dcbebe4afd39e9f8b64a296ef6ea1095",\n'
        '        3145856,\n'
        '        "303990f57fcd277a43df832f1f03ef42a78008f99d0af024fe3e4ed53c977db4",\n'
        '        3145856,\n'
        '        "94c855bbefce29e551ceeb0b9971fcd6fa20adee845ab212147d694f6362d813",\n'
        '    ),\n'
        '    (\n'
        '        "view-034",\n'
        '        "outdoor",\n'
        '        "scene_00023",\n'
        '        "scan_00200",\n'
        '        "00023_00200_outdoor_000_010",\n'
        '        1103637,\n'
        '        "3d1c0f779183fe7eb383e05237a5462dc18fd790f35a80e90787e94d0e43c1c7",\n'
        '        3145856,\n'
        '        "b515576843d3ad6bd718d8e578c7e3bd128f82bcdadc51d2855ce77f65c63279",\n'
        '        3145856,\n'
        '        "23c79f29ae3ea4c3cc4f4e4cbfc4316b5e86a8fb661d1b2cfb2ddaf88f3e12de",\n'
        '    ),\n'
        '    (\n'
        '        "view-035",\n'
        '        "outdoor",\n'
        '        "scene_00023",\n'
        '        "scan_00200",\n'
        '        "00023_00200_outdoor_000_050",\n'
        '        1225980,\n'
        '        "6046d35e5454ceac3691c5086d262431260b9839194be58fd9015415f21b5dba",\n'
        '        3145856,\n'
        '        "672aa46a95e17433425bd0510f502eebb637fe494db3afe41c36b818ab3d7334",\n'
        '        3145856,\n'
        '        "1434e085120fca95356d956046c0b55120d0b4a32a5ec14394d67104f38eb1eb",\n'
        '    ),\n'
        '    (\n'
        '        "view-036",\n'
        '        "outdoor",\n'
        '        "scene_00024",\n'
        '        "scan_00201",\n'
        '        "00024_00201_outdoor_000_000",\n'
        '        1916342,\n'
        '        "2fd43a35faee39cdcf8b08b1f23915c8aabae57601e1c4eb90ac98c2726f6726",\n'
        '        3145856,\n'
        '        "7585ead8e9e351d703220c1fb4aa1a070874e0afcdb676646e8db7e90c57d46a",\n'
        '        3145856,\n'
        '        "4e6f8a5647539f94a7980255bb7c0be7dfdb7fe30395c3fccfdd47fcbddcb734",\n'
        '    ),\n'
        '    (\n'
        '        "view-037",\n'
        '        "outdoor",\n'
        '        "scene_00024",\n'
        '        "scan_00201",\n'
        '        "00024_00201_outdoor_000_020",\n'
        '        1842876,\n'
        '        "08c4b8342eea57a08d4380108ea14f342b43af3ef10fd7d1f9ee28dac1a2af6b",\n'
        '        3145856,\n'
        '        "c15fbde3764e85cb2c2b88a02562101447141b8ed63f0a70fa6da0c8a22a69a1",\n'
        '        3145856,\n'
        '        "5a92fbe940ba37d595197c9caa012526ec6b2f32ebceb5ffadb3b98ecafb80c3",\n'
        '    ),\n'
        '    (\n'
        '        "view-038",\n'
        '        "outdoor",\n'
        '        "scene_00024",\n'
        '        "scan_00202",\n'
        '        "00024_00202_outdoor_000_030",\n'
        '        1911126,\n'
        '        "29640b6d05b7f6f310c31a9a26e160518f679439bf619983dbe0b5fd1afb39aa",\n'
        '        3145856,\n'
        '        "a85149326d869f091b5611e7af0dfe0b0dad770bd8eb92f21bf4be2c76a65887",\n'
        '        3145856,\n'
        '        "f138b3fc58846961bb7193825be86a427d7689cf1f04b23a9eb830988eb93d84",\n'
        '    ),\n'
        '    (\n'
        '        "view-039",\n'
        '        "outdoor",\n'
        '        "scene_00024",\n'
        '        "scan_00202",\n'
        '        "00024_00202_outdoor_070_010",\n'
        '        1877641,\n'
        '        "e6c78b21fdd07ae5feeb21beb3c5f391522f041c9ddb55ba53a47cfd5994e434",\n'
        '        3145856,\n'
        '        "942e5fb2d5e63738fbc09c978d6f8ac878f50313c99bbd73ffc9239d0f44f109",\n'
        '        3145856,\n'
        '        "37ddc9e81d1c9f6abee51157213f4a8d5579ad4994bf5982220a8dc023dcd05a",\n'
        '    ),\n'
        ')\n'
        '\n'
        'DEFAULT_CACHE_DIR = Path("weights") / "diode-sample"  # working-directory-relative, like the notebook\n'
        'SAMPLE_SEED = 42\n'
        'SAMPLE_SPLIT = {"train": 6, "validation": 2, "test": 2}  # scans per domain; 2 domains x 2 views -> 24 / 8 / 8\n'
        'MIN_RECORDS = 4\n'
        'MAX_RECORDS = 2_000\n'
        'MIN_VALID_FRACTION = 0.05  # a depth map must label at least this fraction of its pixels\n'
        'MAX_DEPTH_M = 1_000.0\n'
        '_ID_RE = re.compile(r"^[A-Za-z0-9_.:-]{1,64}$")\n'
        '\n'
        '\n'
        'def _sha256_bytes(data: bytes) -> str:\n'
        '    return hashlib.sha256(data).hexdigest()\n'
        '\n'
        '\n'
        'def file_url(domain: str, scene: str, scan: str, stem: str, suffix: str) -> str:\n'
        '    return f"{CORPUS_BASE_URL}{domain}/{scene}/{scan}/{stem}{suffix}"\n'
        '\n'
        '\n'
        'def _pins(record: tuple) -> dict[str, tuple[int, str]]:\n'
        '    return {\n'
        '        ".png": (record[5], record[6]),\n'
        '        "_depth.npy": (record[7], record[8]),\n'
        '        "_depth_mask.npy": (record[9], record[10]),\n'
        '    }\n'
        '\n'
        '\n'
        'def fetch_corpus(*, cache_dir: str | Path | None = None, fetcher: Any = None) -> dict[str, dict[str, bytes]]:\n'
        '    """Return every pinned file (bytes keyed by record id, then suffix) from the cache or the Hub mirror."""\n'
        '    cache = Path(cache_dir) if cache_dir is not None else DEFAULT_CACHE_DIR\n'
        '    cache.mkdir(parents=True, exist_ok=True)\n'
        '    out: dict[str, dict[str, bytes]] = {}\n'
        '    for record in SAMPLE_RECORDS:\n'
        '        rid, domain, scene, scan, stem = record[:5]\n'
        '        out[rid] = {}\n'
        '        for suffix, (size, digest) in _pins(record).items():\n'
        '            local = cache / f"{stem}{suffix}"\n'
        '            data = local.read_bytes() if local.is_file() else b""\n'
        '            if len(data) != size or _sha256_bytes(data) != digest:\n'
        '                url = file_url(domain, scene, scan, stem, suffix)\n'
        '                if fetcher is not None:\n'
        '                    data = fetcher(url)\n'
        '                else:\n'
        '                    request = urllib.request.Request(\n'
        '                        url, headers={"User-Agent": "dimer-depth-anything-tutorial/1.0"}\n'
        '                    )\n'
        '                    with urllib.request.urlopen(request, timeout=300) as response:  # noqa: S310 (pinned https URL)\n'
        '                        data = response.read()\n'
        '                if len(data) != size or _sha256_bytes(data) != digest:\n'
        '                    raise ValueError(\n'
        '                        f"{rid} ({stem}{suffix}): fetched {len(data)} bytes with sha256 "\n'
        '                        f"{_sha256_bytes(data)[:16]}…, pinned {size} / {digest[:16]}…"\n'
        '                    )\n'
        '                local.write_bytes(data)\n'
        '            out[rid][suffix] = data\n'
        '    return out\n'
        '\n'
        '\n'
        'def _load_npy(data: bytes) -> np.ndarray:\n'
        '    return np.load(io.BytesIO(data), allow_pickle=False)\n'
        '\n'
        '\n'
        'def read_corpus(files: Mapping[str, Mapping[str, bytes]]) -> list[dict[str, Any]]:\n'
        '    """Decode the verified files into `{id, image, depth, mask}` records with their provenance."""\n'
        '    out = []\n'
        '    for record in SAMPLE_RECORDS:\n'
        '        rid, domain, scene, scan, stem = record[:5]\n'
        '        if rid not in files:\n'
        '            raise ValueError(f"corpus is missing {rid}")\n'
        '        image = Image.open(io.BytesIO(files[rid][".png"]))\n'
        '        image.load()\n'
        '        depth = _load_npy(files[rid]["_depth.npy"])\n'
        '        depth = depth[..., 0] if depth.ndim == 3 else depth\n'
        '        mask = _load_npy(files[rid]["_depth_mask.npy"]) > 0\n'
        '        out.append(\n'
        '            {\n'
        '                "id": rid,\n'
        '                "image": image.convert("RGB"),\n'
        '                "depth": np.ascontiguousarray(depth, dtype=np.float32),\n'
        '                "mask": np.ascontiguousarray(mask, dtype=bool),\n'
        '                "domain": domain,\n'
        '                "scene": scene,\n'
        '                "scan": scan,\n'
        '                "source_stem": stem,\n'
        '                "source_url": file_url(domain, scene, scan, stem, ".png"),\n'
        '            }\n'
        '        )\n'
        '    return out\n'
        '\n'
        '\n'
        'def build_sample_dataset(\n'
        '    records: Sequence[Mapping[str, Any]],\n'
        '    *,\n'
        '    seed: int = SAMPLE_SEED,\n'
        '    sizes: Mapping[str, int] | None = None,\n'
        ') -> dict[str, list[dict[str, Any]]]:\n'
        '    """Seeded draw of whole scans per domain: `sizes` = scans per domain for train / validation / test."""\n'
        '    sizes = dict(sizes or SAMPLE_SPLIT)\n'
        '    rng = random.Random(seed)\n'
        '    by_scan: dict[tuple[str, str], list[dict[str, Any]]] = {}\n'
        '    for record in records:\n'
        '        by_scan.setdefault((str(record.get("domain", "")), str(record["scan"])), []).append(dict(record))\n'
        '    out: dict[str, list[dict[str, Any]]] = {name: [] for name in sizes}\n'
        '    for domain in sorted({d for d, _ in by_scan}):\n'
        '        scans = sorted(s for d, s in by_scan if d == domain)\n'
        '        rng.shuffle(scans)\n'
        '        needed = sum(sizes.values())\n'
        '        if len(scans) < needed:\n'
        '            raise ValueError(f"{domain}: only {len(scans)} scans available, need {needed}")\n'
        '        cursor = 0\n'
        '        for name, per_domain in sizes.items():\n'
        '            for scan in scans[cursor : cursor + per_domain]:\n'
        '                out[name].extend(by_scan[(domain, scan)])\n'
        '            cursor += per_domain\n'
        '    for name in out:\n'
        '        rng.shuffle(out[name])\n'
        '        out[name] = [{**r, "id": f"{name}-{i:03d}", "source_id": r["id"]} for i, r in enumerate(out[name])]\n'
        '    return out\n'
        '\n'
        '\n'
        'def fetch_sample_dataset(\n'
        '    *,\n'
        '    cache_dir: str | Path | None = None,\n'
        '    fetcher: Any = None,\n'
        '    seed: int = SAMPLE_SEED,\n'
        '    sizes: Mapping[str, int] | None = None,\n'
        ') -> dict[str, list[dict[str, Any]]]:\n'
        '    """The tutorial splits from the pinned corpus."""\n'
        '    return build_sample_dataset(\n'
        '        read_corpus(fetch_corpus(cache_dir=cache_dir, fetcher=fetcher)), seed=seed, sizes=sizes\n'
        '    )\n'
        '\n'
        '\n'
        'def _as_array(value: Any, label_name: str, key: str) -> np.ndarray:\n'
        '    if isinstance(value, str | Path):\n'
        '        path = Path(value)\n'
        '        if not path.is_file():\n'
        '            raise ValueError(f"{label_name}: {key} file not found: {path}")\n'
        '        value = np.load(path, allow_pickle=False)\n'
        '    if not isinstance(value, np.ndarray):\n'
        '        raise ValueError(f"{label_name}: {key} must be a numpy array or a path to a .npy file")\n'
        '    if value.ndim == 3 and value.shape[-1] == 1:\n'
        '        value = value[..., 0]\n'
        '    if value.ndim != 2:\n'
        '        raise ValueError(f"{label_name}: {key} must be a 2-D H x W array, got shape {value.shape}")\n'
        '    return value\n'
        '\n'
        '\n'
        'def _check_record(record: Any, index: int) -> dict[str, Any]:\n'
        '    label_name = f"records[{index}]"\n'
        '    if not isinstance(record, Mapping):\n'
        '        raise ValueError(f"{label_name} must be a mapping with id/image/depth[/mask]")\n'
        '    for key in ("id", "image", "depth"):\n'
        '        if key not in record:\n'
        '            raise ValueError(f"{label_name} is missing {key!r}")\n'
        '    rid, image = record["id"], record["image"]\n'
        '    if not isinstance(rid, str) or not _ID_RE.match(rid):\n'
        '        raise ValueError(f"{label_name}: id must match {_ID_RE.pattern}")\n'
        '    if isinstance(image, str | Path):\n'
        '        path = Path(image)\n'
        '        if not path.is_file():\n'
        '            raise ValueError(f"{label_name}: image file not found: {path}")\n'
        '        image = Image.open(path)\n'
        '        image.load()\n'
        '    if not isinstance(image, Image.Image):\n'
        '        raise ValueError(f"{label_name}: image must be a PIL.Image.Image or a file path")\n'
        '    width, height = image.size\n'
        '    short, long = min(width, height), max(width, height)\n'
        '    if short < MIN_IMAGE_SIDE or long > MAX_IMAGE_SIDE:\n'
        '        raise ValueError(\n'
        '            f"{label_name}: image side outside {MIN_IMAGE_SIDE}..MAX_IMAGE_SIDE={MAX_IMAGE_SIDE} px: "\n'
        '            f"{image.size}"\n'
        '        )\n'
        '    if long / short > MAX_ASPECT_RATIO:\n'
        '        raise ValueError(\n'
        '            f"{label_name}: aspect ratio {long / short:.2f} > MAX_ASPECT_RATIO {MAX_ASPECT_RATIO}"\n'
        '        )\n'
        '    depth = _as_array(record["depth"], label_name, "depth")\n'
        '    if depth.shape != (height, width):\n'
        '        raise ValueError(\n'
        '            f"{label_name}: depth shape {depth.shape} != image (height, width) {(height, width)}"\n'
        '        )\n'
        '    if not np.issubdtype(depth.dtype, np.number):\n'
        '        raise ValueError(f"{label_name}: depth must be numeric, got {depth.dtype}")\n'
        '    depth = np.ascontiguousarray(depth, dtype=np.float32)\n'
        '    if record.get("mask") is None:\n'
        '        mask = np.isfinite(depth) & (depth > 0)\n'
        '    else:\n'
        '        mask = _as_array(record["mask"], label_name, "mask")\n'
        '        if mask.shape != depth.shape:\n'
        '            raise ValueError(f"{label_name}: mask shape {mask.shape} != depth shape {depth.shape}")\n'
        '        mask = np.ascontiguousarray(mask, dtype=bool) & np.isfinite(depth) & (depth > 0)\n'
        '    fraction = float(mask.mean())\n'
        '    if fraction < MIN_VALID_FRACTION:\n'
        '        raise ValueError(\n'
        '            f"{label_name}: only {fraction:.1%} of pixels carry valid depth; "\n'
        '            f"at least {MIN_VALID_FRACTION:.0%} are required"\n'
        '        )\n'
        '    if float(depth[mask].max()) > MAX_DEPTH_M:\n'
        '        raise ValueError(f"{label_name}: depth exceeds MAX_DEPTH_M={MAX_DEPTH_M} m (metres are expected)")\n'
        '    item = {"id": rid, "image": image.convert("RGB"), "depth": depth, "mask": mask}\n'
        '    for key in ("source_id", "domain", "scene", "scan", "group", "source_stem", "source_url"):\n'
        '        if key in record:\n'
        '            item[key] = record[key]\n'
        '    return item\n'
        '\n'
        '\n'
        'def validate_dataset(\n'
        '    records: Sequence[Mapping[str, Any]], *, min_records: int = MIN_RECORDS, max_records: int = MAX_RECORDS\n'
        ') -> dict[str, Any]:\n'
        '    """Structural validation of a depth-labelled image dataset; raises ValueError before any model import."""\n'
        '    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, str | bytes):\n'
        '        raise ValueError("records must be a list of {id, image, depth[, mask]} mappings")\n'
        '    if not min_records <= len(records) <= max_records:\n'
        '        raise ValueError(f"{len(records)} records; {min_records}..{max_records} are required")\n'
        '    checked = [_check_record(record, index) for index, record in enumerate(records)]\n'
        '    ids = [r["id"] for r in checked]\n'
        '    if len(set(ids)) != len(ids):\n'
        '        duplicate = next(i for i in ids if ids.count(i) > 1)\n'
        '        raise ValueError(f"duplicate id {duplicate!r}")\n'
        '    sides = [max(r["image"].size) for r in checked]\n'
        '    valid = [float(r["mask"].mean()) for r in checked]\n'
        '    depths = [float(np.median(r["depth"][r["mask"]])) for r in checked]\n'
        '    domains: dict[str, int] = {}\n'
        '    for r in checked:\n'
        '        domains[str(r.get("domain", "unspecified"))] = domains.get(str(r.get("domain", "unspecified")), 0) + 1\n'
        '    return {\n'
        '        "records": checked,\n'
        '        "n_records": len(checked),\n'
        '        "domain_counts": domains,\n'
        '        "scans": len({str(r.get("scan", r.get("group", r["id"]))) for r in checked}),\n'
        '        "image_side": {"min": min(sides), "max": max(sides)},\n'
        '        "valid_fraction": {"min": round(min(valid), 4), "max": round(max(valid), 4)},\n'
        '        "median_depth_m": {"min": round(min(depths), 3), "max": round(max(depths), 3)},\n'
        '        "digest": dataset_digest(checked),\n'
        '        "model_id": MODEL_ID,\n'
        '    }\n'
        '\n'
        '\n'
        'def image_digest(image: Image.Image) -> str:\n'
        '    """SHA-256 of the decoded RGB pixels (size + bytes), so a re-encoded copy of the same photo matches."""\n'
        '    rgb = image.convert("RGB")\n'
        '    return _sha256_bytes(f"{rgb.size[0]}x{rgb.size[1]}:".encode() + rgb.tobytes())\n'
        '\n'
        '\n'
        'def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:\n'
        '    payload = [\n'
        '        [\n'
        '            r["id"],\n'
        '            image_digest(r["image"]),\n'
        '            _sha256_bytes(np.ascontiguousarray(r["depth"], dtype=np.float32).tobytes()),\n'
        '        ]\n'
        '        for r in records\n'
        '    ]\n'
        '    return _sha256_bytes(json.dumps(payload, ensure_ascii=False, separators=(",", ":")).encode("utf-8"))\n'
        '\n'
        '\n'
        'def _split_unit(record: Mapping[str, Any]) -> str:\n'
        '    return str(record.get("scan") or record.get("group") or record.get("source_id") or record["id"])\n'
        '\n'
        '\n'
        'def check_split_disjoint(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:\n'
        '    """Assert no image (by decoded-pixel digest) and no scan appears in two splits (leakage check)."""\n'
        '    seen: dict[str, str] = {}\n'
        '    scans: dict[str, str] = {}\n'
        '    for name, records in splits.items():\n'
        '        for record in records:\n'
        '            key = image_digest(record["image"])\n'
        '            if key in seen and seen[key] != name:\n'
        '                raise ValueError(f"image {record[\'id\']!r} appears in both {seen[key]} and {name}")\n'
        '            seen[key] = name\n'
        '            unit = _split_unit(record)\n'
        '            if unit in scans and scans[unit] != name:\n'
        '                raise ValueError(f"scan {unit!r} has views in both {scans[unit]} and {name}")\n'
        '            scans[unit] = name\n'
        '    return {name: len(records) for name, records in splits.items()}\n'
        '\n'
        '\n'
        'def scan_summary(splits: Mapping[str, Sequence[Mapping[str, Any]]]) -> dict[str, Any]:\n'
        '    """Scans and domains per split (an observation of what the split unit was)."""\n'
        '    out: dict[str, Any] = {}\n'
        '    for name, records in splits.items():\n'
        '        out[name] = {\n'
        '            "views": len(records),\n'
        '            "scans": len({_split_unit(r) for r in records}),\n'
        '            "domains": {\n'
        '                d: sum(1 for r in records if str(r.get("domain", "unspecified")) == d)\n'
        '                for d in sorted({str(r.get("domain", "unspecified")) for r in records})\n'
        '            },\n'
        '        }\n'
        '    return out\n'
        '\n'
        '\n'
        'def split_dataset(\n'
        '    records: Sequence[Mapping[str, Any]],\n'
        '    *,\n'
        '    val_fraction: float = 0.15,\n'
        '    test_fraction: float = 0.2,\n'
        '    seed: int = 0,\n'
        ') -> dict[str, list[dict[str, Any]]]:\n'
        '    """Seeded shuffle of a BYOD dataset into train/validation/test by split unit (`scan` / `group`, else the\n'
        '    image itself) after de-duplicating images, so views of one scan never straddle splits."""\n'
        '    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):\n'
        '        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")\n'
        '    checked = validate_dataset(records)["records"]\n'
        '    seen: set[str] = set()\n'
        '    by_unit: dict[str, list[dict[str, Any]]] = {}\n'
        '    for record in checked:\n'
        '        key = image_digest(record["image"])\n'
        '        if key not in seen:\n'
        '            seen.add(key)\n'
        '            by_unit.setdefault(_split_unit(record), []).append(record)\n'
        '    rng = random.Random(seed)\n'
        '    units = sorted(by_unit)\n'
        '    rng.shuffle(units)\n'
        '    n_test = max(1, round(len(units) * test_fraction))\n'
        '    n_val = round(len(units) * val_fraction)\n'
        '    splits: dict[str, list[dict[str, Any]]] = {"test": [], "validation": [], "train": []}\n'
        '    for name, chosen in (\n'
        '        ("test", units[:n_test]),\n'
        '        ("validation", units[n_test : n_test + n_val]),\n'
        '        ("train", units[n_test + n_val :]),\n'
        '    ):\n'
        '        for unit in chosen:\n'
        '            splits[name].extend(by_unit[unit])\n'
        '    for part in splits.values():\n'
        '        rng.shuffle(part)\n'
        '    if len(splits["train"]) < MIN_RECORDS:\n'
        '        raise ValueError(\n'
        '            f"split leaves {len(splits[\'train\'])} training records; at least {MIN_RECORDS} are required"\n'
        '        )\n'
        '    return splits\n'
        '\n'
        '\n'
        'def load_byod_dataset(path: str | Path, *, require_group: bool = True) -> list[dict[str, Any]]:\n'
        '    """Read `{id, image, depth[, mask]}` records from a directory or a zip holding `records.csv` (columns\n'
        '    `id`, `image`, `depth`, `group`, optional `mask`) beside the files; depth and mask are `.npy` arrays\n'
        '    (metres; boolean) decoded from the archive, never extracted to disk. `group` (the scan, capture session\n'
        '    or device the view belongs to) must be non-empty on every row unless `require_group=False`, in which\n'
        '    case the split falls back to one unit per view and the group-disjoint guarantee is gone."""\n'
        '    source = Path(path)\n'
        '    if source.is_dir():\n'
        '        table = (source / "records.csv").read_text(encoding="utf-8")\n'
        '        base_dir = source.resolve()\n'
        '\n'
        '        def loader(name: str) -> bytes:\n'
        '            target = (source / name).resolve()\n'
        '            if base_dir not in target.parents:\n'
        '                raise ValueError(f"BYOD file reference {name!r} leaves the dataset directory")\n'
        '            return target.read_bytes()\n'
        '\n'
        '    elif source.is_file() and source.suffix.lower() == ".zip":\n'
        '        archive = zipfile.ZipFile(source)\n'
        '        names = [n for n in archive.namelist() if not n.endswith("/")]\n'
        '        basenames = [Path(n).name for n in names]\n'
        '        if len(set(basenames)) != len(basenames):\n'
        '            duplicate = next(b for b in basenames if basenames.count(b) > 1)\n'
        '            raise ValueError(f"BYOD zip holds more than one member named {duplicate!r}")\n'
        '        members = dict(zip(basenames, names, strict=True))\n'
        '        if "records.csv" not in members:\n'
        '            raise ValueError("BYOD zip must contain records.csv")\n'
        '        table = archive.read(members["records.csv"]).decode("utf-8")\n'
        '        loader = lambda name: archive.read(members[name])  # noqa: E731\n'
        '    else:\n'
        '        raise ValueError("BYOD datasets must be a directory or a .zip holding records.csv and the files")\n'
        '    rows = list(csv.DictReader(io.StringIO(table)))\n'
        '    missing = {"id", "image", "depth"} - set(rows[0].keys() if rows else set())\n'
        '    if missing:\n'
        '        raise ValueError(f"records.csv is missing columns {sorted(missing)}")\n'
        '    out = []\n'
        '    seen: set[str] = set()\n'
        '    for row in rows:\n'
        '        group = (row.get("group") or "").strip()\n'
        '        if require_group and not group:\n'
        '            raise ValueError(\n'
        '                f"records.csv row for id {row[\'id\']!r} has no `group`; every row needs the scan, capture "\n'
        '                "session or device the view belongs to so the split stays group-disjoint (pass "\n'
        '                "require_group=False to split by view instead, without that guarantee)"\n'
        '            )\n'
        '        if row["id"] in seen:\n'
        '            raise ValueError(f"records.csv lists id {row[\'id\']!r} more than once")\n'
        '        seen.add(row["id"])\n'
        '        image = Image.open(io.BytesIO(loader(row["image"])))\n'
        '        image.load()\n'
        '        item: dict[str, Any] = {\n'
        '            "id": row["id"],\n'
        '            "image": image.convert("RGB"),\n'
        '            "depth": _load_npy(loader(row["depth"])),\n'
        '        }\n'
        '        if row.get("mask"):\n'
        '            item["mask"] = _load_npy(loader(row["mask"]))\n'
        '        if group:\n'
        '            item["group"] = group\n'
        '        out.append(item)\n'
        '    return out\n'
        '\n'
        '\n'
        'def write_dataset_csv(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:\n'
        '    """Write the records table of a split (id, image, depth, mask, group, provenance) as BYOD expects it."""\n'
        '    out = Path(path)\n'
        '    out.parent.mkdir(parents=True, exist_ok=True)\n'
        '    with open(out, "w", encoding="utf-8", newline="") as handle:\n'
        '        writer = csv.DictWriter(\n'
        '            handle, fieldnames=["id", "image", "depth", "mask", "group", "domain", "source_url"]\n'
        '        )\n'
        '        writer.writeheader()\n'
        '        for record in records:\n'
        '            stem = record.get("source_stem") or record["id"]\n'
        '            writer.writerow(\n'
        '                {\n'
        '                    "id": record["id"],\n'
        '                    "image": f"{stem}.png",\n'
        '                    "depth": f"{stem}_depth.npy",\n'
        '                    "mask": f"{stem}_depth_mask.npy",\n'
        '                    "group": _split_unit(record),\n'
        '                    "domain": record.get("domain", ""),\n'
        '                    "source_url": record.get("source_url", ""),\n'
        '                }\n'
        '            )\n'
        '    return out\n'
    ),
    'zoe_reference.py': (
        '"""Monocular metric depth estimation with the pinned ``Intel/zoedepth-nyu-kitti`` checkpoint (ZoeDepth).\n'
        '\n'
        'The class loads the processor and model only from a digest-verified local snapshot (``weights/<key>/``)\n'
        'or, when explicitly allowed, from the Hugging Face Hub at the pinned revision — always with\n'
        '``trust_remote_code=False``: the ZoeDepth architecture (a BEiT-large DPT backbone with metric bin heads\n'
        'for the NYU and KITTI ranges) comes from the pinned ``transformers`` release, the weights are\n'
        'SafeTensors, and no model-repository code is executed. The output is depth in metres — a metric\n'
        "estimate with no confidence, not a measurement — at the caller's resolution.\n"
        '"""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import hashlib\n'
        'import json\n'
        'import math\n'
        'from collections.abc import Callable, Mapping, Sequence\n'
        'from dataclasses import dataclass, field\n'
        'from datetime import UTC, datetime\n'
        'from pathlib import Path\n'
        'from typing import Any\n'
        '\n'
        'import numpy as np\n'
        'from PIL import Image\n'
        '\n'
        'MODEL_ID = "Intel/zoedepth-nyu-kitti"\n'
        'MODEL_REVISION = "f364d4c7936e91f465abba182208dd68142bf0ca"\n'
        'MODEL_LICENSE = "mit"\n'
        'MODEL_KEY = "zoedepth-nyu-kitti"\n'
        'DEFAULT_WEIGHTS_DIR = Path(__file__).resolve().parents[2] / "weights" / MODEL_KEY\n'
        'MANIFEST_NAME = "dimer-base-manifest.json"\n'
        'ARTIFACT_FORMAT = "org.valcorza.zoedepth.metric-head-adapter"\n'
        'ARTIFACT_FORMAT_VERSION = "1.0"\n'
        'ARTIFACT_MANIFEST_NAME = "manifest.json"\n'
        'ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"\n'
        'MAX_ARTIFACT_MANIFEST_BYTES = 64 * 1024\n'
        'MAX_ARTIFACT_HEADER_BYTES = 1024 * 1024\n'
        'TRAINABLE_PREFIXES = ("metric_head.",)\n'
        'MAX_ADAPTATION_RECORDS = 128\n'
        'MAX_ADAPTATION_PIXELS = 32 * 1024 * 1024\n'
        '\n'
        '# Input ceilings. The ZoeDepth processor resizes the image to fit 384x512 while keeping the aspect\n'
        '# ratio (sides rounded to multiples of 32) and pads, so the backbone cost grows with the aspect ratio,\n'
        "# not the pixel count; the prediction is interpolated back to the caller's resolution.\n"
        'MAX_IMAGE_SIDE = 4096\n'
        'MIN_IMAGE_SIDE = 32\n'
        'MAX_ASPECT_RATIO = 4.0\n'
        'DEPTH_KIND = "metric"\n'
        'DEPTH_UNIT = "metres"\n'
        "# The checkpoint's two metric heads (config.json bin_configurations): NYU indoor 0.001-10 m and KITTI\n"
        '# outdoor 0.001-80 m; the model routes each image to one head by its own domain classifier.\n'
        'DEPTH_RANGES_M = {"nyu": (0.001, 10.0), "kitti": (0.001, 80.0)}\n'
        '# Optional horizontal-flip test-time augmentation (the upstream evaluation setting): two forward\n'
        '# passes, predictions averaged. Off by default; a caller-owned request parameter.\n'
        'FLIP_AUGMENTATION = False\n'
        '# delta1 accuracy threshold (the depth-estimation convention): max(pred/ref, ref/pred) < 1.25.\n'
        'DELTA_THRESHOLD = 1.25\n'
        '\n'
        '\n'
        'def _sha256(path: Path) -> str:\n'
        '    digest = hashlib.sha256()\n'
        '    with open(path, "rb") as fh:\n'
        '        for chunk in iter(lambda: fh.read(1 << 20), b""):\n'
        '            digest.update(chunk)\n'
        '    return digest.hexdigest()\n'
        '\n'
        '\n'
        'def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:\n'
        '    """Check a local snapshot against its DIMER manifest; raise naming the first mismatch."""\n'
        '    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID:\n'
        '        raise ValueError(f"manifest modelId {manifest.get(\'modelId\')!r} != {MODEL_ID!r}")\n'
        '    if manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(f"manifest revision {manifest.get(\'revision\')!r} != {MODEL_REVISION!r}")\n'
        '    for entry in manifest["files"]:\n'
        '        file_path = root / entry["path"]\n'
        '        if not file_path.is_file():\n'
        '            raise FileNotFoundError(f"snapshot file missing: {file_path}")\n'
        '        size = file_path.stat().st_size\n'
        '        if size != entry["bytes"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: size {size} != manifest {entry[\'bytes\']}")\n'
        '        digest = _sha256(file_path)\n'
        '        if digest != entry["sha256"]:\n'
        '            raise ValueError(f"{entry[\'path\']}: sha256 {digest} != manifest {entry[\'sha256\']}")\n'
        '    return {\n'
        '        "path": str(root),\n'
        '        "model_id": manifest["modelId"],\n'
        '        "revision": manifest["revision"],\n'
        '        "files": len(manifest["files"]),\n'
        '        "total_bytes": manifest.get("totalBytes"),\n'
        '    }\n'
        '\n'
        '\n'
        'def _hub_download(relative_path: str, root: Path) -> None:\n'
        '    """Fetch one manifest-listed file at MODEL_REVISION straight into the snapshot directory."""\n'
        '    from huggingface_hub import hf_hub_download\n'
        '\n'
        '    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))\n'
        '\n'
        '\n'
        'def stage_missing_files(\n'
        '    path: str | Path | None = None,\n'
        '    *,\n'
        '    allow_download: bool = False,\n'
        '    downloader: Callable[[str, Path], None] | None = None,\n'
        ') -> list[str]:\n'
        '    """Fetch manifest-listed files that are absent locally (a fresh clone commits the manifest but\n'
        '    git-ignores the weights). Returns the relative paths fetched; `verify_snapshot` still runs after."""\n'
        '    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR\n'
        '    manifest_path = root / MANIFEST_NAME\n'
        '    if not manifest_path.is_file():\n'
        '        raise FileNotFoundError(f"manifest not found: {manifest_path}")\n'
        '    with open(manifest_path, encoding="utf-8") as fh:\n'
        '        manifest = json.load(fh)\n'
        '    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:\n'
        '        raise ValueError(\n'
        '            f"manifest names {manifest.get(\'modelId\')}@{manifest.get(\'revision\')}, "\n'
        '            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"\n'
        '        )\n'
        '    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]\n'
        '    if not missing:\n'
        '        return []\n'
        '    if not allow_download:\n'
        '        raise FileNotFoundError(\n'
        '            f"snapshot at {root} is missing {missing}; "\n'
        '            f"pass allow_download=True to fetch them at {MODEL_REVISION}"\n'
        '        )\n'
        '    fetch = downloader or _hub_download\n'
        '    for relative_path in missing:\n'
        '        fetch(relative_path, root)\n'
        '    return missing\n'
        '\n'
        '\n'
        'def _valid_mask(pred: np.ndarray, ref_depth: np.ndarray) -> np.ndarray:\n'
        '    if pred.shape != ref_depth.shape:\n'
        '        raise ValueError(f"shape mismatch: pred {pred.shape} vs ref {ref_depth.shape}")\n'
        '    valid = np.isfinite(ref_depth) & (ref_depth > 0) & np.isfinite(pred) & (pred > 0)\n'
        '    if valid.sum() < 2:\n'
        '        raise ValueError("need at least 2 valid reference pixels (ref_depth > 0 and pred > 0)")\n'
        '    return valid\n'
        '\n'
        '\n'
        'def abs_rel(pred: np.ndarray, ref_depth: np.ndarray) -> float:\n'
        '    """Absolute relative error ``mean(|pred - ref| / ref)`` of metric depth against metric reference depth.\n'
        '\n'
        '    Both arrays are in metres at the same H x W; no scale or shift alignment is applied, because the\n'
        '    model claims metric output — a scale error therefore shows up in the number, as it should.\n'
        '    """\n'
        '    pred = np.asarray(pred, dtype=np.float64)\n'
        '    ref_depth = np.asarray(ref_depth, dtype=np.float64)\n'
        '    valid = _valid_mask(pred, ref_depth)\n'
        '    return float(np.mean(np.abs(pred[valid] - ref_depth[valid]) / ref_depth[valid]))\n'
        '\n'
        '\n'
        'def delta1(pred: np.ndarray, ref_depth: np.ndarray, *, threshold: float = DELTA_THRESHOLD) -> float:\n'
        '    """Fraction of valid pixels whose ratio ``max(pred/ref, ref/pred)`` is below ``threshold`` (1.25)."""\n'
        '    pred = np.asarray(pred, dtype=np.float64)\n'
        '    ref_depth = np.asarray(ref_depth, dtype=np.float64)\n'
        '    valid = _valid_mask(pred, ref_depth)\n'
        '    ratio = np.maximum(pred[valid] / ref_depth[valid], ref_depth[valid] / pred[valid])\n'
        '    return float(np.mean(ratio < threshold))\n'
        '\n'
        '\n'
        'def validate_image(image: Any) -> Image.Image:\n'
        '    """Type- and size-check a caller image and return it as RGB."""\n'
        '    if not isinstance(image, Image.Image):\n'
        '        raise TypeError(f"image must be a PIL.Image.Image, got {type(image).__name__}")\n'
        '    width, height = image.size\n'
        '    short, long = min(width, height), max(width, height)\n'
        '    if short < MIN_IMAGE_SIDE:\n'
        '        raise ValueError(f"image side {short} px < MIN_IMAGE_SIDE {MIN_IMAGE_SIDE}")\n'
        '    if long > MAX_IMAGE_SIDE:\n'
        '        raise ValueError(f"image side {long} px > MAX_IMAGE_SIDE {MAX_IMAGE_SIDE}")\n'
        '    if long / short > MAX_ASPECT_RATIO:\n'
        '        raise ValueError(f"aspect ratio {long / short:.2f} > MAX_ASPECT_RATIO {MAX_ASPECT_RATIO}")\n'
        '    return image.convert("RGB")\n'
        '\n'
        '\n'
        'def _check_flip(value: Any) -> bool:\n'
        '    if not isinstance(value, bool):\n'
        '        raise TypeError("flip_augmentation must be a bool")\n'
        '    return value\n'
        '\n'
        '\n'
        'INPUT_SCHEMA: dict[str, Any] = {\n'
        '    "input": "PIL.Image.Image, or a sequence of them for the validation stage; any mode, converted to RGB",\n'
        '    "short_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],\n'
        '    "long_side_px": [MIN_IMAGE_SIDE, MAX_IMAGE_SIDE],\n'
        '    "aspect_ratio": [1.0, MAX_ASPECT_RATIO],\n'
        '    "flip_augmentation": "bool; two forward passes (image and its mirror) averaged when true",\n'
        '    "output": (\n'
        '        f"{DEPTH_KIND} depth in {DEPTH_UNIT}, float32 H x W at the input resolution (larger = farther); "\n'
        '        "an estimate with no confidence, routed to the NYU (0.001-10 m) or KITTI (0.001-80 m) head by "\n'
        '        "the model\'s own domain classifier"\n'
        '    ),\n'
        '    "preprocessing": (\n'
        '        "convert to RGB; the ZoeDepth processor resizes to fit 384x512 keeping the aspect ratio "\n'
        '        "(sides rounded to multiples of 32), pads, normalises with mean/std 0.5; the prediction is "\n'
        '        "interpolated back to the input resolution by post_process_depth_estimation"\n'
        '    ),\n'
        '}\n'
        '\n'
        '\n'
        'def validate_inputs(\n'
        '    images: Any, *, flip_augmentation: bool = FLIP_AUGMENTATION, names: Sequence[str] | None = None\n'
        ') -> dict[str, Any]:\n'
        '    """Validation stage: return the input manifest (schema, per-input observations, request, verdict).\n'
        '\n'
        '    Each image is routed through the public ``validate_image`` that ``predict`` itself calls, so a\n'
        '    rejection here raises exactly what ``predict`` would; a caller that wants the finding recorded\n'
        '    catches the exception and stores ``str(exc)`` under ``findings``.\n'
        '    """\n'
        '    batch = [images] if isinstance(images, Image.Image) else images\n'
        '    if not isinstance(batch, Sequence) or isinstance(batch, str | bytes):\n'
        '        raise TypeError("images must be a PIL.Image.Image or a sequence of them")\n'
        '    if len(batch) < 1:\n'
        '        raise ValueError("at least one image is required")\n'
        '    if names is not None and len(names) != len(batch):\n'
        '        raise ValueError("names must have one entry per image")\n'
        '    flip = _check_flip(flip_augmentation)\n'
        '    inputs = []\n'
        '    for index, candidate in enumerate(batch):\n'
        '        rgb = validate_image(candidate)\n'
        '        width, height = rgb.size\n'
        '        long_side, short_side = max(width, height), min(width, height)\n'
        '        inputs.append(\n'
        '            {\n'
        '                "id": names[index] if names else f"image-{index}",\n'
        '                "mode": getattr(candidate, "mode", rgb.mode),\n'
        '                "size": [width, height],\n'
        '                "aspect_ratio": round(long_side / short_side, 3),\n'
        '            }\n'
        '        )\n'
        '    return {\n'
        '        "schema": dict(INPUT_SCHEMA),\n'
        '        "inputs": inputs,\n'
        '        "n_images": len(inputs),\n'
        '        "flip_augmentation": flip,\n'
        '        "verdict": "accepted",\n'
        '        "findings": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '\n'
        '\n'
        'def evaluation_report(\n'
        '    result: Mapping[str, Any],\n'
        '    reference_depth: Any | None = None,\n'
        '    *,\n'
        '    sample_kind: str = "synthetic",\n'
        ') -> dict[str, Any]:\n'
        '    """Evaluation stage: a machine-readable report even when nothing is measurable.\n'
        '\n'
        '    With ``reference_depth`` (metric depth in metres, same H x W as the prediction, non-positive or\n'
        '    non-finite pixels ignored) the report carries ``abs_rel`` and ``delta1`` computed without any\n'
        '    alignment — the model claims metres, so scale errors count — as sample-sanity evidence. Without it\n'
        '    the verdict is ``not-measurable`` and the report says what ground truth would make the task\n'
        '    measurable: a depth map in metres has no intrinsic score.\n'
        '    """\n'
        '    depth = np.asarray(result["depth"])\n'
        '    base = {\n'
        '        "task": "monocular metric depth estimation",\n'
        '        "score_semantics": (\n'
        '            f"{result.get(\'depth_kind\', DEPTH_KIND)} depth in {DEPTH_UNIT} with no confidence; the value "\n'
        '            "is an estimate whose scale depends on the model having recognised the scene\'s domain and "\n'
        '            "camera, and a blank image still yields a depth map"\n'
        '        ),\n'
        '        "sample_kind": sample_kind,\n'
        '        "flip_augmentation": bool(result.get("flip_augmentation", FLIP_AUGMENTATION)),\n'
        '        "n_images": 1,\n'
        '        "n_pixels": int(depth.size),\n'
        '        "depth_min_m": float(depth.min()),\n'
        '        "depth_max_m": float(depth.max()),\n'
        '        "depth_median_m": float(np.median(depth)),\n'
        '        "baselines": [],\n'
        '        "model_id": MODEL_ID,\n'
        '        "model_revision": MODEL_REVISION,\n'
        '    }\n'
        '    if reference_depth is None:\n'
        '        return {\n'
        '            **base,\n'
        '            "metrics": [],\n'
        '            "verdict": "not-measurable",\n'
        '            "reason": "no metric reference depth was supplied for the evaluated image",\n'
        '            "needs": (\n'
        '                "a metric depth map in metres with the same height and width as the image, from a depth "\n'
        '                "sensor, LiDAR, stereo or an RGB-D benchmark, scored with abs_rel and delta1 against a "\n'
        '                "constant-depth prior (the reference\'s median) as the trivial baseline"\n'
        '            ),\n'
        '        }\n'
        '    ref = np.asarray(reference_depth, dtype=np.float64)\n'
        '    valid = np.isfinite(ref) & (ref > 0)\n'
        '    n_valid = int(valid.sum())\n'
        '    constant = np.full_like(ref, float(np.median(ref[valid])) if n_valid else 1.0)\n'
        '    return {\n'
        '        **base,\n'
        '        "metrics": [\n'
        '            {\n'
        '                "id": "abs_rel",\n'
        '                "value": abs_rel(depth, ref),\n'
        '                "align": False,\n'
        '                "n_valid_pixels": n_valid,\n'
        '                "estimation": "single image, no alignment (metric output as-is), no dispersion estimate",\n'
        '            },\n'
        '            {\n'
        '                "id": "delta1",\n'
        '                "value": delta1(depth, ref),\n'
        '                "threshold": DELTA_THRESHOLD,\n'
        '                "n_valid_pixels": n_valid,\n'
        '                "estimation": "single image, fraction of valid pixels within the ratio threshold",\n'
        '            },\n'
        '        ],\n'
        '        "baselines": [\n'
        '            {\n'
        '                "id": "constant_median_depth",\n'
        '                "abs_rel": abs_rel(constant, ref),\n'
        '                "delta1": delta1(constant, ref),\n'
        '                "note": "every pixel predicted at the reference\'s median depth",\n'
        '            }\n'
        '        ],\n'
        '        "verdict": "sample-sanity",\n'
        '        "reason": "one image with caller-supplied metric depth; not a benchmark",\n'
        '        "needs": "a held-out set of metric depth maps from the deployment domain for any generalisable claim",\n'
        '    }\n'
        '\n'
        '\n'
        'def validate_depth_dataset(records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n'
        '    """Validate paired RGB images and positive metric-depth targets for adaptation."""\n'
        '    if isinstance(records, str | bytes) or not isinstance(records, Sequence):\n'
        '        raise TypeError("records must be a sequence of mappings")\n'
        '    if not 2 <= len(records) <= MAX_ADAPTATION_RECORDS:\n'
        '        raise ValueError(f"record count {len(records)} outside 2..{MAX_ADAPTATION_RECORDS}")\n'
        '    ids: list[str] = []\n'
        '    digest = hashlib.sha256()\n'
        '    valid_pixels = 0\n'
        '    depth_values: list[np.ndarray] = []\n'
        '    for index, record in enumerate(records):\n'
        '        if not isinstance(record, Mapping):\n'
        '            raise TypeError(f"record {index} must be a mapping")\n'
        '        record_id = record.get("id")\n'
        '        if not isinstance(record_id, str) or not record_id.strip():\n'
        '            raise ValueError(f"record {index} id must be a non-empty string")\n'
        '        if record_id in ids:\n'
        '            raise ValueError(f"duplicate record id: {record_id}")\n'
        '        ids.append(record_id)\n'
        '        image = validate_image(record.get("image"))\n'
        '        depth = np.asarray(record.get("depth_m"), dtype=np.float32)\n'
        '        if depth.shape != (image.height, image.width):\n'
        '            raise ValueError(\n'
        '                f"record {record_id} depth shape {depth.shape} != image {(image.height, image.width)}"\n'
        '            )\n'
        '        if not np.all(np.isfinite(depth)) or np.any(depth <= 0):\n'
        '            raise ValueError(f"record {record_id} depth_m must contain finite positive metres")\n'
        '        if float(depth.max()) > 80.0:\n'
        '            raise ValueError(f"record {record_id} depth_m exceeds the 80 m model ceiling")\n'
        '        valid_pixels += int(depth.size)\n'
        '        if valid_pixels > MAX_ADAPTATION_PIXELS:\n'
        '            raise ValueError(\n'
        '                f"dataset has {valid_pixels} pixels, exceeding MAX_ADAPTATION_PIXELS "\n'
        '                f"{MAX_ADAPTATION_PIXELS}"\n'
        '            )\n'
        '        depth_values.append(depth.reshape(-1))\n'
        '        digest.update(record_id.encode("utf-8"))\n'
        '        digest.update(\n'
        '            json.dumps(\n'
        '                {"image_shape": [image.height, image.width, 3], "depth_shape": list(depth.shape)},\n'
        '                sort_keys=True,\n'
        '                separators=(",", ":"),\n'
        '            ).encode("utf-8")\n'
        '        )\n'
        '        digest.update(np.asarray(image, dtype=np.uint8).tobytes())\n'
        '        digest.update(depth.tobytes())\n'
        '    all_depth = np.concatenate(depth_values)\n'
        '    return {\n'
        '        "records": len(records),\n'
        '        "unique_ids": len(ids),\n'
        '        "valid_depth_pixels": valid_pixels,\n'
        '        "depth_min_m": float(all_depth.min()),\n'
        '        "depth_max_m": float(all_depth.max()),\n'
        '        "depth_median_m": float(np.median(all_depth)),\n'
        '        "dataset_sha256": digest.hexdigest(),\n'
        '        "verdict": "accepted",\n'
        '    }\n'
        '\n'
        '\n'
        'def _depth_record_content_sha256(record: Mapping[str, Any]) -> str:\n'
        '    """Fingerprint one validated RGB/depth pair without trusting its caller-supplied ID."""\n'
        '    image = validate_image(record["image"])\n'
        '    depth = np.asarray(record["depth_m"], dtype=np.float32)\n'
        '    digest = hashlib.sha256()\n'
        '    digest.update(\n'
        '        json.dumps(\n'
        '            {"image_shape": [image.height, image.width, 3], "depth_shape": list(depth.shape)},\n'
        '            sort_keys=True,\n'
        '            separators=(",", ":"),\n'
        '        ).encode("utf-8")\n'
        '    )\n'
        '    digest.update(np.asarray(image, dtype=np.uint8).tobytes())\n'
        '    digest.update(depth.tobytes())\n'
        '    return digest.hexdigest()\n'
        '\n'
        '\n'
        'def _depth_record_rgb_sha256(record: Mapping[str, Any]) -> str:\n'
        '    """Fingerprint a validated RGB input independently of its ID and depth target."""\n'
        '    image = validate_image(record["image"])\n'
        '    digest = hashlib.sha256()\n'
        '    digest.update(\n'
        '        json.dumps(\n'
        '            {"image_shape": [image.height, image.width, 3]},\n'
        '            sort_keys=True,\n'
        '            separators=(",", ":"),\n'
        '        ).encode("utf-8")\n'
        '    )\n'
        '    digest.update(np.asarray(image, dtype=np.uint8).tobytes())\n'
        '    return digest.hexdigest()\n'
        '\n'
        '\n'
        'def _prepare_depth_target(\n'
        '    depth_m: Any,\n'
        '    processor: Any,\n'
        '    *,\n'
        '    output_size: tuple[int, int],\n'
        '    device: Any,\n'
        ') -> Any:\n'
        '    """Apply ZoeDepth\'s spatial pad/resize geometry to a metric-depth target."""\n'
        '    import torch\n'
        '    import torch.nn.functional as F\n'
        '\n'
        '    depth = np.asarray(depth_m, dtype=np.float32)\n'
        '    if getattr(processor, "do_pad", False):\n'
        '        pad_image = getattr(processor, "pad_image", None)\n'
        '        if not callable(pad_image):\n'
        '            raise RuntimeError("ZoeDepth processor enables padding but exposes no pad_image method")\n'
        '        depth = np.asarray(\n'
        '            pad_image(\n'
        '                depth[..., None],\n'
        '                input_data_format="channels_last",\n'
        '                data_format="channels_last",\n'
        '            ),\n'
        '            dtype=np.float32,\n'
        '        )[..., 0]\n'
        '    target = torch.from_numpy(np.ascontiguousarray(depth)).to(device)\n'
        "    # ZoeDepth's image resize uses align_corners=True. Bilinear depth resampling preserves the same\n"
        '    # coordinate grid while avoiding image-specific bicubic overshoot in metric targets.\n'
        '    return F.interpolate(\n'
        '        target[None, None], size=output_size, mode="bilinear", align_corners=True\n'
        '    )[:, 0]\n'
        '\n'
        '\n'
        '@dataclass\n'
        'class ZoeDepthMetricPipeline:\n'
        '    """Monocular metric depth estimation over the pinned ZoeDepth NYU+KITTI checkpoint."""\n'
        '\n'
        '    _runner: Callable[[Image.Image, bool], np.ndarray]\n'
        '    device: str\n'
        '    model: Any | None = None\n'
        '    processor: Any | None = None\n'
        '    adaptation_config: dict[str, Any] = field(default_factory=dict)\n'
        '\n'
        '    @classmethod\n'
        '    def from_pretrained(\n'
        '        cls,\n'
        '        device: str | None = None,\n'
        '        weights_dir: str | Path | None = None,\n'
        '        allow_download: bool = False,\n'
        '    ) -> ZoeDepthMetricPipeline:\n'
        '        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR\n'
        '        if (root / MANIFEST_NAME).is_file():\n'
        '            stage_missing_files(root, allow_download=allow_download)\n'
        '            verify_snapshot(root)\n'
        '            source, kwargs = str(root), {"local_files_only": True}\n'
        '        elif allow_download:\n'
        '            source, kwargs = MODEL_ID, {}\n'
        '        else:\n'
        '            raise FileNotFoundError(\n'
        '                f"no verified snapshot at {root} and allow_download=False; "\n'
        '                f"stage {MODEL_ID}@{MODEL_REVISION} under weights/{MODEL_KEY}"\n'
        '            )\n'
        '        # Refuse invalid snapshots before importing model libraries.\n'
        '        import torch\n'
        '        from transformers import ZoeDepthForDepthEstimation, ZoeDepthImageProcessor\n'
        '\n'
        '        resolved_device = device or ("cuda:0" if torch.cuda.is_available() else "cpu")\n'
        '        processor = ZoeDepthImageProcessor.from_pretrained(\n'
        '            source, revision=MODEL_REVISION, trust_remote_code=False, **kwargs\n'
        '        )\n'
        '        model = ZoeDepthForDepthEstimation.from_pretrained(\n'
        '            source, revision=MODEL_REVISION, trust_remote_code=False, dtype=torch.float32, **kwargs\n'
        '        )\n'
        '        model = model.to(resolved_device).eval()\n'
        '\n'
        '        def runner(image: Image.Image, flip: bool) -> np.ndarray:\n'
        '            inputs = processor(images=image, return_tensors="pt").to(resolved_device)\n'
        '            with torch.inference_mode():\n'
        '                outputs = model(**inputs)\n'
        '                extra = {}\n'
        '                if flip:\n'
        '                    mirrored = torch.flip(inputs["pixel_values"], dims=[3])\n'
        '                    extra["outputs_flipped"] = model(pixel_values=mirrored)\n'
        "            # The pinned processor's post-processing un-pads, un-flips (when given) and interpolates\n"
        '            # the prediction back to the source size; it returns metres.\n'
        '            post = processor.post_process_depth_estimation(\n'
        '                outputs, source_sizes=[(image.height, image.width)], **extra\n'
        '            )\n'
        '            return post[0]["predicted_depth"].float().cpu().numpy()\n'
        '\n'
        '        return cls(runner, resolved_device, model=model, processor=processor)\n'
        '\n'
        '    def freeze_for_adaptation(self) -> dict[str, int]:\n'
        '        """Freeze ZoeDepth except its metric head."""\n'
        '        if self.model is None:\n'
        '            raise RuntimeError("cannot configure adaptation without an underlying torch model")\n'
        '        trainable = frozen = 0\n'
        '        for name, parameter in self.model.named_parameters():\n'
        '            parameter.requires_grad = name.startswith(TRAINABLE_PREFIXES)\n'
        '            if parameter.requires_grad:\n'
        '                trainable += parameter.numel()\n'
        '            else:\n'
        '                frozen += parameter.numel()\n'
        '        if trainable == 0:\n'
        '            raise RuntimeError("ZoeDepth adaptation selected no trainable parameters")\n'
        '        self.adaptation_config = {\n'
        '            "method": "frozen-backbone-metric-head-gradient-adaptation",\n'
        '            "trainable_prefixes": list(TRAINABLE_PREFIXES),\n'
        '            "trainable_parameters": trainable,\n'
        '            "frozen_parameters": frozen,\n'
        '        }\n'
        '        return {"trainable_parameters": trainable, "frozen_parameters": frozen}\n'
        '\n'
        '    def finetune(\n'
        '        self,\n'
        '        train_records: Sequence[Mapping[str, Any]],\n'
        '        val_records: Sequence[Mapping[str, Any]] | None = None,\n'
        '        *,\n'
        '        epochs: int = 2,\n'
        '        learning_rate: float = 1e-5,\n'
        '        seed: int = 42,\n'
        '    ) -> list[dict[str, Any]]:\n'
        '        """Run bounded metric-head adaptation using log-depth L1 loss."""\n'
        '        import random\n'
        '\n'
        '        import torch\n'
        '        from torch.optim import AdamW\n'
        '\n'
        '        if self.model is None or self.processor is None:\n'
        '            raise RuntimeError("cannot fine-tune without the underlying model and processor")\n'
        '        if isinstance(epochs, bool) or not isinstance(epochs, int) or not 1 <= epochs <= 20:\n'
        '            raise ValueError("epochs must be an int in 1..20")\n'
        '        if not isinstance(learning_rate, int | float) or isinstance(learning_rate, bool):\n'
        '            raise TypeError("learning_rate must be numeric")\n'
        '        if not 0 < float(learning_rate) <= 1e-2:\n'
        '            raise ValueError("learning_rate must be in (0, 1e-2]")\n'
        '        train_manifest = validate_depth_dataset(train_records)\n'
        '        val_manifest = validate_depth_dataset(val_records) if val_records else None\n'
        '        total_records = train_manifest["records"] + (\n'
        '            val_manifest["records"] if val_manifest is not None else 0\n'
        '        )\n'
        '        if total_records > MAX_ADAPTATION_RECORDS:\n'
        '            raise ValueError(\n'
        '                f"combined train and validation record count {total_records} exceeds "\n'
        '                f"MAX_ADAPTATION_RECORDS {MAX_ADAPTATION_RECORDS}"\n'
        '            )\n'
        '        total_pixels = train_manifest["valid_depth_pixels"] + (\n'
        '            val_manifest["valid_depth_pixels"] if val_manifest is not None else 0\n'
        '        )\n'
        '        if total_pixels > MAX_ADAPTATION_PIXELS:\n'
        '            raise ValueError(\n'
        '                f"combined train and validation data has {total_pixels} pixels, exceeding "\n'
        '                f"MAX_ADAPTATION_PIXELS {MAX_ADAPTATION_PIXELS}"\n'
        '            )\n'
        '        if val_records:\n'
        '            train_ids = {str(record["id"]) for record in train_records}\n'
        '            val_ids = {str(record["id"]) for record in val_records}\n'
        '            overlapping_ids = sorted(train_ids & val_ids)\n'
        '            if overlapping_ids:\n'
        '                raise ValueError(\n'
        '                    f"train and validation records overlap by id: {overlapping_ids[:5]}"\n'
        '                )\n'
        '            train_rgb = {_depth_record_rgb_sha256(record) for record in train_records}\n'
        '            val_rgb = {_depth_record_rgb_sha256(record) for record in val_records}\n'
        '            if train_rgb & val_rgb:\n'
        '                raise ValueError("train and validation records overlap by RGB content")\n'
        '            train_content = {_depth_record_content_sha256(record) for record in train_records}\n'
        '            val_content = {_depth_record_content_sha256(record) for record in val_records}\n'
        '            if train_content & val_content:\n'
        '                raise ValueError("train and validation records overlap by RGB/depth content")\n'
        '        if not self.adaptation_config:\n'
        '            self.freeze_for_adaptation()\n'
        '        if any(\n'
        '            parameter.requires_grad and not name.startswith(TRAINABLE_PREFIXES)\n'
        '            for name, parameter in self.model.named_parameters()\n'
        '        ):\n'
        '            raise RuntimeError("parameters outside the declared ZoeDepth adapter surface are trainable")\n'
        '\n'
        '        torch.manual_seed(seed)\n'
        '        device = torch.device(self.device)\n'
        '        self.model.to(device).eval()\n'
        '        self.model.metric_head.train()\n'
        '        trainable = {\n'
        '            name: parameter\n'
        '            for name, parameter in self.model.named_parameters()\n'
        '            if parameter.requires_grad\n'
        '        }\n'
        '        if not trainable:\n'
        '            raise RuntimeError("model has no trainable parameters")\n'
        '        before = {name: parameter.detach().cpu().clone() for name, parameter in trainable.items()}\n'
        '        optimizer_betas = (0.9, 0.999)\n'
        '        optimizer_epsilon = 1e-8\n'
        '        optimizer_weight_decay = 0.01\n'
        '        optimizer = AdamW(\n'
        '            list(trainable.values()),\n'
        '            lr=float(learning_rate),\n'
        '            betas=optimizer_betas,\n'
        '            eps=optimizer_epsilon,\n'
        '            weight_decay=optimizer_weight_decay,\n'
        '        )\n'
        '\n'
        '        def loss_for(record: Mapping[str, Any]) -> torch.Tensor:\n'
        '            inputs = self.processor(\n'
        '                images=validate_image(record["image"]), return_tensors="pt"\n'
        '            ).to(device)\n'
        '            prediction = self.model(pixel_values=inputs["pixel_values"]).predicted_depth\n'
        '            target = _prepare_depth_target(\n'
        '                record["depth_m"],\n'
        '                self.processor,\n'
        '                output_size=tuple(prediction.shape[-2:]),\n'
        '                device=device,\n'
        '            )\n'
        '            return torch.mean(torch.abs(torch.log(prediction.clamp_min(1e-3)) - torch.log(target)))\n'
        '\n'
        '        def validation_loss() -> float | None:\n'
        '            if not val_records:\n'
        '                return None\n'
        '            self.model.eval()\n'
        '            with torch.inference_mode():\n'
        '                value = float(np.mean([float(loss_for(record).item()) for record in val_records]))\n'
        '            self.model.metric_head.train()\n'
        '            return value\n'
        '\n'
        '        baseline_val_loss = validation_loss()\n'
        '        history: list[dict[str, Any]] = []\n'
        '        previous_config = dict(self.adaptation_config)\n'
        '        for key in (\n'
        '            "epochs",\n'
        '            "learning_rate",\n'
        '            "batch_size",\n'
        '            "seed",\n'
        '            "optimizer",\n'
        '            "optimizer_betas",\n'
        '            "optimizer_epsilon",\n'
        '            "optimizer_weight_decay",\n'
        '            "loss",\n'
        '            "train_manifest",\n'
        '            "validation_manifest",\n'
        '            "training_median_depth_m",\n'
        '            "baseline_validation_log_l1",\n'
        '            "history",\n'
        '            "weight_delta_l2",\n'
        '        ):\n'
        '            self.adaptation_config.pop(key, None)\n'
        '        try:\n'
        '            for epoch in range(1, epochs + 1):\n'
        '                order = list(range(len(train_records)))\n'
        '                random.Random(seed + epoch * 19).shuffle(order)\n'
        '                total_loss = 0.0\n'
        '                for index in order:\n'
        '                    optimizer.zero_grad(set_to_none=True)\n'
        '                    loss = loss_for(train_records[index])\n'
        '                    if not bool(torch.isfinite(loss)):\n'
        '                        raise RuntimeError("fine-tuning produced a non-finite loss")\n'
        '                    loss.backward()\n'
        '                    optimizer.step()\n'
        '                    if any(\n'
        '                        not bool(torch.isfinite(parameter).all())\n'
        '                        for parameter in trainable.values()\n'
        '                    ):\n'
        '                        raise RuntimeError("fine-tuning produced non-finite adapter weights")\n'
        '                    total_loss += float(loss.item())\n'
        '                epoch_data: dict[str, Any] = {\n'
        '                    "epoch": epoch,\n'
        '                    "train_log_l1": round(total_loss / len(train_records), 6),\n'
        '                    "optimizer_steps": len(train_records),\n'
        '                }\n'
        '                current_val = validation_loss()\n'
        '                if current_val is not None:\n'
        '                    if not math.isfinite(current_val):\n'
        '                        raise RuntimeError("fine-tuning produced a non-finite validation loss")\n'
        '                    epoch_data["val_log_l1"] = round(current_val, 6)\n'
        '                history.append(epoch_data)\n'
        '\n'
        '            delta_sq = 0.0\n'
        '            for name, parameter in trainable.items():\n'
        '                delta_sq += float(\n'
        '                    torch.sum((parameter.detach().cpu() - before[name]) ** 2).item()\n'
        '                )\n'
        '            weight_delta_l2 = delta_sq**0.5\n'
        '            if not math.isfinite(weight_delta_l2):\n'
        '                raise RuntimeError("fine-tuning produced a non-finite weight delta")\n'
        '            if weight_delta_l2 == 0.0:\n'
        '                raise RuntimeError("fine-tuning completed without changing adapter weights")\n'
        '            self.model.eval()\n'
        '            self.adaptation_config.update(\n'
        '                {\n'
        '                    "epochs": epochs,\n'
        '                    "learning_rate": float(learning_rate),\n'
        '                    "batch_size": 1,\n'
        '                    "seed": seed,\n'
        '                    "optimizer": "AdamW",\n'
        '                    "optimizer_betas": list(optimizer_betas),\n'
        '                    "optimizer_epsilon": optimizer_epsilon,\n'
        '                    "optimizer_weight_decay": optimizer_weight_decay,\n'
        '                    "loss": "mean-absolute-log-depth-error",\n'
        '                    "train_manifest": train_manifest,\n'
        '                    "validation_manifest": val_manifest,\n'
        '                    "training_median_depth_m": train_manifest["depth_median_m"],\n'
        '                    "baseline_validation_log_l1": baseline_val_loss,\n'
        '                    "history": history,\n'
        '                    "weight_delta_l2": weight_delta_l2,\n'
        '                }\n'
        '            )\n'
        '        except Exception:\n'
        '            with torch.no_grad():\n'
        '                for name, parameter in trainable.items():\n'
        '                    parameter.copy_(before[name].to(device=parameter.device, dtype=parameter.dtype))\n'
        '            self.adaptation_config = previous_config\n'
        '            self.model.eval()\n'
        '            raise\n'
        '        return history\n'
        '\n'
        '    def evaluate_adaptation(self, records: Sequence[Mapping[str, Any]]) -> dict[str, Any]:\n'
        '        """Score metric depth on labelled records against a constant-median baseline."""\n'
        '        validate_depth_dataset(records)\n'
        '        train_median = self.adaptation_config.get("training_median_depth_m")\n'
        '        if not isinstance(train_median, int | float) or train_median <= 0:\n'
        '            raise RuntimeError("adaptation metadata does not contain a positive training median")\n'
        '        model_abs_rel_sum = 0.0\n'
        '        model_delta1_hits = 0\n'
        '        baseline_abs_rel_sum = 0.0\n'
        '        baseline_delta1_hits = 0\n'
        '        valid_pixels = 0\n'
        '        for record in records:\n'
        '            prediction = np.asarray(self.predict(record["image"])["depth"], dtype=np.float64)\n'
        '            reference = np.asarray(record["depth_m"], dtype=np.float64)\n'
        '            baseline = np.full_like(reference, float(train_median))\n'
        '            valid = _valid_mask(prediction, reference)\n'
        '            count = int(valid.sum())\n'
        '            valid_pixels += count\n'
        '            model_abs_rel_sum += float(\n'
        '                np.sum(np.abs(prediction[valid] - reference[valid]) / reference[valid])\n'
        '            )\n'
        '            model_ratio = np.maximum(\n'
        '                prediction[valid] / reference[valid], reference[valid] / prediction[valid]\n'
        '            )\n'
        '            model_delta1_hits += int(np.sum(model_ratio < DELTA_THRESHOLD))\n'
        '            baseline_abs_rel_sum += float(\n'
        '                np.sum(np.abs(baseline[valid] - reference[valid]) / reference[valid])\n'
        '            )\n'
        '            baseline_ratio = np.maximum(\n'
        '                baseline[valid] / reference[valid], reference[valid] / baseline[valid]\n'
        '            )\n'
        '            baseline_delta1_hits += int(np.sum(baseline_ratio < DELTA_THRESHOLD))\n'
        '        model_abs_rel = model_abs_rel_sum / valid_pixels\n'
        '        model_delta1 = model_delta1_hits / valid_pixels\n'
        '        baseline_abs_rel = baseline_abs_rel_sum / valid_pixels\n'
        '        baseline_delta1 = baseline_delta1_hits / valid_pixels\n'
        '        return {\n'
        '            "records": len(records),\n'
        '            "valid_pixels": valid_pixels,\n'
        '            "abs_rel": model_abs_rel,\n'
        '            "delta1": model_delta1,\n'
        '            "constant_median_baseline_abs_rel": baseline_abs_rel,\n'
        '            "constant_median_baseline_delta1": baseline_delta1,\n'
        '            "abs_rel_delta_vs_baseline": model_abs_rel - baseline_abs_rel,\n'
        '            "delta1_delta_vs_baseline": model_delta1 - baseline_delta1,\n'
        '        }\n'
        '\n'
        '    def save_artifact(self, output_dir: str | Path, *, producer_revision: str) -> Path:\n'
        '        """Write safe metric-head weights plus a closed integrity manifest."""\n'
        '        from safetensors.torch import save_file\n'
        '\n'
        '        weight_delta = self.adaptation_config.get("weight_delta_l2")\n'
        '        training_median = self.adaptation_config.get("training_median_depth_m")\n'
        '        history = self.adaptation_config.get("history")\n'
        '        if (\n'
        '            self.model is None\n'
        '            or not isinstance(weight_delta, int | float)\n'
        '            or isinstance(weight_delta, bool)\n'
        '            or not math.isfinite(float(weight_delta))\n'
        '            or weight_delta <= 0\n'
        '            or not isinstance(training_median, int | float)\n'
        '            or isinstance(training_median, bool)\n'
        '            or not math.isfinite(float(training_median))\n'
        '            or training_median <= 0\n'
        '            or not isinstance(history, list)\n'
        '            or not history\n'
        '        ):\n'
        '            raise RuntimeError("artifact export requires completed finite fine-tuning metadata")\n'
        '        if len(producer_revision) != 40 or any(ch not in "0123456789abcdef" for ch in producer_revision):\n'
        '            raise ValueError("producer_revision must be a lowercase 40-hex Git commit")\n'
        '        root = Path(output_dir)\n'
        '        root.mkdir(parents=True, exist_ok=True)\n'
        '        if any(root.iterdir()):\n'
        '            raise FileExistsError(f"artifact directory is not empty: {root}")\n'
        '        state = {\n'
        '            name: tensor.detach().cpu().contiguous()\n'
        '            for name, tensor in self.model.state_dict().items()\n'
        '            if name.startswith(TRAINABLE_PREFIXES)\n'
        '        }\n'
        '        if not state:\n'
        '            raise RuntimeError("no adapter tensors selected for export")\n'
        '        weights_path = root / ARTIFACT_WEIGHTS_NAME\n'
        '        save_file(state, str(weights_path))\n'
        '        manifest = {\n'
        '            "artifactSpec": "1.0",\n'
        '            "format": ARTIFACT_FORMAT,\n'
        '            "formatVersion": ARTIFACT_FORMAT_VERSION,\n'
        '            "artifactClass": "ADAPTER",\n'
        '            "artifactKind": "zoedepth-metric-head-adapter",\n'
        '            "producer": {\n'
        '                "pipelineId": "zoedepth-metric-depth-pipeline",\n'
        '                "revision": producer_revision,\n'
        '            },\n'
        '            "createdAtUtc": datetime.now(UTC).isoformat(),\n'
        '            "baseModel": {"id": MODEL_ID, "revision": MODEL_REVISION},\n'
        '            "files": [\n'
        '                {\n'
        '                    "path": ARTIFACT_WEIGHTS_NAME,\n'
        '                    "bytes": weights_path.stat().st_size,\n'
        '                    "sha256": _sha256(weights_path),\n'
        '                }\n'
        '            ],\n'
        '            "adaptation": dict(self.adaptation_config),\n'
        '            "trainablePrefixes": list(TRAINABLE_PREFIXES),\n'
        '            "retainedData": {"containsTrainingRecords": False, "containsSupportRecords": False},\n'
        '            "serialization": "safetensors",\n'
        '        }\n'
        '        (root / ARTIFACT_MANIFEST_NAME).write_text(\n'
        '            json.dumps(manifest, indent=2, ensure_ascii=False), encoding="utf-8"\n'
        '        )\n'
        '        return root\n'
        '\n'
        '    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:\n'
        '        """Verify and load a ZoeDepth adapter without code-capable deserialization."""\n'
        '        from safetensors.torch import load_file\n'
        '\n'
        '        if self.model is None:\n'
        '            raise RuntimeError("cannot load an artifact without an underlying torch model")\n'
        '        root = Path(artifact_dir)\n'
        '        manifest_path = root / ARTIFACT_MANIFEST_NAME\n'
        '        if not manifest_path.is_file():\n'
        '            raise FileNotFoundError(f"artifact manifest not found: {manifest_path}")\n'
        '        expected_files = {ARTIFACT_MANIFEST_NAME, ARTIFACT_WEIGHTS_NAME}\n'
        '        actual_entries = {path.name for path in root.iterdir()}\n'
        '        if actual_entries != expected_files:\n'
        '            raise ValueError(\n'
        '                f"artifact directory must contain exactly {sorted(expected_files)}, "\n'
        '                f"found {sorted(actual_entries)}"\n'
        '            )\n'
        '        if manifest_path.stat().st_size > MAX_ARTIFACT_MANIFEST_BYTES:\n'
        '            raise ValueError("artifact manifest exceeds the permitted size")\n'
        '        manifest = json.loads(manifest_path.read_text(encoding="utf-8"))\n'
        '        if manifest.get("format") != ARTIFACT_FORMAT:\n'
        '            raise ValueError(f"unrecognized artifact format: {manifest.get(\'format\')}")\n'
        '        if manifest.get("formatVersion") != ARTIFACT_FORMAT_VERSION:\n'
        '            raise ValueError(f"unsupported artifact formatVersion: {manifest.get(\'formatVersion\')}")\n'
        '        if manifest.get("baseModel") != {"id": MODEL_ID, "revision": MODEL_REVISION}:\n'
        '            raise ValueError("artifact base model identity is incompatible")\n'
        '        if manifest.get("trainablePrefixes") != list(TRAINABLE_PREFIXES):\n'
        '            raise ValueError("artifact trainable prefixes do not match this pipeline")\n'
        '        files = manifest.get("files")\n'
        '        if not isinstance(files, list) or len(files) != 1:\n'
        '            raise ValueError("artifact manifest must inventory exactly one weights file")\n'
        '        entry = files[0]\n'
        '        if entry.get("path") != ARTIFACT_WEIGHTS_NAME:\n'
        '            raise ValueError("artifact manifest names an unexpected weights path")\n'
        '        weights_path = root / ARTIFACT_WEIGHTS_NAME\n'
        '        if not weights_path.is_file():\n'
        '            raise FileNotFoundError(f"artifact weights not found: {weights_path}")\n'
        '        weights_bytes = weights_path.stat().st_size\n'
        '        if weights_bytes != entry.get("bytes"):\n'
        '            raise ValueError("artifact weights failed size or SHA-256 verification")\n'
        '        adaptation = manifest.get("adaptation")\n'
        '        if not isinstance(adaptation, dict):\n'
        '            raise ValueError("artifact adaptation metadata must be a mapping")\n'
        '        destination = self.model.state_dict()\n'
        '        expected = {name for name in destination if name.startswith(TRAINABLE_PREFIXES)}\n'
        '        expected_payload_bytes = sum(\n'
        '            destination[name].numel() * destination[name].element_size() for name in expected\n'
        '        )\n'
        '        with weights_path.open("rb") as stream:\n'
        '            header_prefix = stream.read(8)\n'
        '        if len(header_prefix) != 8:\n'
        '            raise ValueError("artifact weights do not contain a complete SafeTensors header")\n'
        '        header_bytes = int.from_bytes(header_prefix, byteorder="little", signed=False)\n'
        '        if not 0 < header_bytes <= MAX_ARTIFACT_HEADER_BYTES:\n'
        '            raise ValueError("artifact SafeTensors header exceeds the permitted size")\n'
        '        expected_serialized_bytes = 8 + header_bytes + expected_payload_bytes\n'
        '        if weights_bytes != expected_serialized_bytes:\n'
        '            raise ValueError(\n'
        '                "artifact serialized size does not match the declared adapter surface"\n'
        '            )\n'
        '        if _sha256(weights_path) != entry.get("sha256"):\n'
        '            raise ValueError("artifact weights failed size or SHA-256 verification")\n'
        '        # Materialize only after the closed inventory has imposed a tight serialized-size ceiling,\n'
        '        # and stage on CPU so an untrusted artifact cannot allocate directly into accelerator RAM.\n'
        '        state = load_file(str(weights_path), device="cpu")\n'
        '        if set(state) != expected:\n'
        '            raise ValueError("artifact tensor inventory does not match the declared adapter surface")\n'
        '        for name, tensor in state.items():\n'
        '            expected_tensor = destination[name]\n'
        '            if tensor.shape != expected_tensor.shape or tensor.dtype != expected_tensor.dtype:\n'
        '                raise ValueError(\n'
        '                    f"artifact tensor {name} has shape/dtype {tuple(tensor.shape)}/{tensor.dtype}; "\n'
        '                    f"expected {tuple(expected_tensor.shape)}/{expected_tensor.dtype}"\n'
        '                )\n'
        '            if not bool(tensor.isfinite().all()):\n'
        '                raise ValueError(f"artifact tensor {name} contains non-finite values")\n'
        '        self.model.load_state_dict(state, strict=False)\n'
        '        # A freshly constructed base model is fully trainable. Restore the declared adapter surface\n'
        '        # so a verified artifact can be fine-tuned again without exposing the backbone.\n'
        '        self.freeze_for_adaptation()\n'
        '        self.model.to(self.device).eval()\n'
        '        self.adaptation_config = dict(adaptation)\n'
        '        return manifest\n'
        '\n'
        '    @classmethod\n'
        '    def from_artifact(\n'
        '        cls,\n'
        '        artifact_dir: str | Path,\n'
        '        *,\n'
        '        device: str | None = None,\n'
        '        weights_dir: str | Path | None = None,\n'
        '        allow_download: bool = False,\n'
        '    ) -> ZoeDepthMetricPipeline:\n'
        '        """Construct a fresh pinned base model and attach a verified adapter."""\n'
        '        pipeline = cls.from_pretrained(\n'
        '            device=device, weights_dir=weights_dir, allow_download=allow_download\n'
        '        )\n'
        '        pipeline.load_artifact(artifact_dir)\n'
        '        return pipeline\n'
        '\n'
        '    def predict(self, image: Image.Image, *, flip_augmentation: bool = FLIP_AUGMENTATION) -> dict[str, Any]:\n'
        '        """Return metric depth in metres as a float32 H x W array at the input resolution."""\n'
        '        rgb = validate_image(image)\n'
        '        flip = _check_flip(flip_augmentation)\n'
        '        depth = np.asarray(self._runner(rgb, flip), dtype=np.float32)\n'
        '        if depth.shape != (rgb.height, rgb.width):\n'
        '            raise RuntimeError(f"backend returned shape {depth.shape}, expected {(rgb.height, rgb.width)}")\n'
        '        if not np.all(np.isfinite(depth)) or depth.min() <= 0:\n'
        '            raise RuntimeError("backend returned non-finite or non-positive depth values")\n'
        '        return {\n'
        '            "depth": depth,\n'
        '            "depth_kind": DEPTH_KIND,\n'
        '            "depth_unit": DEPTH_UNIT,\n'
        '            "depth_min": float(depth.min()),\n'
        '            "depth_max": float(depth.max()),\n'
        '            "depth_median": float(np.median(depth)),\n'
        '            "flip_augmentation": flip,\n'
        '            "height": rgb.height,\n'
        '            "width": rgb.width,\n'
        '            "model_id": MODEL_ID,\n'
        '            "model_revision": MODEL_REVISION,\n'
        '        }\n'
    ),
    'weights/zoe/dimer-base-manifest.json': (
        '{\n'
        '  "format": "dimer_hf_snapshot",\n'
        '  "formatVersion": 1,\n'
        '  "modelKey": "zoedepth-nyu-kitti",\n'
        '  "modelId": "Intel/zoedepth-nyu-kitti",\n'
        '  "revision": "f364d4c7936e91f465abba182208dd68142bf0ca",\n'
        '  "files": [\n'
        '    {\n'
        '      "path": "README.md",\n'
        '      "bytes": 2508,\n'
        '      "sha256": "3a24f725d6ba1ae7144a50b08eca542965bd99d4c003d2898e77db56cdb6ca9a"\n'
        '    },\n'
        '    {\n'
        '      "path": "config.json",\n'
        '      "bytes": 2225,\n'
        '      "sha256": "58494c160c520023c4d5bdeebb3b2d035e37e48cc81225580f49fcb06e175913"\n'
        '    },\n'
        '    {\n'
        '      "path": "model.safetensors",\n'
        '      "bytes": 1380374404,\n'
        '      "sha256": "c5494fa0938f18d71e215e245472470c3aefebd7b434abd89750e5ae4008e2dc"\n'
        '    },\n'
        '    {\n'
        '      "path": "preprocessor_config.json",\n'
        '      "bytes": 723,\n'
        '      "sha256": "0b64d8edc980d7b7abb819085650c43da8b8183acbd4336ab5a9e0e9caf4648e"\n'
        '    }\n'
        '  ],\n'
        '  "totalBytes": 1380379860\n'
        '}\n'
    ),
    'licenses/zoe.txt': (
        '                                 Apache License\n'
        '                           Version 2.0, January 2004\n'
        '                        http://www.apache.org/licenses/\n'
        '\n'
        '   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n'
        '\n'
        '   1. Definitions.\n'
        '\n'
        '      "License" shall mean the terms and conditions for use, reproduction,\n'
        '      and distribution as defined by Sections 1 through 9 of this document.\n'
        '\n'
        '      "Licensor" shall mean the copyright owner or entity authorized by\n'
        '      the copyright owner that is granting the License.\n'
        '\n'
        '      "Legal Entity" shall mean the union of the acting entity and all\n'
        '      other entities that control, are controlled by, or are under common\n'
        '      control with that entity. For the purposes of this definition,\n'
        '      "control" means (i) the power, direct or indirect, to cause the\n'
        '      direction or management of such entity, whether by contract or\n'
        '      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n'
        '      outstanding shares, or (iii) beneficial ownership of such entity.\n'
        '\n'
        '      "You" (or "Your") shall mean an individual or Legal Entity\n'
        '      exercising permissions granted by this License.\n'
        '\n'
        '      "Source" form shall mean the preferred form for making modifications,\n'
        '      including but not limited to software source code, documentation\n'
        '      source, and configuration files.\n'
        '\n'
        '      "Object" form shall mean any form resulting from mechanical\n'
        '      transformation or translation of a Source form, including but\n'
        '      not limited to compiled object code, generated documentation,\n'
        '      and conversions to other media types.\n'
        '\n'
        '      "Work" shall mean the work of authorship, whether in Source or\n'
        '      Object form, made available under the License, as indicated by a\n'
        '      copyright notice that is included in or attached to the work\n'
        '      (an example is provided in the Appendix below).\n'
        '\n'
        '      "Derivative Works" shall mean any work, whether in Source or Object\n'
        '      form, that is based on (or derived from) the Work and for which the\n'
        '      editorial revisions, annotations, elaborations, or other modifications\n'
        '      represent, as a whole, an original work of authorship. For the purposes\n'
        '      of this License, Derivative Works shall not include works that remain\n'
        '      separable from, or merely link (or bind by name) to the interfaces of,\n'
        '      the Work and Derivative Works thereof.\n'
        '\n'
        '      "Contribution" shall mean any work of authorship, including\n'
        '      the original version of the Work and any modifications or additions\n'
        '      to that Work or Derivative Works thereof, that is intentionally\n'
        '      submitted to Licensor for inclusion in the Work by the copyright owner\n'
        '      or by an individual or Legal Entity authorized to submit on behalf of\n'
        '      the copyright owner. For the purposes of this definition, "submitted"\n'
        '      means any form of electronic, verbal, or written communication sent\n'
        '      to the Licensor or its representatives, including but not limited to\n'
        '      communication on electronic mailing lists, source code control systems,\n'
        '      and issue tracking systems that are managed by, or on behalf of, the\n'
        '      Licensor for the purpose of discussing and improving the Work, but\n'
        '      excluding communication that is conspicuously marked or otherwise\n'
        '      designated in writing by the copyright owner as "Not a Contribution."\n'
        '\n'
        '      "Contributor" shall mean Licensor and any individual or Legal Entity\n'
        '      on behalf of whom a Contribution has been received by Licensor and\n'
        '      subsequently incorporated within the Work.\n'
        '\n'
        '   2. Grant of Copyright License. Subject to the terms and conditions of\n'
        '      this License, each Contributor hereby grants to You a perpetual,\n'
        '      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n'
        '      copyright license to reproduce, prepare Derivative Works of,\n'
        '      publicly display, publicly perform, sublicense, and distribute the\n'
        '      Work and such Derivative Works in Source or Object form.\n'
        '\n'
        '   3. Grant of Patent License. Subject to the terms and conditions of\n'
        '      this License, each Contributor hereby grants to You a perpetual,\n'
        '      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n'
        '      (except as stated in this section) patent license to make, have made,\n'
        '      use, offer to sell, sell, import, and otherwise transfer the Work,\n'
        '      where such license applies only to those patent claims licensable\n'
        '      by such Contributor that are necessarily infringed by their\n'
        '      Contribution(s) alone or by combination of their Contribution(s)\n'
        '      with the Work to which such Contribution(s) was submitted. If You\n'
        '      institute patent litigation against any entity (including a\n'
        '      cross-claim or counterclaim in a lawsuit) alleging that the Work\n'
        '      or a Contribution incorporated within the Work constitutes direct\n'
        '      or contributory patent infringement, then any patent licenses\n'
        '      granted to You under this License for that Work shall terminate\n'
        '      as of the date such litigation is filed.\n'
        '\n'
        '   4. Redistribution. You may reproduce and distribute copies of the\n'
        '      Work or Derivative Works thereof in any medium, with or without\n'
        '      modifications, and in Source or Object form, provided that You\n'
        '      meet the following conditions:\n'
        '\n'
        '      (a) You must give any other recipients of the Work or Derivative\n'
        '          Works a copy of this License; and\n'
        '\n'
        '      (b) You must cause any modified files to carry prominent notices\n'
        '          stating that You changed the files; and\n'
        '\n'
        '      (c) You must retain, in the Source form of any Derivative Works\n'
        '          that You distribute, all copyright, patent, trademark, and\n'
        '          attribution notices from the Source form of the Work,\n'
        '          excluding those notices that do not pertain to any part of\n'
        '          the Derivative Works; and\n'
        '\n'
        '      (d) If the Work includes a "NOTICE" text file as part of its\n'
        '          distribution, then any Derivative Works that You distribute must\n'
        '          include a readable copy of the attribution notices contained\n'
        '          within such NOTICE file, excluding those notices that do not\n'
        '          pertain to any part of the Derivative Works, in at least one\n'
        '          of the following places: within a NOTICE text file distributed\n'
        '          as part of the Derivative Works; within the Source form or\n'
        '          documentation, if provided along with the Derivative Works; or,\n'
        '          within a display generated by the Derivative Works, if and\n'
        '          wherever such third-party notices normally appear. The contents\n'
        '          of the NOTICE file are for informational purposes only and\n'
        '          do not modify the License. You may add Your own attribution\n'
        '          notices within Derivative Works that You distribute, alongside\n'
        '          or as an addendum to the NOTICE text from the Work, provided\n'
        '          that such additional attribution notices cannot be construed\n'
        '          as modifying the License.\n'
        '\n'
        '      You may add Your own copyright statement to Your modifications and\n'
        '      may provide additional or different license terms and conditions\n'
        '      for use, reproduction, or distribution of Your modifications, or\n'
        '      for any such Derivative Works as a whole, provided Your use,\n'
        '      reproduction, and distribution of the Work otherwise complies with\n'
        '      the conditions stated in this License.\n'
        '\n'
        '   5. Submission of Contributions. Unless You explicitly state otherwise,\n'
        '      any Contribution intentionally submitted for inclusion in the Work\n'
        '      by You to the Licensor shall be under the terms and conditions of\n'
        '      this License, without any additional terms or conditions.\n'
        '      Notwithstanding the above, nothing herein shall supersede or modify\n'
        '      the terms of any separate license agreement you may have executed\n'
        '      with Licensor regarding such Contributions.\n'
        '\n'
        '   6. Trademarks. This License does not grant permission to use the trade\n'
        '      names, trademarks, service marks, or product names of the Licensor,\n'
        '      except as required for reasonable and customary use in describing the\n'
        '      origin of the Work and reproducing the content of the NOTICE file.\n'
        '\n'
        '   7. Disclaimer of Warranty. Unless required by applicable law or\n'
        '      agreed to in writing, Licensor provides the Work (and each\n'
        '      Contributor provides its Contributions) on an "AS IS" BASIS,\n'
        '      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n'
        '      implied, including, without limitation, any warranties or conditions\n'
        '      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n'
        '      PARTICULAR PURPOSE. You are solely responsible for determining the\n'
        '      appropriateness of using or redistributing the Work and assume any\n'
        '      risks associated with Your exercise of permissions under this License.\n'
        '\n'
        '   8. Limitation of Liability. In no event and under no legal theory,\n'
        '      whether in tort (including negligence), contract, or otherwise,\n'
        '      unless required by applicable law (such as deliberate and grossly\n'
        '      negligent acts) or agreed to in writing, shall any Contributor be\n'
        '      liable to You for damages, including any direct, indirect, special,\n'
        '      incidental, or consequential damages of any character arising as a\n'
        '      result of this License or out of the use or inability to use the\n'
        '      Work (including but not limited to damages for loss of goodwill,\n'
        '      work stoppage, computer failure or malfunction, or any and all\n'
        '      other commercial damages or losses), even if such Contributor\n'
        '      has been advised of the possibility of such damages.\n'
        '\n'
        '   9. Accepting Warranty or Additional Liability. While redistributing\n'
        '      the Work or Derivative Works thereof, You may choose to offer,\n'
        '      and charge a fee for, acceptance of support, warranty, indemnity,\n'
        '      or other liability obligations and/or rights consistent with this\n'
        '      License. However, in accepting such obligations, You may act only\n'
        '      on Your own behalf and on Your sole responsibility, not on behalf\n'
        '      of any other Contributor, and only if You agree to indemnify,\n'
        '      defend, and hold each Contributor harmless for any liability\n'
        '      incurred by, or claims asserted against, such Contributor by reason\n'
        '      of your accepting any such warranty or additional liability.\n'
        '\n'
        '   END OF TERMS AND CONDITIONS\n'
        '\n'
        '   APPENDIX: How to apply the Apache License to your work.\n'
        '\n'
        '      To apply the Apache License to your work, attach the following\n'
        '      boilerplate notice, with the fields enclosed by brackets "[]"\n'
        "      replaced with your own identifying information. (Don't include\n"
        '      the brackets!)  The text should be enclosed in the appropriate\n'
        '      comment syntax for the file format. We also recommend that a\n'
        '      file or class name and description of purpose be included on the\n'
        '      same "printed page" as the copyright notice for easier\n'
        '      identification within third-party archives.\n'
        '\n'
        '   Copyright 2026 Kurt Valcorza\n'
        '\n'
        '   Licensed under the Apache License, Version 2.0 (the "License");\n'
        '   you may not use this file except in compliance with the License.\n'
        '   You may obtain a copy of the License at\n'
        '\n'
        '       http://www.apache.org/licenses/LICENSE-2.0\n'
        '\n'
        '   Unless required by applicable law or agreed to in writing, software\n'
        '   distributed under the License is distributed on an "AS IS" BASIS,\n'
        '   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n'
        '   See the License for the specific language governing permissions and\n'
        '   limitations under the License.\n'
    ),
    'licenses/depth-code.txt': (
        '                                 Apache License\n'
        '                           Version 2.0, January 2004\n'
        '                        http://www.apache.org/licenses/\n'
        '\n'
        '   TERMS AND CONDITIONS FOR USE, REPRODUCTION, AND DISTRIBUTION\n'
        '\n'
        '   1. Definitions.\n'
        '\n'
        '      "License" shall mean the terms and conditions for use, reproduction,\n'
        '      and distribution as defined by Sections 1 through 9 of this document.\n'
        '\n'
        '      "Licensor" shall mean the copyright owner or entity authorized by\n'
        '      the copyright owner that is granting the License.\n'
        '\n'
        '      "Legal Entity" shall mean the union of the acting entity and all\n'
        '      other entities that control, are controlled by, or are under common\n'
        '      control with that entity. For the purposes of this definition,\n'
        '      "control" means (i) the power, direct or indirect, to cause the\n'
        '      direction or management of such entity, whether by contract or\n'
        '      otherwise, or (ii) ownership of fifty percent (50%) or more of the\n'
        '      outstanding shares, or (iii) beneficial ownership of such entity.\n'
        '\n'
        '      "You" (or "Your") shall mean an individual or Legal Entity\n'
        '      exercising permissions granted by this License.\n'
        '\n'
        '      "Source" form shall mean the preferred form for making modifications,\n'
        '      including but not limited to software source code, documentation\n'
        '      source, and configuration files.\n'
        '\n'
        '      "Object" form shall mean any form resulting from mechanical\n'
        '      transformation or translation of a Source form, including but\n'
        '      not limited to compiled object code, generated documentation,\n'
        '      and conversions to other media types.\n'
        '\n'
        '      "Work" shall mean the work of authorship, whether in Source or\n'
        '      Object form, made available under the License, as indicated by a\n'
        '      copyright notice that is included in or attached to the work\n'
        '      (an example is provided in the Appendix below).\n'
        '\n'
        '      "Derivative Works" shall mean any work, whether in Source or Object\n'
        '      form, that is based on (or derived from) the Work and for which the\n'
        '      editorial revisions, annotations, elaborations, or other modifications\n'
        '      represent, as a whole, an original work of authorship. For the purposes\n'
        '      of this License, Derivative Works shall not include works that remain\n'
        '      separable from, or merely link (or bind by name) to the interfaces of,\n'
        '      the Work and Derivative Works thereof.\n'
        '\n'
        '      "Contribution" shall mean any work of authorship, including\n'
        '      the original version of the Work and any modifications or additions\n'
        '      to that Work or Derivative Works thereof, that is intentionally\n'
        '      submitted to Licensor for inclusion in the Work by the copyright owner\n'
        '      or by an individual or Legal Entity authorized to submit on behalf of\n'
        '      the copyright owner. For the purposes of this definition, "submitted"\n'
        '      means any form of electronic, verbal, or written communication sent\n'
        '      to the Licensor or its representatives, including but not limited to\n'
        '      communication on electronic mailing lists, source code control systems,\n'
        '      and issue tracking systems that are managed by, or on behalf of, the\n'
        '      Licensor for the purpose of discussing and improving the Work, but\n'
        '      excluding communication that is conspicuously marked or otherwise\n'
        '      designated in writing by the copyright owner as "Not a Contribution."\n'
        '\n'
        '      "Contributor" shall mean Licensor and any individual or Legal Entity\n'
        '      on behalf of whom a Contribution has been received by Licensor and\n'
        '      subsequently incorporated within the Work.\n'
        '\n'
        '   2. Grant of Copyright License. Subject to the terms and conditions of\n'
        '      this License, each Contributor hereby grants to You a perpetual,\n'
        '      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n'
        '      copyright license to reproduce, prepare Derivative Works of,\n'
        '      publicly display, publicly perform, sublicense, and distribute the\n'
        '      Work and such Derivative Works in Source or Object form.\n'
        '\n'
        '   3. Grant of Patent License. Subject to the terms and conditions of\n'
        '      this License, each Contributor hereby grants to You a perpetual,\n'
        '      worldwide, non-exclusive, no-charge, royalty-free, irrevocable\n'
        '      (except as stated in this section) patent license to make, have made,\n'
        '      use, offer to sell, sell, import, and otherwise transfer the Work,\n'
        '      where such license applies only to those patent claims licensable\n'
        '      by such Contributor that are necessarily infringed by their\n'
        '      Contribution(s) alone or by combination of their Contribution(s)\n'
        '      with the Work to which such Contribution(s) was submitted. If You\n'
        '      institute patent litigation against any entity (including a\n'
        '      cross-claim or counterclaim in a lawsuit) alleging that the Work\n'
        '      or a Contribution incorporated within the Work constitutes direct\n'
        '      or contributory patent infringement, then any patent licenses\n'
        '      granted to You under this License for that Work shall terminate\n'
        '      as of the date such litigation is filed.\n'
        '\n'
        '   4. Redistribution. You may reproduce and distribute copies of the\n'
        '      Work or Derivative Works thereof in any medium, with or without\n'
        '      modifications, and in Source or Object form, provided that You\n'
        '      meet the following conditions:\n'
        '\n'
        '      (a) You must give any other recipients of the Work or Derivative\n'
        '          Works a copy of this License; and\n'
        '\n'
        '      (b) You must cause any modified files to carry prominent notices\n'
        '          stating that You changed the files; and\n'
        '\n'
        '      (c) You must retain, in the Source form of any Derivative Works\n'
        '          that You distribute, all copyright, patent, trademark, and\n'
        '          attribution notices from the Source form of the Work,\n'
        '          excluding those notices that do not pertain to any part of\n'
        '          the Derivative Works; and\n'
        '\n'
        '      (d) If the Work includes a "NOTICE" text file as part of its\n'
        '          distribution, then any Derivative Works that You distribute must\n'
        '          include a readable copy of the attribution notices contained\n'
        '          within such NOTICE file, excluding those notices that do not\n'
        '          pertain to any part of the Derivative Works, in at least one\n'
        '          of the following places: within a NOTICE text file distributed\n'
        '          as part of the Derivative Works; within the Source form or\n'
        '          documentation, if provided along with the Derivative Works; or,\n'
        '          within a display generated by the Derivative Works, if and\n'
        '          wherever such third-party notices normally appear. The contents\n'
        '          of the NOTICE file are for informational purposes only and\n'
        '          do not modify the License. You may add Your own attribution\n'
        '          notices within Derivative Works that You distribute, alongside\n'
        '          or as an addendum to the NOTICE text from the Work, provided\n'
        '          that such additional attribution notices cannot be construed\n'
        '          as modifying the License.\n'
        '\n'
        '      You may add Your own copyright statement to Your modifications and\n'
        '      may provide additional or different license terms and conditions\n'
        '      for use, reproduction, or distribution of Your modifications, or\n'
        '      for any such Derivative Works as a whole, provided Your use,\n'
        '      reproduction, and distribution of the Work otherwise complies with\n'
        '      the conditions stated in this License.\n'
        '\n'
        '   5. Submission of Contributions. Unless You explicitly state otherwise,\n'
        '      any Contribution intentionally submitted for inclusion in the Work\n'
        '      by You to the Licensor shall be under the terms and conditions of\n'
        '      this License, without any additional terms or conditions.\n'
        '      Notwithstanding the above, nothing herein shall supersede or modify\n'
        '      the terms of any separate license agreement you may have executed\n'
        '      with Licensor regarding such Contributions.\n'
        '\n'
        '   6. Trademarks. This License does not grant permission to use the trade\n'
        '      names, trademarks, service marks, or product names of the Licensor,\n'
        '      except as required for reasonable and customary use in describing the\n'
        '      origin of the Work and reproducing the content of the NOTICE file.\n'
        '\n'
        '   7. Disclaimer of Warranty. Unless required by applicable law or\n'
        '      agreed to in writing, Licensor provides the Work (and each\n'
        '      Contributor provides its Contributions) on an "AS IS" BASIS,\n'
        '      WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or\n'
        '      implied, including, without limitation, any warranties or conditions\n'
        '      of TITLE, NON-INFRINGEMENT, MERCHANTABILITY, or FITNESS FOR A\n'
        '      PARTICULAR PURPOSE. You are solely responsible for determining the\n'
        '      appropriateness of using or redistributing the Work and assume any\n'
        '      risks associated with Your exercise of permissions under this License.\n'
        '\n'
        '   8. Limitation of Liability. In no event and under no legal theory,\n'
        '      whether in tort (including negligence), contract, or otherwise,\n'
        '      unless required by applicable law (such as deliberate and grossly\n'
        '      negligent acts) or agreed to in writing, shall any Contributor be\n'
        '      liable to You for damages, including any direct, indirect, special,\n'
        '      incidental, or consequential damages of any character arising as a\n'
        '      result of this License or out of the use or inability to use the\n'
        '      Work (including but not limited to damages for loss of goodwill,\n'
        '      work stoppage, computer failure or malfunction, or any and all\n'
        '      other commercial damages or losses), even if such Contributor\n'
        '      has been advised of the possibility of such damages.\n'
        '\n'
        '   9. Accepting Warranty or Additional Liability. While redistributing\n'
        '      the Work or Derivative Works thereof, You may choose to offer,\n'
        '      and charge a fee for, acceptance of support, warranty, indemnity,\n'
        '      or other liability obligations and/or rights consistent with this\n'
        '      License. However, in accepting such obligations, You may act only\n'
        '      on Your own behalf and on Your sole responsibility, not on behalf\n'
        '      of any other Contributor, and only if You agree to indemnify,\n'
        '      defend, and hold each Contributor harmless for any liability\n'
        '      incurred by, or claims asserted against, such Contributor by reason\n'
        '      of your accepting any such warranty or additional liability.\n'
        '\n'
        '   END OF TERMS AND CONDITIONS\n'
        '\n'
        '   APPENDIX: How to apply the Apache License to your work.\n'
        '\n'
        '      To apply the Apache License to your work, attach the following\n'
        '      boilerplate notice, with the fields enclosed by brackets "[]"\n'
        "      replaced with your own identifying information. (Don't include\n"
        '      the brackets!)  The text should be enclosed in the appropriate\n'
        '      comment syntax for the file format. We also recommend that a\n'
        '      file or class name and description of purpose be included on the\n'
        '      same "printed page" as the copyright notice for easier\n'
        '      identification within third-party archives.\n'
        '\n'
        '   Copyright 2026 Kurt Valcorza\n'
        '\n'
        '   Licensed under the Apache License, Version 2.0 (the "License");\n'
        '   you may not use this file except in compliance with the License.\n'
        '   You may obtain a copy of the License at\n'
        '\n'
        '       http://www.apache.org/licenses/LICENSE-2.0\n'
        '\n'
        '   Unless required by applicable law or agreed to in writing, software\n'
        '   distributed under the License is distributed on an "AS IS" BASIS,\n'
        '   WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.\n'
        '   See the License for the specific language governing permissions and\n'
        '   limitations under the License.\n'
    ),
    'licenses/weight-data-provenance.md': (
        '# Weight provenance and DIMER hosting\n'
        '\n'
        '- Upstream: `depth-anything/Depth-Anything-V2-Small-hf`\n'
        '- Immutable revision: `5426e4f0f36572d16453bbda7a8389317b1bef99`\n'
        '- Weight format: SafeTensors (`model.safetensors`, 99,173,660 bytes)\n'
        '- Manifest: `weights/depth-anything-v2-small/dimer-base-manifest.json` (4 files, 99,179,785 bytes total, per-file SHA-256)\n'
        '- Upstream weight license: Apache-2.0\n'
        '- DIMER hosting: Apache-2.0 permits use, modification, distribution, and commercial use subject to preservation of the license and notices. The Git repository does not vendor the checkpoint (`weights/**/*.safetensors` is git-ignored); DIMER may mirror the pinned snapshot in its model store under the upstream license.\n'
        '- Loader trust boundary: Transformers `AutoModelForDepthEstimation` / `AutoImageProcessor` with `trust_remote_code=False`, `local_files_only=True` from the verified directory; `verify_snapshot()` must pass before any load.\n'
        '\n'
        '## Adaptation corpus (tutorial data, not weights)\n'
        '\n'
        '- Corpus: 40 views from the DIODE validation release (Vasiljevic et al., 2019; CC BY 4.0) — the first two views in file-name order of each of its 20 laser-scanned scans (10 indoor, 10 outdoor), selected a priori on 2026-09-19 by the fetcher recorded in the build ledger.\n'
        '- Pins: `samples.SAMPLE_RECORDS` holds, per view, the record id, domain, scene, scan, file stem, and the byte size and SHA-256 of each of three files — the RGB PNG (1024 × 768), the depth `.npy` (float32 metres) and the validity-mask `.npy` — as served by the Marigold evaluation mirror `https://huggingface.co/datasets/obukhovai/marigold_depth_eval` at commit `30c5b061` (the full 40-hex commit is pinned in `samples.CORPUS_COMMIT`) (312,448,846 bytes in total, no credential). `samples.fetch_corpus` downloads each file at run time into the git-ignored `weights/diode-sample/` cache and refuses any byte or digest mismatch; `read_corpus` decodes them without re-encoding. The repository redistributes none of the files; every record keeps its source URL. Byte-identity with the original DIODE tarball has not been checked — the pin is to the served mirror files.\n'
        '- Sample: `build_sample_dataset(seed=42)` draws 6 / 2 / 2 whole scans per domain (24 / 8 / 8 views) by a seeded shuffle; `check_split_disjoint` asserts no view (by decoded-pixel digest) and no scan is in two splits, `scan_summary` reports scans and domains per split. DIMER hosting of the weights is unaffected.\n'
        '- Adapter artifacts written by the tutorial (`outputs/depth_anything_depth_estimation_adapter/`, `org.valcorza.depth-anything-v2-small.adapter.v1`, 10.9 MB for the DPT neck and head, about 25.1 MB with two trained blocks) carry the trained tensors with a manifest naming the base `model.safetensors` digest and the selected policy; they are outputs, not hosted weights.\n'
    ),
    'vendor_provenance.json': (
        '{\n'
        '  "zoe": {\n'
        '    "repository": "kurtvalcorza/zoedepth-metric-depth-pipeline",\n'
        '    "revision": "194fbbc6111e36873a84eb78404a687201aa1d73",\n'
        '    "files": {\n'
        '      "zoe_reference.py": {\n'
        '        "source": "src/zoedepth_metric_depth_pipeline/pipeline.py",\n'
        '        "sha256": "1107ff98322c57e76c82ac41172059470ca36163f990de1a1ed0b455c1e07ba2"\n'
        '      },\n'
        '      "zoe_manifest.json": {\n'
        '        "source": "weights/zoedepth-nyu-kitti/dimer-base-manifest.json",\n'
        '        "sha256": "0c3aaba48ba5acbd8c99b30e5a341c6d15ddacd80d7b23608513f275bab853c5",\n'
        '        "carried_sha256": "03b7a287e192b38974a6e6184b0ac2fc8358795bb52ee10f3387af0b62a9e15b",\n'
        '        "line_endings": "upstream commits this manifest with CRLF (weights/** -text); sha256 is the upstream blob, carried_sha256 is the LF copy carried here"\n'
        '      },\n'
        '      "zoe_LICENSE": {\n'
        '        "source": "LICENSE",\n'
        '        "sha256": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138"\n'
        '      }\n'
        '    }\n'
        '  }\n'
        '}'
    ),
    'weights/depth/dimer-base-manifest.json': (
        '{\n'
        '  "format": "dimer_hf_snapshot",\n'
        '  "formatVersion": 1,\n'
        '  "modelKey": "depth-anything-v2-small",\n'
        '  "modelId": "depth-anything/Depth-Anything-V2-Small-hf",\n'
        '  "revision": "5426e4f0f36572d16453bbda7a8389317b1bef99",\n'
        '  "files": [\n'
        '    {\n'
        '      "path": "README.md",\n'
        '      "bytes": 4400,\n'
        '      "sha256": "319566441daaac0c5ed83e75a8fd19bae0863f3275e23f147c06fe0906edf6fb"\n'
        '    },\n'
        '    {\n'
        '      "path": "config.json",\n'
        '      "bytes": 950,\n'
        '      "sha256": "c56698d3643dde1f83ea2212759e6b31a22b8f827246a36dd007ee8a22b3ff75"\n'
        '    },\n'
        '    {\n'
        '      "path": "model.safetensors",\n'
        '      "bytes": 99173660,\n'
        '      "sha256": "3152477ce0d8d6978d76b995120de97cb5b928701fd0f817769f59e249a16b70"\n'
        '    },\n'
        '    {\n'
        '      "path": "preprocessor_config.json",\n'
        '      "bytes": 775,\n'
        '      "sha256": "d41175c0d889477ca8fc67191e540faef14baf6275157b3fdecf78469e6bbf84"\n'
        '    }\n'
        '  ],\n'
        '  "totalBytes": 99179785\n'
        '}'
    ),
    'workshop.py': (
        '"""Standalone depth workshop worker; heavy model imports occur only in GPU stages."""\n'
        '\n'
        'from __future__ import annotations\n'
        '\n'
        'import argparse\n'
        'import csv\n'
        'import hashlib\n'
        'import importlib.metadata\n'
        'import json\n'
        'import os\n'
        'import re\n'
        'import subprocess\n'
        'import sys\n'
        'import time\n'
        'import urllib.error\n'
        'import uuid\n'
        'import zipfile\n'
        'from pathlib import Path\n'
        'from typing import Any\n'
        '\n'
        'import numpy as np\n'
        'from PIL import Image, ImageDraw\n'
        '\n'
        'SCENES = {\n'
        '    "scene_00020": "train",\n'
        '    "scene_00023": "train",\n'
        '    "scene_00019": "validation",\n'
        '    "scene_00024": "validation",\n'
        '    "scene_00021": "test",\n'
        '    "scene_00022": "test",\n'
        '}\n'
        'STAGES = ["prepare", "depth", "reload", "zoe", "activity", "report"]\n'
        'REQUIRES = {\n'
        '    "prepare": [],\n'
        '    "depth": ["prepare"],\n'
        '    "reload": ["depth"],\n'
        '    "zoe": ["prepare"],\n'
        '    "activity": ["reload"],\n'
        '    "report": ["prepare", "depth", "reload", "zoe", "activity"],\n'
        '}\n'
        'CONFIG = {\n'
        '    "trainable_blocks": 0,\n'
        '    "head_epochs": 3,\n'
        '    "epochs": 0,\n'
        '    "lr": 1e-5,\n'
        '    "seed": 42,\n'
        '    "batch_size": 1,\n'
        '    "weight_decay": 0.01,\n'
        '    "gradient_clip": 1.0,\n'
        '    "precision": "float32",\n'
        '    "depth_range_metres": [0.6, 350],\n'
        '    "brightness_factors": [0.6, 1.0, 1.4],\n'
        '    "qualification": "Candidate",\n'
        '    "target_resampling": "source nearest-neighbour depth and mask, then inverse depth",\n'
        '    "prediction_preflight": "reject nonfinite on fixed support; metric predictions positive",\n'
        '    "selection": "validation aligned AbsRel rounded to six decimals, earliest tie incl epoch 0",\n'
        '}\n'
        '\n'
        '\n'
        'def digest(path: Path) -> str:\n'
        '    return hashlib.sha256(path.read_bytes()).hexdigest()\n'
        '\n'
        '\n'
        'def array_digest(array: np.ndarray) -> str:\n'
        '    array = np.ascontiguousarray(array)\n'
        '    return hashlib.sha256(str((array.shape, array.dtype.str)).encode() + array.tobytes()).hexdigest()\n'
        '\n'
        '\n'
        'def write_json(path: Path, data: Any) -> None:\n'
        '    path.parent.mkdir(parents=True, exist_ok=True)\n'
        '    path.write_text(json.dumps(data, indent=2, allow_nan=False) + "\\n", encoding="utf-8")\n'
        '\n'
        '\n'
        'def read_json(path: Path) -> Any:\n'
        '    return json.loads(path.read_text(encoding="utf-8"))\n'
        '\n'
        '\n'
        'def support(depth: np.ndarray, mask: np.ndarray) -> np.ndarray:\n'
        '    depth, mask = np.asarray(depth), np.asarray(mask)\n'
        '    if depth.ndim != 2 or mask.shape != depth.shape or mask.dtype != np.bool_:\n'
        '        raise ValueError("depth must be HxW and mask must be a matching boolean array")\n'
        '    valid = mask & np.isfinite(depth) & (depth >= 0.6) & (depth <= 350)\n'
        '    if int(valid.sum()) < 2:\n'
        '        raise ValueError("need at least two valid reference pixels in [0.6,350] metres")\n'
        '    return valid\n'
        '\n'
        '\n'
        'def aligned_values(pred: np.ndarray, depth: np.ndarray, mask: np.ndarray) -> tuple:\n'
        '    valid = support(depth, mask)\n'
        '    pred = np.asarray(pred, dtype=np.float64)\n'
        '    if pred.shape != depth.shape or not np.isfinite(pred[valid]).all():\n'
        '        raise ValueError("prediction shape or finite fixed-support contract failed")\n'
        '    p, ref = pred[valid], np.asarray(depth, dtype=np.float64)[valid]\n'
        '    design = np.stack([p, np.ones_like(p)], axis=1)\n'
        '    coefficients, _, rank, singular = np.linalg.lstsq(design, 1 / ref, rcond=None)\n'
        '    a, b = coefficients\n'
        '    inverse = a * p + b\n'
        '    estimate = 1 / np.maximum(inverse, 1 / 350)\n'
        '    fit = {\n'
        '        "scale": float(a),\n'
        '        "shift": float(b),\n'
        '        "rank": int(rank),\n'
        '        "singular_values": singular.tolist(),\n'
        '        "degenerate": bool(rank < 2),\n'
        '        "negative_scale": bool(a < 0),\n'
        '        "far_cap_fraction": float(np.mean(inverse <= 1 / 350)),\n'
        '    }\n'
        '    return estimate, ref, valid, fit\n'
        '\n'
        '\n'
        'def score(pred: np.ndarray, depth: np.ndarray, mask: np.ndarray, aligned: bool = False) -> dict:\n'
        '    valid = support(depth, mask)\n'
        '    pred = np.asarray(pred, dtype=np.float64)\n'
        '    if pred.shape != depth.shape or not np.isfinite(pred[valid]).all():\n'
        '        raise ValueError("prediction shape or finite fixed-support contract failed")\n'
        '    fit = {}\n'
        '    if aligned:\n'
        '        estimate, ref, valid, fit = aligned_values(pred, depth, mask)\n'
        '    else:\n'
        '        estimate, ref = pred[valid], np.asarray(depth, dtype=np.float64)[valid]\n'
        '        if np.any(estimate <= 0):\n'
        '            raise ValueError("metric predictions must be positive on fixed support")\n'
        '    return {\n'
        '        "abs_rel": float(np.mean(np.abs(estimate - ref) / ref)),\n'
        '        "delta1": float(np.mean(np.maximum(estimate / ref, ref / estimate) < 1.25)),\n'
        '        "n_valid": int(valid.sum()),\n'
        '        "valid_fraction": float(valid.mean()),\n'
        '        "excluded_predictions": 0,\n'
        '        "protocol": "reference-assisted affine alignment" if aligned else "unaligned metric metres",\n'
        '        **fit,\n'
        '    }\n'
        '\n'
        '\n'
        'def aggregate(rows: list[dict]) -> dict:\n'
        '    if not rows:\n'
        '        return {"n": 0, "abs_rel": None, "delta1": None, "reason": "no records"}\n'
        '    result = {\n'
        '        "n": len(rows),\n'
        '        "abs_rel": float(np.mean([r["abs_rel"] for r in rows])),\n'
        '        "delta1": float(np.mean([r["delta1"] for r in rows])),\n'
        '        "aggregation": "equal-image mean",\n'
        '    }\n'
        '    for key in ("scene", "domain"):\n'
        '        result[f"per_{key}"] = {}\n'
        '        for value in sorted({str(r.get(key, "unspecified")) for r in rows}):\n'
        '            group = [r for r in rows if str(r.get(key, "unspecified")) == value]\n'
        '            result[f"per_{key}"][value] = {\n'
        '                "n": len(group),\n'
        '                "abs_rel": float(np.mean([r["abs_rel"] for r in group])),\n'
        '                "delta1": float(np.mean([r["delta1"] for r in group])),\n'
        '            }\n'
        '    return result\n'
        '\n'
        '\n'
        'def brightness(image: Image.Image, factor: float) -> tuple[Image.Image, dict]:\n'
        '    # The canonical activity uses exactly 0.6, 1.0 and 1.4; the optional own-factor activity may use 0.2..2.0.\n'
        '    if not 0.2 <= factor <= 2.0:\n'
        '        raise ValueError(f"brightness factor {factor} is outside 0.2..2.0")\n'
        '    values = np.asarray(image.convert("RGB"), dtype=np.float32) * factor\n'
        '    output = np.round(np.clip(values, 0, 255)).astype(np.uint8)\n'
        '    return Image.fromarray(output), {\n'
        '        "factor": factor,\n'
        '        "clipped_pixel_fraction": float(np.any((values < 0) | (values > 255), axis=2).mean()),\n'
        '        "input_sha256": array_digest(output),\n'
        '    }\n'
        '\n'
        '\n'
        'def verify_parity(before: np.ndarray, after: np.ndarray) -> dict:\n'
        '    before, after = np.asarray(before), np.asarray(after)\n'
        '    if before.shape != after.shape or not np.isfinite(before).all() or not np.isfinite(after).all():\n'
        '        raise ValueError("reload shape/nonfinite failure")\n'
        '    error = float(np.max(np.abs(before.astype(float) - after.astype(float))))\n'
        '    if not np.allclose(before, after, atol=1e-5, rtol=1e-4):\n'
        '        raise ValueError(f"reload prediction parity failed: max abs {error}")\n'
        '    return {"max_absolute_error": error, "atol": 1e-5, "rtol": 1e-4, "passed": True}\n'
        '\n'
        '\n'
        'def validate_roles(records: list[dict], byod: bool = False) -> dict:\n'
        '    ids, images, contents, groups = set(), {}, {}, {}\n'
        '    counts = dict.fromkeys(("train", "validation", "test"), 0)\n'
        '    for r in records:\n'
        '        rid, role = r.get("id"), r.get("role")\n'
        '        if not isinstance(rid, str) or not re.fullmatch(r"[A-Za-z0-9_-]{1,64}", rid):\n'
        '            raise ValueError(f"record id {rid!r}: ids must be 1-64 characters from A-Z a-z 0-9 _ -")\n'
        '        if rid in ids:\n'
        '            raise ValueError(f"record id {rid!r} appears more than once")\n'
        '        ids.add(rid)\n'
        '        if role not in counts:\n'
        '            raise ValueError(f"record {rid!r}: role {role!r} must be \'train\', \'validation\' or \'test\'")\n'
        '        counts[role] += 1\n'
        '        group = r.get("group_id") or (str(r.get("domain", "")) + "/" + str(r.get("scene", "")))\n'
        '        if not group or group == "/":\n'
        '            raise ValueError(f"record {rid!r}: an acquisition group (group_id) is required")\n'
        '        if group in groups and groups[group] != role:\n'
        '            raise ValueError(\n'
        '                f"record {rid!r}: group {group!r} appears in both {groups[group]!r} and {role!r}; "\n'
        '                "keep every record of an acquisition group in one role"\n'
        '            )\n'
        '        groups[group] = role\n'
        '        try:\n'
        '            valid = support(r["depth"], r["mask"])\n'
        '        except ValueError as error:\n'
        '            raise ValueError(f"record {rid!r}: {error}") from error\n'
        '        if r["depth"].shape != (r["image"].height, r["image"].width):\n'
        '            raise ValueError(\n'
        '                f"record {rid!r}: depth shape {r[\'depth\'].shape} differs from the image "\n'
        '                f"(height, width) {(r[\'image\'].height, r[\'image\'].width)}"\n'
        '            )\n'
        '        rgb_hash = array_digest(np.asarray(r["image"].convert("RGB")))\n'
        '        rgbd_hash = hashlib.sha256(\n'
        '            (rgb_hash + array_digest(r["depth"]) + array_digest(r["mask"])).encode()\n'
        '        ).hexdigest()\n'
        '        for table, key in ((images, rgb_hash), (contents, rgbd_hash)):\n'
        '            if key in table:\n'
        '                raise ValueError(f"record {rid!r}: decoded content duplicates record {table[key]!r}")\n'
        '            table[key] = rid\n'
        '        r.update(\n'
        '            group_id=group,\n'
        '            image_sha256=rgb_hash,\n'
        '            rgbd_sha256=rgbd_hash,\n'
        '            n_valid=int(valid.sum()),\n'
        '            valid_fraction=float(valid.mean()),\n'
        '            reference_above_80m=int(np.sum(valid & (r["depth"] > 80))),\n'
        '        )\n'
        '    expected = (\n'
        '        {"train": 6, "validation": 2, "test": 4} if byod else {"train": 14, "validation": 6, "test": 20}\n'
        '    )\n'
        '    if any(counts[k] < v if byod else counts[k] != v for k, v in expected.items()):\n'
        '        raise ValueError(\n'
        '            f"wrong role counts: {counts}"\n'
        '            + ("; BYOD needs at least 6 train, 2 validation and 4 test records" if byod else "")\n'
        '        )\n'
        '    group_counts = {role: sum(v == role for v in groups.values()) for role in counts}\n'
        '    if any(n < 2 for n in group_counts.values()):\n'
        '        raise ValueError(\n'
        '            f"at least two independent acquisition groups required in every role; found {group_counts}"\n'
        '        )\n'
        '    return {\n'
        '        "counts": counts,\n'
        '        "group_counts": group_counts,\n'
        '        "exclusions": [],\n'
        '        "reference_above_80m_by_role": {\n'
        '            role: sum(r["reference_above_80m"] for r in records if r["role"] == role) for role in counts\n'
        '        },\n'
        '    }\n'
        '\n'
        '\n'
        'def safe_file(root: Path, name: str) -> Path:\n'
        '    if not isinstance(name, str) or not name or "\\\\" in name or ":" in name:\n'
        '        raise ValueError(f"{name!r}: only relative paths inside the BYOD directory, with \'/\', are allowed")\n'
        '    rel = Path(name)\n'
        '    path = (root / rel).resolve()\n'
        '    if rel.is_absolute() or ".." in rel.parts or not path.is_relative_to(root.resolve()):\n'
        '        raise ValueError(f"{name!r}: path escapes the BYOD directory")\n'
        '    if not path.is_file():\n'
        '        raise ValueError(f"{name!r}: file not found in the BYOD directory")\n'
        '    if path.stat().st_size > 32 * 1024**2:\n'
        '        raise ValueError(f"{name!r}: file exceeds 32 MiB")\n'
        '    return path\n'
        '\n'
        '\n'
        'def bounded_image(path: Path) -> Image.Image:\n'
        '    if path.stat().st_size > 32 * 1024**2:\n'
        '        raise ValueError(f"{path.name}: image file exceeds 32 MiB")\n'
        '    with Image.open(path) as image:\n'
        '        w, h = image.size\n'
        '        if min(w, h) < 32 or max(w, h) > 2048 or max(w, h) / min(w, h) > 4:\n'
        '            raise ValueError(\n'
        '                f"{path.name}: {w}x{h} px is outside the limits (sides 32..2048 px, aspect ratio at most 4); "\n'
        '                "save a resized copy whose longest side is at most 2048 px"\n'
        '            )\n'
        '        orientation = image.getexif().get(274, 1)\n'
        '        if orientation != 1:\n'
        '            raise ValueError(\n'
        '                f"{path.name}: EXIF orientation is {orientation}; save a copy with the rotation applied "\n'
        '                "(for example PIL.ImageOps.exif_transpose) and, for labelled data, rotate its depth and mask "\n'
        '                "arrays the same way"\n'
        '            )\n'
        '        return image.convert("RGB")\n'
        '\n'
        '\n'
        'def bounded_array(path: Path, shape: tuple, dtype: np.dtype) -> np.ndarray:\n'
        '    with path.open("rb") as handle:\n'
        '        version = np.lib.format.read_magic(handle)\n'
        '        if version == (1, 0):\n'
        '            got_shape, _, got_dtype = np.lib.format.read_array_header_1_0(handle)\n'
        '        elif version == (2, 0):\n'
        '            got_shape, _, got_dtype = np.lib.format.read_array_header_2_0(handle)\n'
        '        else:\n'
        '            raise ValueError(f"{path.name}: only NPY format versions 1.0 and 2.0 are supported")\n'
        '        if got_shape != shape or got_dtype != dtype or got_dtype.hasobject:\n'
        '            raise ValueError(\n'
        '                f"{path.name}: expected a {dtype} array of shape {shape} (the image\'s height x width), "\n'
        '                f"found {got_dtype} {got_shape}"\n'
        '            )\n'
        '        if int(np.prod(got_shape)) * got_dtype.itemsize > 32 * 1024**2:\n'
        '            raise ValueError(f"{path.name}: array allocation exceeds 32 MiB")\n'
        '    return np.load(path, allow_pickle=False)\n'
        '\n'
        '\n'
        'def check_source_contract(record: dict) -> None:\n'
        '    """The carried Depth Anything per-record contract (>=5% valid coverage, depth <= 1000 m), unprefixed."""\n'
        '    from depth_reference import samples\n'
        '\n'
        '    try:\n'
        '        samples.validate_dataset([record], min_records=1)\n'
        '    except ValueError as error:\n'
        '        raise ValueError(str(error).removeprefix("records[0]: ").removeprefix("records[0] ")) from error\n'
        '\n'
        '\n'
        'def load_byod(root: Path) -> list[dict]:\n'
        '    root = Path(root)\n'
        '    manifest = safe_file(root, "manifest.json")\n'
        '    if manifest.stat().st_size > 1024**2:\n'
        '        raise ValueError("manifest exceeds 1 MiB")\n'
        '    data = read_json(manifest)\n'
        '    if data.get("units") != "metres" or not isinstance(data.get("records"), list):\n'
        '        raise ValueError("manifest requires units=\'metres\' and records list")\n'
        '    if not 12 <= len(data["records"]) <= 128:\n'
        '        raise ValueError("BYOD requires 12..128 records")\n'
        '    records, pixels = [], 0\n'
        '    for index, row in enumerate(data["records"]):\n'
        '        if not isinstance(row, dict):\n'
        '            raise ValueError(f"record {index}: each record must be a JSON object")\n'
        '        label = f"record {row.get(\'id\', index)!r}"\n'
        '        try:\n'
        '            if not row.get("group_id"):\n'
        '                raise ValueError("an explicit group_id is required")\n'
        '            for key in ("image", "depth", "mask", "role"):\n'
        '                if key not in row:\n'
        '                    raise ValueError(f"missing required key {key!r}")\n'
        '            image = bounded_image(safe_file(root, row["image"]))\n'
        '            shape = (image.height, image.width)\n'
        '            pixels += image.height * image.width\n'
        '            if pixels > 64_000_000:\n'
        '                raise ValueError("total image pixels across records exceed 64 million")\n'
        '            depth = bounded_array(safe_file(root, row["depth"]), shape, np.dtype("float32"))\n'
        '            mask = bounded_array(safe_file(root, row["mask"]), shape, np.dtype("bool"))\n'
        '            record = {\n'
        '                **row,\n'
        '                "image": image,\n'
        '                "depth": depth,\n'
        '                "mask": mask,\n'
        '                "scene": row["group_id"],\n'
        '                "domain": str(row.get("domain", "BYOD")),\n'
        '            }\n'
        '            check_source_contract(record)\n'
        '        except ValueError as error:\n'
        '            raise ValueError(f"{label}: {error}") from error\n'
        '        records.append(record)\n'
        '    validate_roles(records, byod=True)\n'
        '    return records\n'
        '\n'
        '\n'
        'def record_metadata(r: dict) -> dict:\n'
        '    return {k: v for k, v in r.items() if k not in ("image", "depth", "mask")}\n'
        '\n'
        '\n'
        'def load_records(root: Path) -> list[dict]:\n'
        '    rows = read_json(root / "data/records.json")\n'
        '    return [\n'
        '        {\n'
        '            **r,\n'
        '            "image": Image.open(root / r["image"]).convert("RGB"),\n'
        '            "depth": np.load(root / r["depth"], allow_pickle=False),\n'
        '            "mask": np.load(root / r["mask"], allow_pickle=False),\n'
        '        }\n'
        '        for r in rows\n'
        '    ]\n'
        '\n'
        '\n'
        'def panel(root: Path, record: dict, name: str, predictions: dict | None = None) -> None:\n'
        '    valid = support(record["depth"], record["mask"])\n'
        '    images = [("RGB", record["image"])]\n'
        '\n'
        '    def heat(\n'
        '        array: np.ndarray, low: float, high: float, use_mask: bool = True, log_scale: bool = False\n'
        '    ) -> Image.Image:\n'
        '        if log_scale:\n'
        '            scaled = np.log(np.maximum(array, low) / low) / np.log(high / low)\n'
        '        else:\n'
        '            scaled = (array - low) / (high - low)\n'
        '        scaled = np.clip(scaled, 0, 1)\n'
        '        scaled = np.nan_to_num(scaled)\n'
        '        rgb = np.stack([scaled, 1 - np.abs(2 * scaled - 1), 1 - scaled], axis=-1)\n'
        '        rgb[~valid if use_mask else np.zeros_like(valid)] = 0.35\n'
        '        return Image.fromarray(np.round(rgb * 255).astype(np.uint8))\n'
        '\n'
        '    images += [\n'
        '        ("Reference metres (log colour: 0.6 to 350)", heat(record["depth"], 0.6, 350, log_scale=True)),\n'
        '        ("Valid reference support", Image.fromarray(valid.astype(np.uint8) * 255).convert("RGB")),\n'
        '    ]\n'
        '    for label, (prediction, aligned) in (predictions or {}).items():\n'
        '        if aligned:\n'
        '            values, _, _, _ = aligned_values(prediction, record["depth"], record["mask"])\n'
        '            raw = np.asarray(prediction)\n'
        '            images.append(\n'
        '                (\n'
        '                    label + " arbitrary units (display min/max)",\n'
        '                    heat(raw, float(raw.min()), float(raw.max()) + 1e-12, False),\n'
        '                )\n'
        '            )\n'
        '            metric = np.full(valid.shape, np.nan)\n'
        '            metric[valid] = values\n'
        '        else:\n'
        '            metric = prediction\n'
        '            images.append(\n'
        '                (label + " metres (log colour: 0.6 to 350)", heat(metric, 0.6, 350, log_scale=True))\n'
        '            )\n'
        '        images.append(\n'
        '            (label + " absolute error metres [0,50]", heat(np.abs(metric - record["depth"]), 0, 50))\n'
        '        )\n'
        '    columns = min(3, len(images))\n'
        '    rows = (len(images) + columns - 1) // columns\n'
        '    canvas = Image.new("RGB", (420 * columns, 370 * rows + 65), "white")\n'
        '    draw = ImageDraw.Draw(canvas)\n'
        '    for i, (title, image) in enumerate(images):\n'
        '        image = image.copy()\n'
        '        image.thumbnail((410, 300))\n'
        '        x, y = (i % columns) * 420, (i // columns) * 370\n'
        '        canvas.paste(image, (x, y + 40))\n'
        '        draw.text((x + 4, y + 4), title, fill="black")\n'
        '    legend_y = rows * 370\n'
        '    for column, caption in enumerate(\n'
        '        ("Metric metres (log)", "Absolute error metres (linear)", "Relative display (normalized)")\n'
        '    ):\n'
        '        if column >= columns:\n'
        '            break\n'
        '        x = column * 420\n'
        '        draw.text((x + 4, legend_y), caption, fill="black")\n'
        '        for pixel in range(400):\n'
        '            fraction = pixel / 399\n'
        '            colour = (int(255 * fraction), int(255 * (1 - abs(2 * fraction - 1))), int(255 * (1 - fraction)))\n'
        '            draw.line((x + pixel, legend_y + 18, x + pixel, legend_y + 28), fill=colour)\n'
        '        ticks = ((0.6, 2, 10, 50, 350), (0, 12.5, 25, 37.5, 50), (0, 0.25, 0.5, 0.75, 1))[column]\n'
        '        for value in ticks:\n'
        '            position = (\n'
        '                np.log(value / 0.6) / np.log(350 / 0.6) if column == 0 else value / (50 if column == 1 else 1)\n'
        '            )\n'
        '            draw.text((x + min(374, int(position * 399)), legend_y + 33), str(value), fill="black")\n'
        '    path = root / "outputs" / name\n'
        '    path.parent.mkdir(parents=True, exist_ok=True)\n'
        '    canvas.save(path)\n'
        '\n'
        '\n'
        'def prepare(root: Path, byod: str | None = None) -> None:\n'
        '    import zoe_reference\n'
        '    from depth_reference import pipeline as source\n'
        '    from depth_reference import samples\n'
        '\n'
        '    if byod:\n'
        '        records = load_byod(Path(byod))\n'
        '    else:\n'
        '        for attempt in range(3):\n'
        '            try:\n'
        '                records = samples.read_corpus(samples.fetch_corpus(cache_dir=root / "cache"))\n'
        '                break\n'
        '            except (urllib.error.URLError, TimeoutError, ConnectionError):\n'
        '                if attempt == 2:\n'
        '                    raise\n'
        '                time.sleep(2**attempt)\n'
        '        for r in records:\n'
        '            r["role"] = SCENES[r["scene"]]\n'
        '    result = validate_roles(records, byod=bool(byod))\n'
        '    for r in records:\n'
        '        try:\n'
        '            source.validate_image(r["image"])\n'
        '            zoe_reference.validate_image(r["image"])\n'
        '            # Preserve source adaptation validation, including >=5% sensor-valid coverage.\n'
        '            check_source_contract(r)\n'
        '        except ValueError as error:\n'
        '            raise ValueError(f"record {r[\'id\']!r}: {error}") from error\n'
        '    data_dir = root / "data"\n'
        '    data_dir.mkdir(parents=True, exist_ok=True)\n'
        '    rows = []\n'
        '    for r in records:\n'
        '        image_path, depth_path, mask_path = (\n'
        '            f"data/{r[\'id\']}{s}" for s in (".png", "_depth.npy", "_mask.npy")\n'
        '        )\n'
        '        r["image"].save(root / image_path)\n'
        '        np.save(root / depth_path, r["depth"], allow_pickle=False)\n'
        '        np.save(root / mask_path, r["mask"], allow_pickle=False)\n'
        '        rows.append({**record_metadata(r), "image": image_path, "depth": depth_path, "mask": mask_path})\n'
        '    write_json(data_dir / "records.json", rows)\n'
        '    manifest = {\n'
        '        **result,\n'
        '        "private_byod": bool(byod),\n'
        '        "units": "metres",\n'
        '        "records": [\n'
        '            {\n'
        '                **record_metadata(r),\n'
        '                "shape": list(r["depth"].shape),\n'
        '                "depth_sha256": array_digest(r["depth"]),\n'
        '                "mask_sha256": array_digest(r["mask"]),\n'
        '            }\n'
        '            for r in records\n'
        '        ],\n'
        '        "dataset": None\n'
        '        if byod\n'
        '        else {\n'
        '            "release": samples.CORPUS_RELEASE,\n'
        '            "revision": samples.CORPUS_COMMIT,\n'
        '            "license": samples.CORPUS_LICENSE,\n'
        '            "bytes": samples.CORPUS_BYTES,\n'
        '        },\n'
        '    }\n'
        '    write_json(root / "outputs/sample_manifest.json", manifest)\n'
        '    write_json(root / "outputs/config.json", CONFIG)\n'
        '    medians = [\n'
        '        float(np.median(r["depth"][support(r["depth"], r["mask"])])) for r in records if r["role"] == "train"\n'
        '    ]\n'
        '    constant = float(np.median(medians))\n'
        '    baseline_rows = []\n'
        '    for r in records:\n'
        '        if r["role"] == "train":\n'
        '            continue\n'
        '        shape = r["depth"].shape\n'
        '        oracle = float(np.median(r["depth"][support(r["depth"], r["mask"])]))\n'
        '        ramp = np.repeat(np.linspace(0, 1, shape[0], dtype=np.float32)[:, None], shape[1], axis=1)\n'
        '        for label, prediction, aligned in (\n'
        '            ("training_median", np.full(shape, constant), False),\n'
        '            ("oracle_median", np.full(shape, oracle), False),\n'
        '            ("vertical_ramp", ramp, True),\n'
        '        ):\n'
        '            row = {**record_metadata(r), "model": label, **score(prediction, r["depth"], r["mask"], aligned)}\n'
        '            if label == "oracle_median":\n'
        '                row["protocol"] = "reference-assisted oracle median"\n'
        '            baseline_rows.append(row)\n'
        '    write_json(root / "outputs/baselines.json", {"training_constant_metres": constant, "rows": baseline_rows})\n'
        '    write_json(\n'
        '        root / "outputs/prepare.json",\n'
        '        {**result, "private_byod": bool(byod), "training_constant_metres": constant},\n'
        '    )\n'
        '    for domain in sorted({r["domain"] for r in records}):\n'
        '        example = next(r for r in records if r["domain"] == domain)\n'
        '        panel(root, example, f"reference_{example[\'id\']}.png")\n'
        '    print("Prepared", result, flush=True)\n'
        '\n'
        '\n'
        'def device() -> str:\n'
        '    import torch\n'
        '\n'
        '    if not torch.cuda.is_available():\n'
        '        raise RuntimeError("This model stage requires a Colab CUDA GPU; CPU validation never runs models")\n'
        '    torch.cuda.reset_peak_memory_stats()\n'
        '    torch.cuda.synchronize()\n'
        '    return "cuda:0"\n'
        '\n'
        '\n'
        'def predict_rows(root: Path, model: Any, records: list[dict], label: str, aligned: bool) -> list[dict]:\n'
        '    import torch\n'
        '\n'
        '    rows = []\n'
        '    folder = root / "outputs/raw" / label\n'
        '    folder.mkdir(parents=True, exist_ok=True)\n'
        '    for r in records:\n'
        '        torch.cuda.synchronize()\n'
        '        started = time.perf_counter()\n'
        '        prediction = model.predict(r["image"])["depth"]\n'
        '        torch.cuda.synchronize()\n'
        '        row = {\n'
        '            **record_metadata(r),\n'
        '            "model": label,\n'
        '            "seconds": time.perf_counter() - started,\n'
        '            **score(prediction, r["depth"], r["mask"], aligned),\n'
        '        }\n'
        '        np.save(folder / f"{r[\'id\']}.npy", prediction, allow_pickle=False)\n'
        '        rows.append(row)\n'
        '    write_json(\n'
        '        root / "outputs" / f"{label}.json",\n'
        '        {\n'
        '            "rows": rows,\n'
        '            "test": aggregate([r for r in rows if r["role"] == "test"]),\n'
        '            "validation": aggregate([r for r in rows if r["role"] == "validation"]),\n'
        '        },\n'
        '    )\n'
        '    return rows\n'
        '\n'
        '\n'
        'def tensor_hashes(model: Any, names: list[str]) -> dict:\n'
        '    state = model.state_dict()\n'
        '    return {name: array_digest(state[name].detach().cpu().numpy()) for name in names}\n'
        '\n'
        '\n'
        'def depth(root: Path) -> None:\n'
        '    import torch\n'
        '    from depth_reference.pipeline import DepthAnythingPipeline\n'
        '\n'
        '    runtime_device = device()\n'
        '    records = load_records(root)\n'
        '    train = [r for r in records if r["role"] == "train"]\n'
        '    val = [r for r in records if r["role"] == "validation"]\n'
        '    evaluation = [r for r in records if r["role"] != "train"]\n'
        '    model = DepthAnythingPipeline.from_pretrained(\n'
        '        device=runtime_device, weights_dir=root / "weights/depth", allow_download=True\n'
        '    )\n'
        '    model.predict(val[0]["image"])\n'
        '    predict_rows(root, model, evaluation, "depth_frozen", True)\n'
        '    params = dict(model._model.named_parameters())\n'
        '    names = model._trainable_names(0)\n'
        '    initial = {n: params[n].detach().cpu().clone() for n in names}\n'
        '    frozen_names = [n for n in model._model.state_dict() if n not in names]\n'
        '    before_frozen = tensor_hashes(model._model, frozen_names)\n'
        '    updates, observations = [0], []\n'
        '    real_adamw = torch.optim.AdamW\n'
        '\n'
        '    def counted_adamw(parameters: Any, *args: Any, **kwargs: Any) -> Any:\n'
        '        parameters = list(parameters)\n'
        '        if {id(p) for p in parameters} != {id(params[n]) for n in names}:\n'
        '            raise RuntimeError("unexpected trainable tensor set")\n'
        '        optimiser = real_adamw(parameters, *args, **kwargs)\n'
        '\n'
        '        def step_hook(*_: Any) -> None:\n'
        '            updates[0] += 1\n'
        '\n'
        '        optimiser.register_step_post_hook(step_hook)\n'
        '        return optimiser\n'
        '\n'
        '    def progress(entry: dict) -> None:\n'
        '        delta = max(float((params[n].detach().cpu() - initial[n]).abs().max()) for n in names)\n'
        '        observations.append({**entry, "max_weight_delta_before_selection": delta, "actual_steps": updates[0]})\n'
        '        print("Epoch", entry["epoch"], "loss", entry["train_loss"], "validation", entry["val"], flush=True)\n'
        '        write_json(root / "outputs/training_progress.json", observations)\n'
        '\n'
        '    # Strict workshop support replaces only evaluation; source rounding/selection remains unchanged.\n'
        '    def strict_evaluate(rows: list[dict]) -> dict:\n'
        '        metrics = [score(model.predict(r["image"])["depth"], r["depth"], r["mask"], True) for r in rows]\n'
        '        return aggregate(metrics)\n'
        '\n'
        '    model.evaluate = strict_evaluate\n'
        '    torch.optim.AdamW = counted_adamw\n'
        '    try:\n'
        '        history = model.adapt(\n'
        '            train, val, trainable_blocks=0, head_epochs=3, epochs=0, lr=1e-5, seed=42, progress=progress\n'
        '        )\n'
        '    finally:\n'
        '        torch.optim.AdamW = real_adamw\n'
        '    if (\n'
        '        updates[0] != 3 * len(train)\n'
        '        or len(observations) != 3\n'
        '        or not any(r["max_weight_delta_before_selection"] > 0 for r in observations)\n'
        '    ):\n'
        '        raise RuntimeError("expected real optimizer updates were not observed")\n'
        '    after_frozen = tensor_hashes(model._model, frozen_names)\n'
        '    if before_frozen != after_frozen:\n'
        '        raise RuntimeError("frozen tensors changed")\n'
        '    history.update(\n'
        '        actual_optimizer_steps=updates[0],\n'
        '        epoch_observations=observations,\n'
        '        selected_max_weight_delta=max(\n'
        '            float((params[n].detach().cpu() - initial[n]).abs().max()) for n in names\n'
        '        ),\n'
        '        frozen_tensor_hashes_before=before_frozen,\n'
        '        frozen_tensor_hashes_after=after_frozen,\n'
        '        training_pid=os.getpid(),\n'
        '        trainable_tensor_names=names,\n'
        '        trainable_parameters=sum(params[n].numel() for n in names),\n'
        '        selection_rounding="source rounds to six decimals; earliest tie; epoch zero competes",\n'
        '    )\n'
        '    write_json(root / "outputs/training_history.json", history)\n'
        '    write_json(\n'
        '        root / "outputs/selection.json",\n'
        '        {\n'
        '            "selected_epoch": history["best_epoch"],\n'
        '            "epoch_zero_selected": history["best_epoch"] == 0,\n'
        '            "selection": CONFIG["selection"],\n'
        '        },\n'
        '    )\n'
        '    model.save_artifact(\n'
        '        root / "outputs/adapter", metadata={"workshop": CONFIG, "source": read_json(root / "source.json")}\n'
        '    )\n'
        '    predict_rows(root, model, evaluation, "depth_selected_live", True)\n'
        '    example = evaluation[0]\n'
        '    panel(\n'
        '        root,\n'
        '        example,\n'
        '        "depth_first.png",\n'
        '        {"Selected depth": (np.load(root / f"outputs/raw/depth_selected_live/{example[\'id\']}.npy"), True)},\n'
        '    )\n'
        '\n'
        '\n'
        'def reload(root: Path) -> None:\n'
        '    from depth_reference.pipeline import DepthAnythingPipeline\n'
        '\n'
        '    if read_json(root / "outputs/training_history.json")["training_pid"] == os.getpid():\n'
        '        raise RuntimeError("reload must run in a new process")\n'
        '    model = DepthAnythingPipeline.from_artifact(\n'
        '        root / "outputs/adapter", device=device(), weights_dir=root / "weights/depth"\n'
        '    )\n'
        '    records = [r for r in load_records(root) if r["role"] != "train"]\n'
        '    rows = predict_rows(root, model, records, "depth_reloaded", True)\n'
        '    checks = []\n'
        '    original = read_json(root / "outputs/depth_selected_live.json")\n'
        '    for r in records:\n'
        '        checks.append(\n'
        '            {\n'
        '                "id": r["id"],\n'
        '                **verify_parity(\n'
        '                    np.load(root / f"outputs/raw/depth_selected_live/{r[\'id\']}.npy", allow_pickle=False),\n'
        '                    np.load(root / f"outputs/raw/depth_reloaded/{r[\'id\']}.npy", allow_pickle=False),\n'
        '                ),\n'
        '            }\n'
        '        )\n'
        '    drift = {}\n'
        '    for role in ("validation", "test"):\n'
        '        measured = aggregate([r for r in rows if r["role"] == role])\n'
        '        for key in ("abs_rel", "delta1"):\n'
        '            value = abs(measured[key] - original[role][key])\n'
        '            drift[f"{role}_{key}"] = value\n'
        '            if value > 1e-6:\n'
        '                raise RuntimeError("independently recomputed metric reload parity failed")\n'
        '    write_json(\n'
        '        root / "outputs/reload_parity.json",\n'
        '        {\n'
        '            "passed": True,\n'
        '            "checks": checks,\n'
        '            "metric_drift": drift,\n'
        '            "metric_atol": 1e-6,\n'
        '            "reload_pid": os.getpid(),\n'
        '        },\n'
        '    )\n'
        '\n'
        '\n'
        'def zoe(root: Path) -> None:\n'
        '    from zoe_reference import ZoeDepthMetricPipeline\n'
        '\n'
        '    model = ZoeDepthMetricPipeline.from_pretrained(\n'
        '        device=device(), weights_dir=root / "weights/zoe", allow_download=True\n'
        '    )\n'
        '    records = [r for r in load_records(root) if r["role"] != "train"]\n'
        '    model.predict(records[0]["image"])\n'
        '    predict_rows(root, model, records, "zoe_metrics", False)\n'
        '    example = records[0]\n'
        '    panel(\n'
        '        root,\n'
        '        example,\n'
        '        "zoe_first.png",\n'
        '        {"ZoeDepth": (np.load(root / f"outputs/raw/zoe_metrics/{example[\'id\']}.npy"), False)},\n'
        '    )\n'
        '\n'
        '\n'
        'def paired(before: list[dict], after: list[dict]) -> dict:\n'
        '    old = {r["id"]: r for r in before}\n'
        '    rows = [\n'
        '        {\n'
        '            "id": r["id"],\n'
        '            "scene": r.get("scene", "unspecified"),\n'
        '            "domain": r.get("domain", "unspecified"),\n'
        '            "abs_rel_delta": r["abs_rel"] - old[r["id"]]["abs_rel"],\n'
        '            "delta1_delta": r["delta1"] - old[r["id"]]["delta1"],\n'
        '        }\n'
        '        for r in after\n'
        '    ]\n'
        '    return {\n'
        '        "rows": rows,\n'
        '        "improved": sum(r["abs_rel_delta"] < -1e-12 for r in rows),\n'
        '        "unchanged": sum(abs(r["abs_rel_delta"]) <= 1e-12 for r in rows),\n'
        '        "worsened": sum(r["abs_rel_delta"] > 1e-12 for r in rows),\n'
        '        "per_scene": {\n'
        '            scene: {\n'
        '                "n": sum(r["scene"] == scene for r in rows),\n'
        '                "mean_abs_rel_delta": float(\n'
        '                    np.mean([r["abs_rel_delta"] for r in rows if r["scene"] == scene])\n'
        '                ),\n'
        '            }\n'
        '            for scene in sorted({r["scene"] for r in rows})\n'
        '        },\n'
        '    }\n'
        '\n'
        '\n'
        'def activity(root: Path) -> None:\n'
        '    from depth_reference.pipeline import DepthAnythingPipeline\n'
        '\n'
        '    model = DepthAnythingPipeline.from_artifact(\n'
        '        root / "outputs/adapter", device=device(), weights_dir=root / "weights/depth"\n'
        '    )\n'
        '    records = [r for r in load_records(root) if r["role"] == "validation"]\n'
        '    all_rows = []\n'
        '    for factor in (0.6, 1.0, 1.4):\n'
        '        changed, metadata = [], {}\n'
        '        for r in records:\n'
        '            image, info = brightness(r["image"], factor)\n'
        '            metadata[r["id"]] = info\n'
        '            changed.append({**r, "image": image})\n'
        '        rows = predict_rows(root, model, changed, f"brightness_{factor}", True)\n'
        '        all_rows.extend([{**r, **metadata[r["id"]]} for r in rows])\n'
        '    base = [r for r in all_rows if r["factor"] == 1.0]\n'
        '    comparisons = {str(f): paired(base, [r for r in all_rows if r["factor"] == f]) for f in (0.6, 1.4)}\n'
        '    write_json(\n'
        '        root / "outputs/brightness_experiment.json",\n'
        '        {\n'
        '            "rows": all_rows,\n'
        '            "comparisons": comparisons,\n'
        '            "role": "reused validation; descriptive activity only",\n'
        '            "selection_changed": False,\n'
        '        },\n'
        '    )\n'
        '    example = records[0]\n'
        '    panel(\n'
        '        root,\n'
        '        example,\n'
        '        "activity_first.png",\n'
        '        {\n'
        '            f"factor {f}": (np.load(root / f"outputs/raw/brightness_{f}/{example[\'id\']}.npy"), True)\n'
        '            for f in (0.6, 1.0, 1.4)\n'
        '        },\n'
        '    )\n'
        '\n'
        '\n'
        'def select_figures(test: list[dict], change_rows: list[dict]) -> dict:\n'
        '    """Fixed figure rule: first test record, lowest/highest paired AbsRel change, first record per domain.\n'
        '\n'
        '    When adaptation changed no test image (epoch 0 selected), the extremes are only the lowest/highest id\n'
        '    and are labelled that way instead of best/worst.\n'
        '    """\n'
        '    ranked = sorted(change_rows, key=lambda r: (r["abs_rel_delta"], r["id"]))\n'
        '    changed = any(abs(r["abs_rel_delta"]) > 1e-12 for r in change_rows)\n'
        '    selected = {\n'
        '        test[0]["id"]: "first test record",\n'
        '        ranked[0]["id"]: "best adaptation delta" if changed else "no adaptation change; lowest id",\n'
        '        ranked[-1]["id"]: "worst adaptation delta" if changed else "no adaptation change; highest id",\n'
        '    }\n'
        '    for domain in sorted({r["domain"] for r in test}):\n'
        '        selected.setdefault(\n'
        '            next(r["id"] for r in test if r["domain"] == domain), "first test record in domain"\n'
        '        )\n'
        '    return selected\n'
        '\n'
        '\n'
        'def report(root: Path) -> None:\n'
        '    outputs = root / "outputs"\n'
        '    records = load_records(root)\n'
        '    rows = read_json(outputs / "baselines.json")["rows"]\n'
        '    for label in ("depth_frozen", "depth_reloaded", "zoe_metrics"):\n'
        '        rows += read_json(outputs / f"{label}.json")["rows"]\n'
        '    summaries = {\n'
        '        label: aggregate([r for r in rows if r["model"] == label and r["role"] == "test"])\n'
        '        for label in sorted({r["model"] for r in rows})\n'
        '    }\n'
        '    write_json(outputs / "metrics.json", {"rows": rows, "test_equal_image_means": summaries})\n'
        '    fields = [\n'
        '        "id",\n'
        '        "role",\n'
        '        "scene",\n'
        '        "domain",\n'
        '        "model",\n'
        '        "protocol",\n'
        '        "abs_rel",\n'
        '        "delta1",\n'
        '        "n_valid",\n'
        '        "valid_fraction",\n'
        '    ]\n'
        '    with (outputs / "metrics.csv").open("w", newline="", encoding="utf-8") as handle:\n'
        '        writer = csv.DictWriter(handle, fieldnames=fields, extrasaction="ignore")\n'
        '        writer.writeheader()\n'
        '        writer.writerows(rows)\n'
        '    with (outputs / "metrics.csv").open(encoding="utf-8", newline="") as handle:\n'
        '        restored = list(csv.DictReader(handle))\n'
        '    if len(restored) != len(rows) or any(\n'
        '        float(a["abs_rel"]) != b["abs_rel"] for a, b in zip(restored, rows, strict=True)\n'
        '    ):\n'
        '        raise RuntimeError("exported table reload failed")\n'
        '    change = paired(\n'
        '        [r for r in rows if r["model"] == "depth_frozen" and r["role"] == "test"],\n'
        '        [r for r in rows if r["model"] == "depth_reloaded" and r["role"] == "test"],\n'
        '    )\n'
        '    write_json(outputs / "paired_changes.json", change)\n'
        '    sample = read_json(outputs / "sample_manifest.json")\n'
        '    if sample["private_byod"]:\n'
        '        sample_note = (\n'
        '            f"Your test split has {sample[\'counts\'][\'test\']} records in {sample[\'group_counts\'][\'test\']} "\n'
        '            "acquisition groups; overlap with upstream pretraining data is unknown."\n'
        '        )\n'
        '    else:\n'
        '        sample_note = (\n'
        '            "Default test views cover only two scenes; "\n'
        '            "contamination from upstream pretraining is not ruled out."\n'
        '        )\n'
        '    lines = [\n'
        '        "# Depth estimation notebook: measured results",\n'
        '        "",\n'
        '        "Candidate: execution does not confer release approval.",\n'
        '        "",\n'
        '    ]\n'
        '    for heading, labels in (\n'
        '        (\n'
        '            "Reference-assisted affine alignment / oracle baseline",\n'
        '            ["depth_frozen", "depth_reloaded", "vertical_ramp", "oracle_median"],\n'
        '        ),\n'
        '        ("Unaligned metric metres", ["zoe_metrics", "training_median"]),\n'
        '    ):\n'
        '        lines += [\n'
        '            "## " + heading,\n'
        '            "",\n'
        '            "| Method | Views | AbsRel (lower) | Delta-1 (higher) |",\n'
        '            "|---|---:|---:|---:|",\n'
        '        ]\n'
        '        for label in labels:\n'
        '            row = summaries[label]\n'
        '            lines.append(f"| {label} | {row[\'n\']} | {row[\'abs_rel\']:.6f} | {row[\'delta1\']:.6f} |")\n'
        '        lines.append("")\n'
        '    lines += [\n'
        '        "Equal-image means; see metrics.json for scene/domain means. No cross-protocol ranking.",\n'
        '        "Reference alignment uses test references and is not a deployable metre calibration.",\n'
        '        sample_note,\n'
        '        f"Adaptation: {change[\'improved\']} improved, {change[\'unchanged\']} unchanged, "\n'
        '        f"{change[\'worsened\']} worsened (AbsRel).",\n'
        '    ]\n'
        '    (outputs / "summary.md").write_text("\\n".join(lines) + "\\n", encoding="utf-8")\n'
        '    test = [r for r in records if r["role"] == "test"]\n'
        '    selected = select_figures(test, change["rows"])\n'
        '    write_json(outputs / "figure_selection.json", selected)\n'
        '    for rid in selected:\n'
        '        r = next(r for r in test if r["id"] == rid)\n'
        '        panel(\n'
        '            root,\n'
        '            r,\n'
        '            f"figures/{rid}.png",\n'
        '            {\n'
        '                label: (\n'
        '                    np.load(outputs / f"raw/{label}/{rid}.npy", allow_pickle=False),\n'
        '                    label != "zoe_metrics",\n'
        '                )\n'
        '                for label in ("depth_frozen", "depth_reloaded", "zoe_metrics")\n'
        '            },\n'
        '        )\n'
        '    private = read_json(outputs / "sample_manifest.json")["private_byod"]\n'
        '    provenance = {\n'
        '        "python": sys.version,\n'
        '        "packages": {\n'
        '            name: importlib.metadata.version(name)\n'
        '            for name in (\n'
        '                "numpy",\n'
        '                "Pillow",\n'
        '                "torch",\n'
        '                "torchvision",\n'
        '                "transformers",\n'
        '                "safetensors",\n'
        '                "huggingface-hub",\n'
        '            )\n'
        '        },\n'
        '        "source": read_json(root / "source.json"),\n'
        '        "worker_sha256": digest(Path(__file__)),\n'
        '        "lock_sha256": digest(root / "requirements.txt"),\n'
        '        "model_manifests": {\n'
        '            name: read_json(root / f"weights/{name}/dimer-base-manifest.json") for name in ("depth", "zoe")\n'
        '        },\n'
        '        "stage_receipts_snapshot_before_report_completion": read_json(root / "stage_receipts.json"),\n'
        '        "strict_prediction_support": CONFIG["prediction_preflight"],\n'
        '    }\n'
        '    write_json(outputs / "provenance.json", provenance)\n'
        '    write_json(\n'
        '        outputs / "run_summary.json",\n'
        '        {\n'
        '            "execution": "required model stages completed and parity passed",\n'
        '            "qualification": (\n'
        '                "Candidate; fresh hosted run and representative labelled BYOD require maintainer review"\n'
        '            ),\n'
        '            "selected_epoch": read_json(outputs / "selection.json")["selected_epoch"],\n'
        '            "actual_updates": read_json(outputs / "training_history.json")["actual_optimizer_steps"],\n'
        '            "protocols": ["reference-assisted affine alignment", "unaligned metric metres"],\n'
        '            "private_byod": private,\n'
        '        },\n'
        '    )\n'
        '    allow = [\n'
        '        "sample_manifest.json",\n'
        '        "config.json",\n'
        '        "prepare.json",\n'
        '        "baselines.json",\n'
        '        "training_history.json",\n'
        '        "selection.json",\n'
        '        "reload_parity.json",\n'
        '        "brightness_experiment.json",\n'
        '        "paired_changes.json",\n'
        '        "metrics.json",\n'
        '        "metrics.csv",\n'
        '        "summary.md",\n'
        '        "figure_selection.json",\n'
        '        "run_summary.json",\n'
        '        "provenance.json",\n'
        '        "adapter/manifest.json",\n'
        '        "adapter/adapter.safetensors",\n'
        '    ]\n'
        '    if not private:\n'
        '        allow += [str(p.relative_to(outputs)).replace("\\\\", "/") for p in (outputs / "figures").glob("*.png")]\n'
        '    checksums = {name: digest(outputs / name) for name in allow}\n'
        '    write_json(outputs / "checksums.json", checksums)\n'
        '    archive = outputs / "depth_results.zip"\n'
        '    with zipfile.ZipFile(archive, "w", zipfile.ZIP_DEFLATED) as bundle:\n'
        '        for name in [*allow, "checksums.json"]:\n'
        '            bundle.write(outputs / name, name)\n'
        '    with zipfile.ZipFile(archive) as bundle:\n'
        '        for name, checksum in checksums.items():\n'
        '            if hashlib.sha256(bundle.read(name)).hexdigest() != checksum:\n'
        '                raise RuntimeError("ZIP checksum mismatch")\n'
        '    # Raw arrays remain separate from displays/shareable ZIP; reread every array for export integrity.\n'
        '    raw_checks = {}\n'
        '    for path in (outputs / "raw").glob("*/*.npy"):\n'
        '        array = np.load(path, allow_pickle=False)\n'
        '        if array.ndim != 2 or not np.isfinite(array).all():\n'
        '            raise RuntimeError("raw array export invalid")\n'
        '        raw_checks[str(path.relative_to(outputs))] = digest(path)\n'
        '    write_json(outputs / "raw_checksums.json", raw_checks)\n'
        '    print((outputs / "summary.md").read_text(encoding="utf-8"), flush=True)\n'
        '\n'
        '\n'
        'def stage_signature(root: Path) -> str:\n'
        '    paths = [root / "outputs/sample_manifest.json", root / "outputs/config.json", root / "data/records.json"]\n'
        '    paths += [root / "source.json", root / "requirements.txt", Path(__file__).resolve()]\n'
        '    paths += sorted((root / "weights").glob("*/dimer-base-manifest.json"))\n'
        '    paths += sorted((root / "data").glob("*.npy")) + sorted((root / "data").glob("*.png"))\n'
        '    return hashlib.sha256("".join(digest(p) for p in paths).encode()).hexdigest()\n'
        '\n'
        '\n'
        'def begin_stage(root: Path, stage: str) -> dict:\n'
        '    path = root / "stage_receipts.json"\n'
        '    state = read_json(path) if path.exists() else {"run_id": uuid.uuid4().hex, "stages": {}}\n'
        '    if stage == "prepare":\n'
        '        state = {"run_id": uuid.uuid4().hex, "stages": {}}\n'
        '    else:\n'
        '        for required in REQUIRES[stage]:\n'
        '            if state["stages"].get(required, {}).get("status") != "complete":\n'
        '                raise RuntimeError(f"missing current prerequisite {required}")\n'
        '            products = state["stages"][required].get("product_sha256", {})\n'
        '            for relative, expected in products.items():\n'
        '                product_path = root / "outputs" / relative\n'
        '                if not product_path.is_file() or digest(product_path) != expected:\n'
        '                    raise RuntimeError(f"prerequisite output changed: {relative}; rerun {required}")\n'
        '        if state.get("data_signature") != stage_signature(root):\n'
        '            raise RuntimeError("prepared data/config changed; rerun prepare")\n'
        '        descendants = {stage}\n'
        '        for _ in STAGES:\n'
        '            descendants.update(s for s, parents in REQUIRES.items() if any(p in descendants for p in parents))\n'
        '        for item in descendants:\n'
        '            state["stages"].pop(item, None)\n'
        '    state["stages"][stage] = {"status": "running", "pid": os.getpid()}\n'
        '    write_json(path, state)\n'
        '    return state\n'
        '\n'
        '\n'
        'def finish_stage(root: Path, stage: str, state: dict, elapsed: float) -> None:\n'
        '    receipt = {"status": "complete", "pid": os.getpid(), "seconds": elapsed}\n'
        '    if stage == "prepare":\n'
        '        state["data_signature"] = stage_signature(root)\n'
        '    if "torch" in sys.modules and sys.modules["torch"].cuda.is_available():\n'
        '        torch = sys.modules["torch"]\n'
        '        torch.cuda.synchronize()\n'
        '        receipt.update(\n'
        '            gpu=torch.cuda.get_device_name(0),\n'
        '            peak_gpu_bytes=torch.cuda.max_memory_allocated(),\n'
        '            precision="float32",\n'
        '        )\n'
        '    try:\n'
        '        import resource\n'
        '\n'
        '        receipt["peak_host_rss_bytes"] = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss * 1024\n'
        '    except ImportError:\n'
        '        receipt["peak_host_rss_bytes"] = None\n'
        '    products = {\n'
        '        "prepare": ["sample_manifest.json", "config.json", "prepare.json", "baselines.json"],\n'
        '        "depth": [\n'
        '            "depth_frozen.json",\n'
        '            "depth_selected_live.json",\n'
        '            "training_history.json",\n'
        '            "selection.json",\n'
        '            "adapter/*",\n'
        '            "raw/depth_frozen/*.npy",\n'
        '            "raw/depth_selected_live/*.npy",\n'
        '        ],\n'
        '        "reload": ["depth_reloaded.json", "reload_parity.json", "raw/depth_reloaded/*.npy"],\n'
        '        "zoe": ["zoe_metrics.json", "raw/zoe_metrics/*.npy"],\n'
        '        "activity": ["brightness_experiment.json", "brightness_*.json", "raw/brightness_*/*.npy"],\n'
        '        "report": ["metrics.json", "metrics.csv", "summary.md", "run_summary.json", "depth_results.zip"],\n'
        '    }\n'
        '    outputs = root / "outputs"\n'
        '    receipt["product_sha256"] = {\n'
        '        path.relative_to(outputs).as_posix(): digest(path)\n'
        '        for pattern in products[stage]\n'
        '        for path in outputs.glob(pattern)\n'
        '        if path.is_file()\n'
        '    }\n'
        '    state["stages"][stage] = receipt\n'
        '    write_json(root / "stage_receipts.json", state)\n'
        '\n'
        '\n'
        'def byod_run(root: Path, directory: str) -> None:\n'
        '    load_byod(Path(directory))  # Refuse malformed data before any child model stage.\n'
        '    child = root / "byod_runs" / uuid.uuid4().hex\n'
        '    child.mkdir(parents=True)\n'
        '    (child / "weights").symlink_to((root / "weights").resolve(), target_is_directory=True)\n'
        '    for name in ("source.json", "requirements.txt"):\n'
        '        (child / name).write_bytes((root / name).read_bytes())\n'
        '    for stage in STAGES:\n'
        '        command = [sys.executable, str(Path(__file__).resolve()), "--root", str(child), "--stage", stage]\n'
        '        if stage == "prepare":\n'
        '            command += ["--byod", directory]\n'
        '        subprocess.run(command, check=True)\n'
        '    write_json(\n'
        '        root / "byod_output.json", {"root": str(child), "archive": str(child / "outputs/depth_results.zip")}\n'
        '    )\n'
        '\n'
        '\n'
        'def unlabelled_preview(image: Image.Image, relative: np.ndarray, metric: np.ndarray, path: Path) -> None:\n'
        '    """RGB | relative inverse depth (display min/max, arbitrary units) | ZoeDepth metres (log 0.6..350)."""\n'
        '\n'
        '    def colour(scaled: np.ndarray) -> Image.Image:\n'
        '        scaled = np.clip(np.nan_to_num(scaled), 0, 1)\n'
        '        rgb = np.stack([scaled, 1 - np.abs(2 * scaled - 1), 1 - scaled], axis=-1)\n'
        '        return Image.fromarray(np.round(rgb * 255).astype(np.uint8))\n'
        '\n'
        '    rel = np.asarray(relative, dtype=np.float64)\n'
        '    low, high = float(rel.min()), float(rel.max())\n'
        '    panels = [\n'
        '        ("RGB input", image.convert("RGB")),\n'
        '        (\n'
        '            "Depth Anything: relative inverse depth\\n(arbitrary units, display min/max; larger = nearer)",\n'
        '            colour((rel - low) / (high - low + 1e-12)),\n'
        '        ),\n'
        '        (\n'
        '            "ZoeDepth: predicted metres\\n(log colour 0.6 to 350 m; blue = far)",\n'
        '            colour(np.log(np.maximum(np.asarray(metric, dtype=np.float64), 0.6) / 0.6) / np.log(350 / 0.6)),\n'
        '        ),\n'
        '    ]\n'
        '    canvas = Image.new("RGB", (420 * len(panels), 350), "white")\n'
        '    draw = ImageDraw.Draw(canvas)\n'
        '    for i, (title, panel_image) in enumerate(panels):\n'
        '        panel_image = panel_image.copy()\n'
        '        panel_image.thumbnail((410, 300))\n'
        '        canvas.paste(panel_image, (i * 420, 40))\n'
        '        draw.text((i * 420 + 4, 4), title, fill="black")\n'
        '    path.parent.mkdir(parents=True, exist_ok=True)\n'
        '    canvas.save(path)\n'
        '\n'
        '\n'
        'def own_activity(root: Path, factor: float) -> None:\n'
        '    """Optional learner-chosen brightness factor on the reused validation views.\n'
        '\n'
        '    Writes only under ``own_activity/<run>/`` and ``own_activity_output.json``; canonical outputs, stage\n'
        '    receipts and the results ZIP are untouched, and nothing here can change model selection.\n'
        '    """\n'
        '    if not 0.2 <= factor <= 2.0:\n'
        '        raise ValueError(f"OWN_BRIGHTNESS_FACTOR {factor} is outside 0.2..2.0; choose a value in that range")\n'
        '    receipts = root / "stage_receipts.json"\n'
        '    stages = read_json(receipts)["stages"] if receipts.is_file() else {}\n'
        '    experiment = root / "outputs/brightness_experiment.json"\n'
        '    if stages.get("activity", {}).get("status") != "complete" or not experiment.is_file():\n'
        '        raise RuntimeError("Run the section 8 brightness activity first (Run all), then choose a factor")\n'
        '    from depth_reference.pipeline import DepthAnythingPipeline\n'
        '\n'
        '    base = [r for r in read_json(experiment)["rows"] if r["factor"] == 1.0]\n'
        '    target = root / "own_activity" / uuid.uuid4().hex\n'
        '    model = DepthAnythingPipeline.from_artifact(\n'
        '        root / "outputs/adapter", device=device(), weights_dir=root / "weights/depth"\n'
        '    )\n'
        '    records = [r for r in load_records(root) if r["role"] == "validation"]\n'
        '    changed, inputs = [], {}\n'
        '    for r in records:\n'
        '        image, info = brightness(r["image"], factor)\n'
        '        inputs[r["id"]] = info\n'
        '        changed.append({**r, "image": image})\n'
        '    rows = predict_rows(target, model, changed, "own_factor", True)\n'
        '    result = {\n'
        '        "root": str(target),\n'
        '        "factor": factor,\n'
        '        "role": "reused validation; descriptive activity only",\n'
        '        "selection_changed": False,\n'
        '        "compared_with": "factor 1.0 rows of outputs/brightness_experiment.json",\n'
        '        "inputs": inputs,\n'
        '        "comparison": paired(base, rows),\n'
        '    }\n'
        '    write_json(target / "own_activity.json", result)\n'
        '    write_json(root / "own_activity_output.json", result)\n'
        '    example = records[0]\n'
        '    panel(\n'
        '        target,\n'
        '        example,\n'
        '        "own_factor.png",\n'
        '        {\n'
        '            "factor 1.0": (np.load(root / f"outputs/raw/brightness_1.0/{example[\'id\']}.npy"), True),\n'
        '            f"factor {factor}": (np.load(target / f"outputs/raw/own_factor/{example[\'id\']}.npy"), True),\n'
        '        },\n'
        '    )\n'
        '\n'
        '\n'
        'def infer(root: Path, image_path: str) -> None:\n'
        '    import gc\n'
        '\n'
        '    if not (root / "outputs/adapter/manifest.json").is_file():\n'
        '        raise RuntimeError(\n'
        '            "No selected adapter yet: complete Run all through section 5 (adaptation) before this cell"\n'
        '        )\n'
        '    image = bounded_image(Path(image_path))\n'
        '    from depth_reference.pipeline import DepthAnythingPipeline, validate_image\n'
        '    from zoe_reference import ZoeDepthMetricPipeline\n'
        '    from zoe_reference import validate_image as validate_zoe\n'
        '\n'
        '    validate_image(image)\n'
        '    validate_zoe(image)\n'
        '    target = root / "unlabelled" / uuid.uuid4().hex\n'
        '    target.mkdir(parents=True)\n'
        '    model = DepthAnythingPipeline.from_artifact(\n'
        '        root / "outputs/adapter", device=device(), weights_dir=root / "weights/depth"\n'
        '    )\n'
        '    relative = model.predict(image)["depth"]\n'
        '    if not np.isfinite(relative).all():\n'
        '        raise ValueError("nonfinite relative prediction")\n'
        '    np.save(target / "relative_inverse_depth.npy", relative, allow_pickle=False)\n'
        '    del model\n'
        '    gc.collect()\n'
        '    import torch\n'
        '\n'
        '    torch.cuda.empty_cache()\n'
        '    model = ZoeDepthMetricPipeline.from_pretrained(\n'
        '        device=device(), weights_dir=root / "weights/zoe", allow_download=True\n'
        '    )\n'
        '    metric = model.predict(image)["depth"]\n'
        '    if not np.isfinite(metric).all() or np.any(metric <= 0):\n'
        '        raise ValueError("invalid metric prediction")\n'
        '    np.save(target / "predicted_metres.npy", metric, allow_pickle=False)\n'
        '    unlabelled_preview(image, relative, metric, target / "preview.png")\n'
        '    result = {\n'
        '        "root": str(target),\n'
        '        "input_sha256": array_digest(np.asarray(image)),\n'
        '        "abs_rel": None,\n'
        '        "delta1": None,\n'
        '        "reason": "no reference; accuracy not measurable",\n'
        '        "alignment": "none",\n'
        '        "outputs": {\n'
        '            "relative_inverse_depth.npy": "relative inverse depth; arbitrary units",\n'
        '            "predicted_metres.npy": "ZoeDepth predicted metres",\n'
        '            "preview.png": "display only: RGB, relative inverse depth (display min/max), metres (log colour)",\n'
        '        },\n'
        '    }\n'
        '    write_json(target / "result.json", result)\n'
        '    write_json(root / "unlabelled_output.json", result)\n'
        '\n'
        '\n'
        'def main() -> None:\n'
        '    parser = argparse.ArgumentParser()\n'
        '    parser.add_argument("--root", type=Path, required=True)\n'
        '    parser.add_argument("--stage", choices=[*STAGES, "byod", "infer", "own_activity"], required=True)\n'
        '    parser.add_argument("--byod")\n'
        '    parser.add_argument("--image")\n'
        '    parser.add_argument("--factor", type=float)\n'
        '    args = parser.parse_args()\n'
        '    root = args.root.resolve()\n'
        '    root.mkdir(parents=True, exist_ok=True)\n'
        '    if args.stage == "own_activity":\n'
        '        if args.factor is None:\n'
        '            parser.error("--factor required")\n'
        '        own_activity(root, args.factor)\n'
        '        return\n'
        '    if args.stage == "byod":\n'
        '        if not args.byod:\n'
        '            parser.error("--byod directory required")\n'
        '        byod_run(root, args.byod)\n'
        '        return\n'
        '    if args.stage == "infer":\n'
        '        if not args.image:\n'
        '            parser.error("--image path required")\n'
        '        infer(root, args.image)\n'
        '        return\n'
        '    state = begin_stage(root, args.stage)\n'
        '    start = time.perf_counter()\n'
        '    try:\n'
        '        if args.stage == "prepare":\n'
        '            prepare(root, args.byod)\n'
        '        else:\n'
        '            globals()[args.stage](root)\n'
        '        finish_stage(root, args.stage, state, time.perf_counter() - start)\n'
        '    except BaseException as error:\n'
        '        state["stages"][args.stage] = {\n'
        '            "status": "failed",\n'
        '            "pid": os.getpid(),\n'
        '            "error_type": type(error).__name__,\n'
        '        }\n'
        '        write_json(root / "stage_receipts.json", state)\n'
        '        raise\n'
        '\n'
        '\n'
        'if __name__ == "__main__":\n'
        '    main()\n'
    ),
    'requirements.txt': (
        '# This file was autogenerated by uv via the following command:\n'
        '#    uv pip compile tools/depth-workshop-requirements.in --python-version 3.12 --python-platform x86_64-manylinux_2_28 --generate-hashes --output-file tools/depth-workshop-requirements.lock\n'
        'certifi==2026.7.22 \\\n'
        '    --hash=sha256:62f22742b58a1a33014a2b6b706588a8d7e2a88ae7bd1a6ebe8c992928483775 \\\n'
        '    --hash=sha256:741e2c3b351ddf169a738da9f2c048608ff7f2c5cc02f1ebc6b118bb090d5d55\n'
        '    # via requests\n'
        'charset-normalizer==3.5.1 \\\n'
        '    --hash=sha256:00668ebb0609751758682eb0b5857e7c35b9f00e84dfdef062e103244ec94d45 \\\n'
        '    --hash=sha256:012a22b88a77ca2e59b98ac5889b0deb604147666032f45e6d6e217634d2550d \\\n'
        '    --hash=sha256:01e93745f7f219b703b60ba7afead36cfc4242782be5af484673fc500df12da5 \\\n'
        '    --hash=sha256:04368edf83514385ffc3e1cfd4546e595f4f1272dd23ba437a93a9cc3741d47b \\\n'
        '    --hash=sha256:0722590aabf9dc6a6c0343d523c05458fa2b5047dbe6302fd526bb570600753f \\\n'
        '    --hash=sha256:07ffd07412fc5d5e84cd8952acf9ff7e4ed7a708e69d1bada19d8ba91711353f \\\n'
        '    --hash=sha256:09a7bba9f739468c8e78c36a75c33768e53cb1959fc638f510454c14683f00d5 \\\n'
        '    --hash=sha256:0b2b1b3fa5670c127b246df1d0c059defd41f689a868a3b9d79df9b1cac42d22 \\\n'
        '    --hash=sha256:0c6dfb5ca6723eeed15aa8e564a014d69fcb8812f94eef11fe3631e0508199f5 \\\n'
        '    --hash=sha256:0d929fc574b4d6fd9e7c0f5c2ede8716a41911923aa7fa5fce38e0818aa4a1ac \\\n'
        '    --hash=sha256:13e3afe97712e8887cd516e960c63f0b93122971e5b5e4b2622fe7701771e838 \\\n'
        '    --hash=sha256:15f024313246a4ed976c60f440bb8d257815513a681d212ff74fd46f7d715a90 \\\n'
        '    --hash=sha256:195ce897c6153c0700078142cf8efe3e6454ca4cf4357499e4078dfd83396626 \\\n'
        '    --hash=sha256:19a3dd5aa73cef1c99687c4fc57db016a9c17104ae1185da88ba566a5d3bebe4 \\\n'
        '    --hash=sha256:1d1c7a53a6c2103925cdd6d7229f8c567379f211c869793df679f2e9f738c369 \\\n'
        '    --hash=sha256:1f5883d77fd409a261abb5dc8ccbe335720d798b1de4abb3b1d47ccbbc76b53b \\\n'
        '    --hash=sha256:21b82d8082f6f5e7f456ef0bd16323d08de1266efbfeb476e64b2a91d1471a4e \\\n'
        '    --hash=sha256:252d099029bcbea642f2a06c4ed5046bdf8b5a8150b64afa5e027e88b106e5ee \\\n'
        '    --hash=sha256:256dd4d85d9e4dc595e2bc983c980e73f62ddeb3165c58b4c3dfe78c5c8548c1 \\\n'
        '    --hash=sha256:26422d45fd13551cf564c58932f7d72b4f58b93b0fcf18c35ba6be12b46bb102 \\\n'
        '    --hash=sha256:2679de311c7946dde5d3b6f44941844133ff5c7cb86099c0061ab1e8901c20a8 \\\n'
        '    --hash=sha256:29880d17a8eb0b5cfdfd8944b468322928059aa35f1f5fa8ff22b149ec0b42f8 \\\n'
        '    --hash=sha256:2bced4061f000f7187254a02ad3433ae17eaf991747ceea2f478422590a5bba9 \\\n'
        '    --hash=sha256:2e9cf9253119d8e5d111f05d71626786fd3d6193817316eab1ca088cdb8593cf \\\n'
        '    --hash=sha256:2f06b7eae9dbe77fe1d644ca244dad508de8d302870a43f3c559b521270938a0 \\\n'
        '    --hash=sha256:2f293479cce755c75f1697e87c409b7ae4c555c7dfecb6e988ad13abba943031 \\\n'
        '    --hash=sha256:329fc3ccb63ad22d867d84c2adea759a64079a37ba4a343433b02c7a2816871e \\\n'
        '    --hash=sha256:343fb4f2821043bd87095f7b08a1a181febc8e36ac64212143bbfd0a0e1bc235 \\\n'
        '    --hash=sha256:3588e376b3ea2eea84976f67273d679f229e24c66dce7b82ae45aef04ff6e072 \\\n'
        '    --hash=sha256:35aea775dc2bd5f54cd84a1cd2696cc3207c479cb9cf0bd346f0d343e4300ddb \\\n'
        '    --hash=sha256:35fe081843b35aad20ffeccec3eeffbe637b15d14f3fb22cc1b59cd8ec17e93c \\\n'
        '    --hash=sha256:36047af20e17097c3bb9476c2b7655f2f7aa51322c0ba58c07695bedf755a950 \\\n'
        '    --hash=sha256:3617ac3cfd8b9888f145ad89dd6e692285834b0201c6074a5eeaad3fd4d668c2 \\\n'
        '    --hash=sha256:366ec70f5547c640d3ce1985722490f23faf4eb5216a7eeba78277490e78dacb \\\n'
        '    --hash=sha256:394fea06235c8543390050ed5f529187074b029fb027213f6c46ac11ab5d950e \\\n'
        '    --hash=sha256:3d27167433c0d5f18dc850f07d0b3816221984fecdc405d6c157a6f0b8f8e9e6 \\\n'
        '    --hash=sha256:3e5e1224c0a6a90e05843e07adfec669edebec17801c67072f51e59561d63c0b \\\n'
        '    --hash=sha256:41876ee62a3dddf48ff1121ad8f0798032aa03f2fd35f21f34a4cab14f18d8d2 \\\n'
        '    --hash=sha256:433c5a81eade63b47e522303bad236f59dba55ea6951746f5558355eeed8c75d \\\n'
        '    --hash=sha256:4582c27e8c889d64811987b5967fbd3ae0c823fe1fd933b543d55ac20bb475fa \\\n'
        '    --hash=sha256:485a0d363cafefcd2538a73c7c838daa2035f09b2c9f9b5e3133f80c6aeb84c2 \\\n'
        '    --hash=sha256:494b70049a4d69aec6e8137c13af4cf8db8c9f9820a1392ac293b0dd2987a818 \\\n'
        '    --hash=sha256:496846868fea80e479324862fa877f02411f2fd0f83b79ccee2607aa68b2a032 \\\n'
        '    --hash=sha256:4abdc5f9ad448c1ecbfae2974b820535d6bc6e7eef63babbab3d81cf46968c71 \\\n'
        '    --hash=sha256:4b599739b93b2cbeded49645ae3c8d1405c29ddfbceac1545c87a3f9580a9e96 \\\n'
        '    --hash=sha256:4bea7f8ebe90bbd7f0e4a2de42ca6924ba23e3e76418c408ff82f1d46fabd687 \\\n'
        '    --hash=sha256:4c4fb141a727957c93edfe5c32a26ceb6b5f6461d67146e2d39f51e16170bea8 \\\n'
        '    --hash=sha256:4c9548dc78002099910abaebc0a72ac58b7d30931869e0351c09b507dff4ece3 \\\n'
        '    --hash=sha256:4d26f14f041e83dd8edfd61f4cd4fa7285d31798b5bf1f28e70c367ba6c41d61 \\\n'
        '    --hash=sha256:4f298bdadb8f0b9e5672877f647d1be9373ef5320c9e2f049795e26cad28b6a9 \\\n'
        '    --hash=sha256:52ec005752a56ae79547a05c0139ca2501a0c866390b6115008456b9f0e7cde1 \\\n'
        '    --hash=sha256:55261ac0d2941c42f196dd576f543d87a8ee03cd6f5e30dfb4d807b2e3b9121a \\\n'
        '    --hash=sha256:56490c595a28b1bb27dfc583e816152a9767721ef58b2c03b13f954d2f707420 \\\n'
        '    --hash=sha256:58d3e12c88e0950bca850ae1f7c256055c097639c2edb9eb123af9807d8b15e4 \\\n'
        '    --hash=sha256:58d4aa13a59c969dbfdf9e6a9560e242cbfd9e8a8f50c2747714df1a423adf65 \\\n'
        '    --hash=sha256:59171c6e45bf07d0d5cab3b0bf81d945035530f6873398b3b531c31184d46663 \\\n'
        '    --hash=sha256:5b6d1386bf0096d26d3a863dc0a487a5b4eb9aa93cf5ba69683d29dde6b9d60f \\\n'
        '    --hash=sha256:5c0ea61a470e070686aa30892fed79e297d2c8d0ab46b8bcdf027d38c51da591 \\\n'
        '    --hash=sha256:5c84bec0ab5ae0c64bfe73a7d2adcb5ce73b467523fc27fd6a28ab2aa6cbe35a \\\n'
        '    --hash=sha256:5ca0555312ae2fe82715cada7fac375530c2f3349e1eaa1bcb33d0283ac79a18 \\\n'
        '    --hash=sha256:5d8531a6569d025f68e2321e7638fb7978f23db58e5f69f56913837aae03816e \\\n'
        '    --hash=sha256:5e2d0e146dcb57034f8b97dc58d2d512cb90aba253960ce449f695fec6a82c6f \\\n'
        '    --hash=sha256:5fc45d653ea8c9a20479167e11d4a0f8cb2fa3470737ab6f9c827532313187b7 \\\n'
        '    --hash=sha256:6117b84ea48435e5356dc737f5121485c30920ba43375fa7b434fd753df0eac3 \\\n'
        '    --hash=sha256:6199d5606e2bbf2b096cf64d03f8b6790c91081d5ac866b8e7bb6422738cc60c \\\n'
        '    --hash=sha256:62b55f6722735a6c472f88361cde6640608773d9443cebdbb51abf436a1fcdd3 \\\n'
        '    --hash=sha256:687c9ca3035544b113bea2055e180af96fb63c0c476e22a9180f51925186e7b7 \\\n'
        '    --hash=sha256:6b7430cf5728e68f6c462254009a6ef4086e1bea43cf2f57aa9c55fb4f50ff96 \\\n'
        '    --hash=sha256:6ba32c4d2abf1d2fe7cf27d280f4cca5664233b0f885549c7761719eb977f486 \\\n'
        '    --hash=sha256:6c9cdde8becb25a7fde49924511aa2644d6f8081cc8df8e9452724303348d8e3 \\\n'
        '    --hash=sha256:6df0ec430f9a831772c23ca5a224cba36517a58a84bb32c32bb59a9fa67c47f6 \\\n'
        '    --hash=sha256:6e2912d4babbc65196ac13c2f53468dc57fb8b9c25ef913e8c59ddf7c6dc0e1b \\\n'
        '    --hash=sha256:6e5e4d73d588ca5ed09df1b7dcd1b203d1df3c542e3f50d126c947d432b10731 \\\n'
        '    --hash=sha256:70055ff39b97c99e7ae40ea3e393fb62aa2e44dbd9b29f8d14f42fb0025c3959 \\\n'
        '    --hash=sha256:706bfd38730a5ac7a365793269a00f4e988178cec121391f4248d84ad8c972e9 \\\n'
        '    --hash=sha256:7235dc28fc6dd9d832ac7c7bce95367dedb85929f17368a0c2bee1e080b9acbf \\\n'
        '    --hash=sha256:774d157f112367ff4abd29019f38f023c24e00e56edc7829c20e358a5a913ad8 \\\n'
        '    --hash=sha256:77efcff2b23071c349402ac1066667a3d011f62398d81408c9b88ad991747c9e \\\n'
        '    --hash=sha256:789b8982559ae28dad2356519f841655756cdcd96616410590ae0b17454ee64f \\\n'
        '    --hash=sha256:7ac76cf9afd34929d76eb7fcb63be476a4853d8a96f0dcf2d0db68a0cbdf9885 \\\n'
        '    --hash=sha256:7c0c10730342b0c9b35dd1d619beb8214e520bd96a1f870f452680b238aab3e0 \\\n'
        '    --hash=sha256:823f82903d189af463d7df250ef1f7f696f3cee08cc8d91deb565e8d425f6506 \\\n'
        '    --hash=sha256:838648accb3a7fd9803fd45c87bce8509648eb0c11bc34e216141300977244f2 \\\n'
        '    --hash=sha256:854066be00447fa8de2ccbbe893e2ffc4b123ef16d897af794c1e18bd4a714b0 \\\n'
        '    --hash=sha256:85d5855daafc240cc045c026d7a15fd198a09b0fc8ff6f5ecbb5297b509cb11e \\\n'
        '    --hash=sha256:85de3134b5379856e323ba37c19c9256d39425f7b76a63af52b09fb4664c2e8f \\\n'
        '    --hash=sha256:87e4f41d375c0b9be2fb5251aee4b8a689169e134535aed81bf085c3b647451e \\\n'
        '    --hash=sha256:88ca277405c2d3b71c4e1c2ee0e7966e807bcba86a69d11e19ba199d18ae4491 \\\n'
        '    --hash=sha256:88e85ab89cb822c1e635f51d6d32e488f94e002e70e2f492bdb8b945543f345a \\\n'
        '    --hash=sha256:8ac8c94b6539074e0f40899301273ac8402b9b3e01c7b7ba269ff30340aaaf20 \\\n'
        '    --hash=sha256:8fe532b3c966d1fb794e0698e4589d0444017ae77fc0b31edea13c0e35bcc449 \\\n'
        '    --hash=sha256:9085f87b0e38a2b92b8923059b4e8789fe40d9279712d15dcc670048d77079af \\\n'
        '    --hash=sha256:90b7481fb62fbe172c558bc6fd1c4c98d82004a54a7551f20e11ac9bf0b8708c \\\n'
        '    --hash=sha256:92caef967d287a407085d61176fce4012b1dd62daed4eb6d5ceb26d3d2538712 \\\n'
        '    --hash=sha256:9362dd90aa7dab48c0054a21187791ccf05473f7dba5d92b8033ae62164675e7 \\\n'
        '    --hash=sha256:94d78ecec2605a8d0398b0f365d5f12a63248438516f5dac536a5eff7337df4a \\\n'
        '    --hash=sha256:94fbf1c0c6cc0d3d5e50f9a9313a8cdca90dd696d34b381cd1704f8c9e939f20 \\\n'
        '    --hash=sha256:950f23cb393f85543777b0433f082cddd25b51ab398eac7971146495679efe5f \\\n'
        '    --hash=sha256:96eefc178f8636b9c760c5829345307fd81cfae9ab1e80997dbddeb0f54ee9a3 \\\n'
        '    --hash=sha256:96fef3e886d6a9874b14f27fc193fbdc69d5d8035783d86aa4e1cea594e695f9 \\\n'
        '    --hash=sha256:977cdbd483a9cff38179bea4fd754289a6f2195c7abd414aba85410b3e66cc5e \\\n'
        '    --hash=sha256:978eab16f55b4ab2c2a745be9a0a840bf8f09a7f227d9c76eb30214d078865a5 \\\n'
        '    --hash=sha256:994e883d17c559cdfd38c84003c8b27d25424a1077272a17e7cd27bfe0bf57b2 \\\n'
        '    --hash=sha256:9ac4444d8d4fd4c4bd08bf451ed3167aa9e7ec6cdb41b648794f1d1103652e36 \\\n'
        '    --hash=sha256:9b5db6052055d34d41230fb78d7c439c23dc536a9896f6cb039e8dd92cfc1263 \\\n'
        '    --hash=sha256:9d9a0dc7cbe9bec24c3f767c9122c41fe5a1bc43f47cd099d00d393e09769de4 \\\n'
        '    --hash=sha256:9dbdd9205662134957cf0c324f639bdc5031c0ca056e2369e238db75187c0f11 \\\n'
        '    --hash=sha256:9eea3ab2597a5e65fe65296e2d6a84570845a6b55532d90333d740d48bbc850a \\\n'
        '    --hash=sha256:a2028475ba855475b8b4d3cfeb4994269c967aea8b9892dfba907f4263a863a3 \\\n'
        '    --hash=sha256:a3a370082ce34d0612f421e15fe011c53bb1feff21a26d06ad4fb244dab5a375 \\\n'
        '    --hash=sha256:a545775cfe815855ea32d7c27731d79da358ef2055b4a25830231b1622dd18aa \\\n'
        '    --hash=sha256:a5cbd90ecf0fc62e64726917ad083b73001f0563657a87ec3c0b504e277dc90d \\\n'
        '    --hash=sha256:a6d095662e73e74f0a49988e0593373e243e3a52e27bfeea0a859e88acf4a0f5 \\\n'
        '    --hash=sha256:a6dac12ff6b846103483683f60c5f8fee205121adc58ffd87e90a90a3af69e99 \\\n'
        '    --hash=sha256:a951ad59cad9145664a730d3036b40b844e74d2d3683da40111463cd3a83845d \\\n'
        '    --hash=sha256:aa1099b956fb795e686d073568f6dc002a0bb89765ea6d5b055dd7d9bf1b116c \\\n'
        '    --hash=sha256:aa2bb0b37202dca27175591f761108b5d34096ade1191ffe4808bdf6b1571488 \\\n'
        '    --hash=sha256:aae2ee51122d3ae968a3837d97dc24a0aeebb0dea23694422cd172bd30017cd6 \\\n'
        '    --hash=sha256:ab743e9bc90c1f73552ec33e10e3331315acd2c397b36065b591b0181de533cc \\\n'
        '    --hash=sha256:ac00177c4831ffa650f8609e4bdddd5fe09c03b1c0c47acece7e6ea20421598b \\\n'
        '    --hash=sha256:ac13b004224fb341e1e25a1ed5e19d32f57cdb2a403e01f003b46f051a550f6f \\\n'
        '    --hash=sha256:acaf604462bf330b0d07e7a07c1d6e4adac79e5fb13e9c5140590542cafacc00 \\\n'
        '    --hash=sha256:ae31a1a1db2ee6cc2942fccaf695c934bc7f3db9f2133a3fef1f367cf1a4ab10 \\\n'
        '    --hash=sha256:ae4a097991662cd4fff0ddc74e0fe7874f82e00042fa0ea00855645ed0c79598 \\\n'
        '    --hash=sha256:aea996a6aba25260827c9ea511d1addfde2da9eb686ac961838509086188b7e6 \\\n'
        '    --hash=sha256:b39b69b347e5e47a3b5b8cfc005c68c1ba347474e3960236c4944a8ecd174962 \\\n'
        '    --hash=sha256:b54e7e13267d49ffbfe68e25b3cbd774dab38fa37238f71265e91b36146eb21c \\\n'
        '    --hash=sha256:b9af956078716df40d985fb0dfeb2c2120c5ca92ba4ff4b388acfd01cdc14d08 \\\n'
        '    --hash=sha256:ba2f37ee79e6338845261a3c5b1784e5d1acdff2c0785b284f1b633033d136ab \\\n'
        '    --hash=sha256:ba501e667c17d8411f98e67a022d9604ef179aff0e459b7e292c796837c13573 \\\n'
        '    --hash=sha256:baf3775a2635e5a11fbd5e4e64ee69c7e86875d224a5c72aca4c141064589a90 \\\n'
        '    --hash=sha256:bb57753e36e4855b8ca375069482250a6246372331a3e4f3407eaebb007443f5 \\\n'
        '    --hash=sha256:bd6c173f04743d483881bffa1478d5a4624475b8cd1d2194956a75548e191c18 \\\n'
        '    --hash=sha256:be47f99644b208bff7766314013f9acf57b056b04191d570d68ad14022cf5b1d \\\n'
        '    --hash=sha256:c010f5581d9c612804cc59fcf7b524b707fbcb72828551237ab545bb5c7034af \\\n'
        '    --hash=sha256:c1dcc36dcb96abc02236e182d17e0f71430152a6c2c7447421da2d2dc144edea \\\n'
        '    --hash=sha256:c428c6c31eb5f4277d7f8eccaf767fbd548ddd5ce3c8b4f4cbbfab3d96b5904c \\\n'
        '    --hash=sha256:c658c50ac0c98cd755a2dd50b7977d3bca7df401dcc47fbdfa87db53ef7d4e8b \\\n'
        '    --hash=sha256:c71fb0d56c920c269cd3e2e3fe7c610e3f1fdb21a6ce60efa6430ff63676cea6 \\\n'
        '    --hash=sha256:c7b742bf31c88566b4bb6335a7f393bb322e580b6bb98df7bd0c25e6e3519ce8 \\\n'
        '    --hash=sha256:cc0329df4caaceb950d2f580b5ac716a377f7059624a0bafaeaf8a218c6ed774 \\\n'
        '    --hash=sha256:cc5d36d96478aa9c60654bd932525bf32964c62a7281eafdf16d85003a8d6004 \\\n'
        '    --hash=sha256:ce854f5f478050ade5a238731c4ca985a7d3b3cb53ff600a9b5c3b689b5f0a7a \\\n'
        '    --hash=sha256:ced3fdd71aaa83ce593746c2edb42b7a59cb4c19c8b5c407781c72e493aae55a \\\n'
        '    --hash=sha256:cee5dd7c6fb5dd52a0fe2a740f9bc6e3593f5f8b1788bde49de02086f30182b2 \\\n'
        '    --hash=sha256:cfa1c0cc3a8f9f53f1243a5a99ac36fd003880199383b37672e86ddda9cb07e2 \\\n'
        '    --hash=sha256:d1ee1e296209fdce05b81b663250eefa02213a2da7b41bf26f7829b8ba3545aa \\\n'
        '    --hash=sha256:d59b75732e9b6f27388e10c14b0259cc5f2e48c78627d185e6a177b58ad3cffe \\\n'
        '    --hash=sha256:d63600d620ad0064c3a748b950ac5ea38a80190e5498532efefa4b7b3f1da1f3 \\\n'
        '    --hash=sha256:dd732602a7009217f658d5863d12d79d373a4de0eebc111094bcdd3bb8e0a6cc \\\n'
        '    --hash=sha256:e06efa066f7dbadbc84ebc126a97c452a6451dfcf589d89d788484949e1cf795 \\\n'
        '    --hash=sha256:e199fb99720074809a7720f1c0b4d919eea8b87e88713e0f8f602f7bef543d9d \\\n'
        '    --hash=sha256:e4b018dc5a0eee4676e38fe84a47a427816c590b93b55d9025274ec4d6ffc2dc \\\n'
        '    --hash=sha256:e6621fb2a4988d6e53eedc455e5903e2679f3967b8acb3d639f1b63c14a2e893 \\\n'
        '    --hash=sha256:e71c909f353863b2b89c83de2ebed71ea6d0df8a6ef65a128193c5e650766bef \\\n'
        '    --hash=sha256:e90251c0c7bdd54a100a0dce3c07b7e637278c93af29dbf78ebb89a58c4bac7d \\\n'
        '    --hash=sha256:e9fbdce1e47394b09bc9f26ab117dfc8d6491977a11d86f592bb42c779db2fda \\\n'
        '    --hash=sha256:eb12fb2ba69ffa05f8695f61c69e591dc4b4a12ac3757ac8af8adb259bf56d17 \\\n'
        '    --hash=sha256:eda059b6bc8bc0812d626fd91a7ce01bf583df0a61296eff390fd94141a34e30 \\\n'
        '    --hash=sha256:f03ac127268b43ef4fe9e6ab6794a6794b49485a0cc0c1db79876d2f33f75bc7 \\\n'
        '    --hash=sha256:f298e218441525d3794428b4c8b8fb8662c6d3ea79925d4807ee6b9a96a3bca5 \\\n'
        '    --hash=sha256:f5542f9b941279d82d41eb0aa9f98eba36fe4df5c7086c651df7944935b37182 \\\n'
        '    --hash=sha256:f6f7deae3feb4edfa2efaf7c574fe88cbf055038a6abdb40188e4fff66d5699f \\\n'
        '    --hash=sha256:f9b1e28d0e8dbfa858abdba91d6b547beaf2df1a59bec6da6faae7b96a4991a9 \\\n'
        '    --hash=sha256:f9f8405c2c758532c74fed975dbee57be1f31a6e865c031870c79a6ed3212ada \\\n'
        '    --hash=sha256:fa48b1b63d639f9483e0633e092f5851e2348c352f1f9bb6c8182f87884ef876 \\\n'
        '    --hash=sha256:fb78f6e7fcd8ad785d28cd577168bc1aaee827b25bb8755638f694794ea98f0a \\\n'
        '    --hash=sha256:fbc597639158fd7c14d55e808718848319540f51b0e6746e3eefa59723a4a348 \\\n'
        '    --hash=sha256:fce8cbd4997efeb450bd298b54f755dcdff18d496f7a5ddbb4867c6d7c88fdc3 \\\n'
        '    --hash=sha256:fd0350afdc3aabd5576f60ea109228bd5538139713c7b094c5cd27c73a98bc6f \\\n'
        '    --hash=sha256:fd0a274c0e5f9a21565cd9d3dd749b61f96b7aa1e20a93aa1ba4029518f2e5c0 \\\n'
        '    --hash=sha256:fdb8a068947befafba9952162645dc2fecaeb400e64584829ed5e9b2fbe21a7f\n'
        '    # via requests\n'
        'cuda-bindings==13.4.3 \\\n'
        '    --hash=sha256:044c03b056dcc5cecfad426a071187dd9e1e6817fbb363bc2f3a7520170b3e67 \\\n'
        '    --hash=sha256:0deff5b22462bb410859684299b1fc6a621780c43a4729342aa6ec28783485b4 \\\n'
        '    --hash=sha256:130ff1daae550db2cef559ba477f3a63d040756bd135f5deba4b5bdc4e110252 \\\n'
        '    --hash=sha256:145cc9b02dfb7bcc3288701b6f19c69c590e4451c62916ba0a8a3db9a64a91a8 \\\n'
        '    --hash=sha256:1fd7d8459b364aedc11f3e59703453ced823135f78a9111ca70feef8d56d4d21 \\\n'
        '    --hash=sha256:3d7a6506e625be59cdc119ef4423afa0fc3a4830b129dc010cbd51324b93d66d \\\n'
        '    --hash=sha256:4796864ce829bd95ef2ef0d23c6ba21bb64e08f7fab0a377302ed1affb6605c7 \\\n'
        '    --hash=sha256:52d7f3f5f7f014dddddc66cd802ed0ddf65ae99ad42b5619fae7b82e5ddd6771 \\\n'
        '    --hash=sha256:6c6bb1a4c4f7520e062669c6f3605a7016b2e65ba51b8922f92edd91d5fadf8d \\\n'
        '    --hash=sha256:7e11cfe8fec4c85ce79feda18124971c52596f0cbd642a94f5dafc257124a4b3 \\\n'
        '    --hash=sha256:81eef62bbb95cb4a705fb423b2ad1c63d716edb352ee2af0c28bf8a29cc8258c \\\n'
        '    --hash=sha256:a9ea13b9cfe711515ae83b5efc99101af4d3f8ad882f99724a839c8b5417fb7e \\\n'
        '    --hash=sha256:b89d6e738494b7b95c38e3413f86d32c24682c8e714870531a5a2b193a2fc50a \\\n'
        '    --hash=sha256:bbacde6f75665b197016b986164cfdaa33b17515e5e635a63ddb75926aaa71c3 \\\n'
        '    --hash=sha256:bc51990309b416e0780a11a921ac597ef9c0e52e1bdf5ae0b8e8c4766b66442b \\\n'
        '    --hash=sha256:bfbd3f7d4ac04dd41dc49121b9e408c8283992f47124c2290ecb79bbbadcca8e \\\n'
        '    --hash=sha256:c2af7e69d2557fdb4e5fc1169159fd08da7e861aa84fb83307a1a0396b6b0973 \\\n'
        '    --hash=sha256:d5f72bcfcdf3be23e1da3c792f68f508586f48d037bca8b10f552c4cca5971f2 \\\n'
        '    --hash=sha256:d6eb969920e28f66f8fc3b0b3afcb6e09381cc96bf8e8158d774e9488ae89980 \\\n'
        '    --hash=sha256:d7c6c9f46fca7f3fc61959ef9a2398ac656172145b43f408e0a6492360cf1c0c \\\n'
        '    --hash=sha256:df8b3767facca8acde216460df684dfc3826d519096c13adfd3030a55870dc79 \\\n'
        '    --hash=sha256:e52f66340785a51b8b77f4487329de188f3cabbf37c62e68f410a8299e8677ae \\\n'
        '    --hash=sha256:f8519603001c92bf83e7095df3b8211e3999a9c4ded096b57de0f3ff52b66368\n'
        '    # via torch\n'
        'cuda-pathfinder==1.8.2 \\\n'
        '    --hash=sha256:4e65059febdb4d19d5cbc4798677e19db2b582f2f702f457b609e571690d357e\n'
        '    # via cuda-bindings\n'
        'cuda-toolkit==13.0.3.0 \\\n'
        '    --hash=sha256:d693caaa261214ddd7dbb60d68e71cbed884e68c2be7509778f3051da0b91c3f\n'
        '    # via torch\n'
        'filelock==4.0.4 \\\n'
        '    --hash=sha256:0df72be195ca7892216d16f2edce8d9b93a571f02402972020a8cff84c594c7b \\\n'
        '    --hash=sha256:90999ed63a26ccf86b93b959ab10cf1017f422d816be454ed54cbed263e71ab5\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   torch\n'
        '    #   transformers\n'
        'fsspec==2026.9.0 \\\n'
        '    --hash=sha256:0f08147951c8cb31d844c3547d631053b127863b60be04cf06e121333ee0e2fe \\\n'
        '    --hash=sha256:8dd6e646e99ea382bd85f97a45e6b526a442d79423a7dc673f1e2756d05fcb5f\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   torch\n'
        'hf-xet==1.6.0 \\\n'
        '    --hash=sha256:0e6e21fa3cdfcdcd76748564bf593870a5e013f47d97cf10aed63aa222cff5b7 \\\n'
        '    --hash=sha256:23379c2f9ec8696d952b16414a2bae72cad86a52df869b050698ba60f538c675 \\\n'
        '    --hash=sha256:2e58454a340b3556dfa4972d5451aff4fba8dd42a236600ba1a1d2b1514f0fef \\\n'
        '    --hash=sha256:35cec30d75c6f9eb9c16a77cef68e85a103b72e24d4b473714ec9ff06428bab9 \\\n'
        '    --hash=sha256:3dc3e35441ba395006af5aaacc40ef2e603c51ef46c3530b9156185f00935ea3 \\\n'
        '    --hash=sha256:4fc74352a17015bd0ee90038bc9efe38db894cde45f268b6712b04fce8cd0acb \\\n'
        '    --hash=sha256:5153e6bb103ad49d6ea9f1b2e230db5a2ea32551ad09a706d2f61d7c7c80d80e \\\n'
        '    --hash=sha256:5789835d7c6bc9436962853192082374297fb72d7eff7e7762ec25ceb7e25338 \\\n'
        '    --hash=sha256:633dc0cd71d32da58ab8c03ad38e2fac452c15c2b0a2866ebf6ededfe0a5061d \\\n'
        '    --hash=sha256:70cbb9c896901600128cb9b6f06e132954fbede1db30f31f7c6c63f84cb7c31d \\\n'
        '    --hash=sha256:75765820ce4700db3750c94acc8fe27c5fae4c9ec000a0dbac3ca082acf97765 \\\n'
        '    --hash=sha256:8fb4f71cba6129110c3374a33f919001ff130488fc23553698e34cc1c2a1198c \\\n'
        '    --hash=sha256:948f15d3a9545cfe5932f6bd8b440f6ae630aee108f14b7bd6c561f7c2dcc522 \\\n'
        '    --hash=sha256:d62671bb130879cef0ee4c9ebe47a14af6c66ec53e6d84dc15936e5ffdfac82f \\\n'
        '    --hash=sha256:f0906082d9932ae0c0057fa194041c22b4e2cdb46b2592ef3b91f020d62a081a \\\n'
        '    --hash=sha256:f2f7278c05c22fd60cb436cda1269649b3e81db65ecdc8496e5e164aa4143e7b \\\n'
        '    --hash=sha256:fb4fadde1b2b70bf4c0c14a6dccbe7194b1c28947fefd5bbe3fed9d940676c3b\n'
        '    # via huggingface-hub\n'
        'huggingface-hub==0.36.2 \\\n'
        '    --hash=sha256:1934304d2fb224f8afa3b87007d58501acfda9215b334eed53072dd5e815ff7a \\\n'
        '    --hash=sha256:48f0c8eac16145dfce371e9d2d7772854a4f591bcb56c9cf548accf531d54270\n'
        '    # via\n'
        '    #   -r tools/depth-workshop-requirements.in\n'
        '    #   tokenizers\n'
        '    #   transformers\n'
        'idna==3.20 \\\n'
        '    --hash=sha256:a7db850025b95ded1eae8a46181a1a6c56c92c96f0e2b005d9ff8dc0210cab44 \\\n'
        '    --hash=sha256:ab7ae7122974553370f0bdb919e1a960b2cd1bc1ef0276416d896db81c14582c\n'
        '    # via requests\n'
        'jinja2==3.1.6 \\\n'
        '    --hash=sha256:0137fb05990d35f1275a587e9aee6d56da821fc83491a0fb838183be43f66d6d \\\n'
        '    --hash=sha256:85ece4451f492d0c13c5dd7c13a64681a86afae63a5f347908daf103ce6d2f67\n'
        '    # via torch\n'
        'markupsafe==3.0.3 \\\n'
        '    --hash=sha256:0303439a41979d9e74d18ff5e2dd8c43ed6c6001fd40e5bf2e43f7bd9bbc523f \\\n'
        '    --hash=sha256:068f375c472b3e7acbe2d5318dea141359e6900156b5b2ba06a30b169086b91a \\\n'
        '    --hash=sha256:0bf2a864d67e76e5c9a34dc26ec616a66b9888e25e7b9460e1c76d3293bd9dbf \\\n'
        '    --hash=sha256:0db14f5dafddbb6d9208827849fad01f1a2609380add406671a26386cdf15a19 \\\n'
        '    --hash=sha256:0eb9ff8191e8498cca014656ae6b8d61f39da5f95b488805da4bb029cccbfbaf \\\n'
        '    --hash=sha256:0f4b68347f8c5eab4a13419215bdfd7f8c9b19f2b25520968adfad23eb0ce60c \\\n'
        '    --hash=sha256:1085e7fbddd3be5f89cc898938f42c0b3c711fdcb37d75221de2666af647c175 \\\n'
        '    --hash=sha256:116bb52f642a37c115f517494ea5feb03889e04df47eeff5b130b1808ce7c219 \\\n'
        '    --hash=sha256:12c63dfb4a98206f045aa9563db46507995f7ef6d83b2f68eda65c307c6829eb \\\n'
        '    --hash=sha256:133a43e73a802c5562be9bbcd03d090aa5a1fe899db609c29e8c8d815c5f6de6 \\\n'
        '    --hash=sha256:1353ef0c1b138e1907ae78e2f6c63ff67501122006b0f9abad68fda5f4ffc6ab \\\n'
        '    --hash=sha256:15d939a21d546304880945ca1ecb8a039db6b4dc49b2c5a400387cdae6a62e26 \\\n'
        '    --hash=sha256:177b5253b2834fe3678cb4a5f0059808258584c559193998be2601324fdeafb1 \\\n'
        '    --hash=sha256:1872df69a4de6aead3491198eaf13810b565bdbeec3ae2dc8780f14458ec73ce \\\n'
        '    --hash=sha256:1b4b79e8ebf6b55351f0d91fe80f893b4743f104bff22e90697db1590e47a218 \\\n'
        '    --hash=sha256:1b52b4fb9df4eb9ae465f8d0c228a00624de2334f216f178a995ccdcf82c4634 \\\n'
        '    --hash=sha256:1ba88449deb3de88bd40044603fafffb7bc2b055d626a330323a9ed736661695 \\\n'
        '    --hash=sha256:1cc7ea17a6824959616c525620e387f6dd30fec8cb44f649e31712db02123dad \\\n'
        '    --hash=sha256:218551f6df4868a8d527e3062d0fb968682fe92054e89978594c28e642c43a73 \\\n'
        '    --hash=sha256:26a5784ded40c9e318cfc2bdb30fe164bdb8665ded9cd64d500a34fb42067b1c \\\n'
        '    --hash=sha256:2713baf880df847f2bece4230d4d094280f4e67b1e813eec43b4c0e144a34ffe \\\n'
        '    --hash=sha256:2a15a08b17dd94c53a1da0438822d70ebcd13f8c3a95abe3a9ef9f11a94830aa \\\n'
        '    --hash=sha256:2f981d352f04553a7171b8e44369f2af4055f888dfb147d55e42d29e29e74559 \\\n'
        '    --hash=sha256:32001d6a8fc98c8cb5c947787c5d08b0a50663d139f1305bac5885d98d9b40fa \\\n'
        '    --hash=sha256:3524b778fe5cfb3452a09d31e7b5adefeea8c5be1d43c4f810ba09f2ceb29d37 \\\n'
        '    --hash=sha256:3537e01efc9d4dccdf77221fb1cb3b8e1a38d5428920e0657ce299b20324d758 \\\n'
        '    --hash=sha256:35add3b638a5d900e807944a078b51922212fb3dedb01633a8defc4b01a3c85f \\\n'
        '    --hash=sha256:38664109c14ffc9e7437e86b4dceb442b0096dfe3541d7864d9cbe1da4cf36c8 \\\n'
        '    --hash=sha256:3a7e8ae81ae39e62a41ec302f972ba6ae23a5c5396c8e60113e9066ef893da0d \\\n'
        '    --hash=sha256:3b562dd9e9ea93f13d53989d23a7e775fdfd1066c33494ff43f5418bc8c58a5c \\\n'
        '    --hash=sha256:457a69a9577064c05a97c41f4e65148652db078a3a509039e64d3467b9e7ef97 \\\n'
        '    --hash=sha256:4bd4cd07944443f5a265608cc6aab442e4f74dff8088b0dfc8238647b8f6ae9a \\\n'
        '    --hash=sha256:4e885a3d1efa2eadc93c894a21770e4bc67899e3543680313b09f139e149ab19 \\\n'
        '    --hash=sha256:4faffd047e07c38848ce017e8725090413cd80cbc23d86e55c587bf979e579c9 \\\n'
        '    --hash=sha256:509fa21c6deb7a7a273d629cf5ec029bc209d1a51178615ddf718f5918992ab9 \\\n'
        '    --hash=sha256:5678211cb9333a6468fb8d8be0305520aa073f50d17f089b5b4b477ea6e67fdc \\\n'
        '    --hash=sha256:591ae9f2a647529ca990bc681daebdd52c8791ff06c2bfa05b65163e28102ef2 \\\n'
        '    --hash=sha256:5a7d5dc5140555cf21a6fefbdbf8723f06fcd2f63ef108f2854de715e4422cb4 \\\n'
        '    --hash=sha256:69c0b73548bc525c8cb9a251cddf1931d1db4d2258e9599c28c07ef3580ef354 \\\n'
        '    --hash=sha256:6b5420a1d9450023228968e7e6a9ce57f65d148ab56d2313fcd589eee96a7a50 \\\n'
        '    --hash=sha256:722695808f4b6457b320fdc131280796bdceb04ab50fe1795cd540799ebe1698 \\\n'
        '    --hash=sha256:729586769a26dbceff69f7a7dbbf59ab6572b99d94576a5592625d5b411576b9 \\\n'
        '    --hash=sha256:77f0643abe7495da77fb436f50f8dab76dbc6e5fd25d39589a0f1fe6548bfa2b \\\n'
        '    --hash=sha256:795e7751525cae078558e679d646ae45574b47ed6e7771863fcc079a6171a0fc \\\n'
        '    --hash=sha256:7be7b61bb172e1ed687f1754f8e7484f1c8019780f6f6b0786e76bb01c2ae115 \\\n'
        '    --hash=sha256:7c3fb7d25180895632e5d3148dbdc29ea38ccb7fd210aa27acbd1201a1902c6e \\\n'
        '    --hash=sha256:7e68f88e5b8799aa49c85cd116c932a1ac15caaa3f5db09087854d218359e485 \\\n'
        '    --hash=sha256:83891d0e9fb81a825d9a6d61e3f07550ca70a076484292a70fde82c4b807286f \\\n'
        '    --hash=sha256:8485f406a96febb5140bfeca44a73e3ce5116b2501ac54fe953e488fb1d03b12 \\\n'
        '    --hash=sha256:8709b08f4a89aa7586de0aadc8da56180242ee0ada3999749b183aa23df95025 \\\n'
        '    --hash=sha256:8f71bc33915be5186016f675cd83a1e08523649b0e33efdb898db577ef5bb009 \\\n'
        '    --hash=sha256:915c04ba3851909ce68ccc2b8e2cd691618c4dc4c4232fb7982bca3f41fd8c3d \\\n'
        '    --hash=sha256:949b8d66bc381ee8b007cd945914c721d9aba8e27f71959d750a46f7c282b20b \\\n'
        '    --hash=sha256:94c6f0bb423f739146aec64595853541634bde58b2135f27f61c1ffd1cd4d16a \\\n'
        '    --hash=sha256:9a1abfdc021a164803f4d485104931fb8f8c1efd55bc6b748d2f5774e78b62c5 \\\n'
        '    --hash=sha256:9b79b7a16f7fedff2495d684f2b59b0457c3b493778c9eed31111be64d58279f \\\n'
        '    --hash=sha256:a320721ab5a1aba0a233739394eb907f8c8da5c98c9181d1161e77a0c8e36f2d \\\n'
        '    --hash=sha256:a4afe79fb3de0b7097d81da19090f4df4f8d3a2b3adaa8764138aac2e44f3af1 \\\n'
        '    --hash=sha256:ad2cf8aa28b8c020ab2fc8287b0f823d0a7d8630784c31e9ee5edea20f406287 \\\n'
        '    --hash=sha256:b8512a91625c9b3da6f127803b166b629725e68af71f8184ae7e7d54686a56d6 \\\n'
        '    --hash=sha256:bc51efed119bc9cfdf792cdeaa4d67e8f6fcccab66ed4bfdd6bde3e59bfcbb2f \\\n'
        '    --hash=sha256:bdc919ead48f234740ad807933cdf545180bfbe9342c2bb451556db2ed958581 \\\n'
        '    --hash=sha256:bdd37121970bfd8be76c5fb069c7751683bdf373db1ed6c010162b2a130248ed \\\n'
        '    --hash=sha256:be8813b57049a7dc738189df53d69395eba14fb99345e0a5994914a3864c8a4b \\\n'
        '    --hash=sha256:c0c0b3ade1c0b13b936d7970b1d37a57acde9199dc2aecc4c336773e1d86049c \\\n'
        '    --hash=sha256:c47a551199eb8eb2121d4f0f15ae0f923d31350ab9280078d1e5f12b249e0026 \\\n'
        '    --hash=sha256:c4ffb7ebf07cfe8931028e3e4c85f0357459a3f9f9490886198848f4fa002ec8 \\\n'
        '    --hash=sha256:ccfcd093f13f0f0b7fdd0f198b90053bf7b2f02a3927a30e63f3ccc9df56b676 \\\n'
        '    --hash=sha256:d2ee202e79d8ed691ceebae8e0486bd9a2cd4794cec4824e1c99b6f5009502f6 \\\n'
        '    --hash=sha256:d53197da72cc091b024dd97249dfc7794d6a56530370992a5e1a08983ad9230e \\\n'
        '    --hash=sha256:d6dd0be5b5b189d31db7cda48b91d7e0a9795f31430b7f271219ab30f1d3ac9d \\\n'
        '    --hash=sha256:d88b440e37a16e651bda4c7c2b930eb586fd15ca7406cb39e211fcff3bf3017d \\\n'
        '    --hash=sha256:de8a88e63464af587c950061a5e6a67d3632e36df62b986892331d4620a35c01 \\\n'
        '    --hash=sha256:df2449253ef108a379b8b5d6b43f4b1a8e81a061d6537becd5582fba5f9196d7 \\\n'
        '    --hash=sha256:e1c1493fb6e50ab01d20a22826e57520f1284df32f2d8601fdd90b6304601419 \\\n'
        '    --hash=sha256:e1cf1972137e83c5d4c136c43ced9ac51d0e124706ee1c8aa8532c1287fa8795 \\\n'
        '    --hash=sha256:e2103a929dfa2fcaf9bb4e7c091983a49c9ac3b19c9061b6d5427dd7d14d81a1 \\\n'
        '    --hash=sha256:e56b7d45a839a697b5eb268c82a71bd8c7f6c94d6fd50c3d577fa39a9f1409f5 \\\n'
        '    --hash=sha256:e8afc3f2ccfa24215f8cb28dcf43f0113ac3c37c2f0f0806d8c70e4228c5cf4d \\\n'
        '    --hash=sha256:e8fc20152abba6b83724d7ff268c249fa196d8259ff481f3b1476383f8f24e42 \\\n'
        '    --hash=sha256:eaa9599de571d72e2daf60164784109f19978b327a3910d3e9de8c97b5b70cfe \\\n'
        '    --hash=sha256:ec15a59cf5af7be74194f7ab02d0f59a62bdcf1a537677ce67a2537c9b87fcda \\\n'
        '    --hash=sha256:f190daf01f13c72eac4efd5c430a8de82489d9cff23c364c3ea822545032993e \\\n'
        '    --hash=sha256:f34c41761022dd093b4b6896d4810782ffbabe30f2d443ff5f083e0cbbb8c737 \\\n'
        '    --hash=sha256:f3e98bb3798ead92273dc0e5fd0f31ade220f59a266ffd8a4f6065e0a3ce0523 \\\n'
        '    --hash=sha256:f42d0984e947b8adf7dd6dde396e720934d12c506ce84eea8476409563607591 \\\n'
        '    --hash=sha256:f71a396b3bf33ecaa1626c255855702aca4d3d9fea5e051b41ac59a9c1c41edc \\\n'
        '    --hash=sha256:f9e130248f4462aaa8e2552d547f36ddadbeaa573879158d721bbd33dfe4743a \\\n'
        '    --hash=sha256:fed51ac40f757d41b7c48425901843666a6677e3e8eb0abcff09e4ba6e664f50\n'
        '    # via jinja2\n'
        'mpmath==1.3.0 \\\n'
        '    --hash=sha256:7a28eb2a9774d00c7bc92411c19a89209d5da7c4c9a9e227be8330a23a25b91f \\\n'
        '    --hash=sha256:a0b2b9fe80bbcd81a6647ff13108738cfb482d481d826cc0e02f5b35e5c88d2c\n'
        '    # via sympy\n'
        'networkx==3.7 \\\n'
        '    --hash=sha256:e3fd2c13a7814cee3746340d8d7f8598a67f16a58bf47fb7f8793fab6efca1b0 \\\n'
        '    --hash=sha256:fd77a511bd90f39f3d016351345b52cf5319b813bdca01de3f755d3cca62e96a\n'
        '    # via torch\n'
        'numpy==2.5.3 \\\n'
        '    --hash=sha256:012e66aca395d795496446e52aeeb5866312a5d4d3f27da270e5a0b43f70dc5c \\\n'
        '    --hash=sha256:09d5a423c71ad5feb5625844ad58050e35df43871004b52ac9c0ad44a56775be \\\n'
        '    --hash=sha256:09ffa5d903faeaa5c4dd05009cf81c8bab9f2cb37c548b8d39b65b4cfa7c97f7 \\\n'
        '    --hash=sha256:0a59a421a32580a009e8a1751345bf829631b990dc1794b80514ab722b435def \\\n'
        '    --hash=sha256:116f96cadd935c6122e9228d676fe7ede19e741f5c8bb1c3cddbe0c51ccebea2 \\\n'
        '    --hash=sha256:1302b90c0e52281681b2975adfe8a860cb7b12216a27b4b0b4207c44bf7bccf0 \\\n'
        '    --hash=sha256:15aa985ac73a8db02db7663381aa109510449d3819d37206caed27b33a65a8a6 \\\n'
        '    --hash=sha256:1aad64d99730d013cfc6debafed22783b4fc5a7f4b8bc744d2d8cf7dcc880551 \\\n'
        '    --hash=sha256:1c80eabb4035ecf4ca9cd49cde8a9fdd69a729e63e6474887d1523ade7aa277f \\\n'
        '    --hash=sha256:1f3ed25271581281f2fccb1adcedfcde4c07362eec69189b50baf6f90e3ae159 \\\n'
        '    --hash=sha256:1fb6f8fb9ff0b3a69f52c66ce397b0246583e9f28616231b0e32ca49259a5fa6 \\\n'
        '    --hash=sha256:214045a5bf00113a146ab9ee9730c44501af6723cdf1f6830932f7b5ef2e7af0 \\\n'
        '    --hash=sha256:26e15e4aecd8617dfbaecb37d223e365d7b39411fba20454be2670a96aa74cb5 \\\n'
        '    --hash=sha256:2c25dfa72943e4336ddb6b0ee4277b47a0c85bede0807530ec68103bf58e2c10 \\\n'
        '    --hash=sha256:2d8240cb4c16fd831074aa2b2cf9fc54664d826341d61c372245b96a74a49a9a \\\n'
        '    --hash=sha256:350ba9783ce969cf9f7ce6e6a9a58e1a6e2a19ca025b7ee448c4db727706212a \\\n'
        '    --hash=sha256:4c8a6d2ebce6305fd82fbefca827775437147052a976ee7c94b36a0c1b52ac6c \\\n'
        '    --hash=sha256:4f8929ee6c96bfbd7b4ed2032e0c03af86fe1826740ab61ddabf9072d06e57ff \\\n'
        '    --hash=sha256:536f963710a4e63934d80ac0dc4f478804a83e9a84b6828018f25d09953ada33 \\\n'
        '    --hash=sha256:54a115e5a73b8fc44f0cebef486365a1894b5c9760685d4558b72b7c3eb846e0 \\\n'
        '    --hash=sha256:595d020938c84e320bcf40ad71089e108eac0d377cd018e14a8c094f39e98d85 \\\n'
        '    --hash=sha256:66a78fe4556c60aceda5916f9eacd638b18e9e681016ec302dcb4682d6d4d034 \\\n'
        '    --hash=sha256:6b05c171afb3aa07adbd20abc00aea86fe375beb0fdb9ef780ec5b7f63bab1c0 \\\n'
        '    --hash=sha256:6cef4bb1706dfec49243c05d921eefb4e190d41e2528b30d8035ea1f36b4c24a \\\n'
        '    --hash=sha256:6f24021b9f22bc6301c37b196974a92c1c18dccedb6fef3dd252e95f2d6adbe4 \\\n'
        '    --hash=sha256:71b39d9f935b6ec0f8753e3e2afb51e3efba6f2e05b68b32a40754d24bcd4a3c \\\n'
        '    --hash=sha256:71cad2b2a7451ab79d8f5e71b453485b6775963d5cf794179144a7463fe6e8ec \\\n'
        '    --hash=sha256:76c2c1e6bfa5c84adc6434dfbf013aa92096a7985221762c8f11fedfd20fff58 \\\n'
        '    --hash=sha256:8617bbfae4486cf99c9f899966699428d19da931d06ca94ad3da986c76e15997 \\\n'
        '    --hash=sha256:86bff898a431c0fb71f7610b75726e75a54d47b37edc9d537f48de63bb3c0b90 \\\n'
        '    --hash=sha256:8e4dd766076855b5ff7ea52fa5f07ce26286726e0f8bff446b7739d02e6ea204 \\\n'
        '    --hash=sha256:92f30e89b8ee0ecf363033576c422b2f58fed6a80bed0aa48dff6d14c654663e \\\n'
        '    --hash=sha256:93e1f5447e2b1e479d7bd74701e84746b86450cff1fc368b132d195e2b8f8211 \\\n'
        '    --hash=sha256:9a37475425b431b4d060f23b4f52cd2f3aef6bc7c654bd760adf0040eec9d435 \\\n'
        '    --hash=sha256:9deb49575e5b0b94ed72c8a64ec4d033381adc27e9060ae842971f697ba96104 \\\n'
        '    --hash=sha256:a5fa86b80fd24bcd1aff83ad23be44ea323de3f787be8f8b15d4a65621e25321 \\\n'
        '    --hash=sha256:a6391fafaba97500887132cd582abc6e19452b1ac775a47caa7b24490e152058 \\\n'
        '    --hash=sha256:a72f874bc9e10e4b8f80426fb49716d5141f64442a0c8418065093ec8017fbb0 \\\n'
        '    --hash=sha256:ac7bb1c52d445bd4f8f7f97fefe6abc3a084dc4d63df50d79b17fa2b78e89297 \\\n'
        '    --hash=sha256:adc1ada2662f8a5f960b8a10d9986897e7499ef07e06d4cfe7197f8cce923c07 \\\n'
        '    --hash=sha256:b00eefbcf0f292945c4b4dec2ae845389ef5bcdcd596e6e4328051db5b5ba694 \\\n'
        '    --hash=sha256:b0521d0f4aebb6e06189451025fa17a913287b13c03d5fe05c017333b654ea5b \\\n'
        '    --hash=sha256:b5d93cf48f687479941d12b69c873ad2cc76bbd487f0091c2200636497f34034 \\\n'
        '    --hash=sha256:b7e18c623bb5c95acb3b3328861272816ba199fb531921c5d6d0b675f1fde9e3 \\\n'
        '    --hash=sha256:bd4cb9ad3c7889b9b3fe0a9a9fb5d2ed26f9879bff2608d9f01aed147a20d231 \\\n'
        '    --hash=sha256:be5a8381859b6da607c84f4f7d6847725f1cf1853ef8a2c9e115b7d58bef47dc \\\n'
        '    --hash=sha256:befa1ae5bd6030b3f512b43ff3fa5290bbed6b84411a44244b14adf835f5b89d \\\n'
        '    --hash=sha256:bf63afbe037eb5d2fe87fbcc7778e61da53ebaf21d938a4515aa73b62532a5d4 \\\n'
        '    --hash=sha256:c00abe94c1a69d75d827dcf1c025b25c8a45d230b3bcd77a9020883a1b047653 \\\n'
        '    --hash=sha256:c2381f82999704f818e2c987a865050e285ec3621262c66d40f5a96c8f899f8e \\\n'
        '    --hash=sha256:c76d5dde9f445058f83d0c02af00557a4db91de9a9a57c0df87d1535001d654b \\\n'
        '    --hash=sha256:cb189f09db39283b26bfd061ec16189e14f71c6755207f72a0f7540867afe5b9 \\\n'
        '    --hash=sha256:ccb32e0525d29e8b0572eb84c9a57af0e7a4e615726927506f55063c62414034 \\\n'
        '    --hash=sha256:ccbc4665079665c3cf3bab4db9f6b095370cd6437d66be549b6c2a1fd19e1958 \\\n'
        '    --hash=sha256:d1c89973648c85069c5046ad460f7b8a00218b29a2e42359ac8cc63e9ab94832 \\\n'
        '    --hash=sha256:df2d5874ff183595a4ba404edd04f6bd9b5505c1d7708573f6a6c17489a67563 \\\n'
        '    --hash=sha256:e01c918ac3d48e18a927cf7b14a26a3e29ff2bdf2eacb976da0aecd6a43ed034 \\\n'
        '    --hash=sha256:e6ab667ba76450084eb64013762c438ea76d9d29cc676dcd6c2e9892ba37f841 \\\n'
        '    --hash=sha256:e931e4f499e0dc7ef29d269a8e5b35dd722e5d14be07df6240166ea7c6532fae \\\n'
        '    --hash=sha256:f54660b0eb6b0b9f36e7fe1cdfdff472028dd0d14acd9b9b65098efbad059469 \\\n'
        '    --hash=sha256:f59a878c33d6b88122d80d239bb3b845d58708750b0cb06a09aebb9b18ec696c \\\n'
        '    --hash=sha256:f7fabeb6cea87d65f3b926de33d03fb016cfdc29314c90974383b5582ae72891 \\\n'
        '    --hash=sha256:f9579f383d1bf9df80081e72760e84960a7fd4f88cf0c9e535a8597c9bb646f5 \\\n'
        '    --hash=sha256:f9a2353b37a1a9e78fd82b27ad7e2a32a2d036604d18f02b05e3136c62ca3b09 \\\n'
        '    --hash=sha256:fc36dc566135b5eceec4cf89758fcb719266a019ef07dae1754ae7c9f617ef3e \\\n'
        '    --hash=sha256:ffdc76bfcae6b255dff75202c5e7feaf95b40246bc0a17944facc1fecf9f79ab\n'
        '    # via\n'
        '    #   -r tools/depth-workshop-requirements.in\n'
        '    #   torchvision\n'
        '    #   transformers\n'
        'nvidia-cublas==13.1.1.3 \\\n'
        '    --hash=sha256:37936a16db8fe4ac1f065c2139360608a543a09275cb1a1af612e08cfa065436 \\\n'
        '    --hash=sha256:b6cdce694e47ff6aadf0a69df1cab6628d696f5ff56e8d16af50309d855fa20f \\\n'
        '    --hash=sha256:b7a210458267ac818974c53038fbec2e969d5c99f305ab15c72522fa9f001dd5\n'
        '    # via\n'
        '    #   cuda-toolkit\n'
        '    #   nvidia-cudnn-cu13\n'
        '    #   nvidia-cusolver\n'
        'nvidia-cuda-cupti==13.0.85 \\\n'
        '    --hash=sha256:4eb01c08e859bf924d222250d2e8f8b8ff6d3db4721288cf35d14252a4d933c8 \\\n'
        '    --hash=sha256:683f58d301548deeefcb8f6fac1b8d907691b9d8b18eccab417f51e362102f00 \\\n'
        '    --hash=sha256:796bd679890ee55fb14a94629b698b6db54bcfd833d391d5e94017dd9d7d3151\n'
        '    # via cuda-toolkit\n'
        'nvidia-cuda-nvrtc==13.0.88 \\\n'
        '    --hash=sha256:6bcd4e7f8e205cbe644f5a98f2f799bef9556fefc89dd786e79a16312ce49872 \\\n'
        '    --hash=sha256:ad9b6d2ead2435f11cbb6868809d2adeeee302e9bb94bcf0539c7a40d80e8575 \\\n'
        '    --hash=sha256:d27f20a0ca67a4bb34268a5e951033496c5b74870b868bacd046b1b8e0c3267b\n'
        '    # via\n'
        '    #   cuda-toolkit\n'
        '    #   nvidia-cublas\n'
        'nvidia-cuda-runtime==13.0.96 \\\n'
        '    --hash=sha256:7f82250d7782aa23b6cfe765ecc7db554bd3c2870c43f3d1821f1d18aebf0548 \\\n'
        '    --hash=sha256:ef9bcbe90493a2b9d810e43d249adb3d02e98dd30200d86607d8d02687c43f55 \\\n'
        '    --hash=sha256:f79298c8a098cec150a597c8eba58ecdab96e3bdc4b9bc4f9983635031740492\n'
        '    # via cuda-toolkit\n'
        'nvidia-cudnn-cu13==9.24.0.43 \\\n'
        '    --hash=sha256:67a7273b5cf062f9446fd76cf464351a1c0f66501e6cd78f6675c0d604d8ac87 \\\n'
        '    --hash=sha256:71f181cd810e90f9b6023b01186fe82d13d65f0ec098581ee201d39fad769e4b \\\n'
        '    --hash=sha256:a6812a554a1ff0413e9c52b84c26c050380649ab9615f9c16bded368ce9f421f\n'
        '    # via torch\n'
        'nvidia-cufft==12.0.0.61 \\\n'
        '    --hash=sha256:2708c852ef8cd89d1d2068bdbece0aa188813a0c934db3779b9b1faa8442e5f5 \\\n'
        '    --hash=sha256:2abce5b39d2f5ae12730fb7e5db6696533e36c26e2d3e8fd1750bdd2853364eb \\\n'
        '    --hash=sha256:6c44f692dce8fd5ffd3e3df134b6cdb9c2f72d99cf40b62c32dde45eea9ddad3\n'
        '    # via cuda-toolkit\n'
        'nvidia-cufile==1.15.1.6 \\\n'
        '    --hash=sha256:08a3ecefae5a01c7f5117351c64f17c7c62efa5fffdbe24fc7d298da19cd0b44 \\\n'
        '    --hash=sha256:bdc0deedc61f548bddf7733bdc216456c2fdb101d020e1ab4b88d232d5e2f6d1\n'
        '    # via cuda-toolkit\n'
        'nvidia-curand==10.4.0.35 \\\n'
        '    --hash=sha256:133df5a7509c3e292aaa2b477afd0194f06ce4ea24d714d616ff36439cee349a \\\n'
        '    --hash=sha256:1aee33a5da6e1db083fe2b90082def8915f30f3248d5896bcec36a579d941bfc \\\n'
        '    --hash=sha256:65b1710aa6961d326b411e314b374290904c5ddf41dc3f766ebc3f1d7d4ca69f\n'
        '    # via cuda-toolkit\n'
        'nvidia-cusolver==12.0.4.66 \\\n'
        '    --hash=sha256:02c2457eaa9e39de20f880f4bd8820e6a1cfb9f9a34f820eb12a155aa5bc92d2 \\\n'
        '    --hash=sha256:0a759da5dea5c0ea10fd307de75cdeb59e7ea4fcb8add0924859b944babf1112 \\\n'
        '    --hash=sha256:16515bd33a8e76bb54d024cfa068fa68d30e80fc34b9e1090813ea9362e0cb65\n'
        '    # via cuda-toolkit\n'
        'nvidia-cusparse==12.6.3.3 \\\n'
        '    --hash=sha256:2b3c89c88d01ee0e477cb7f82ef60a11a4bcd57b6b87c33f789350b59759360b \\\n'
        '    --hash=sha256:80bcc4662f23f1054ee334a15c72b8940402975e0eab63178fc7e670aa59472c \\\n'
        '    --hash=sha256:cbcf42feb737bd7ec15b4c0a63e62351886bd3f975027b8815d7f720a2b5ea79\n'
        '    # via\n'
        '    #   cuda-toolkit\n'
        '    #   nvidia-cusolver\n'
        'nvidia-cusparselt-cu13==0.8.1 \\\n'
        '    --hash=sha256:4dca476c50bf4780d46cd0bfbd82e2bc10a08e4fef7950917ce8d7578d22a23f \\\n'
        '    --hash=sha256:786ce87568c303fadb5afcc7102d454cd3040d75f6f8626f5db460d1871f4dd0 \\\n'
        '    --hash=sha256:dccbd362f91a7b9024d1f55ee9f548ac065027ff15d8c8b0db889ab3a8f31215\n'
        '    # via torch\n'
        'nvidia-nccl-cu13==2.30.7 \\\n'
        '    --hash=sha256:ca786ffa5a647c75d4d1f5cc72a6c4f537947e2ba8823d7c8aaf768e7a7b9f77 \\\n'
        '    --hash=sha256:cefa7fdb9710efd0f39c5f1be1d61ff6fc9a996c451265bd7fbdcf9455ed4b50\n'
        '    # via torch\n'
        'nvidia-nvjitlink==13.4.92 \\\n'
        '    --hash=sha256:25f74fad0d654271c921ac4dca614bd6258bc21791242fc7b2289dad7ae9c099 \\\n'
        '    --hash=sha256:9e4a7ff4f0cafa8c624917055b863dc11f5c2912ead23c462889e166f3b0e57d \\\n'
        '    --hash=sha256:b286f3a4f227a9363efdec263c7b91788cef1478d2b8a5fa8bab7f3e82ff82fd \\\n'
        '    --hash=sha256:e0391f24ed94ec879b84e3da4d4ec320c879aff681f2c7a638462f7199284323\n'
        '    # via\n'
        '    #   cuda-toolkit\n'
        '    #   nvidia-cufft\n'
        '    #   nvidia-cusolver\n'
        '    #   nvidia-cusparse\n'
        'nvidia-nvshmem-cu13==3.4.5 \\\n'
        '    --hash=sha256:290f0a2ee94c9f3687a02502f3b9299a9f9fe826e6d0287ee18482e78d495b80 \\\n'
        '    --hash=sha256:6dc2a197f38e5d0376ad52cd1a2a3617d3cdc150fd5966f4aee9bcebb1d68fe9\n'
        '    # via torch\n'
        'nvidia-nvtx==13.0.85 \\\n'
        '    --hash=sha256:4936d1d6780fbe68db454f5e72a42ff64d1fd6397df9f363ae786930fd5c1cd4 \\\n'
        '    --hash=sha256:cb7780edb6b14107373c835bf8b72e7a178bac7367e23da7acb108f973f157a6 \\\n'
        '    --hash=sha256:d66ea44254dd3c6eacc300047af6e1288d2269dd072b417e0adffbf479e18519\n'
        '    # via cuda-toolkit\n'
        'packaging==26.3 \\\n'
        '    --hash=sha256:94edc256424af38762eb31306eed28beb9f0efc50a8837492c9d6fd6004aed79 \\\n'
        '    --hash=sha256:d7193f7c8e4e93f444fde0262bf90af30e16fa0ad0ad44cb553c87339b23cd1c\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'pillow==11.3.0 \\\n'
        '    --hash=sha256:023f6d2d11784a465f09fd09a34b150ea4672e85fb3d05931d89f373ab14abb2 \\\n'
        '    --hash=sha256:02a723e6bf909e7cea0dac1b0e0310be9d7650cd66222a5f1c571455c0a45214 \\\n'
        '    --hash=sha256:040a5b691b0713e1f6cbe222e0f4f74cd233421e105850ae3b3c0ceda520f42e \\\n'
        '    --hash=sha256:05f6ecbeff5005399bb48d198f098a9b4b6bdf27b8487c7f38ca16eeb070cd59 \\\n'
        '    --hash=sha256:068d9c39a2d1b358eb9f245ce7ab1b5c3246c7c8c7d9ba58cfa5b43146c06e50 \\\n'
        '    --hash=sha256:0743841cabd3dba6a83f38a92672cccbd69af56e3e91777b0ee7f4dba4385632 \\\n'
        '    --hash=sha256:092c80c76635f5ecb10f3f83d76716165c96f5229addbd1ec2bdbbda7d496e06 \\\n'
        '    --hash=sha256:0b275ff9b04df7b640c59ec5a3cb113eefd3795a8df80bac69646ef699c6981a \\\n'
        '    --hash=sha256:0bce5c4fd0921f99d2e858dc4d4d64193407e1b99478bc5cacecba2311abde51 \\\n'
        '    --hash=sha256:1019b04af07fc0163e2810167918cb5add8d74674b6267616021ab558dc98ced \\\n'
        '    --hash=sha256:106064daa23a745510dabce1d84f29137a37224831d88eb4ce94bb187b1d7e5f \\\n'
        '    --hash=sha256:118ca10c0d60b06d006be10a501fd6bbdfef559251ed31b794668ed569c87e12 \\\n'
        '    --hash=sha256:13f87d581e71d9189ab21fe0efb5a23e9f28552d5be6979e84001d3b8505abe8 \\\n'
        '    --hash=sha256:155658efb5e044669c08896c0c44231c5e9abcaadbc5cd3648df2f7c0b96b9a6 \\\n'
        '    --hash=sha256:1904e1264881f682f02b7f8167935cce37bc97db457f8e7849dc3a6a52b99580 \\\n'
        '    --hash=sha256:19d2ff547c75b8e3ff46f4d9ef969a06c30ab2d4263a9e287733aa8b2429ce8f \\\n'
        '    --hash=sha256:1a992e86b0dd7aeb1f053cd506508c0999d710a8f07b4c791c63843fc6a807ac \\\n'
        '    --hash=sha256:1b9c17fd4ace828b3003dfd1e30bff24863e0eb59b535e8f80194d9cc7ecf860 \\\n'
        '    --hash=sha256:1c627742b539bba4309df89171356fcb3cc5a9178355b2727d1b74a6cf155fbd \\\n'
        '    --hash=sha256:1cd110edf822773368b396281a2293aeb91c90a2db00d78ea43e7e861631b722 \\\n'
        '    --hash=sha256:1f85acb69adf2aaee8b7da124efebbdb959a104db34d3a2cb0f3793dbae422a8 \\\n'
        '    --hash=sha256:23cff760a9049c502721bdb743a7cb3e03365fafcdfc2ef9784610714166e5a4 \\\n'
        '    --hash=sha256:2465a69cf967b8b49ee1b96d76718cd98c4e925414ead59fdf75cf0fd07df673 \\\n'
        '    --hash=sha256:2a3117c06b8fb646639dce83694f2f9eac405472713fcb1ae887469c0d4f6788 \\\n'
        '    --hash=sha256:2aceea54f957dd4448264f9bf40875da0415c83eb85f55069d89c0ed436e3542 \\\n'
        '    --hash=sha256:2d6fcc902a24ac74495df63faad1884282239265c6839a0a6416d33faedfae7e \\\n'
        '    --hash=sha256:30807c931ff7c095620fe04448e2c2fc673fcbb1ffe2a7da3fb39613489b1ddd \\\n'
        '    --hash=sha256:30b7c02f3899d10f13d7a48163c8969e4e653f8b43416d23d13d1bbfdc93b9f8 \\\n'
        '    --hash=sha256:3828ee7586cd0b2091b6209e5ad53e20d0649bbe87164a459d0676e035e8f523 \\\n'
        '    --hash=sha256:3cee80663f29e3843b68199b9d6f4f54bd1d4a6b59bdd91bceefc51238bcb967 \\\n'
        '    --hash=sha256:3e184b2f26ff146363dd07bde8b711833d7b0202e27d13540bfe2e35a323a809 \\\n'
        '    --hash=sha256:41342b64afeba938edb034d122b2dda5db2139b9a4af999729ba8818e0056477 \\\n'
        '    --hash=sha256:41742638139424703b4d01665b807c6468e23e699e8e90cffefe291c5832b027 \\\n'
        '    --hash=sha256:4445fa62e15936a028672fd48c4c11a66d641d2c05726c7ec1f8ba6a572036ae \\\n'
        '    --hash=sha256:45dfc51ac5975b938e9809451c51734124e73b04d0f0ac621649821a63852e7b \\\n'
        '    --hash=sha256:465b9e8844e3c3519a983d58b80be3f668e2a7a5db97f2784e7079fbc9f9822c \\\n'
        '    --hash=sha256:48d254f8a4c776de343051023eb61ffe818299eeac478da55227d96e241de53f \\\n'
        '    --hash=sha256:4c834a3921375c48ee6b9624061076bc0a32a60b5532b322cc0ea64e639dd50e \\\n'
        '    --hash=sha256:4c96f993ab8c98460cd0c001447bff6194403e8b1d7e149ade5f00594918128b \\\n'
        '    --hash=sha256:504b6f59505f08ae014f724b6207ff6222662aab5cc9542577fb084ed0676ac7 \\\n'
        '    --hash=sha256:527b37216b6ac3a12d7838dc3bd75208ec57c1c6d11ef01902266a5a0c14fc27 \\\n'
        '    --hash=sha256:5418b53c0d59b3824d05e029669efa023bbef0f3e92e75ec8428f3799487f361 \\\n'
        '    --hash=sha256:59a03cdf019efbfeeed910bf79c7c93255c3d54bc45898ac2a4140071b02b4ae \\\n'
        '    --hash=sha256:5e05688ccef30ea69b9317a9ead994b93975104a677a36a8ed8106be9260aa6d \\\n'
        '    --hash=sha256:6359a3bc43f57d5b375d1ad54a0074318a0844d11b76abccf478c37c986d3cfc \\\n'
        '    --hash=sha256:643f189248837533073c405ec2f0bb250ba54598cf80e8c1e043381a60632f58 \\\n'
        '    --hash=sha256:65dc69160114cdd0ca0f35cb434633c75e8e7fad4cf855177a05bf38678f73ad \\\n'
        '    --hash=sha256:67172f2944ebba3d4a7b54f2e95c786a3a50c21b88456329314caaa28cda70f6 \\\n'
        '    --hash=sha256:676b2815362456b5b3216b4fd5bd89d362100dc6f4945154ff172e206a22c024 \\\n'
        '    --hash=sha256:6a418691000f2a418c9135a7cf0d797c1bb7d9a485e61fe8e7722845b95ef978 \\\n'
        '    --hash=sha256:6abdbfd3aea42be05702a8dd98832329c167ee84400a1d1f61ab11437f1717eb \\\n'
        '    --hash=sha256:6be31e3fc9a621e071bc17bb7de63b85cbe0bfae91bb0363c893cbe67247780d \\\n'
        '    --hash=sha256:7107195ddc914f656c7fc8e4a5e1c25f32e9236ea3ea860f257b0436011fddd0 \\\n'
        '    --hash=sha256:71f511f6b3b91dd543282477be45a033e4845a40278fa8dcdbfdb07109bf18f9 \\\n'
        '    --hash=sha256:7859a4cc7c9295f5838015d8cc0a9c215b77e43d07a25e460f35cf516df8626f \\\n'
        '    --hash=sha256:7966e38dcd0fa11ca390aed7c6f20454443581d758242023cf36fcb319b1a874 \\\n'
        '    --hash=sha256:79ea0d14d3ebad43ec77ad5272e6ff9bba5b679ef73375ea760261207fa8e0aa \\\n'
        '    --hash=sha256:7aee118e30a4cf54fdd873bd3a29de51e29105ab11f9aad8c32123f58c8f8081 \\\n'
        '    --hash=sha256:7b161756381f0918e05e7cb8a371fff367e807770f8fe92ecb20d905d0e1c149 \\\n'
        '    --hash=sha256:7c8ec7a017ad1bd562f93dbd8505763e688d388cde6e4a010ae1486916e713e6 \\\n'
        '    --hash=sha256:7d1aa4de119a0ecac0a34a9c8bde33f34022e2e8f99104e47a3ca392fd60e37d \\\n'
        '    --hash=sha256:7db51d222548ccfd274e4572fdbf3e810a5e66b00608862f947b163e613b67dd \\\n'
        '    --hash=sha256:819931d25e57b513242859ce1876c58c59dc31587847bf74cfe06b2e0cb22d2f \\\n'
        '    --hash=sha256:83e1b0161c9d148125083a35c1c5a89db5b7054834fd4387499e06552035236c \\\n'
        '    --hash=sha256:857844335c95bea93fb39e0fa2726b4d9d758850b34075a7e3ff4f4fa3aa3b31 \\\n'
        '    --hash=sha256:8797edc41f3e8536ae4b10897ee2f637235c94f27404cac7297f7b607dd0716e \\\n'
        '    --hash=sha256:8924748b688aa210d79883357d102cd64690e56b923a186f35a82cbc10f997db \\\n'
        '    --hash=sha256:89bd777bc6624fe4115e9fac3352c79ed60f3bb18651420635f26e643e3dd1f6 \\\n'
        '    --hash=sha256:8dc70ca24c110503e16918a658b869019126ecfe03109b754c402daff12b3d9f \\\n'
        '    --hash=sha256:91da1d88226663594e3f6b4b8c3c8d85bd504117d043740a8e0ec449087cc494 \\\n'
        '    --hash=sha256:921bd305b10e82b4d1f5e802b6850677f965d8394203d182f078873851dada69 \\\n'
        '    --hash=sha256:932c754c2d51ad2b2271fd01c3d121daaa35e27efae2a616f77bf164bc0b3e94 \\\n'
        '    --hash=sha256:93efb0b4de7e340d99057415c749175e24c8864302369e05914682ba642e5d77 \\\n'
        '    --hash=sha256:97afb3a00b65cc0804d1c7abddbf090a81eaac02768af58cbdcaaa0a931e0b6d \\\n'
        '    --hash=sha256:97f07ed9f56a3b9b5f49d3661dc9607484e85c67e27f3e8be2c7d28ca032fec7 \\\n'
        '    --hash=sha256:98a9afa7b9007c67ed84c57c9e0ad86a6000da96eaa638e4f8abe5b65ff83f0a \\\n'
        '    --hash=sha256:9ab6ae226de48019caa8074894544af5b53a117ccb9d3b3dcb2871464c829438 \\\n'
        '    --hash=sha256:9c412fddd1b77a75aa904615ebaa6001f169b26fd467b4be93aded278266b288 \\\n'
        '    --hash=sha256:a1bc6ba083b145187f648b667e05a2534ecc4b9f2784c2cbe3089e44868f2b9b \\\n'
        '    --hash=sha256:a418486160228f64dd9e9efcd132679b7a02a5f22c982c78b6fc7dab3fefb635 \\\n'
        '    --hash=sha256:a4d336baed65d50d37b88ca5b60c0fa9d81e3a87d4a7930d3880d1624d5b31f3 \\\n'
        '    --hash=sha256:a6444696fce635783440b7f7a9fc24b3ad10a9ea3f0ab66c5905be1c19ccf17d \\\n'
        '    --hash=sha256:a7bc6e6fd0395bc052f16b1a8670859964dbd7003bd0af2ff08342eb6e442cfe \\\n'
        '    --hash=sha256:b4b8f3efc8d530a1544e5962bd6b403d5f7fe8b9e08227c6b255f98ad82b4ba0 \\\n'
        '    --hash=sha256:b5f56c3f344f2ccaf0dd875d3e180f631dc60a51b314295a3e681fe8cf851fbe \\\n'
        '    --hash=sha256:be5463ac478b623b9dd3937afd7fb7ab3d79dd290a28e2b6df292dc75063eb8a \\\n'
        '    --hash=sha256:c37d8ba9411d6003bba9e518db0db0c58a680ab9fe5179f040b0463644bc9805 \\\n'
        '    --hash=sha256:c84d689db21a1c397d001aa08241044aa2069e7587b398c8cc63020390b1c1b8 \\\n'
        '    --hash=sha256:c96d333dcf42d01f47b37e0979b6bd73ec91eae18614864622d9b87bbd5bbf36 \\\n'
        '    --hash=sha256:cadc9e0ea0a2431124cde7e1697106471fc4c1da01530e679b2391c37d3fbb3a \\\n'
        '    --hash=sha256:cc3e831b563b3114baac7ec2ee86819eb03caa1a2cef0b481a5675b59c4fe23b \\\n'
        '    --hash=sha256:cd8ff254faf15591e724dc7c4ddb6bf4793efcbe13802a4ae3e863cd300b493e \\\n'
        '    --hash=sha256:d000f46e2917c705e9fb93a3606ee4a819d1e3aa7a9b442f6444f07e77cf5e25 \\\n'
        '    --hash=sha256:d9da3df5f9ea2a89b81bb6087177fb1f4d1c7146d583a3fe5c672c0d94e55e12 \\\n'
        '    --hash=sha256:e5c5858ad8ec655450a7c7df532e9842cf8df7cc349df7225c60d5d348c8aada \\\n'
        '    --hash=sha256:e67d793d180c9df62f1f40aee3accca4829d3794c95098887edc18af4b8b780c \\\n'
        '    --hash=sha256:ea944117a7974ae78059fcc1800e5d3295172bb97035c0c1d9345fca1419da71 \\\n'
        '    --hash=sha256:eb76541cba2f958032d79d143b98a3a6b3ea87f0959bbe256c0b5e416599fd5d \\\n'
        '    --hash=sha256:ec1ee50470b0d050984394423d96325b744d55c701a439d2bd66089bff963d3c \\\n'
        '    --hash=sha256:ee92f2fd10f4adc4b43d07ec5e779932b4eb3dbfbc34790ada5a6669bc095aa6 \\\n'
        '    --hash=sha256:f0f5d8f4a08090c6d6d578351a2b91acf519a54986c055af27e7a93feae6d3f1 \\\n'
        '    --hash=sha256:f1f182ebd2303acf8c380a54f615ec883322593320a9b00438eb842c1f37ae50 \\\n'
        '    --hash=sha256:f8a5827f84d973d8636e9dc5764af4f0cf2318d26744b3d902931701b0d46653 \\\n'
        '    --hash=sha256:f944255db153ebb2b19c51fe85dd99ef0ce494123f21b9db4877ffdfc5590c7c \\\n'
        '    --hash=sha256:fdae223722da47b024b867c1ea0be64e0df702c5e0a60e27daad39bf960dd1e4 \\\n'
        '    --hash=sha256:fe27fb049cdcca11f11a7bfda64043c37b30e6b91f10cb5bab275806c32f6ab3\n'
        '    # via\n'
        '    #   -r tools/depth-workshop-requirements.in\n'
        '    #   torchvision\n'
        'pyyaml==6.0.3 \\\n'
        '    --hash=sha256:00c4bdeba853cc34e7dd471f16b4114f4162dc03e6b7afcc2128711f0eca823c \\\n'
        '    --hash=sha256:0150219816b6a1fa26fb4699fb7daa9caf09eb1999f3b70fb6e786805e80375a \\\n'
        '    --hash=sha256:02893d100e99e03eda1c8fd5c441d8c60103fd175728e23e431db1b589cf5ab3 \\\n'
        '    --hash=sha256:02ea2dfa234451bbb8772601d7b8e426c2bfa197136796224e50e35a78777956 \\\n'
        '    --hash=sha256:0f29edc409a6392443abf94b9cf89ce99889a1dd5376d94316ae5145dfedd5d6 \\\n'
        '    --hash=sha256:10892704fc220243f5305762e276552a0395f7beb4dbf9b14ec8fd43b57f126c \\\n'
        '    --hash=sha256:16249ee61e95f858e83976573de0f5b2893b3677ba71c9dd36b9cf8be9ac6d65 \\\n'
        '    --hash=sha256:1d37d57ad971609cf3c53ba6a7e365e40660e3be0e5175fa9f2365a379d6095a \\\n'
        '    --hash=sha256:1ebe39cb5fc479422b83de611d14e2c0d3bb2a18bbcb01f229ab3cfbd8fee7a0 \\\n'
        '    --hash=sha256:214ed4befebe12df36bcc8bc2b64b396ca31be9304b8f59e25c11cf94a4c033b \\\n'
        '    --hash=sha256:2283a07e2c21a2aa78d9c4442724ec1eb15f5e42a723b99cb3d822d48f5f7ad1 \\\n'
        '    --hash=sha256:22ba7cfcad58ef3ecddc7ed1db3409af68d023b7f940da23c6c2a1890976eda6 \\\n'
        '    --hash=sha256:27c0abcb4a5dac13684a37f76e701e054692a9b2d3064b70f5e4eb54810553d7 \\\n'
        '    --hash=sha256:28c8d926f98f432f88adc23edf2e6d4921ac26fb084b028c733d01868d19007e \\\n'
        '    --hash=sha256:2e71d11abed7344e42a8849600193d15b6def118602c4c176f748e4583246007 \\\n'
        '    --hash=sha256:34d5fcd24b8445fadc33f9cf348c1047101756fd760b4dacb5c3e99755703310 \\\n'
        '    --hash=sha256:37503bfbfc9d2c40b344d06b2199cf0e96e97957ab1c1b546fd4f87e53e5d3e4 \\\n'
        '    --hash=sha256:3c5677e12444c15717b902a5798264fa7909e41153cdf9ef7ad571b704a63dd9 \\\n'
        '    --hash=sha256:3ff07ec89bae51176c0549bc4c63aa6202991da2d9a6129d7aef7f1407d3f295 \\\n'
        '    --hash=sha256:41715c910c881bc081f1e8872880d3c650acf13dfa8214bad49ed4cede7c34ea \\\n'
        '    --hash=sha256:418cf3f2111bc80e0933b2cd8cd04f286338bb88bdc7bc8e6dd775ebde60b5e0 \\\n'
        '    --hash=sha256:44edc647873928551a01e7a563d7452ccdebee747728c1080d881d68af7b997e \\\n'
        '    --hash=sha256:4a2e8cebe2ff6ab7d1050ecd59c25d4c8bd7e6f400f5f82b96557ac0abafd0ac \\\n'
        '    --hash=sha256:4ad1906908f2f5ae4e5a8ddfce73c320c2a1429ec52eafd27138b7f1cbe341c9 \\\n'
        '    --hash=sha256:501a031947e3a9025ed4405a168e6ef5ae3126c59f90ce0cd6f2bfc477be31b7 \\\n'
        '    --hash=sha256:5190d403f121660ce8d1d2c1bb2ef1bd05b5f68533fc5c2ea899bd15f4399b35 \\\n'
        '    --hash=sha256:5498cd1645aa724a7c71c8f378eb29ebe23da2fc0d7a08071d89469bf1d2defb \\\n'
        '    --hash=sha256:5cf4e27da7e3fbed4d6c3d8e797387aaad68102272f8f9752883bc32d61cb87b \\\n'
        '    --hash=sha256:5e0b74767e5f8c593e8c9b5912019159ed0533c70051e9cce3e8b6aa699fcd69 \\\n'
        '    --hash=sha256:5ed875a24292240029e4483f9d4a4b8a1ae08843b9c54f43fcc11e404532a8a5 \\\n'
        '    --hash=sha256:5fcd34e47f6e0b794d17de1b4ff496c00986e1c83f7ab2fb8fcfe9616ff7477b \\\n'
        '    --hash=sha256:5fdec68f91a0c6739b380c83b951e2c72ac0197ace422360e6d5a959d8d97b2c \\\n'
        '    --hash=sha256:6344df0d5755a2c9a276d4473ae6b90647e216ab4757f8426893b5dd2ac3f369 \\\n'
        '    --hash=sha256:64386e5e707d03a7e172c0701abfb7e10f0fb753ee1d773128192742712a98fd \\\n'
        '    --hash=sha256:652cb6edd41e718550aad172851962662ff2681490a8a711af6a4d288dd96824 \\\n'
        '    --hash=sha256:66291b10affd76d76f54fad28e22e51719ef9ba22b29e1d7d03d6777a9174198 \\\n'
        '    --hash=sha256:66e1674c3ef6f541c35191caae2d429b967b99e02040f5ba928632d9a7f0f065 \\\n'
        '    --hash=sha256:6adc77889b628398debc7b65c073bcb99c4a0237b248cacaf3fe8a557563ef6c \\\n'
        '    --hash=sha256:79005a0d97d5ddabfeeea4cf676af11e647e41d81c9a7722a193022accdb6b7c \\\n'
        '    --hash=sha256:7c6610def4f163542a622a73fb39f534f8c101d690126992300bf3207eab9764 \\\n'
        '    --hash=sha256:7f047e29dcae44602496db43be01ad42fc6f1cc0d8cd6c83d342306c32270196 \\\n'
        '    --hash=sha256:8098f252adfa6c80ab48096053f512f2321f0b998f98150cea9bd23d83e1467b \\\n'
        '    --hash=sha256:850774a7879607d3a6f50d36d04f00ee69e7fc816450e5f7e58d7f17f1ae5c00 \\\n'
        '    --hash=sha256:8d1fab6bb153a416f9aeb4b8763bc0f22a5586065f86f7664fc23339fc1c1fac \\\n'
        '    --hash=sha256:8da9669d359f02c0b91ccc01cac4a67f16afec0dac22c2ad09f46bee0697eba8 \\\n'
        '    --hash=sha256:8dc52c23056b9ddd46818a57b78404882310fb473d63f17b07d5c40421e47f8e \\\n'
        '    --hash=sha256:9149cad251584d5fb4981be1ecde53a1ca46c891a79788c0df828d2f166bda28 \\\n'
        '    --hash=sha256:93dda82c9c22deb0a405ea4dc5f2d0cda384168e466364dec6255b293923b2f3 \\\n'
        '    --hash=sha256:96b533f0e99f6579b3d4d4995707cf36df9100d67e0c8303a0c55b27b5f99bc5 \\\n'
        '    --hash=sha256:9c57bb8c96f6d1808c030b1687b9b5fb476abaa47f0db9c0101f5e9f394e97f4 \\\n'
        '    --hash=sha256:9c7708761fccb9397fe64bbc0395abcae8c4bf7b0eac081e12b809bf47700d0b \\\n'
        '    --hash=sha256:9f3bfb4965eb874431221a3ff3fdcddc7e74e3b07799e0e84ca4a0f867d449bf \\\n'
        '    --hash=sha256:a33284e20b78bd4a18c8c2282d549d10bc8408a2a7ff57653c0cf0b9be0afce5 \\\n'
        '    --hash=sha256:a80cb027f6b349846a3bf6d73b5e95e782175e52f22108cfa17876aaeff93702 \\\n'
        '    --hash=sha256:b30236e45cf30d2b8e7b3e85881719e98507abed1011bf463a8fa23e9c3e98a8 \\\n'
        '    --hash=sha256:b3bc83488de33889877a0f2543ade9f70c67d66d9ebb4ac959502e12de895788 \\\n'
        '    --hash=sha256:b865addae83924361678b652338317d1bd7e79b1f4596f96b96c77a5a34b34da \\\n'
        '    --hash=sha256:b8bb0864c5a28024fac8a632c443c87c5aa6f215c0b126c449ae1a150412f31d \\\n'
        '    --hash=sha256:ba1cc08a7ccde2d2ec775841541641e4548226580ab850948cbfda66a1befcdc \\\n'
        '    --hash=sha256:bdb2c67c6c1390b63c6ff89f210c8fd09d9a1217a465701eac7316313c915e4c \\\n'
        '    --hash=sha256:c1ff362665ae507275af2853520967820d9124984e0f7466736aea23d8611fba \\\n'
        '    --hash=sha256:c2514fceb77bc5e7a2f7adfaa1feb2fb311607c9cb518dbc378688ec73d8292f \\\n'
        '    --hash=sha256:c3355370a2c156cffb25e876646f149d5d68f5e0a3ce86a5084dd0b64a994917 \\\n'
        '    --hash=sha256:c458b6d084f9b935061bc36216e8a69a7e293a2f1e68bf956dcd9e6cbcd143f5 \\\n'
        '    --hash=sha256:d0eae10f8159e8fdad514efdc92d74fd8d682c933a6dd088030f3834bc8e6b26 \\\n'
        '    --hash=sha256:d76623373421df22fb4cf8817020cbb7ef15c725b9d5e45f17e189bfc384190f \\\n'
        '    --hash=sha256:ebc55a14a21cb14062aa4162f906cd962b28e2e9ea38f9b4391244cd8de4ae0b \\\n'
        '    --hash=sha256:eda16858a3cab07b80edaf74336ece1f986ba330fdb8ee0d6c0d68fe82bc96be \\\n'
        '    --hash=sha256:ee2922902c45ae8ccada2c5b501ab86c36525b883eff4255313a253a3160861c \\\n'
        '    --hash=sha256:efd7b85f94a6f21e4932043973a7ba2613b059c4a000551892ac9f1d11f5baf3 \\\n'
        '    --hash=sha256:f7057c9a337546edc7973c0d3ba84ddcdf0daa14533c2065749c9075001090e6 \\\n'
        '    --hash=sha256:fa160448684b4e94d80416c0fa4aac48967a969efe22931448d853ada8baf926 \\\n'
        '    --hash=sha256:fc09d0aa354569bc501d4e787133afc08552722d3ab34836a80547331bb5d4a0\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'regex==2026.9.10 \\\n'
        '    --hash=sha256:030fa9e23624e39b3b94e46b90a5abd1a1678eb2f58fcdd3fd6c27526bf91c7e \\\n'
        '    --hash=sha256:032da15431c890d376f53547f0a6219f4f4cd19f3e4f11bdc321453b5bd207e4 \\\n'
        '    --hash=sha256:044bd4639b6bb409ec9e5d8b7accd57e02b4c4a4e2eafde916f8ae8006b3e40b \\\n'
        '    --hash=sha256:048a89ee797db10160bd2bd519286577a6b43a100279bd4b7d8456a3d69c80a0 \\\n'
        '    --hash=sha256:05fb018cfe7144585fc83882405906ff84994a2d154afc2509ecc7752c51f864 \\\n'
        '    --hash=sha256:07b45ba5c94b8fcb30cb6c56a11f715c57533a3017964504322ea52690a27b72 \\\n'
        '    --hash=sha256:0aa7589394230e0f0a422ab6b90841ff12c87e855e7aaf75d192a54a5f124548 \\\n'
        '    --hash=sha256:0acee94b480dd853e39434aa9a575f95385b1b4b8fa3feae56db363ca5cad782 \\\n'
        '    --hash=sha256:0b9ba3b2765cdfe18f0f561a69f78a69701f2896654a81c711108d35d14e5099 \\\n'
        '    --hash=sha256:0c32480f3371b75068decaf9e5da72c224e953830dd71e36e06cf80e30ea39d8 \\\n'
        '    --hash=sha256:1270cdec69248592bbe38a0b263ed58d907b891bd2b93703e225c317e421bda1 \\\n'
        '    --hash=sha256:13c52fc377792675f604a207a2ae5958c080f6854f7698d40d9ff034d95b1e76 \\\n'
        '    --hash=sha256:14caa05ce39ec70437af5aac8814c50ee6628f4a90353871c059692f448a164f \\\n'
        '    --hash=sha256:1562aabd9d4eb09bd88a62ad97ed06800094b529ac43419e43020b9cefec79b0 \\\n'
        '    --hash=sha256:175cf49ce7a994c88b8f15e3cb17cdb66a48ebb2d36de736b8205033db950f89 \\\n'
        '    --hash=sha256:1aa309ab7ba89a62d6cf70dbd38d4176440bce3c7001ab86256704cf4c18c6eb \\\n'
        '    --hash=sha256:1ad10a135fa0b4e4a462a61d07c6654d7518cfdb5cb8da08f9ff7d61384af1fe \\\n'
        '    --hash=sha256:1b891f77554bff991804cee24b78b40789f7d5993a24c7907bc7025fd2a70c8d \\\n'
        '    --hash=sha256:1e321e2c84f0e52c457f5ea5944f796d6e8e09cb99738ea98dcc1bfe402a128d \\\n'
        '    --hash=sha256:1e954e246466d5a1a78f563ce8364b5d7cb19e7adb0ccdec8f9c9610083187bc \\\n'
        '    --hash=sha256:1f0a8b4928823bc8b217a1ab7bf3d90598909dec9a70fbbfe9a52cc4eca55990 \\\n'
        '    --hash=sha256:1fbc8314436353e097c050e11b01a6c11433579437ed0579730157676ef59e2f \\\n'
        '    --hash=sha256:20e8bfb07ad79a282f8b95b56fe67f9750b1b7f775724e4ba1f23cb296115ce4 \\\n'
        '    --hash=sha256:217e98ba5fc8908ed8ffd4ebac04753a0c831067cbfb495b9821b94cc61eaa76 \\\n'
        '    --hash=sha256:239620b0e0681669367c0e218c8eb2551d9f8fe3b9fccfc8d0003377804e8348 \\\n'
        '    --hash=sha256:23ac9a28180f274d7dd7651fa131ad5b02d343b75df4b040737f0356223895dd \\\n'
        '    --hash=sha256:2479171edccced52ef02b899558f88ab2c235fe05b93180fdcae1670aacd89e1 \\\n'
        '    --hash=sha256:24d12a625a37c89c2b09303402a06942f55f071b95a7916a49c17034c3d47cd5 \\\n'
        '    --hash=sha256:2dd9286093c71afc8f55ef035c5b9d2776641fd72c6535f1febc92d0b0be9666 \\\n'
        '    --hash=sha256:2e67f8843f0e4b931f1fa860bf3bbe4134b714c0155cc5c7c0d7ea450230aae0 \\\n'
        '    --hash=sha256:31e4df2b11d48f61d511019bc1ee9b477055f17c352b68fe72db7a98b14d603c \\\n'
        '    --hash=sha256:3264132d576847ab5f88bb83e7debe67854bf165b3ea613bd467312b6099536a \\\n'
        '    --hash=sha256:3540734dbe241ebb3b87d5713781f6749a3e4d45480f506aa5fb5cbb0c37d249 \\\n'
        '    --hash=sha256:35ba3bab0c45079735f55ac61526774de1d84bc4a0333cc554e1a4ab74913924 \\\n'
        '    --hash=sha256:3a66e40a1a20de96a2fee00ed67e11012b62d85b277688258677fd19997addb7 \\\n'
        '    --hash=sha256:3bdeed3318a8eb2bbadc9c56347e0ff651639e934a47e168d05a3b12929fd0e7 \\\n'
        '    --hash=sha256:3fb4ae8cf83ef4e9addd43b2da31a9f45be816a8036fae8af59c8998b72718e2 \\\n'
        '    --hash=sha256:4971776b4f2bd7fd9a83eceb2cb2592cbe2924f639fe8045e6a9de5ba4bfcf25 \\\n'
        '    --hash=sha256:4a761ea45f2ad74c575ef5850ea514cef97302a552d3c7c9d1a1a870d4661d6c \\\n'
        '    --hash=sha256:4c66d54042a14a503907d81861b8a5235e6d1f03d4fbc1d8767f652eaf957ac1 \\\n'
        '    --hash=sha256:4db7d00c4afbfbb55b8e17b1e371da11418ea9389b030acec63c1fa4c7ad4b86 \\\n'
        '    --hash=sha256:4f0407474ffac8e5e89d93ca41d60891e29f0ab8423eb66ff292d850a86a0843 \\\n'
        '    --hash=sha256:53e182b6b04d0011909b47d51a2d72d908de07c7b1c7f16b3adda2204d723bc1 \\\n'
        '    --hash=sha256:5847e22bbf959764d776937d791d034cc2d19b787e361c88d97e859e8dc68502 \\\n'
        '    --hash=sha256:58c01f7b81079cf0817ba831ff4d9eff5d28be4a3ac76c353e6f09bd63f4c386 \\\n'
        '    --hash=sha256:58da726d3e766c0b3f5a3997dfaf0275898a1107b8191cdd6b0437fe45fd817d \\\n'
        '    --hash=sha256:5bef622850cf760154719d4e0d74b0a855962432995168e250069899ae12fe8f \\\n'
        '    --hash=sha256:5ccd139b2061132e7b265cfb4b4721baeb9f8928b81415304abf1ec7e3181c26 \\\n'
        '    --hash=sha256:5cef9f3d14796500ea834c41dbe688f1f6b23c7024dc23e8a794d7ebaf5d71d0 \\\n'
        '    --hash=sha256:63bb62cf62217dc38c8a6b2b61b165b0e4eb8fa93b0aba12139251c0986a8fa3 \\\n'
        '    --hash=sha256:681ed38664b64c6617d3c3c332018d1948c77e139c5ea667c1886efa671e426f \\\n'
        '    --hash=sha256:6888065672b341e5246f391ec16dc258a29218ac784172fd67c30d941544755b \\\n'
        '    --hash=sha256:6aebdd9a946de328b3f6f61dbf48dd064a36eb6dddf96e34ae6651d37f6e9383 \\\n'
        '    --hash=sha256:6afcad14310f1311d077553ed374b42a5e538f85a8c884b4e38e52de091c8077 \\\n'
        '    --hash=sha256:6b34a778c695d24e77c140e3b4c95da69282e34f2f6b02b55656aa4a0379f643 \\\n'
        '    --hash=sha256:6fd555fc9abef50c530869690b2daca054c8811a7aff632d11f9a7b2590b2742 \\\n'
        '    --hash=sha256:71879292c9c7ac67b1680345b16daba1be937cb027362cfa04e68f65db2dcfdd \\\n'
        '    --hash=sha256:75242f44a3e283106077be4ab717bc535e4701c9d54ad69e195945c22f137a1d \\\n'
        '    --hash=sha256:75aa39d3f4f1650eea84e46b0d8cefe77dd5478c10e3d0aaf0b0f00493475a7a \\\n'
        '    --hash=sha256:75f9297b16fcb588a1f8d8a55dabef3c0c20b0c7bac43c87ceaaaf1a825c12f4 \\\n'
        '    --hash=sha256:79e9432995e14c749d34209413de5e621ec8e67789bf4f46dbfabea9d06a2406 \\\n'
        '    --hash=sha256:7abb38b8c40f3a235235a44da452c64b7b5c1d650ec6351027db0e090804f2e5 \\\n'
        '    --hash=sha256:7dcad477c49c4c626a6c4fcd71b39a971aa217060cc40a6569fd24edcc0fa509 \\\n'
        '    --hash=sha256:7e6c0b5ec6ddee4032247585dc491b0fa58627745b66a705728703a3f0331231 \\\n'
        '    --hash=sha256:7f8f10015866608fe4c043cec2e4fe4c39a94bb50e45091de4cdf4004b9ae4b0 \\\n'
        '    --hash=sha256:866de9f98df0611d7b62b3a8729d3284a64c0cc6edd90bb95a533e443a4939cb \\\n'
        '    --hash=sha256:87f5f75c109f08f5c602d68e1af54cead8165189c727b6ac946b30b9833a3ba4 \\\n'
        '    --hash=sha256:880ac684c27176464c00c3fdc456116364f5ebc70da07aad0c2d4a7ba45e98db \\\n'
        '    --hash=sha256:88b02aa8d0ec9b6189fe933d425775882271c23700ac11fd26d1779b0f56fde3 \\\n'
        '    --hash=sha256:8ba1f78bd4fef2d8f84b894ec28ac3481afe6cc07aaa253ad4717ef7b3fe6bcb \\\n'
        '    --hash=sha256:8c07021a4faa3f092869adbd1f35cdc7a592276c807aeebc3ceb8ff1a638f0b4 \\\n'
        '    --hash=sha256:8d5c4518235a2ec1611e57af85fa488d529c1106aacff12adadcedf8687012cd \\\n'
        '    --hash=sha256:8e127d9a80cbf1c3276bb465c6d047e8705e97b58c2b8f2f0c0a69c336b44b37 \\\n'
        '    --hash=sha256:94c5ce3bc41d226b4eb89ca3f842b2e28c031487fb1f34eb2153d98235831325 \\\n'
        '    --hash=sha256:94d096369b7cd96d15343fef5257fe39eff9d0e8758b92a0e15e358b92cdb2fc \\\n'
        '    --hash=sha256:968c1e33edd9a104d1bf24c8d476c72de7e3839ae7f894b37e9e4f4739fdeeca \\\n'
        '    --hash=sha256:990797e765d89a423880052c68b61c31afe701de94a8c060f61c40605ca6c727 \\\n'
        '    --hash=sha256:9ce239acb15843ab03976626af810a4424b0409689ec2bbc52088ab5479ab487 \\\n'
        '    --hash=sha256:9d772586951d7d6a5d162d48f414065e483b1c81ab38fd8ed97c78b05883421a \\\n'
        '    --hash=sha256:9fbd2e5d8002dc49a6129fb321ec51c57a025e752ed525ddce0ba9223c4350a7 \\\n'
        '    --hash=sha256:a41693eb3fc4b92e6127d113813c6c395237f7edd3224abf67609af48c690d11 \\\n'
        '    --hash=sha256:abbfc1c33bf8efddcc43844aba61e036d74a918680dc3ce8ce2538b004eda0f9 \\\n'
        '    --hash=sha256:b298cdc33c5cc6969ff07f0fba19cc73e0fd8576373c50935feadaca2f6b4405 \\\n'
        '    --hash=sha256:b43456de605c8ee77eb75f07bc1ee44ba27f9cee22207deb77d495e954b7d953 \\\n'
        '    --hash=sha256:b71649169a9fcf30b395ee01047fa7ad6654a4c900ca75b23c04dedcce6a1f8c \\\n'
        '    --hash=sha256:b91c37551bf39d75116c02b146956f65b9aa0337a4a652f4ae186983789d4001 \\\n'
        '    --hash=sha256:b9d36b03dc362aa40ffaaec9d9bd75e87763529563ec008c43b0e07782f5be7a \\\n'
        '    --hash=sha256:bafa41b0dd63669e5c0f8adf3d24819efeb73c847f492eb011212eb352e69041 \\\n'
        '    --hash=sha256:bb7774924f8cd69f49cba0b3c2d679a6326f777e0e67d130ad5203e4df53f0d3 \\\n'
        '    --hash=sha256:bf29611e5376fec8f795879bb5c6153a76c3a292573d173c26784042b01eb840 \\\n'
        '    --hash=sha256:c014641157e9049b0603b8daa5343bd408d9b757b709aaa0f373cd3fab2d7944 \\\n'
        '    --hash=sha256:c103b3b14e011774af4fb7e4617ad4d72b9171905cd3b231a70a4efd76e477d7 \\\n'
        '    --hash=sha256:c22df8dd6373bbe3898e77429ffc85594300e39d752fd0e68a31e59d37899376 \\\n'
        '    --hash=sha256:c25a754bb81a2edcfc3b65eda50f017d736f818112ed43e8aafd595cb00678ae \\\n'
        '    --hash=sha256:c32818b28bcd153b25b63038348a9fe9b9fbcddb60df43f204c3ab55eeb57f77 \\\n'
        '    --hash=sha256:c37fa93bf18bf4f90b01c0fa9f11ea567ee4b7dd8bf96e63663e5edc37aa38cf \\\n'
        '    --hash=sha256:c3d95d7d9538b5b726dd6fcd7b6117a71e6565202f6d64f5845fb4d8f203f533 \\\n'
        '    --hash=sha256:c8fbd9cb30c68c1686b94029b9ef845d5870d3d65baf66cb126b676849b9d72b \\\n'
        '    --hash=sha256:cb76a9c4e07a6a47849726af0ed14c41741a182f097f134a8cf29c1bc0f4dde8 \\\n'
        '    --hash=sha256:ce7c118cb102975f974585688357a717ffbf9dddd64ab0bb1bc93eb5b367cf95 \\\n'
        '    --hash=sha256:cf377960d2ac37d987394a9dbaa75e91338c41a46d41e1d25e90125e7b3ee2dc \\\n'
        '    --hash=sha256:d278ad30ec83b6b9202685b0f80b741a51ea3ca7f0595ebda96e7628b6398876 \\\n'
        '    --hash=sha256:d2d377fd1cad611b806cdd732d86b65f536c768209890cb442556548daa65a23 \\\n'
        '    --hash=sha256:d414c411c06fe0009eac33488fb1591c66b5c2673e342e452e7bb2fe63da8194 \\\n'
        '    --hash=sha256:d8c668af8f7bdb1d18739c27d30cd9f4b371495a883f75a002fb7a39d740fecd \\\n'
        '    --hash=sha256:dce932f8e3ba936475ea3d0d8b59f7b050a9e206e994f53f8fd80299871e87da \\\n'
        '    --hash=sha256:debc629e98b95abaea1cf3057ca296151f348c697c9b8a59d18013adb302c0dd \\\n'
        '    --hash=sha256:e0dc78251154b66dc60211563fc115345da332eaa881e4e2523fb1edae3772f4 \\\n'
        '    --hash=sha256:e5e4a6e0734a685d13b9685622bb503bdbb2927f8b0df025a5085f0ea067475b \\\n'
        '    --hash=sha256:e6b99181d184d0f5c7b36b8d12b94d1e9499cce6246594331f9edc5d2ea9fceb \\\n'
        '    --hash=sha256:e7327795089ddb44912dce1434e1d7244be2e9fb48fcc2d6782936af7a3062db \\\n'
        '    --hash=sha256:ebb2ba68e4641a994061f70bf44ed448fba0b9b1d18c94ffb9efc1cca805b39b \\\n'
        '    --hash=sha256:ec8855f08c17895a26fbf5f19ed829722e19b34a96629e49a43c92974924026b \\\n'
        '    --hash=sha256:ecb2e7acb18f8cc4a67f0ad986c0af291ea4dd385d0614ba9bc09d7f8bbb478c \\\n'
        '    --hash=sha256:ef4c0a9dfdc90581b90b1b95a8c3d1557f8ff8f5a2a53536d26314de699d1468 \\\n'
        '    --hash=sha256:ef4ce69ff97fbb44b46751cfea5e859ad0b66d1a50abf34954f0645f51e81671 \\\n'
        '    --hash=sha256:ef5a059ea1c6ee5d1c7e99a2484e628608d010921efe876c6f0e2029d2f35eca \\\n'
        '    --hash=sha256:f0e2e5d23448b660d60a6ed85c46cc03b4b48bd276b8f4041d4a5fe2a4a0626b \\\n'
        '    --hash=sha256:f2374c27deb189b282ec7e16106752c22ad39b056bbd8018960b1e4cc95d67a1 \\\n'
        '    --hash=sha256:f2f43bf4e47ff7ce9e585558706d698c6204d0f80bf2207766382ed817c8e9f4 \\\n'
        '    --hash=sha256:f5c629df03adec31ee505dda3c8988f106c9390e4cbd343600036eb8b3d6724f \\\n'
        '    --hash=sha256:f70b9f0e39c2dba1d9da6bf7ef7c377cad7277f8440e9a69be05ede529ff024c \\\n'
        '    --hash=sha256:f7d4656e17ab736e9415a6442a345bfc97bb8b7dcce47884bb74a37f70f08d0c \\\n'
        '    --hash=sha256:f8bdec659a8fa7af51a32b224b3b7c02bc415d54ffd35187b1d224176b17d607 \\\n'
        '    --hash=sha256:faa911fbbcf8ac90bda0e0657d60768e3390954ef0588211d63a22add1cb1cd1 \\\n'
        '    --hash=sha256:fbc4e2f3cb7ce8436154e6483079e7d35eeb321a952fa936e180300630d8b873 \\\n'
        '    --hash=sha256:fd6bd89b9fc06018d35851cab0240adb7dd84d51941b19f6574ac90cd54e3ae5 \\\n'
        '    --hash=sha256:ff4d7b14ea19e50c8d9d6d83f45bd9b45cbb624c07ac1fa54db0a019049abed7 \\\n'
        '    --hash=sha256:ff6b3267318661dfddf6b3628663e00e5946bd0a5c8fa678537a1401f0388f91 \\\n'
        '    --hash=sha256:ffc2da104e43db716ce30cef9f28049a1faa6aca385dd8771b033268d0730b07\n'
        '    # via transformers\n'
        'requests==2.34.2 \\\n'
        '    --hash=sha256:2a0d60c172f83ac6ab31e4554906c0f3b3588d37b5cb939b1c061f4907e278e0 \\\n'
        '    --hash=sha256:f288924cae4e29463698d6d60bc6a4da69c89185ad1e0bcc4104f584e960b9ed\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'safetensors==0.8.0 \\\n'
        '    --hash=sha256:040070828e36dc8e122178bbbd5830ff9e97920affb84cbe0f46442497bed358 \\\n'
        '    --hash=sha256:096ec1a98435df7beb08853bb5aa9081a84f23d0adc67ed1a0a10550f608373f \\\n'
        '    --hash=sha256:2ddf52eac562eda224f99acfa7889d02968c1fd59a5b011ae7d8137c37e9c02d \\\n'
        '    --hash=sha256:3ae091f16662658bdc019a4ff6cb4c085bb7d725eb5978b183ffd265863b6d2d \\\n'
        '    --hash=sha256:4124502b78f03534117c848f87a39b8f31e577b15eff423bf8bfb95f2a8c30d0 \\\n'
        '    --hash=sha256:4a95ae2b05d7726d751da4ebf626a2ca782b706e101bd894c95bc2450b1cffcc \\\n'
        '    --hash=sha256:7a46e5ff292c356d6991e60942ba7f79817682d3a2cef0702136448cb9c4d235 \\\n'
        '    --hash=sha256:7bc0a787ba8a35be368ee3574edfa2b1ad389eebd0a72e482ae275490e3f6c98 \\\n'
        '    --hash=sha256:87eec7ffed2b809f05a398a8becb7d013f19f7837cd15d9748580d6cf30dbaf4 \\\n'
        '    --hash=sha256:8e080062fcde23be189565e1c3305d16751a218ecf9412c8601e64204eb6f846 \\\n'
        '    --hash=sha256:8e9f537aa183a38ace122d27303dcd986b26bd2a7591f9181d7f0c396f4677ca \\\n'
        '    --hash=sha256:c554f85858e05226d3c2828e32395e677434685d6d94594a41643361c5e837f0 \\\n'
        '    --hash=sha256:c80201d22cbf405b80647a60ada77bba06c8fba2da2743ba1e89cdcc39a81f25 \\\n'
        '    --hash=sha256:f7838e5135a406ad3e02efdcb8cf2e5397d368b0154537c4fec682dbc544d452 \\\n'
        '    --hash=sha256:fabaf3e0f18a6618d9b36560682562157f77c2b71fcffc7b432be2baed9d753d \\\n'
        '    --hash=sha256:fcdd41ec4628fee5799f807c73c353629130fbd942aa23d83c623dd6c9d52d78 \\\n'
        '    --hash=sha256:fd6f3f93c9a0a7cc2788ee63fb763353d4bd2e89b0751bc78fcf7dda00bea774\n'
        '    # via\n'
        '    #   -r tools/depth-workshop-requirements.in\n'
        '    #   transformers\n'
        'setuptools==84.0.0 \\\n'
        '    --hash=sha256:51a52592b3b99e102b609654876bd65f19f999935166d1352678931132b0c670 \\\n'
        '    --hash=sha256:f4695c21257f0d9b537ec2692c941d02ee143b7cc1276941349a546573b2ef73\n'
        '    # via torch\n'
        'sympy==1.14.0 \\\n'
        '    --hash=sha256:d3d3fe8df1e5a0b42f0e7bdf50541697dbe7d23746e894990c030e2b05e72517 \\\n'
        '    --hash=sha256:e091cc3e99d2141a0ba2847328f5479b05d94a6635cb96148ccb3f34671bd8f5\n'
        '    # via torch\n'
        'tokenizers==0.22.2 \\\n'
        '    --hash=sha256:143b999bdc46d10febb15cbffb4207ddd1f410e2c755857b5a0797961bbdc113 \\\n'
        '    --hash=sha256:1a62ba2c5faa2dd175aaeed7b15abf18d20266189fb3406c5d0550dd34dd5f37 \\\n'
        '    --hash=sha256:1c774b1276f71e1ef716e5486f21e76333464f47bece56bbd554485982a9e03e \\\n'
        '    --hash=sha256:1e418a55456beedca4621dbab65a318981467a2b188e982a23e117f115ce5001 \\\n'
        '    --hash=sha256:1e50f8554d504f617d9e9d6e4c2c2884a12b388a97c5c77f0bc6cf4cd032feee \\\n'
        '    --hash=sha256:2249487018adec45d6e3554c71d46eb39fa8ea67156c640f7513eb26f318cec7 \\\n'
        '    --hash=sha256:25b85325d0815e86e0bac263506dd114578953b7b53d7de09a6485e4a160a7dd \\\n'
        '    --hash=sha256:29c30b83d8dcd061078b05ae0cb94d3c710555fbb44861139f9f83dcca3dc3e4 \\\n'
        '    --hash=sha256:319f659ee992222f04e58f84cbf407cfa66a65fe3a8de44e8ad2bc53e7d99012 \\\n'
        '    --hash=sha256:369cc9fc8cc10cb24143873a0d95438bb8ee257bb80c71989e3ee290e8d72c67 \\\n'
        '    --hash=sha256:37ae80a28c1d3265bb1f22464c856bd23c02a05bb211e56d0c5301a435be6c1a \\\n'
        '    --hash=sha256:38337540fbbddff8e999d59970f3c6f35a82de10053206a7562f1ea02d046fa5 \\\n'
        '    --hash=sha256:473b83b915e547aa366d1eee11806deaf419e17be16310ac0a14077f1e28f917 \\\n'
        '    --hash=sha256:544dd704ae7238755d790de45ba8da072e9af3eea688f698b137915ae959281c \\\n'
        '    --hash=sha256:64d94e84f6660764e64e7e0b22baa72f6cd942279fdbb21d46abd70d179f0195 \\\n'
        '    --hash=sha256:753d47ebd4542742ef9261d9da92cd545b2cacbb48349a1225466745bb866ec4 \\\n'
        '    --hash=sha256:791135ee325f2336f498590eb2f11dc5c295232f288e75c99a36c5dbce63088a \\\n'
        '    --hash=sha256:9ce725d22864a1e965217204946f830c37876eee3b2ba6fc6255e8e903d5fcbc \\\n'
        '    --hash=sha256:a6bf3f88c554a2b653af81f3204491c818ae2ac6fbc09e76ef4773351292bc92 \\\n'
        '    --hash=sha256:bfb88f22a209ff7b40a576d5324bf8286b519d7358663db21d6246fb17eea2d5 \\\n'
        '    --hash=sha256:c9ea31edff2968b44a88f97d784c2f16dc0729b8b143ed004699ebca91f05c48 \\\n'
        '    --hash=sha256:df6c4265b289083bf710dff49bc51ef252f9d5be33a45ee2bed151114a56207b \\\n'
        '    --hash=sha256:e10bf9113d209be7cd046d40fbabbaf3278ff6d18eb4da4c500443185dc1896c \\\n'
        '    --hash=sha256:f01a9c019878532f98927d2bacb79bbb404b43d3437455522a00a30718cdedb5\n'
        '    # via transformers\n'
        'torch==2.14.0 \\\n'
        '    --hash=sha256:0e7cf18cb0d8bd666b6120932e29c7aef3502b61a08b44da4839580c539a7cdb \\\n'
        '    --hash=sha256:2fb30099be1fec9d163a6dd05ace176305ce8cf914b4d7a5f4f6d7a326aa4b60 \\\n'
        '    --hash=sha256:44b044b9f6f633d982839422a57433d6a1da520037fd88e0c8a47efde589b3b8 \\\n'
        '    --hash=sha256:4dbea8a10d80b10dff2be1ed467d2617b704fe6bf76bbfad4194774c8f6d886b \\\n'
        '    --hash=sha256:553aec938d37d77b783bcf801e638cad068870e7643c47eadac21eb180f551ed \\\n'
        '    --hash=sha256:6981872df75eb5409c439050d36b67cb57d88b4e11080516e2aa7410f6730705 \\\n'
        '    --hash=sha256:731784e3914843c6bcc7aba3987ff7610ac57dbbc816a5d6b9b62e04c240a641 \\\n'
        '    --hash=sha256:731b9ebdea402b8b1996d4c2ae613b16660e559b19e47bdc45d970568bc91c53 \\\n'
        '    --hash=sha256:84bf384779a10c02fc3c6bdbab71a9cb66b0dd93c652d1ed5d6dfc0cb37e5962 \\\n'
        '    --hash=sha256:860423e970f2ce02c4476e8e2d1350131b1c5c5a5e4912180e78b50b53241efa \\\n'
        '    --hash=sha256:8d9e232b6376c62f3090237889fb1cb6887c0ece9f73e6c1052e80fc1481f999 \\\n'
        '    --hash=sha256:96383c62423c3f2767023c4a94774b6435e2c006691daf396e9ddb55d412fbc6 \\\n'
        '    --hash=sha256:9d4b1022a5d9b71282ec67ad0d9e7235870096b8a246dc1c32d6ea1fc83dc998 \\\n'
        '    --hash=sha256:ada340e62591d06a2bcc2d68170f20f45f0b0665d372dc510a8ed7eb3b1d609a \\\n'
        '    --hash=sha256:ae530ddd3f3b94248b77f1fd3313c4f3405bd0d17283d505b81574816767133a \\\n'
        '    --hash=sha256:b2cd92bce63d40bf6fc2e5d840fc2f0063bd180a242daa761cb6088cc2f46e27 \\\n'
        '    --hash=sha256:b985c7defeb8d28691b7513ebaecc64946c5f68ec770e9f4ddba39688864f46a \\\n'
        '    --hash=sha256:c1f844f1c750e87df4b68bc3afbc0e2b0c7ef19d7b8f666e48bdcf6a0c4f0056 \\\n'
        '    --hash=sha256:cad84f41bbdf3dcf333ce394aeeaf25237c4d94fd6b659ba5eb813c829978823 \\\n'
        '    --hash=sha256:cd8cd8f714d511ccdca907282d1da3d9be8322d4e1520b9c3bce39a5c1318a4b \\\n'
        '    --hash=sha256:d2526f71e6638133b97b3cd2881ece3df521460a4727030e8ae2a72a7d3ae31d \\\n'
        '    --hash=sha256:d30207cb89e713dfd6afb5d592b30d9ef252acf80407c5543157ea7b07c4d9a0 \\\n'
        '    --hash=sha256:d70c2b41bde81efde59fcefb147b9247fd062497e778677661b7de1bee5c8c99 \\\n'
        '    --hash=sha256:fecffb58f51fd643d213acd68da21cc3fc19bea05a3bc64b4ee55128f47a4963\n'
        '    # via\n'
        '    #   -r tools/depth-workshop-requirements.in\n'
        '    #   torchvision\n'
        'torchvision==0.29.0 \\\n'
        '    --hash=sha256:01a029fa1b2eacac27e1e4f9e0dba24d112bcbd523261b294e4b4a1c7f8330bc \\\n'
        '    --hash=sha256:0e3b99c62f7f095153887330c2c60c9ebc2da84dfbe08eea564a41ec361629cf \\\n'
        '    --hash=sha256:0e631a2f8b24732672d35224d2574ff89a78d56bef5364ef6094a625fa5f1771 \\\n'
        '    --hash=sha256:2428a13706a354ee3901fd63e3f1ef8c2a59f6e20d24cce13210b4e61be08f7a \\\n'
        '    --hash=sha256:28a3964a9e6db34354d4ca440d2c2038deeef24e7219e8d6f49d9d174f44830c \\\n'
        '    --hash=sha256:710df2c4f03ab6da161520c10d9467f4c0201ca314fd1432499a48c9d7a5818f \\\n'
        '    --hash=sha256:784c8b8de9e81e02dea093373bc35040791378df2daec0017e657985183bf862 \\\n'
        '    --hash=sha256:80789f9b50f5277302e7020c8128344e0269b12f6e3b5855034e56bb7081b874 \\\n'
        '    --hash=sha256:83db0299170502b038640444214c99d7b54bad72db447f0554deb8d2c4f02f94 \\\n'
        '    --hash=sha256:85fa54bec1f7d9227b5e4e201ed4bac92247ee10fb19148d92f9836e3e91d5c3 \\\n'
        '    --hash=sha256:942a3b3fb4e981e1abb7846679ee3882dfc733533a30b68df0dadf6c06f238a9 \\\n'
        '    --hash=sha256:994687b818cac0e6d34cb407a41458e2c3262cb7db927465841c9522ba7ebb92 \\\n'
        '    --hash=sha256:9b99a7385da8d706f2dc14cacffb8cbf653d48fc37013a38c41a36659acc3731 \\\n'
        '    --hash=sha256:a292ca7044236a2702ec3d4664de135e01fc9fb3ae9422db3fecc819546e831c \\\n'
        '    --hash=sha256:add88479361b2bf08790338fce4ba11a11fdc1ae508e80a3a013daf1c0d2893d \\\n'
        '    --hash=sha256:b7a736eaa6b2e22476c95ceb62986d195f3e600cdd7d196d7329aa2dfc951994 \\\n'
        '    --hash=sha256:bb74d08d65785b51c61d4d4ec167bfa50a5ea644b57555a33ba2f4435b0d05dd \\\n'
        '    --hash=sha256:bbe15455b59a6c9d822584fd3faaaac50ee972101d7bb7e2887da456e148a3ea \\\n'
        '    --hash=sha256:bd12e152640d1024fca15deaf6f39c53702e12eb398db4b2b038a5a97ab85f7f \\\n'
        '    --hash=sha256:ce3b58631f0b7c8828e8675301bff5cb0623aaed704e3ac91079e89607753205 \\\n'
        '    --hash=sha256:dc5717feef9a0b430b052895db71f1d695939575b74344da981120e9f4da7620 \\\n'
        '    --hash=sha256:e8fdf234d76dca6fc47bc5f4af86ecf1ef57fab6285a7b21681bdcbb494e4c1b \\\n'
        '    --hash=sha256:ebf54f6744556ebd8d7b5f9b6515e2060e5d3156723970dc257bf093f63b2170 \\\n'
        '    --hash=sha256:f9e9627d5036cac8e6de76aeb2f1acc67841ff5ff4b63c4cf78ff954efeb073e\n'
        '    # via -r tools/depth-workshop-requirements.in\n'
        'tqdm==4.70.1 \\\n'
        '    --hash=sha256:c293e525e6fef9c20e8728fd4612df02a0aa31bb5fe91ecd93e123b1b7bffa73 \\\n'
        '    --hash=sha256:cefd0eca11b2a37a3aee776544d4f4ae913f02688135b5556b8788dfa474afc4\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   transformers\n'
        'transformers==4.57.6 \\\n'
        '    --hash=sha256:4c9e9de11333ddfe5114bc872c9f370509198acf0b87a832a0ab9458e2bd0550 \\\n'
        '    --hash=sha256:55e44126ece9dc0a291521b7e5492b572e6ef2766338a610b9ab5afbb70689d3\n'
        '    # via -r tools/depth-workshop-requirements.in\n'
        'triton==3.8.0 \\\n'
        '    --hash=sha256:1b84e7d512490ba529111260fa6f7cad8b254a6bb5fbdf41d5ef9a5e57f52d0a \\\n'
        '    --hash=sha256:1f0497218e26b7d79773ad9c2a3fa3b539ee69f587a13fac2e552b1d322a8015 \\\n'
        '    --hash=sha256:1f6b48d0591929a3867973acac3dccd4e058585f91bfb41022de496c9ffab304 \\\n'
        '    --hash=sha256:372285307d4c44ee74cee32de0b4f04bd157e071427e38c6f4ee3e3beb2194f4 \\\n'
        '    --hash=sha256:387dae4cb0089a7b6ba1a428ae0782b65c4c58f57d94617cb22ca8593d8ccbca \\\n'
        '    --hash=sha256:398f4b009c7ab08ed9aeb1d1282dee822945b53f399a5e12a4fecb643cf2007d \\\n'
        '    --hash=sha256:68988ac85d5e7086baeda0ddc175af9667db7529b3c5e11a5c0601b8bef2200a \\\n'
        '    --hash=sha256:6d914c52f89dc942b1819db959e7c07a5999f678ec760f677625c706b7e0d743 \\\n'
        '    --hash=sha256:74217bb56ed8692759227758e4c4b3bd2d608a209c1a7a081bf361fb4c2c1bf9 \\\n'
        '    --hash=sha256:a9c404c69ed4a39e8ec632eaf6b9fe058a060bf98979c177f6ef666f06bb8d50 \\\n'
        '    --hash=sha256:b7004666652f500ed854a86988e4b3d69d247188b5d2092b5df1e44f4a954099 \\\n'
        '    --hash=sha256:e91ffa46d095b252248297292dd22bcbacd53a125a0c2eefbbbf74925a320bc3\n'
        '    # via torch\n'
        'typing-extensions==4.16.0 \\\n'
        '    --hash=sha256:481caa481374e813c1b176ada14e97f1f67a4539ce9cfeb3f350d78d6370c2e8 \\\n'
        '    --hash=sha256:dc983d19a509c94dba722ee6abd33940f7c05a89e243c47e907eb4db6f1a43e5\n'
        '    # via\n'
        '    #   huggingface-hub\n'
        '    #   torch\n'
        'urllib3==2.8.0 \\\n'
        '    --hash=sha256:0cf3cae568d36aa9576b28dfb35f11328f1cb974ca7647d9475ebb86c75ac6e3 \\\n'
        '    --hash=sha256:63bf2ead4c879426ebf22ef2a781eeb4aa3b4ae798a0435506f8687fd5bb9b63\n'
        '    # via requests\n'
    ),
    'source.json': (
        '{\n'
        '  "repository": "kurtvalcorza/depth-anything-depth-estimation-pipeline",\n'
        '  "base_revision": "07c4a9858d3d12f2f1924ffb8a23bad10cd992fb",\n'
        '  "generator": "build_depth_workshop.py/1",\n'
        '  "generator_sha256": "242ed84cd3ff6247417ad305604ca395ee3dfbf3105b2b6f8850d478f8f46174",\n'
        '  "files": {\n'
        '    "depth_reference/__init__.py": "a08fb132505bbc5e5221a766e841407be5f1036b1b4529706ac63b21028e9306",\n'
        '    "depth_reference/metrics.py": "cf1f7a3433c3a433124ebfe525b7d1ed7953017938ed002e447a791e9ec7c396",\n'
        '    "depth_reference/pipeline.py": "1f4cbfe386c1b7e7d23a06181d6b1a346d8d6667b9738038f6efaf3fcada7bee",\n'
        '    "depth_reference/samples.py": "ee1bc73a87a3c9a0773a3904aef0b176669e985fa17a9b007ad1b6b21a9af214",\n'
        '    "zoe_reference.py": "1107ff98322c57e76c82ac41172059470ca36163f990de1a1ed0b455c1e07ba2",\n'
        '    "weights/zoe/dimer-base-manifest.json": "03b7a287e192b38974a6e6184b0ac2fc8358795bb52ee10f3387af0b62a9e15b",\n'
        '    "licenses/zoe.txt": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138",\n'
        '    "licenses/depth-code.txt": "f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138",\n'
        '    "licenses/weight-data-provenance.md": "fbcaa05cb3067b13ec16db19146e5ce5aa4322c5dfedbcdbc5c7dec925c2c15d",\n'
        '    "vendor_provenance.json": "4543cb358caf7aacd634a6247efb08d939e2ef09d23de036bddf2396b01a7722",\n'
        '    "weights/depth/dimer-base-manifest.json": "284c7f6ffafcb843b4db0ba6d389af114419ff8f5fe6dfdd6be06f07402c69c0",\n'
        '    "workshop.py": "fbfb4d51b736ec88e3f28f6f394ebcfc6408260d53ac76ca70c5fbce789290a8",\n'
        '    "requirements.txt": "6f670565defbe15a0fc3f7684133fd3c6878ff9befa59295a27cb2c9f25ce0c0"\n'
        '  }\n'
        '}'
    ),
}
CARRIED_HASHES = {'depth_reference/__init__.py': 'a08fb132505bbc5e5221a766e841407be5f1036b1b4529706ac63b21028e9306', 'depth_reference/metrics.py': 'cf1f7a3433c3a433124ebfe525b7d1ed7953017938ed002e447a791e9ec7c396', 'depth_reference/pipeline.py': '1f4cbfe386c1b7e7d23a06181d6b1a346d8d6667b9738038f6efaf3fcada7bee', 'depth_reference/samples.py': 'ee1bc73a87a3c9a0773a3904aef0b176669e985fa17a9b007ad1b6b21a9af214', 'zoe_reference.py': '1107ff98322c57e76c82ac41172059470ca36163f990de1a1ed0b455c1e07ba2', 'weights/zoe/dimer-base-manifest.json': '03b7a287e192b38974a6e6184b0ac2fc8358795bb52ee10f3387af0b62a9e15b', 'licenses/zoe.txt': 'f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138', 'licenses/depth-code.txt': 'f9da6567cb5a1828ccec41c7b72b70997fe6d9d63811b3f3177a7b01da52d138', 'licenses/weight-data-provenance.md': 'fbcaa05cb3067b13ec16db19146e5ce5aa4322c5dfedbcdbc5c7dec925c2c15d', 'vendor_provenance.json': '4543cb358caf7aacd634a6247efb08d939e2ef09d23de036bddf2396b01a7722', 'weights/depth/dimer-base-manifest.json': '284c7f6ffafcb843b4db0ba6d389af114419ff8f5fe6dfdd6be06f07402c69c0', 'workshop.py': 'fbfb4d51b736ec88e3f28f6f394ebcfc6408260d53ac76ca70c5fbce789290a8', 'requirements.txt': '6f670565defbe15a0fc3f7684133fd3c6878ff9befa59295a27cb2c9f25ce0c0', 'source.json': 'a397a869f564a3e1ec0ee19ac6c811b7e8a429581de4314c1964e440b8104cdc'}
for name, text in CARRIED_FILES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(text, encoding='utf-8')
    if hashlib.sha256(path.read_bytes()).hexdigest() != CARRIED_HASHES[name]:
        raise RuntimeError('Carried source integrity failure: ' + name)
print('Separate Depth Anything and ZoeDepth source namespaces materialized and verified.')

In [ ]:
# @title Infrastructure: build the isolated Python 3.12.12 environment and define helpers
UV_URL = 'https://files.pythonhosted.org/packages/1e/fd/432451d732917c49152a291de3ef171aa6b0f1a22d39780fb2c1f085ca4c/uv-0.12.15-py3-none-manylinux_2_17_x86_64.manylinux2014_x86_64.whl'
UV_SHA256 = 'aee9802f46bae436bd91751bb33ddeb379ef1596b5c19df193219d545d244b60'
for attempt in range(3):
    try:
        with urllib.request.urlopen(UV_URL, timeout=90) as response:
            wheel = response.read(20081405)
        break
    except (urllib.error.URLError, TimeoutError, ConnectionError):
        if attempt == 2:
            raise
        time.sleep(2 ** attempt)
if len(wheel) != 20081404 or hashlib.sha256(wheel).hexdigest() != UV_SHA256:
    raise RuntimeError('uv wheel size/hash mismatch')
import io
with zipfile.ZipFile(io.BytesIO(wheel)) as archive:
    member = next(n for n in archive.namelist() if n.endswith('.data/scripts/uv'))
    UV = ROOT / 'uv'
    UV.write_bytes(archive.read(member))
UV.chmod(0o700)
ENV = dict(os.environ, HF_HUB_DISABLE_IMPLICIT_TOKEN='1', HF_HUB_DISABLE_TELEMETRY='1', DO_NOT_TRACK='1')
ENV.pop('HF_TOKEN', None)
ENV.pop('HUGGING_FACE_HUB_TOKEN', None)
subprocess.run([str(UV), 'venv', '--managed-python', '--python', '3.12.12', str(ROOT / 'env')], env=ENV, check=True)
PYTHON = ROOT / 'env/bin/python'
subprocess.run([str(UV), 'pip', 'install', '--python', str(PYTHON), '--require-hashes', '--only-binary', ':all:',
                '--index-url', 'https://pypi.org/simple', '-r', str(ROOT / 'requirements.txt')], env=ENV, check=True)
subprocess.run([str(PYTHON), '-c', 'import sys,torch; print(sys.version); print(torch.__version__); assert torch.cuda.is_available()'], env=ENV, check=True)
BOOTSTRAP_SECONDS = time.perf_counter() - SESSION_START

def run_stage(stage, *options, root=None):
    selected_root = ROOT if root is None else root
    log = selected_root / (stage + '.log')
    print('Running', stage, '— one model per process; log:', log, flush=True)
    with log.open('w', encoding='utf-8') as output:
        result = subprocess.Popen([str(PYTHON), '-u', str(ROOT / 'workshop.py'), '--root', str(selected_root), '--stage', stage, *options],
                                  env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True)
        for line in result.stdout:
            output.write(line)
            output.flush()
            if line.strip() and len(line) < 2000:
                print(line.rstrip(), flush=True)
        result.wait()
    print(log.read_text(encoding='utf-8')[-7000:])
    if result.returncode:
        raise RuntimeError(f'{stage} failed with exit {result.returncode}. Read the diagnostics above.')

def show_json(relative):
    value = json.loads((ROOT / relative).read_text())
    print(json.dumps(value, indent=2))
    return value

def show_previews(pattern):
    from IPython.display import Image, display
    for path in sorted((ROOT / 'outputs').glob(pattern))[:6]:
        print(path.name)
        display(Image(filename=str(path)))

def show_metric_tables(filenames, role='test', breakdown=False):
    from IPython.display import Markdown, display
    rows = []
    for filename in filenames:
        rows.extend(json.loads((ROOT / 'outputs' / filename).read_text())['rows'])
    rows = [r for r in rows if r['role'] == role]
    for protocol in sorted({r['protocol'] for r in rows}):
        selected = [r for r in rows if r['protocol'] == protocol]
        for grouping in (['all', 'domain', 'scene'] if breakdown else ['all']):
            groups = {}
            for row in selected:
                key = (row['model'], row.get(grouping, 'all') if grouping != 'all' else 'all')
                groups.setdefault(key, []).append(row)
            table = [f'**{protocol} — {role}; equal-image means ({grouping})**', '',
                     '| Model | Group | Images | AbsRel ↓ | Delta-1 ↑ |', '|---|---|---:|---:|---:|']
            for (model, group), records in sorted(groups.items()):
                a = sum(r['abs_rel'] for r in records) / len(records)
                d = sum(r['delta1'] for r in records) / len(records)
                table.append(f'| {model} | {group} | {len(records)} | {a:.5f} | {d:.5f} |')
            display(Markdown('\n'.join(table)))

def show_activity():
    from IPython.display import Markdown, display
    activity = json.loads((ROOT / 'outputs/brightness_experiment.json').read_text())
    table = ['| Brightness factor | Improved AbsRel | Unchanged | Worsened | Mean paired AbsRel delta |',
             '|---|---:|---:|---:|---:|']
    for factor, comparison in sorted(activity['comparisons'].items()):
        delta = sum(r['abs_rel_delta'] for r in comparison['rows']) / len(comparison['rows'])
        table.append(f"| {factor} | {comparison['improved']} | {comparison['unchanged']} | {comparison['worsened']} | {delta:+.5f} |")
    display(Markdown('\n'.join(table)))
    scene_table = ['| Factor | Scene | Images | Mean paired AbsRel delta |', '|---|---|---:|---:|']
    for factor, comparison in sorted(activity['comparisons'].items()):
        for scene, values in sorted(comparison['per_scene'].items()):
            scene_table.append(f"| {factor} | {scene} | {values['n']} | {values['mean_abs_rel_delta']:+.5f} |")
    display(Markdown('\n'.join(scene_table)))
    print('Role:', activity['role'], '| Selection changed:', activity['selection_changed'])
    print('Full paired rows, clipping and input hashes are in brightness_experiment.json.')

## 2. Inspect RGB, reference depth and sensor validity

**Input:** 40 digest-pinned DIODE views, each with RGB, metric depth and a sensor mask. **System:** byte verification, original-resolution decoding, content deduplication and scene-disjoint role assignment. **Output:** reference panels, fixed roles and an auditable sample manifest.

| Role | Indoor scene / views | Outdoor scene / views | Total |
|---|---|---|---:|
| Training | scene_00020 / 8 | scene_00023 / 6 | 14 |
| Validation | scene_00019 / 2 | scene_00024 / 4 | 6 |
| Test | scene_00021 / 10 | scene_00022 / 10 | 20 |

All scans/views of a `(domain, scene)` stay together. This is a notebook-specific regrouping of a published validation release, **not an official DIODE benchmark split**. The 20 test views represent only two independent scenes. Upstream pretraining overlap has not been ruled out.

Canonical evaluation support is **sensor-valid AND finite reference depth in [0.6, 350] metres**, with at least two pixels. Zero depth is not a real observation. Every model is scored on exactly this fixed support; nonfinite predictions are rejected, and ZoeDepth must be positive there. No model-dependent pixel exclusions are allowed.

**Predict:** would two adjacent views of the same room count as independent test scenes? Why should missing sensor pixels remain unscored?

In [ ]:
run_stage('prepare')
show_json('outputs/prepare.json')
show_previews('reference*.png')

**Notice:** the RGB/reference alignment, missing sensor regions, valid counts/fractions and scene identities. Reference depth is not resized or interpolated for scoring; output maps return to the original pixel coordinates.

<details><summary>Interpretation hint</summary>Nearby views share geometry and texture. Splitting them across training and test can exaggerate generalization. A mask records where the sensor supports a comparison; filling holes does not create new observations.</details>

## 3. Work through two depth metrics

**AbsRel** is the average of `abs(predicted_metres − reference_metres) / reference_metres`; lower is better. **Delta-1** is the fraction satisfying `max(pred/reference, reference/pred) < 1.25`; higher is better. The inequality is strict, so an exact ratio of 1.25 does not pass.

**Predict:** references are 2 and 4 metres, predictions are 2.5 and 4 metres. What are AbsRel and delta-1? Does a larger inverse-depth value mean farther away?

In [ ]:
reference = [2.0, 4.0]
prediction = [2.5, 4.0]
errors = [abs(p - r) / r for p, r in zip(prediction, reference)]
passes = [max(p / r, r / p) < 1.25 for p, r in zip(prediction, reference)]
print({'AbsRel': sum(errors) / len(errors), 'delta1': sum(passes) / len(passes)})
print('Inverse depths of 2 m and 4 m:', [1 / r for r in reference])

<details><summary>Worked answer</summary>AbsRel = 0.125 and delta-1 = 0.5. The first prediction has ratio exactly 1.25 and fails the strict test. Inverse depth is 0.5 at 2 m and 0.25 at 4 m: the nearer surface has the larger inverse depth.</details>

We report per-image values and **equal-image means**, plus scene/domain means and support counts. These are not pooled-pixel scores. Undefined values carry a reason; a default image that cannot be scored fails the run instead of disappearing.

## 4. Establish baselines and keep protocols separate

**A — Relative structure: reference-assisted affine alignment.** Fit `1/reference = a × predicted_inverse_depth + b` separately on each image's fixed valid pixels, then convert as `1 / max(a*p+b, 1/350)`. This deliberately uses the evaluated reference, including test references. It is **not an inference-time calibration** and cannot give valid metres for a new image. Rank/degeneracy, negative fitted scale and far-cap fractions are recorded. No positive-scale constraint is silently added.

Compare Depth Anything only with the **oracle per-image reference median** and a **fixed vertical inverse-depth ramp** under this reference-assisted protocol. The median receives reference information; the ramp checks a simple positional prior. Neither is a learned deployable distance estimator.

**B — Metric distance: raw, unaligned metres.** ZoeDepth is compared with a constant equal to the **median of the 14 training-image valid-depth medians**. This constant uses training only and weights images equally. ZoeDepth receives no fitted alignment, median scaling or beneficial output clipping.

**Predict:** could a wrongly scaled prediction score well after reference-assisted alignment? Would comparing its score directly with raw ZoeDepth establish the better distance estimator?

In [ ]:
baselines = json.loads((ROOT / 'outputs/baselines.json').read_text())
print('Training-only constant (metres):', baselines['training_constant_metres'])
show_metric_tables(['baselines.json'])

**Read the scoring code (optional).** The tables above, and every score in this notebook, come from three short functions in the carried worker. `support` builds the fixed evaluation pixels (sensor-valid, finite, 0.6–350 m, at least two). `score` computes AbsRel and delta-1 on exactly those pixels, in raw metres unless `aligned=True`. `aligned_values` fits the per-image `1/reference = a × prediction + b` alignment used by protocol A and records the fit diagnostics. The next cell prints them; running it changes nothing.

In [ ]:
import ast
worker_source = (ROOT / 'workshop.py').read_text(encoding='utf-8')
for node in ast.parse(worker_source).body:
    if isinstance(node, ast.FunctionDef) and node.name in ('support', 'aligned_values', 'score'):
        print(ast.get_source_segment(worker_source, node), end='\n\n')


<details><summary>Interpretation hint</summary>Alignment supplies reference information and can correct scale/shift. A visually plausible or well-aligned map is not evidence that the model knows metres. Keep the two protocols in separate tables and interpret degenerate or reversed fits explicitly.</details>

## 5. Capture frozen predictions, then adapt and lock selection

**Input:** 14 training views and six validation views in disjoint scenes. **Model:** Depth Anything V2 Small, backbone frozen; neck/head trainable. **Output:** untouched frozen predictions, three training epochs, update evidence, selected adapter and paired held-out results.

Before adaptation, the worker captures untouched validation/test predictions for the preregistered comparison. It then calls `adapt(train, val, trainable_blocks=0, head_epochs=3, epochs=0, lr=1e-5, seed=42)`: batch one, AdamW weight decay 0.01, gradient clipping 1.0, no augmentation, float32, source scale-and-shift-invariant loss. The default run expects **42 actual optimizer updates**. Adaptation target resampling is recorded separately from original-resolution scoring.

Selection uses lowest source validation aligned AbsRel over epochs **0–3**, rounded to six decimal places by the source, with earliest ties. Test data never selects an epoch. There is no fallback unfreezing or test-driven parameter search.

**Predict before running:** will training help both held-out scenes? What distinguishes actual training updates from the final artifact if epoch zero wins?

In [ ]:
run_stage('depth')
from IPython.display import Markdown, display
history = json.loads((ROOT / 'outputs/training_history.json').read_text())
observed = {o['epoch']: o for o in history['epoch_observations']}
table = ['| Epoch | Training loss | Validation aligned AbsRel | Delta-1 | Optimizer steps so far | '
         'Max weight change before selection |', '|---:|---:|---:|---:|---:|---:|']
for entry in history['history']:
    loss = 'baseline' if entry['train_loss'] is None else f"{entry['train_loss']:.6f}"
    seen = observed.get(entry['epoch'])
    steps = 0 if seen is None else seen['actual_steps']
    change = '0 (starting weights)' if seen is None else f"{seen['max_weight_delta_before_selection']:.3e}"
    table.append(f"| {entry['epoch']} | {loss} | {entry['val']['abs_rel']:.6f} | {entry['val']['delta1']:.6f} "
                 f"| {steps} | {change} |")
display(Markdown('\n'.join(table)))
print({'actual_optimizer_steps': history['actual_optimizer_steps'],
       'trainable_parameters': history['trainable_parameters'],
       'selected_max_weight_delta_after_restoring_selected_epoch': history['selected_max_weight_delta']})
print('Full tensor hashes and training evidence are saved in training_history.json.')
show_json('outputs/selection.json')
show_metric_tables(['depth_frozen.json', 'depth_selected_live.json'], breakdown=True)
show_previews('depth*.png')

**Notice:** trainable names/counts, frozen tensor hashes, actual optimizer steps, losses, validation scores and the **max weight change before selection** column. That column is non-zero after each training epoch, which shows the optimizer really changed the weights; if epoch 0 is selected, the starting weights are restored afterwards, so the value printed below the table is 0. A zero-epoch selection is valid, even when training genuinely updated parameters. Improvement is not guaranteed or required.

<details><summary>Interpretation hint</summary>Optimization minimizes a training objective; validation selects a checkpoint. Test scenes assess the already locked choice. Report worsening or unchanged predictions as faithfully as improvements; do not change settings after seeing test results.</details>

## 6. Reload the selected adapter in a fresh process

**Input:** the exported neck/head safetensors adapter, artifact manifest and verified compatible base snapshot. **System:** a new Python process, with no live model state. **Output:** raw inverse-depth parity on fixed validation probes and all test images, then independently recomputed metrics.

Prediction tolerance is fixed at `atol=1e-5, rtol=1e-4`; aggregate-metric tolerance is `1e-6` absolute. The worker logs maximum differences and fails excess drift. Base weights are not redistributed with the adapter.

**Predict:** if parity passes, does it show that the model is accurate, or that the saved artifact reconstructs the selected behavior?

In [ ]:
run_stage('reload')
parity = json.loads((ROOT / 'outputs/reload_parity.json').read_text())
print({'passed': parity['passed'], 'images_checked': len(parity['checks']),
       'max_absolute_prediction_error': max(r['max_absolute_error'] for r in parity['checks']),
       'metric_drift': parity['metric_drift'], 'reload_pid': parity['reload_pid']})

**Notice:** the separate process IDs, checked probes and errors. Parity is an artifact reproducibility check; it is not independent validation of the model's predictions. Keep the fixed tolerances when investigating a failure.

## 7. Inspect ZoeDepth's direct distance estimates

**Input:** the same original RGB images and fixed reference support. **Model:** frozen ZoeDepth NYU+KITTI. **Output:** positive raw metres and unaligned error metrics, compared with the training-only constant.

Only one model is resident on the GPU at a time. This stage does not train ZoeDepth and does not pass DIODE through its separate training-target validator. Its routed NYU/KITTI heads have nominal ranges of 0.001–10 m and 0.001–80 m, while our fixed DIODE reference support extends to 350 m. Reference pixels above 80 m are counted as a range limitation; predictions and errors are not clipped to hide it.

**Predict:** where might a distance model struggle when moving from indoor to outdoor scenes? Treat that as a hypothesis until the maps and per-domain results support it.

In [ ]:
run_stage('zoe')
show_previews('zoe*.png')
show_metric_tables(['zoe_metrics.json'], breakdown=True)

**Notice:** distances and errors, not just attractive colours. Metric plots use documented shared metre ranges; error maps share a scale. Relative display normalization, where used, is labelled arbitrary units and never enters scoring. Invalid reference pixels are neutral.

<details><summary>Interpretation hint</summary>A correct ordering of near/far surfaces can coexist with wrong distances. Conversely, a useful average may conceal errors at boundaries or in one domain. Inspect actual cases before naming a failure mode.</details>

## 8. Controlled activity: change only RGB brightness

**Input:** the six validation views, openly reused for a descriptive activity. **Model:** the locked selected Depth Anything adapter. **Change one thing:** multiply RGB by 0.6, 1.0 or 1.4, clip to [0,255], round, and convert to uint8. Geometry, references, masks and scoring stay fixed.

This is a pixel-brightness transform, not physically accurate exposure. These images are **not additional held-out observations**, and the activity cannot change model selection. Input hashes, actual factors and clipped-pixel fractions are exported.

**Predict:** must darker or brighter always be worse? Could per-image reference alignment hide a change in output scale?

In [ ]:
run_stage('activity')
show_activity()
show_previews('activity*.png')

**Notice:** paired AbsRel/delta-1 changes against factor 1.0, improved/unchanged/worsened counts and scene results. Preserve counterexamples to your prediction.

<details><summary>Interpretation hint</summary>A brightness transform can help, hurt or leave a case unchanged. Alignment can absorb scale changes, so these aligned scores cannot establish stable distance calibration. Six correlated views do not establish camera robustness.</details>


### Optional: choose your own brightness factor

**Predict → change one thing → run → observe → explain.** Pick a factor between 0.2 (much darker) and 2.0 (much brighter), predict how many of the six validation views will improve or worsen against factor 1.0, then set `RUN_OWN_FACTOR` and run the next cell after Run all has finished section 8. It applies the same pixel transform with the locked adapter, compares each view with its factor-1.0 result, and writes to its own folder. The canonical results, completion receipts and results ZIP are not changed, and model selection cannot change. Explain any difference from your prediction using the hint above.

In [ ]:
RUN_OWN_FACTOR = False  # @param {type:"boolean"}
OWN_BRIGHTNESS_FACTOR = 0.8  # @param {type:"number"}
if RUN_OWN_FACTOR:
    if not 0.2 <= float(OWN_BRIGHTNESS_FACTOR) <= 2.0:
        raise ValueError('Choose OWN_BRIGHTNESS_FACTOR between 0.2 and 2.0, then rerun this cell.')
    run_stage('own_activity', '--factor', str(float(OWN_BRIGHTNESS_FACTOR)))
    own = json.loads((ROOT / 'own_activity_output.json').read_text())
    comparison = own['comparison']
    mean_delta = sum(r['abs_rel_delta'] for r in comparison['rows']) / len(comparison['rows'])
    print({'factor': own['factor'], 'improved': comparison['improved'],
           'unchanged': comparison['unchanged'], 'worsened': comparison['worsened'],
           'mean_paired_abs_rel_delta': round(mean_delta, 5)})
    from IPython.display import Image as ShowImage, display
    display(ShowImage(filename=str(Path(own['root']) / 'outputs/own_factor.png')))
else:
    print('Optional own-factor activity disabled.')


## 9. Export results and conclude with evidence

The report presents **two separate protocol tables**, per-image/scene/domain metrics, frozen-versus-selected changes, fixed examples (the first test view, the largest improvement and worsening, or the lowest and highest id when adaptation changed nothing) and both domains, with figure selection rules. It reloads exported tables/arrays and checks ZIP checksums. It includes sample identities, source/dependency/model pins, training/selection/update evidence, activity, runtime/memory, adapter and reload checks.

Raw numeric outputs are retained separately from display images. A complete execution remains **Candidate** until the exact fresh hosted run and real labelled BYOD evidence are reviewed; it is not release approval.

In [ ]:
run_stage('report')
import shutil
from IPython.display import Markdown, display
display(Markdown((ROOT / 'outputs/summary.md').read_text()))
show_previews('figures/*.png')
receipts = json.loads((ROOT / 'stage_receipts.json').read_text())['stages']
resources = {stage: {'seconds': round(r.get('seconds', 0.0), 1),
                     'peak_gpu_mib': None if r.get('peak_gpu_bytes') is None else round(r['peak_gpu_bytes'] / 2**20, 1)}
             for stage, r in receipts.items()}
run_folder_bytes = sum(p.stat().st_size for p in ROOT.rglob('*') if p.is_file())
disk = shutil.disk_usage(ROOT)
print('This run, per stage (wall seconds, peak GPU MiB):', json.dumps(resources, indent=2))
print(f'Run folder on disk: {run_folder_bytes / 2**30:.2f} GiB (environment, models, data and outputs); '
      f'free disk: {disk.free / 2**30:.1f} GiB')
print('Results ZIP:', ROOT / 'outputs/depth_results.zip')
print('To download it in Colab: open the Files pane (folder icon, left), browse to that path, right-click > Download.')
session = {'bootstrap_seconds': BOOTSTRAP_SECONDS, 'elapsed_seconds': time.perf_counter() - SESSION_START,
           'stage_resources': resources, 'run_folder_bytes': run_folder_bytes, 'disk_free_bytes': disk.free,
           'source_files': CARRIED_HASHES, 'qualification': 'Candidate until exact hosted and BYOD evidence reviewed'}
(ROOT / 'session_summary.json').write_text(json.dumps(session, indent=2))
print('Session summary (not inside the ZIP):', ROOT / 'session_summary.json')
print(session)

Use your measured results in this optional personal note:

> On [views and independent scenes], [model/state] obtained [AbsRel/delta-1] under [exact protocol], compared with [appropriate baseline]. Adaptation [improved/worsened/preserved] [images/scenes]. Brightness factor [value] changed [paired result]. A case requiring further study is [example]. This does not establish [population or deployment claim].

**Self-check:** explain inverse depth versus metres; identify reference-assisted alignment; name the scenes that selected the epoch; explain valid support and why 20 views are not 20 independent scenes; distinguish adapter parity from predictive accuracy.

## 10. Optional: your own labelled RGB-D data

Use only data you are authorized to process in a hosted Colab runtime; do not use confidential, personal or regulated data without that authorization. Your files stay inside this runtime: nothing is sent to a DIMER service or any other external service, and only the results ZIP you choose to download leaves it. Supply a directory with `manifest.json`, RGB images, float32 `.npy` depth in **metres**, and boolean `.npy` masks. Related camera sequences/acquisition scenes must share `group_id` and stay in one role. Require explicit units and groups; units are never guessed. Arrays load with `allow_pickle=False`.

Each record names `id`, relative `image`, relative `depth`, relative `mask`, `group_id`, `role` and optional `domain`. The manifest declares `"units": "metres"` exactly. Ids are 1–64 characters from `A-Za-z0-9_-` and unique; `role` is `train`, `validation` or `test`. Use **12–128 records**, at least **6 train / 2 validation / 4 test**, and at least two distinct groups in each role. Minimums enable execution, not strong generalization claims.

Limits: sides 32–2048 pixels, aspect ratio at most 4, EXIF orientation 1 (apply any rotation before saving, to the image and its arrays alike), 32 MiB per file, 1 MiB JSON, 64 million total RGB pixels. Depth and mask arrays are exactly `float32` and `bool`, with the image's height × width. At least 5% of each image's pixels must carry valid (masked, finite, positive) depth, masked depth may not exceed 1000 m, and at least two masked reference pixels must lie in the scored range of 0.6–350 m. Paths must remain inside the directory and use `/`. Headers/allocation bounds, units, shapes, image contracts, duplicate content, group overlap and positive reference support are checked before GPU work. References remain in original coordinates.

The optional route performs the **same full baselines → adaptation → validation selection → test → brightness → export → fresh reload** in a separate output directory. It never overwrites the canonical run. Private images/depth arrays are excluded from the shareable ZIP by default, but IDs, labels and metrics may remain: review exports before sharing. Default Run all never asks for uploads.

A minimal `manifest.json` (one record shown; list all of them):

```json
{"units": "metres",
 "records": [{"id": "kitchen_01", "image": "rgb/kitchen_01.png", "depth": "depth/kitchen_01.npy",
              "mask": "mask/kitchen_01.npy", "group_id": "kitchen", "role": "train", "domain": "indoor"}]}
```

To get the directory into Colab, upload it with the **Files** pane (for example to `/content/my_rgbd`) or mount Google Drive, then set `BYOD_PATH` to that directory. A refusal names the record and file that failed and the rule it broke; fix that item and rerun the cell. Results appear below the cell, and the BYOD run's summary describes your own split.

In [ ]:
RUN_BYOD = False  # @param {type:"boolean"}
BYOD_PATH = ''  # @param {type:"string"}
if RUN_BYOD:
    if not BYOD_PATH.strip():
        raise ValueError('Set BYOD_PATH to a directory containing manifest.json and labelled RGB-D files.')
    run_stage('byod', '--byod', str(Path(BYOD_PATH).resolve()))
    byod_root = Path(json.loads((ROOT / 'byod_output.json').read_text())['root'])
    from IPython.display import Markdown, display
    display(Markdown((byod_root / 'outputs/summary.md').read_text()))
    print('Your results ZIP:', byod_root / 'outputs/depth_results.zip')
    print('Download it from the Files pane (right-click > Download). Review it before sharing: '
          'record IDs, labels and metrics remain inside it.')
else:
    print('Optional labelled BYOD disabled.')

## 11. Optional: one image without a reference

Use the selected Depth Anything adapter and frozen ZoeDepth on an unlabelled image. The first output is **raw relative inverse depth**, the second is **predicted metres**. No prior image's fitted alignment is reused. Without a reference, accuracy metrics are **null / not measurable**. This inference-only option supplements the full labelled workflow; it makes no new adaptation claim.

Run it after Run all has completed section 5, because it uses the selected adapter. The image must have sides of 32–2048 pixels, an aspect ratio of at most 4, and no EXIF rotation: a full-size phone photo (for example 4032 × 3024) is refused, so save a resized copy whose longest side is at most 2048 px, with any rotation applied. Upload the image with the Files pane and set `IMAGE_PATH` to it. The cell shows the photo, the relative map and the metre map side by side; the `.npy` files hold the actual numbers.

In [ ]:
RUN_UNLABELLED = False  # @param {type:"boolean"}
IMAGE_PATH = ''  # @param {type:"string"}
if RUN_UNLABELLED:
    if not IMAGE_PATH.strip():
        raise ValueError('Set IMAGE_PATH to an RGB image within the documented size limits.')
    run_stage('infer', '--image', str(Path(IMAGE_PATH).resolve()))
    inference_root = Path(json.loads((ROOT / 'unlabelled_output.json').read_text())['root'])
    print((inference_root / 'result.json').read_text())
    from IPython.display import Image as ShowImage, display
    display(ShowImage(filename=str(inference_root / 'preview.png')))
    print('Files (download from the Files pane):', *sorted(str(p) for p in inference_root.iterdir()), sep='\n  ')
else:
    print('Optional unlabelled inference disabled.')

## Troubleshooting, limitations and attribution

| Observation | Action |
|---|---|
| No GPU or out of memory | Use a fresh T4 runtime; close competing work. Do not silently skip training or a model. |
| Download/package failure | Read stage logs; retry transient acquisition while preserving pins. Never bypass integrity checks. |
| Missing reference regions | Inspect sensor validity; do not fill holes and count them as observations. |
| Nonfinite/nonpositive prediction | Preserve diagnostics and investigate. Do not remove difficult pixels to improve the score. |
| Degenerate/negative alignment | Inspect coefficients and rank flags; an aligned number may not reflect meaningful near/far ordering. |
| Epoch zero wins or test worsens | Report it. Do not alter training after viewing final test outcomes. |
| Reload mismatch | Preserve errors and fixed tolerances; investigate before qualification. |
| Own data refused | The message names the record, the file and the rule (units, path, dtype/shape, coverage, grouping or counts); fix that item and rerun the cell. Validation runs before any model loads. |
| Own image refused | Resize so the longest side is at most 2048 px and apply any EXIF rotation, then save a copy and rerun. Run the unlabelled cell only after section 5 has completed. |

The sample contains only six independent scenes and cannot establish population accuracy, safety-critical ranging or autonomous navigation suitability. Sensor depth has missing regions; reflective surfaces, boundaries and domain shift are hypotheses to inspect, not assumed observed failures. Reference-assisted alignment can hide global-scale errors. Upstream training contamination has not been ruled out.

**Attribution and licenses:** DIODE (Vasiljevic et al., 2019), distributed here via the pinned Marigold evaluation sample, is recorded as CC BY 4.0. Depth Anything V2 Small weights are recorded as Apache-2.0; ZoeDepth NYU+KITTI as MIT. Carried manifests/license files retain source attribution; base weights are downloaded, never bundled with the exported adapter.

**AI Assistance Disclosure:** This notebook's code and explanations were developed with generative AI assistance under maintainer direction. The maintainer is responsible for review, result validation and release decisions. AI assistance does not constitute independent verification, provider endorsement or release approval.

References: [Depth Anything V2 Small](https://huggingface.co/depth-anything/Depth-Anything-V2-Small-hf), [ZoeDepth](https://huggingface.co/Intel/zoedepth-nyu-kitti), [DIODE](https://diode-dataset.org/), [pinned sample distribution](https://huggingface.co/datasets/obukhovai/marigold_depth_eval/tree/30c5b061d863e383e3ea9fa14555737a199d7ad9), [DIMER notebook specification](https://github.com/kurtvalcorza/ml-worker/blob/main/integrations/dimer/fleet-specs/NOTEBOOK_SPEC.md). Exact generating file hashes accompany revision labels in `source.json` and exported provenance.